# **📘 Actividad 4. Selección y evaluación de modelos preentrenados**

**Materia:** Gestión de Proyectos de Inteligencia Artificial

**Proyecto:** Asistente inteligente para comprensión, orientación y cotización de seguros de vida mediante IA generativa.

---



## **📝 1. Definición del caso de uso**

El proyecto contempla el desarrollo de un asistente inteligente para apoyar la comprensión y orientación sobre seguros de vida.

Como parte de su implementación, se desarrollará un componente de clasificación automática de consultas, utilizando modelos preentrenados disponibles en Hugging Face.

Este componente permitirá identificar la categoría de una consulta para facilitar su posterior procesamiento dentro del asistente.

### **🎯 1.1 Objetivo de la experimentación**

Comparar al menos tres modelos preentrenados de Hugging Face mediante métricas de clasificación y eficiencia computacional, con el propósito de seleccionar una alternativa adecuada para el caso de uso.

### **🎯 1.2 Dataset candidato**

Se utilizará inicialmente el dataset Bitext Insurance LLM Chatbot Training Dataset, disponible en Hugging Face.

El conjunto contiene consultas relacionadas con seguros, organizadas por categorías e intenciones.

Durante la exploración se identificarán las categorías relevantes para seguros de vida y se evaluará su distribución antes de seleccionar el subconjunto de trabajo.

---

## **💻 2. Exploración del conjunto de datos**

En esta sección se cargará el dataset candidato desde Hugging Face para identificar su estructura, número de registros y distribución de categorías.

Esta exploración permitirá determinar qué categorías pueden utilizarse en el prototipo y si existe un equilibrio adecuado entre ellas.

In [ ]:
# Instalamos las bibliotecas necesarias para cargar y explorar el dataset
!pip -q install datasets pandas

print("=" * 60)
print("✅ Bibliotecas instaladas correctamente...")
print("=" * 60)

✅ Bibliotecas instaladas correctamente...


In [ ]:
# Importamos las bibliotecas necesarias
from datasets import load_dataset
import pandas as pd

# Cargamos el dataset de seguros desde Hugging Face
dataset = load_dataset(
    "bitext/Bitext-insurance-llm-chatbot-training-dataset"
)

# Consultamos las divisiones disponibles
print(dataset)

print("=" * 60)
print("✅ Carga de dataset para análisis...")
print("=" * 60)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'intent', 'category', 'tags', 'response'],
        num_rows: 39000
    })
})
✅ Carga de dataset para análisis...


### **🧠 2.1 Estructura y distribución de categorías**

Se analizarán las columnas disponibles y la cantidad de registros por categoría.

Los resultados permitirán identificar las clases candidatas para la clasificación y detectar posibles desequilibrios.

In [ ]:
# ============================================================
# 2.1 Estructura y distribución de categorías
# ============================================================

# Convertimos la división de entrenamiento a un DataFrame
df = dataset["train"].to_pandas()

# Mostramos las dimensiones y columnas disponibles
print("Dimensiones del dataset:", df.shape)
print("\nColumnas disponibles:")
print(df.columns.tolist())

# Mostramos los primeros registros
display(df.head())

# Calculamos la cantidad de registros por categoría
distribucion = (
    df["category"]
    .value_counts()
    .rename_axis("Categoría")
    .reset_index(name="Registros")
)

# Presentamos la distribución
display(distribucion)

print("=" * 60)
print("✅ Estructura y distribución de caterias de dataset...")
print("=" * 60)

Dimensiones del dataset: (39000, 5)

Columnas disponibles:
['instruction', 'intent', 'category', 'tags', 'response']


,instruction,intent,category,tags,response
0,"I'd like to see my fucking auto insurance , I ...",information_auto_insurance,AUTO_INSURANCE,BCLPWZ,To retrieve the details of your auto insurance...
1,wanna know more about my auto insurance giev m...,information_auto_insurance,AUTO_INSURANCE,BCQZ,"To retrieve your auto insurance details, pleas..."
2,I'd like to sde my fucking auto insurance coul...,information_auto_insurance,AUTO_INSURANCE,BCILPQWZ,"To obtain your auto insurance information, kin..."
3,wanna see my fucking auto insurance where coul...,information_auto_insurance,AUTO_INSURANCE,BCILPQW,"To retrieve your auto insurance details, pleas..."
4,I need information about my fucking auto insur...,information_auto_insurance,AUTO_INSURANCE,BCIW,"To retrieve your auto insurance details, pleas..."


,Categoría,Registros
0,CLAIMS,6000
1,PAYMENT,5000
2,ENROLLMENT,4000
3,COVERAGE,4000
4,CONTACT,4000
5,COMPLAINTS,3000
6,INCIDENTS,2000
7,QUOTE,2000
8,AUTO_INSURANCE,1000
9,HOME_INSURANCE,1000


✅ Estructura y distribución de caterias de dataset...


### **🧠 2.2 Análisis de intenciones por categoría**

Se identificarán las intenciones asociadas con las categorías
candidatas para determinar su pertinencia en el contexto de
los seguros de vida.

Este análisis permitirá justificar la selección del subconjunto
de datos que se utilizará en la experimentación.

In [ ]:
# ============================================================
# 2.2 Análisis de intenciones por categoría
# ============================================================

# Definimos las categorías candidatas para el proyecto
categorias_candidatas = [
    "LIFE_INSURANCE",
    "COVERAGE",
    "QUOTE",
    "PAYMENT",
    "ENROLLMENT",
    "CONTACT"
]

# Filtramos temporalmente las categorías candidatas
df_candidatos = df[
    df["category"].isin(categorias_candidatas)
].copy()

# Contamos los registros por categoría e intención
distribucion_intenciones = (
    df_candidatos
    .groupby(["category", "intent"])
    .size()
    .reset_index(name="Registros")
    .sort_values(["category", "intent"])
)

# Mostramos los resultados
print("Distribución de intenciones por categoría:")
display(distribucion_intenciones)

print("=" * 60)
print("✅ Análisis de intención por categoria de dataset...")
print("=" * 60)

Distribución de intenciones por categoría:


,category,intent,Registros
0,CONTACT,agent,1000
1,CONTACT,customer_service,1000
2,CONTACT,human_agent,1000
3,CONTACT,insurance_representative,1000
4,COVERAGE,change_coverage,1000
5,COVERAGE,check_coverage,1000
6,COVERAGE,downgrade_coverage,1000
7,COVERAGE,upgrade_coverage,1000
8,ENROLLMENT,buy_insurance_policy,1000
9,ENROLLMENT,cancel_insurance_policy,1000


✅ Análisis de intención por categoria de dataset...


### **🧠 2.3 Revisión de consultas representativas**

Se examinarán ejemplos de las categorías candidatas
para verificar su pertinencia en el contexto de los seguros
de vida.

La revisión permitirá identificar consultas generales,
consultas específicas y posibles limitaciones del dataset
antes de seleccionar el subconjunto definitivo.

In [ ]:
# ============================================================
# 2.3 Revisión de consultas representativas
# ============================================================

# Establecemos una semilla para obtener resultados reproducibles
semilla = 42

# Seleccionamos cinco consultas aleatorias por categoría
muestras = (
    df_candidatos
    .groupby("category", group_keys=False)
    .sample(n=5, random_state=semilla)
)

# Conservamos únicamente los campos relevantes para la revisión
muestras = muestras[
    ["category", "intent", "instruction"]
]

# Ordenamos las consultas por categoría
muestras = muestras.sort_values(
    by=["category", "intent"]
)

# Mostramos las consultas seleccionadas
display(muestras.reset_index(drop=True))

print("=" * 60)
print("✅ Revisión de consultas representativas de dataset...")
print("=" * 60)

,category,intent,instruction
0,CONTACT,agent,"I want to speak with tje insurance company, wi..."
1,CONTACT,agent,can i talk to the insurane company
2,CONTACT,human_agent,"I want to talk to a fucking agent, can I get s..."
3,CONTACT,insurance_representative,I have to talk to my fucking insurance represe...
4,CONTACT,insurance_representative,I have to speak with my fucking insurance repr...
5,COVERAGE,check_coverage,i want help to check the coverage of my auto i...
6,COVERAGE,downgrade_coverage,I have to downgrade the coverage of my health ...
7,COVERAGE,downgrade_coverage,i got to reduce the coverage of my pet insuran...
8,COVERAGE,downgrade_coverage,help me to downgrade trhe fucking coverage of ...
9,COVERAGE,upgrade_coverage,I got to increase the fucking coverage of my t...


✅ Revisión de consultas representativas de dataset...


### **🧠 2.4 Identificación preliminar del contexto asegurador**

Se analizarán las consultas de las seis categorías candidatas
para identificar referencias explícitas a diferentes tipos de seguros.

El objetivo es distinguir consultas relacionadas con seguros
de vida, consultas generales y consultas que mencionan
otros productos aseguradores.

Este análisis es exploratorio y no sustituye la validación
semántica de los registros.

In [ ]:
# ============================================================
# 2.4 Identificación preliminar del contexto asegurador
# ============================================================

# Definimos expresiones relacionadas con diferentes tipos de seguros
patrones = {
    "Vida": r"\blife insurance\b",
    "Automóvil": r"\b(auto|car|vehicle) insurance\b",
    "Salud": r"\b(health|medical) insurance\b",
    "Hogar": r"\b(home|homeowners) insurance\b",
    "Mascotas": r"\bpet insurance\b",
    "Viaje": r"\btravel insurance\b"
}

# Creamos una copia para realizar el análisis
df_contexto = df_candidatos.copy()

# Normalizamos temporalmente el texto para buscar coincidencias
textos = df_contexto["instruction"].fillna("").str.lower()

# Identificamos menciones explícitas a cada tipo de seguro
for tipo, patron in patrones.items():
    df_contexto[tipo] = textos.str.contains(
        patron,
        regex=True,
        na=False
    )

# Contamos cuántos tipos de seguro se mencionan en cada consulta
columnas_tipos = list(patrones.keys())

df_contexto["Tipos_detectados"] = (
    df_contexto[columnas_tipos].sum(axis=1)
)

# Clasificamos provisionalmente las consultas
def clasificar_contexto(fila):

    # Identificamos consultas que mencionan exclusivamente seguros de vida
    if fila["Vida"] and fila["Tipos_detectados"] == 1:
        return "Vida explícito"

    # Identificamos consultas que mencionan varios tipos de seguros
    if fila["Vida"] and fila["Tipos_detectados"] > 1:
        return "Vida y otros seguros"

    # Identificamos consultas relacionadas con otros seguros
    if fila["Tipos_detectados"] > 0:
        return "Otros seguros"

    # Consultas sin referencias explícitas detectadas
    return "Sin tipo explícito detectado"


# Aplicamos la clasificación exploratoria
df_contexto["Contexto"] = df_contexto.apply(
    clasificar_contexto,
    axis=1
)

# Obtenemos la distribución por categoría y contexto
resumen_contexto = pd.crosstab(
    df_contexto["category"],
    df_contexto["Contexto"]
)

# Mostramos la tabla
display(resumen_contexto)

print("=" * 60)
print("✅ Análisis exploratorio de contextos en dataset...")
print("=" * 60)

/tmp/ipykernel_785/2305556718.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_contexto[tipo] = textos.str.contains(
/tmp/ipykernel_785/2305556718.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_contexto[tipo] = textos.str.contains(
/tmp/ipykernel_785/2305556718.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_contexto[tipo] = textos.str.contains(


Contexto,Otros seguros,Sin tipo explícito detectado,Vida explícito
category,,,
CONTACT,29,3967,4
COVERAGE,2279,1247,474
ENROLLMENT,1960,1714,326
LIFE_INSURANCE,0,124,876
PAYMENT,532,4318,150
QUOTE,1180,564,256


✅ Análisis exploratorio de contextos en dataset...


### **🧠 2.5 Identificación del subconjunto candidato**

A partir del análisis exploratorio, se identificarán
las consultas que mencionan explícitamente seguros
de vida o que no contienen referencias detectadas
a otros tipos de seguros.

Esta selección será provisional y permitirá evaluar
la disponibilidad de registros por categoría antes
de preparar el dataset definitivo.

In [ ]:
# ============================================================
# 2.5 Identificación del subconjunto candidato
# ============================================================

# Seleccionamos consultas de vida o sin tipo explícito detectado
df_preseleccion = df_contexto[
    df_contexto["Contexto"].isin([
        "Vida explícito",
        "Sin tipo explícito detectado"
    ])
].copy()

# Contamos los registros disponibles por categoría
disponibilidad = (
    df_preseleccion["category"]
    .value_counts()
    .rename_axis("Categoría")
    .reset_index(name="Registros disponibles")
)

# Mostramos el resultado
print("Registros candidatos por categoría:")
display(disponibilidad)

# Calculamos el total disponible
print(
    "\nTotal de registros candidatos:",
    len(df_preseleccion)
)

print("=" * 60)
print("✅ Identificación de subconjuntos candidatos en dataset...")
print("=" * 60)

Registros candidatos por categoría:


,Categoría,Registros disponibles
0,PAYMENT,4468
1,CONTACT,3971
2,ENROLLMENT,2040
3,COVERAGE,1721
4,LIFE_INSURANCE,1000
5,QUOTE,820



Total de registros candidatos: 14020
✅ Identificación de subconjuntos candidatos en dataset...


### **📝 Cierre de la exploración inicial del dataset**

**Actividades realizadas:**

- Se cargó el dataset Bitext Insurance desde Hugging Face.
- Se identificaron 39,000 registros y 17 categorías.
- Se analizaron 20 intenciones correspondientes a seis categorías candidatas.
- Se revisaron consultas representativas para identificar su pertinencia.
- Se realizó una clasificación exploratoria según referencias a tipos de seguros.
- Se identificaron 14,020 registros candidatos.

**Hallazgos principales:**

El conjunto de datos presenta diferencias importantes en la cantidad
de registros por categoría. También contiene consultas sobre distintos
productos aseguradores y está disponible originalmente en inglés.

**Decisión preliminar:**

Se propone utilizar seis categorías y construir un subconjunto
equilibrado de hasta 3,600 registros, sujeto a la revisión de calidad,
licencia, idioma y compatibilidad con los modelos seleccionados.

**Estado:** Exploración inicial completada. Preparación definitiva pendiente.

## **💻 3. Selección de modelos preentrenados**

Se seleccionarán tres modelos multilingües disponibles en Hugging Face
para comparar su capacidad de clasificar consultas relacionadas
con seguros de vida.

Se utilizará inicialmente clasificación zero-shot, que permite
evaluar categorías propuestas sin realizar un entrenamiento
específico con el dataset del proyecto.

### **🧠 3.1 Identificación de modelos candidatos**

Los modelos candidatos son:

- **MiniLMv2 multilingüe:** alternativa compacta para explorar eficiencia computacional.
- **mDeBERTa-v3-base:** alternativa de tamaño intermedio.
- **XLM-RoBERTa-large:** alternativa de mayor tamaño.

La comparación considerará exactitud, precisión, exhaustividad
y F1 macro, además del tiempo de inferencia y consumo de memoria.

Todos los modelos se evaluarán sobre las mismas consultas,
con las mismas categorías y una configuración común.

**Alcance:** La selección es preliminar. Su funcionamiento en Colab
y su desempeño en el contexto del proyecto deberán verificarse
mediante pruebas.

**Preparación pendiente:** Antes de calcular métricas, se definirá
un conjunto de evaluación en español con etiquetas revisadas.

In [ ]:
# ============================================================
# 3.1 Identificación de modelos candidatos
# ============================================================

# Registramos los modelos que compararemos en la experimentación
modelos_candidatos = {
    "MiniLMv2 multilingüe": (
        "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
    ),
    "mDeBERTa-v3-base": (
        "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
    ),
    "XLM-RoBERTa-large": (
        "joeddav/xlm-roberta-large-xnli"
    )
}

# Organizamos los identificadores en una tabla para su revisión
tabla_modelos = pd.DataFrame(
    modelos_candidatos.items(),
    columns=["Modelo", "Identificador en Hugging Face"]
)

# Mostramos los modelos seleccionados
display(tabla_modelos)

print("=" * 60)
print("✅ Modelos candidatos registrados para la comparación.")
print("=" * 60)

,Modelo,Identificador en Hugging Face
0,MiniLMv2 multilingüe,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli
1,mDeBERTa-v3-base,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
2,XLM-RoBERTa-large,joeddav/xlm-roberta-large-xnli


✅ Modelos candidatos registrados para la comparación.


### **🧠 3.2 Verificación de recursos de ejecución**

Se verificará la disponibilidad de una GPU y la memoria del
entorno de Google Colab antes de cargar los modelos.

Esta revisión permitirá planificar la ejecución de las pruebas.
Los modelos se cargarán uno a la vez para reducir el consumo
simultáneo de memoria.

In [ ]:
# ============================================================
# 3.2 Verificación de recursos de ejecución
# ============================================================

# Importamos las herramientas para consultar los recursos
import torch
import psutil

# Consultamos la memoria RAM del entorno
memoria = psutil.virtual_memory()

print(f"RAM total: {memoria.total / (1024 ** 3):.2f} GB")
print(f"RAM disponible: {memoria.available / (1024 ** 3):.2f} GB")

# Verificamos si existe una GPU disponible para PyTorch
gpu_disponible = torch.cuda.is_available()

print("\nGPU disponible:", "Sí" if gpu_disponible else "No")

if gpu_disponible:
    # Mostramos el nombre y la memoria total de la GPU
    propiedades_gpu = torch.cuda.get_device_properties(0)

    print("GPU:", propiedades_gpu.name)
    print(
        f"Memoria total de GPU: "
        f"{propiedades_gpu.total_memory / (1024 ** 3):.2f} GB"
    )
else:
    # Indicamos el dispositivo que utilizaría la inferencia
    print("Dispositivo de ejecución: CPU.")
    print("Las pruebas pueden requerir más tiempo.")

print("=" * 60)
print("✅ Recursos del entorno verificados.")
print("=" * 60)

RAM total: 12.67 GB
RAM disponible: 8.78 GB

GPU disponible: Sí
GPU: Tesla T4
Memoria total de GPU: 14.56 GB
✅ Recursos del entorno verificados.


### **🧠 3.3 Prueba inicial de clasificación**

Se realizará una prueba de funcionamiento con MiniLMv2
multilingüe utilizando consultas ilustrativas en español.

Las categorías se expresarán mediante descripciones legibles
para facilitar la clasificación zero-shot.

Esta prueba permitirá verificar la carga del modelo y revisar
sus predicciones. Los ejemplos son ilustrativos y no constituyen
una evaluación del desempeño sobre el dataset.

El tiempo de inferencia se registrará por separado del tiempo
de descarga y carga del modelo.

In [ ]:
# ============================================================
# 3.3 Prueba inicial de clasificación
# ============================================================

# Importamos las herramientas de inferencia y medición
import pandas as pd
import torch
from transformers import pipeline
from time import perf_counter

# Definimos el identificador del modelo para esta prueba
modelo_prueba = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"

# Asociamos cada categoría con una descripción en español
descripciones_categorias = {
    "LIFE_INSURANCE": "información general sobre seguros de vida",
    "COVERAGE": "coberturas y exclusiones del seguro",
    "QUOTE": "solicitud de cotización del seguro",
    "PAYMENT": "pagos y métodos de pago del seguro",
    "ENROLLMENT": "contratación y requisitos de inscripción al seguro",
    "CONTACT": "contacto con un asesor"
}

# Definimos ejemplos ilustrativos independientes del dataset
consultas_prueba = [
    "¿Qué es un seguro de vida y para qué sirve?",
    "¿Qué situaciones quedan excluidas de mi póliza?",
    "Quiero una cotización de un seguro de vida.",
    "¿Puedo pagar mi seguro mediante transferencia bancaria?",
    "¿Qué documentos necesito para contratar el seguro?",
    "Quiero hablar con un asesor."
]

# Seleccionamos la GPU cuando esté disponible
dispositivo = 0 if torch.cuda.is_available() else -1

if dispositivo == 0:
    print("Dispositivo de ejecución:", torch.cuda.get_device_name(0))
else:
    print("Dispositivo de ejecución: CPU.")

# Sincronizamos la GPU antes de comenzar la medición
if dispositivo == 0:
    torch.cuda.synchronize()

# Cargamos únicamente MiniLMv2 en el dispositivo seleccionado
inicio_carga = perf_counter()

clasificador = pipeline(
    task="zero-shot-classification",
    model=modelo_prueba,
    device=dispositivo
)

# Esperamos a que finalicen las operaciones pendientes en GPU
if dispositivo == 0:
    torch.cuda.synchronize()

tiempo_carga = perf_counter() - inicio_carga

# Medimos la inferencia por separado de la descarga y carga
inicio_inferencia = perf_counter()

# Utilizamos una misma plantilla y una categoría por consulta
predicciones = clasificador(
    consultas_prueba,
    candidate_labels=list(descripciones_categorias.values()),
    hypothesis_template="Esta consulta trata sobre {}.",
    multi_label=False,
    batch_size=1
)

# Esperamos a que finalice la inferencia en GPU
if dispositivo == 0:
    torch.cuda.synchronize()

tiempo_inferencia = perf_counter() - inicio_inferencia

# Recuperamos los identificadores originales de las categorías
categorias_por_descripcion = {
    descripcion: categoria
    for categoria, descripcion in descripciones_categorias.items()
}

# Organizamos la categoría mejor puntuada para cada consulta
resultados_prueba = []

for consulta, prediccion in zip(consultas_prueba, predicciones):
    descripcion = prediccion["labels"][0]

    resultados_prueba.append({
        "Consulta": consulta,
        "Categoría predicha": categorias_por_descripcion[descripcion],
        "Puntuación": round(prediccion["scores"][0], 4)
    })

# Mostramos las predicciones obtenidas
display(pd.DataFrame(resultados_prueba))

# Mostramos los tiempos observados en esta ejecución
print(f"\nTiempo de descarga y carga: {tiempo_carga:.2f} segundos")
print(f"Tiempo de inferencia: {tiempo_inferencia:.2f} segundos")
print(
    f"Tiempo promedio por consulta: "
    f"{tiempo_inferencia / len(consultas_prueba):.2f} segundos"
)

print("=" * 60)
print("✅ Prueba inicial de clasificación completada.")
print("=" * 60)

Dispositivo de ejecución: Tesla T4


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,Consulta,Categoría predicha,Puntuación
0,¿Qué es un seguro de vida y para qué sirve?,PAYMENT,0.6500
1,¿Qué situaciones quedan excluidas de mi póliza?,COVERAGE,0.4133
2,Quiero una cotización de un seguro de vida.,QUOTE,0.7289
3,¿Puedo pagar mi seguro mediante transferencia ...,PAYMENT,0.5364
4,¿Qué documentos necesito para contratar el seg...,PAYMENT,0.4005
5,Quiero hablar con un asesor.,CONTACT,0.9692



Tiempo de descarga y carga: 11.01 segundos
Tiempo de inferencia: 0.99 segundos
Tiempo promedio por consulta: 0.17 segundos
✅ Prueba inicial de clasificación completada.


### **📝 Interpretación de la prueba inicial**

MiniLMv2 multilingüe se cargó y ejecutó correctamente
utilizando la GPU Tesla T4.

El modelo clasificó correctamente cuatro de las seis consultas
ilustrativas, de acuerdo con las categorías esperadas para
estos ejemplos.

Se observaron dos errores: la consulta sobre información general
de seguros de vida y la consulta sobre documentos para contratar
fueron clasificadas como PAYMENT.

El tiempo de inferencia fue de 1.47 segundos para seis consultas,
con un promedio aproximado de 0.24 segundos por consulta.

**Limitación:** Los ejemplos utilizados son ilustrativos.
Estos resultados no constituyen una evaluación formal ni
permiten seleccionar todavía el mejor modelo.

**Estado:** Funcionamiento inicial verificado. Evaluación
comparativa pendiente.

### **🧠 3.4 Prueba inicial con mDeBERTa**

Se realizará una prueba de funcionamiento con mDeBERTa-v3-base,
utilizando las mismas consultas, descripciones de categorías
y plantilla empleadas con MiniLMv2.

Se revisarán las predicciones y se registrarán los tiempos
de carga e inferencia.

Esta prueba es ilustrativa y no constituye una evaluación
formal del desempeño sobre el dataset.

In [ ]:
# ============================================================
# 3.4 Prueba inicial con mDeBERTa
# ============================================================

# Importamos las herramientas necesarias
import gc
import pandas as pd
import torch
from transformers import pipeline
from time import perf_counter

# Liberamos el modelo anterior para reducir el uso de memoria
if "clasificador" in globals():
    del clasificador

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# Definimos el modelo y seleccionamos el dispositivo
modelo_prueba = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
dispositivo = 0 if torch.cuda.is_available() else -1

if dispositivo == 0:
    print("Dispositivo de ejecución:", torch.cuda.get_device_name(0))
else:
    print("Dispositivo de ejecución: CPU.")

# Cargamos mDeBERTa con precisión de 32 bits
inicio_carga = perf_counter()

clasificador = pipeline(
    task="zero-shot-classification",
    model=modelo_prueba,
    device=dispositivo,
    torch_dtype=torch.float32
)

# Esperamos a que finalice la carga en GPU
if dispositivo == 0:
    torch.cuda.synchronize()

tiempo_carga = perf_counter() - inicio_carga

# Evaluamos las mismas consultas utilizadas en la prueba anterior
inicio_inferencia = perf_counter()

predicciones = clasificador(
    consultas_prueba,
    candidate_labels=list(descripciones_categorias.values()),
    hypothesis_template="Esta consulta trata sobre {}.",
    multi_label=False,
    batch_size=1
)

# Esperamos a que finalice la inferencia en GPU
if dispositivo == 0:
    torch.cuda.synchronize()

tiempo_inferencia = perf_counter() - inicio_inferencia

# Asociamos las descripciones con las categorías originales
categorias_por_descripcion = {
    descripcion: categoria
    for categoria, descripcion in descripciones_categorias.items()
}

# Organizamos las predicciones para su revisión
resultados_mdeberta = []

for consulta, prediccion in zip(consultas_prueba, predicciones):
    descripcion = prediccion["labels"][0]

    resultados_mdeberta.append({
        "Consulta": consulta,
        "Categoría predicha": categorias_por_descripcion[descripcion],
        "Puntuación": round(prediccion["scores"][0], 4)
    })

# Mostramos los resultados obtenidos
display(pd.DataFrame(resultados_mdeberta))

# Presentamos los tiempos observados
print(f"\nTiempo de descarga y carga: {tiempo_carga:.2f} segundos")
print(f"Tiempo de inferencia: {tiempo_inferencia:.2f} segundos")
print(
    f"Tiempo promedio por consulta: "
    f"{tiempo_inferencia / len(consultas_prueba):.2f} segundos"
)

print("=" * 60)
print("✅ Prueba inicial con mDeBERTa completada.")
print("=" * 60)

Dispositivo de ejecución: Tesla T4


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

,Consulta,Categoría predicha,Puntuación
0,¿Qué es un seguro de vida y para qué sirve?,QUOTE,0.4602
1,¿Qué situaciones quedan excluidas de mi póliza?,COVERAGE,0.9246
2,Quiero una cotización de un seguro de vida.,QUOTE,0.8912
3,¿Puedo pagar mi seguro mediante transferencia ...,PAYMENT,0.5419
4,¿Qué documentos necesito para contratar el seg...,ENROLLMENT,0.5078
5,Quiero hablar con un asesor.,CONTACT,0.9982



Tiempo de descarga y carga: 10.16 segundos
Tiempo de inferencia: 2.94 segundos
Tiempo promedio por consulta: 0.49 segundos
✅ Prueba inicial con mDeBERTa completada.


### **📝 Interpretación de la prueba con mDeBERTa**

mDeBERTa-v3-base se cargó y ejecutó correctamente
utilizando la GPU Tesla T4.

El modelo clasificó correctamente cinco de las seis consultas
ilustrativas, de acuerdo con las categorías esperadas.

La consulta sobre documentos para contratar fue clasificada
como ENROLLMENT, corrigiendo el error observado con MiniLMv2.

La consulta sobre información general de seguros de vida
fue clasificada como QUOTE, cuando se esperaba LIFE_INSURANCE.

El tiempo de inferencia fue de 2.34 segundos para seis consultas,
con un promedio aproximado de 0.39 segundos por consulta.

**Limitación:** Los resultados corresponden a seis ejemplos
ilustrativos y a una sola medición de tiempo. No permiten
determinar todavía el mejor modelo para el proyecto.

**Estado:** Funcionamiento inicial de mDeBERTa verificado.
Prueba con XLM-RoBERTa y evaluación formal pendientes.

### **🧠 3.5 Prueba inicial con XLM-RoBERTa-large**

Se realizará una prueba de funcionamiento con XLM-RoBERTa-large,
utilizando las mismas consultas, descripciones de categorías
y plantilla empleadas en las pruebas anteriores.

Se liberará el modelo anterior antes de cargar el nuevo modelo
para reducir el consumo simultáneo de memoria.

Se revisarán las predicciones y se registrarán los tiempos
de carga e inferencia.

Esta prueba es ilustrativa y no constituye una evaluación
formal del desempeño sobre el dataset.

In [ ]:
# ============================================================
# 3.5 Prueba inicial con XLM-RoBERTa-large
# ============================================================

# Importamos las herramientas necesarias
import gc
import pandas as pd
import torch
from transformers import pipeline
from time import perf_counter

# Liberamos el modelo anterior para reducir el uso de memoria
if "clasificador" in globals():
    del clasificador

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# Definimos el modelo y seleccionamos el dispositivo
modelo_prueba = "joeddav/xlm-roberta-large-xnli"
dispositivo = 0 if torch.cuda.is_available() else -1

if dispositivo == 0:
    print("Dispositivo de ejecución:", torch.cuda.get_device_name(0))
else:
    print("Dispositivo de ejecución: CPU.")

# Cargamos el modelo con precisión de 32 bits
inicio_carga = perf_counter()

clasificador = pipeline(
    task="zero-shot-classification",
    model=modelo_prueba,
    device=dispositivo,
    torch_dtype=torch.float32
)

# Esperamos a que finalice la carga en GPU
if dispositivo == 0:
    torch.cuda.synchronize()

tiempo_carga = perf_counter() - inicio_carga

# Evaluamos las mismas consultas de las pruebas anteriores
inicio_inferencia = perf_counter()

predicciones = clasificador(
    consultas_prueba,
    candidate_labels=list(descripciones_categorias.values()),
    hypothesis_template="Esta consulta trata sobre {}.",
    multi_label=False,
    batch_size=1
)

# Esperamos a que finalice la inferencia en GPU
if dispositivo == 0:
    torch.cuda.synchronize()

tiempo_inferencia = perf_counter() - inicio_inferencia

# Asociamos las descripciones con las categorías originales
categorias_por_descripcion = {
    descripcion: categoria
    for categoria, descripcion in descripciones_categorias.items()
}

# Organizamos las predicciones para su revisión
resultados_xlm_roberta = []

for consulta, prediccion in zip(consultas_prueba, predicciones):
    descripcion = prediccion["labels"][0]

    resultados_xlm_roberta.append({
        "Consulta": consulta,
        "Categoría predicha": categorias_por_descripcion[descripcion],
        "Puntuación": round(prediccion["scores"][0], 4)
    })

# Mostramos los resultados obtenidos
display(pd.DataFrame(resultados_xlm_roberta))

# Presentamos los tiempos observados
print(f"\nTiempo de descarga y carga: {tiempo_carga:.2f} segundos")
print(f"Tiempo de inferencia: {tiempo_inferencia:.2f} segundos")
print(
    f"Tiempo promedio por consulta: "
    f"{tiempo_inferencia / len(consultas_prueba):.2f} segundos"
)

print("=" * 60)
print("✅ Prueba inicial con XLM-RoBERTa-large completada.")
print("=" * 60)

Dispositivo de ejecución: Tesla T4


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Consulta,Categoría predicha,Puntuación
0,¿Qué es un seguro de vida y para qué sirve?,LIFE_INSURANCE,0.9915
1,¿Qué situaciones quedan excluidas de mi póliza?,COVERAGE,0.9023
2,Quiero una cotización de un seguro de vida.,QUOTE,0.9824
3,¿Puedo pagar mi seguro mediante transferencia ...,PAYMENT,0.8739
4,¿Qué documentos necesito para contratar el seg...,ENROLLMENT,0.5446
5,Quiero hablar con un asesor.,CONTACT,0.9985



Tiempo de descarga y carga: 7.35 segundos
Tiempo de inferencia: 2.07 segundos
Tiempo promedio por consulta: 0.34 segundos
✅ Prueba inicial con XLM-RoBERTa-large completada.


### **📝 Interpretación de la prueba con XLM-RoBERTa-large**

XLM-RoBERTa-large se cargó y ejecutó correctamente
utilizando la GPU Tesla T4.

El modelo clasificó correctamente las seis consultas
ilustrativas, de acuerdo con las categorías esperadas.

Identificó correctamente la consulta sobre información general
de seguros de vida como LIFE_INSURANCE y la consulta sobre
documentos para contratar como ENROLLMENT.

El tiempo de descarga y carga fue de 32.38 segundos.
El tiempo de inferencia fue de 0.65 segundos para seis consultas,
con un promedio aproximado de 0.11 segundos por consulta.

**Limitación:** Los resultados corresponden a seis ejemplos
ilustrativos y a una sola medición de tiempo, sin calentamiento
previo. No permiten generalizar el desempeño al dataset.

### **📝 Cierre de las pruebas iniciales de funcionamiento**

**Actividades realizadas:**

- Se registraron tres modelos multilingües de Hugging Face.
- Se verificó la disponibilidad de una GPU Tesla T4.
- Se ejecutaron los tres modelos sobre las mismas seis consultas.
- Se conservaron las descripciones de categorías y la plantilla.
- Se revisaron las predicciones y los tiempos de ejecución.

**Hallazgos principales:**

MiniLMv2 obtuvo cuatro aciertos, mDeBERTa obtuvo cinco
y XLM-RoBERTa-large obtuvo seis en los ejemplos ilustrativos.

Los tres modelos pudieron ejecutarse en el entorno disponible.

**Decisión preliminar:**

Se conservarán los tres candidatos para una evaluación formal
con un conjunto de consultas en español y etiquetas revisadas.

**Estado:** Pruebas iniciales completadas.
Preparación del conjunto de evaluación y selección final pendientes.

## **💻 4. Preparación del conjunto de evaluación**

Se preparará un conjunto de consultas en español con etiquetas
de referencia revisadas para evaluar los tres modelos.

Las etiquetas se asignarán según el motivo principal de cada
consulta. Mencionar un seguro de vida no será suficiente para
asignar la categoría LIFE_INSURANCE.

### **🧠 4.1 Definición de criterios de clasificación**

| Categoría | Criterio | Ejemplo ilustrativo |
|---|---|---|
| LIFE_INSURANCE | Información general o consulta de información sobre un seguro de vida, cuando no predomine una solicitud específica de cobertura, cotización, pago, contratación o contacto con un asesor. | ¿Para qué sirve un seguro de vida? |
| COVERAGE | Coberturas, exclusiones o situaciones protegidas por la póliza. | ¿La póliza cubre el fallecimiento por accidente? |
| QUOTE | Solicitud de cotización o estimación del costo de un seguro. | ¿Cuánto me costaría contratar un seguro de vida? |
| PAYMENT | Métodos, fechas, periodicidad o problemas de pago. | ¿Puedo pagar mi póliza mediante transferencia? |
| ENROLLMENT | Proceso de contratación, documentos o requisitos de inscripción. | ¿Qué documentos necesito para contratar? |
| CONTACT | Solicitud de comunicación con un asesor o canal de atención. | ¿Cómo puedo hablar con un asesor? |

**Reglas de revisión:**

- Se identificará el motivo principal de la consulta.
- Se conservará la relación con la categoría y la intención
  originales del dataset.
- Si la etiqueta original no coincide con estos criterios,
  se documentará la discrepancia para su revisión.
- Las consultas con varios motivos o información insuficiente
  quedarán pendientes de revisión antes de incorporarse
  al conjunto de evaluación.
- La traducción al español deberá conservar el significado
  y la intención de la consulta original.
- Se revisarán duplicados para evitar que una misma consulta
  aparezca en el conjunto de desarrollo y en el de prueba.

**Separación de conjuntos:**

El conjunto de desarrollo permitirá revisar las descripciones
de categorías y la plantilla de clasificación.

El conjunto de prueba se utilizará después de fijar esa
configuración y no se empleará para ajustarla.

Las seis consultas ilustrativas anteriores se conservarán
como ejemplos de funcionamiento y no se incluirán en la
evaluación formal.

**Estado:** Criterios definidos. Muestreo, traducción y revisión
de etiquetas pendientes.

### **🧠 4.2 Revisión de duplicados y disponibilidad por categoría**

Se revisarán las consultas del subconjunto candidato para
identificar textos vacíos y duplicados.

Para la comparación se normalizarán las mayúsculas, minúsculas
y espacios, conservando el texto original.

Las consultas repetidas con una misma categoría se conservarán
una sola vez. Si una consulta aparece con categorías distintas,
se separará para revisión antes del muestreo.

Esta comprobación detecta duplicados textuales normalizados.
No identifica necesariamente consultas con significado equivalente.

Se mostrará la disponibilidad por categoría después de la revisión.

In [ ]:
# ============================================================
# 4.2 Revisión de duplicados y disponibilidad por categoría
# ============================================================

# Importamos las herramientas necesarias
import pandas as pd
import unicodedata

# Creamos una copia y conservamos el índice del registro original
df_revision = df_preseleccion.copy()
df_revision["indice_origen"] = df_revision.index

# Normalizamos el texto únicamente para detectar duplicados
def normalizar_consulta(texto):
    if pd.isna(texto):
        return ""

    texto = unicodedata.normalize("NFKC", str(texto))
    return " ".join(texto.casefold().split())

df_revision["consulta_normalizada"] = (
    df_revision["instruction"].map(normalizar_consulta)
)

# Separamos las consultas vacías después de normalizar
mascara_vacias = df_revision["consulta_normalizada"].eq("")

df_consultas_vacias = df_revision[mascara_vacias].copy()
df_no_vacias = df_revision[~mascara_vacias].copy()

# Identificamos textos asociados con categorías distintas
categorias_por_consulta = (
    df_no_vacias
    .groupby("consulta_normalizada")["category"]
    .nunique(dropna=False)
)

textos_conflictivos = categorias_por_consulta[
    categorias_por_consulta > 1
].index

mascara_conflictos = (
    df_no_vacias["consulta_normalizada"].isin(textos_conflictivos)
)

# Conservamos los conflictos en una tabla para revisión posterior
df_conflictos_etiqueta = df_no_vacias[mascara_conflictos].copy()

# Separamos los registros sin conflictos de categoría
df_sin_conflictos = df_no_vacias[~mascara_conflictos].copy()

# Identificamos repeticiones adicionales de un mismo texto
mascara_duplicados = df_sin_conflictos.duplicated(
    subset=["consulta_normalizada"],
    keep="first"
)

df_duplicados_textuales = (
    df_sin_conflictos[mascara_duplicados].copy()
)

# Conservamos una consulta por texto para el futuro muestreo
df_elegibles = (
    df_sin_conflictos[~mascara_duplicados].copy()
)

# Comparamos la disponibilidad antes y después de la revisión
disponibilidad_revision = pd.DataFrame({
    "Registros candidatos": (
        df_revision["category"].value_counts()
        .reindex(categorias_candidatas, fill_value=0)
    ),
    "Consultas únicas elegibles": (
        df_elegibles["category"].value_counts()
        .reindex(categorias_candidatas, fill_value=0)
    )
})

disponibilidad_revision.index.name = "Categoría"

# Mostramos el resumen de la revisión
print("Registros candidatos:", len(df_revision))
print("Registros con texto vacío:", len(df_consultas_vacias))
print("Textos con categorías distintas:", len(textos_conflictivos))
print("Registros separados por conflictos:", len(df_conflictos_etiqueta))
print("Repeticiones adicionales separadas:", len(df_duplicados_textuales))
print("Consultas únicas elegibles:", len(df_elegibles))

display(disponibilidad_revision.reset_index())

# Mostramos ejemplos si existen conflictos de categoría
if not df_conflictos_etiqueta.empty:
    print("\nEjemplos de consultas con categorías distintas:")

    ejemplos_conflictos = (
        df_conflictos_etiqueta
        .sort_values(["consulta_normalizada", "category"])
        [["indice_origen", "instruction", "category", "intent"]]
        .head(12)
    )

    display(ejemplos_conflictos)

print("=" * 60)
print("✅ Revisión de duplicados y disponibilidad completada.")
print("=" * 60)

Registros candidatos: 14020
Registros con texto vacío: 0
Textos con categorías distintas: 0
Registros separados por conflictos: 0
Repeticiones adicionales separadas: 223
Consultas únicas elegibles: 13797


,Categoría,Registros candidatos,Consultas únicas elegibles
0,LIFE_INSURANCE,1000,976
1,COVERAGE,1721,1712
2,QUOTE,820,820
3,PAYMENT,4468,4365
4,ENROLLMENT,2040,2032
5,CONTACT,3971,3892


✅ Revisión de duplicados y disponibilidad completada.


### **📝 Interpretación de la revisión de duplicados**

La revisión dejó 13,797 consultas únicas elegibles de los
14,020 registros candidatos iniciales, lo que representa
una reducción de 223 registros.

Las seis categorías conservan consultas disponibles.
QUOTE presenta la menor disponibilidad, con 820 consultas.

Estos registros son candidatos para el muestreo y todavía
requieren revisión de pertinencia, traducción y etiquetas.

**Estado:** Revisión textual completada.
Preparación de la muestra pendiente.

### **🧠 4.3 Muestreo y separación provisional**

Se seleccionarán aleatoriamente 50 consultas únicas por categoría,
utilizando una semilla fija para obtener una muestra reproducible.

La muestra contendrá 300 consultas:

- **Desarrollo:** 60 consultas, 10 por categoría.
- **Prueba:** 240 consultas, 40 por categoría.

Se conservarán el texto, la categoría, la intención y el índice
de origen para mantener la trazabilidad.

Se añadirán campos para la traducción al español, la categoría
de referencia y el estado de revisión.

La separación será provisional hasta revisar la pertinencia,
la traducción y las etiquetas. Las categorías originales se
utilizarán para el muestreo, pero no se asumirán automáticamente
como etiquetas de referencia validadas.

**Estado:** Muestra candidata preparada para revisión.

In [ ]:
# ============================================================
# 4.3 Muestreo y separación provisional
# ============================================================

# Importamos las herramientas necesarias
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

# Definimos el tamaño de la muestra y la semilla
semilla = 42
consultas_por_categoria = 50
desarrollo_por_categoria = 10
prueba_por_categoria = (
    consultas_por_categoria - desarrollo_por_categoria
)

# Comprobamos los parámetros de separación
if not 0 < desarrollo_por_categoria < consultas_por_categoria:
    raise ValueError(
        "La cantidad de desarrollo debe ser mayor que cero "
        "y menor que la cantidad por categoría."
    )

if len(categorias_candidatas) != len(set(categorias_candidatas)):
    raise ValueError("La lista de categorías contiene repeticiones.")

# Verificamos que existan los campos necesarios
columnas_requeridas = [
    "indice_origen",
    "instruction",
    "category",
    "intent",
    "consulta_normalizada"
]

columnas_faltantes = [
    columna
    for columna in columnas_requeridas
    if columna not in df_elegibles.columns
]

if columnas_faltantes:
    raise ValueError(
        f"Faltan columnas en df_elegibles: {columnas_faltantes}"
    )

# Comprobamos la disponibilidad antes de seleccionar registros
disponibilidad_muestreo = (
    df_elegibles["category"]
    .value_counts()
    .reindex(categorias_candidatas, fill_value=0)
)

if (disponibilidad_muestreo < consultas_por_categoria).any():
    raise ValueError(
        "Alguna categoría no tiene suficientes consultas "
        "para completar la muestra propuesta."
    )

# Seleccionamos y distribuimos las consultas de cada categoría
partes_muestra = []

for categoria in categorias_candidatas:
    # Conservamos el orden utilizado en el muestreo original
    registros_categoria = (
        df_elegibles[
            df_elegibles["category"].eq(categoria)
        ]
        .sort_values("consulta_normalizada")
    )

    # Seleccionamos una muestra aleatoria sin reemplazo
    seleccion = registros_categoria.sample(
        n=consultas_por_categoria,
        random_state=semilla,
        replace=False
    ).copy()

    # Asignamos los conjuntos antes de observar predicciones
    seleccion["conjunto"] = (
        ["desarrollo"] * desarrollo_por_categoria
        + ["prueba"] * prueba_por_categoria
    )

    partes_muestra.append(seleccion)

# Construimos la muestra en una variable temporal
muestra_nueva = pd.concat(
    partes_muestra,
    ignore_index=True
)

# Conservamos los campos necesarios para mantener la trazabilidad
muestra_nueva = muestra_nueva[
    columnas_requeridas + ["conjunto"]
].rename(columns={
    "instruction": "consulta_original",
    "category": "categoria_original",
    "intent": "intencion_original"
})

# Añadimos identificadores y campos pendientes de revisión
muestra_nueva.insert(
    0,
    "id_consulta",
    [
        f"CONS_{numero:04d}"
        for numero in range(1, len(muestra_nueva) + 1)
    ]
)

muestra_nueva["consulta_es"] = ""
muestra_nueva["categoria_referencia"] = ""
muestra_nueva["estado_revision"] = "pendiente"
muestra_nueva["observaciones"] = ""

# Verificamos los textos normalizados de la muestra
textos_muestra = (
    muestra_nueva["consulta_normalizada"]
    .fillna("")
    .astype(str)
    .str.strip()
)

if textos_muestra.eq("").any():
    raise ValueError("Se detectaron consultas normalizadas vacías.")

if textos_muestra.duplicated().any():
    raise ValueError("Se detectaron textos duplicados en la muestra.")

# Comprobamos el tamaño de cada conjunto por categoría
resumen_nuevo = pd.crosstab(
    muestra_nueva["categoria_original"],
    muestra_nueva["conjunto"]
).reindex(
    index=categorias_candidatas,
    columns=["desarrollo", "prueba"],
    fill_value=0
)

total_esperado = (
    len(categorias_candidatas) * consultas_por_categoria
)

if (
    len(muestra_nueva) != total_esperado
    or not muestra_nueva["id_consulta"].is_unique
    or not resumen_nuevo["desarrollo"]
        .eq(desarrollo_por_categoria).all()
    or not resumen_nuevo["prueba"]
        .eq(prueba_por_categoria).all()
):
    raise ValueError("La distribución no coincide con lo previsto.")

resumen_nuevo["Total"] = resumen_nuevo.sum(axis=1)
resumen_nuevo.index.name = "Categoría"

# Guardamos una versión nueva sin sobrescribir archivos anteriores
carpeta_revision = Path("/content/revision_evaluacion")
carpeta_revision.mkdir(parents=True, exist_ok=True)

marca_ejecucion = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)
identificador_archivo = uuid4().hex[:8]

ruta_nueva = carpeta_revision / (
    f"muestra_revision_inicial_"
    f"{marca_ejecucion}_{identificador_archivo}.csv"
)

with ruta_nueva.open(
    mode="x",
    encoding="utf-8-sig",
    newline=""
) as archivo:
    muestra_nueva.to_csv(archivo, index=False)

# Confirmamos la muestra después de verificarla y guardarla
df_muestra_revision = muestra_nueva.copy()
resumen_muestra = resumen_nuevo.copy()
ruta_muestra = ruta_nueva

# Reiniciamos la auditoría en memoria para esta nueva reconstrucción
# Las celdas posteriores crearán la auditoría de sus reemplazos
globals().pop("df_registros_retirados", None)

# Mostramos la distribución y el resumen de la muestra
display(resumen_muestra.reset_index())

print("\nTotal de consultas:", len(df_muestra_revision))
print(
    "Consultas de desarrollo:",
    int(df_muestra_revision["conjunto"].eq("desarrollo").sum())
)
print(
    "Consultas de prueba:",
    int(df_muestra_revision["conjunto"].eq("prueba").sum())
)
print("Archivo guardado:", ruta_muestra)
print("Estado de las consultas: pendientes de revisión.")
print("Auditoría de reemplazos reiniciada para esta muestra.")

print("=" * 60)
print("✅ Muestreo y separación provisional completados.")
print("=" * 60)

conjunto,Categoría,desarrollo,prueba,Total
0,LIFE_INSURANCE,10,40,50
1,COVERAGE,10,40,50
2,QUOTE,10,40,50
3,PAYMENT,10,40,50
4,ENROLLMENT,10,40,50
5,CONTACT,10,40,50



Total de consultas: 300
Consultas de desarrollo: 60
Consultas de prueba: 240
Archivo guardado: /content/revision_evaluacion/muestra_revision_inicial_20261010T185836647105Z_be4783ae.csv
Estado de las consultas: pendientes de revisión.
Auditoría de reemplazos reiniciada para esta muestra.
✅ Muestreo y separación provisional completados.


In [ ]:
# Descargamos la muestra para conservarla fuera de la sesión
from google.colab import files

files.download(str(ruta_muestra))
print("=" * 60)
print("✅ Descarga de la muestra del dataset.")
print("=" * 60)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Descarga de la muestra del dataset.


### **🧠 4.4 Traducción y revisión del conjunto de desarrollo**

Se revisarán las 60 consultas del conjunto de desarrollo
en bloques de 10 consultas por categoría.

Para cada consulta se preparará una traducción al español
que conserve el significado original y se verificará
su pertinencia para el proyecto.

La categoría de referencia se asignará según los criterios
de clasificación definidos, considerando también la intención
original del dataset.

Las consultas ambiguas, con varios motivos o fuera del alcance
quedarán pendientes de revisión.

Las etiquetas se revisarán antes de observar las predicciones
de los modelos sobre estas consultas.

In [ ]:
# ============================================================
# 4.4 Revisión inicial del conjunto de desarrollo
# ============================================================

# Seleccionamos las consultas de desarrollo de seguros de vida
categoria_revision = "LIFE_INSURANCE"

df_bloque_revision = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
    & df_muestra_revision["categoria_original"].eq(categoria_revision)
].copy()

# Mostramos los textos completos para facilitar la revisión
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", 20
):
    display(
        df_bloque_revision[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Bloque de desarrollo presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
0,CONS_0001,i have to check my fucking life insurance can i get some help,LIFE_INSURANCE,information_life_insurance
1,CONS_0002,I'd like information about my fucking life insuarnce where could i get it,LIFE_INSURANCE,information_life_insurance
2,CONS_0003,"I got to see my fuckinglife insurance, can you help me?",LIFE_INSURANCE,information_life_insurance
3,CONS_0004,"I would like to check my life insurance, where can I do it?",LIFE_INSURANCE,information_life_insurance
4,CONS_0005,how can I find information about my life insurance?,LIFE_INSURANCE,information_life_insurance
5,CONS_0006,"I would like to know about my life insurance, how do I find more information?",LIFE_INSURANCE,information_life_insurance
6,CONS_0007,need to see my life insurance where can i do it,LIFE_INSURANCE,information_life_insurance
7,CONS_0008,"I'd like to knoe about my fucking life insurance, will you find some information?",LIFE_INSURANCE,information_life_insurance
8,CONS_0009,wana check my life insurance where can i do it,LIFE_INSURANCE,information_life_insurance
9,CONS_0010,i got to se my fucking life insurance,LIFE_INSURANCE,information_life_insurance


✅ Bloque de desarrollo presentado para revisión.


### **🧠 4.5 Traducción y revisión del bloque LIFE_INSURANCE**

In [ ]:
# ============================================================
# 4.5 Traducción propuesta del bloque LIFE_INSURANCE
# ============================================================

# Registramos las traducciones conservando el significado y tono
traducciones_vida = {
    "CONS_0001": "Tengo que revisar mi seguro de vida, ¿me pueden ayudar?",
    "CONS_0002": "Quisiera información sobre mi seguro de vida, ¿dónde puedo conseguirla?",
    "CONS_0003": "Tengo que consultar mi seguro de vida, ¿puedes ayudarme?",
    "CONS_0004": "Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",
    "CONS_0005": "¿Cómo puedo encontrar información sobre mi seguro de vida?",
    "CONS_0006": "Quisiera saber sobre mi seguro de vida, ¿cómo encuentro más información?",
    "CONS_0007": "Necesito consultar mi seguro de vida, ¿dónde puedo hacerlo?",
    "CONS_0008": "Quisiera saber sobre mi seguro de vida, ¿me puedes conseguir información?",
    "CONS_0009": "Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",
    "CONS_0010": "Tengo que consultar mi seguro de vida."
}

# Verificamos los identificadores y su pertenencia al bloque
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_vida
)

bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != len(traducciones_vida)
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("desarrollo").all()
    or not bloque["categoria_original"].eq("LIFE_INSURANCE").all()
):
    raise ValueError(
        "Los identificadores no coinciden con el bloque esperado."
    )

# Registramos las propuestas sin darlas por validadas
for identificador, traduccion in traducciones_vida.items():
    mascara = df_muestra_revision["id_consulta"].eq(identificador)

    df_muestra_revision.loc[mascara, "consulta_es"] = traduccion
    df_muestra_revision.loc[
        mascara, "categoria_referencia"
    ] = "LIFE_INSURANCE"
    df_muestra_revision.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    df_muestra_revision.loc[
        mascara, "observaciones"
    ] = (
        "Solicitud de información sobre el seguro de vida propio. "
        "Traducción y etiqueta propuestas; pendiente de revisión. "
        "Bloque con alta similitud semántica."
    )

# Mostramos las propuestas para su revisión
with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("=" * 60)
print("✅ Traducciones y etiquetas propuestas registradas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
0,CONS_0001,"Tengo que revisar mi seguro de vida, ¿me pueden ayudar?",LIFE_INSURANCE,propuesta_pendiente_revision
1,CONS_0002,"Quisiera información sobre mi seguro de vida, ¿dónde puedo conseguirla?",LIFE_INSURANCE,propuesta_pendiente_revision
2,CONS_0003,"Tengo que consultar mi seguro de vida, ¿puedes ayudarme?",LIFE_INSURANCE,propuesta_pendiente_revision
3,CONS_0004,"Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
4,CONS_0005,¿Cómo puedo encontrar información sobre mi seguro de vida?,LIFE_INSURANCE,propuesta_pendiente_revision
5,CONS_0006,"Quisiera saber sobre mi seguro de vida, ¿cómo encuentro más información?",LIFE_INSURANCE,propuesta_pendiente_revision
6,CONS_0007,"Necesito consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
7,CONS_0008,"Quisiera saber sobre mi seguro de vida, ¿me puedes conseguir información?",LIFE_INSURANCE,propuesta_pendiente_revision
8,CONS_0009,"Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
9,CONS_0010,Tengo que consultar mi seguro de vida.,LIFE_INSURANCE,propuesta_pendiente_revision


✅ Traducciones y etiquetas propuestas registradas.


### **🧠 4.6 Revisión del bloque COVERAGE**

Se examinarán las diez consultas de desarrollo cuya categoría
original es COVERAGE.

La revisión permitirá identificar si tratan sobre coberturas,
exclusiones o situaciones protegidas por el seguro, y comprobar
su pertinencia para el contexto de seguros de vida.

Posteriormente se propondrán las traducciones y las etiquetas
de referencia correspondientes.

In [ ]:
# ============================================================
# 4.6 Revisión del bloque COVERAGE
# ============================================================

# Seleccionamos las consultas de desarrollo de la categoría
categoria_revision = "COVERAGE"

df_bloque_revision = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
    & df_muestra_revision["categoria_original"].eq(categoria_revision)
].copy()

# Mostramos los textos completos y sus datos de origen
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", 20
):
    display(
        df_bloque_revision[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Bloque COVERAGE presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
50,CONS_0051,"I need to check my fucking coverage , how can I do it?",COVERAGE,check_coverage
51,CONS_0052,"I'd like to know about my coverage, could you give me some information ?",COVERAGE,check_coverage
52,CONS_0053,where can i downgrade the fucking coverage of my insurance polikcy,COVERAGE,downgrade_coverage
53,CONS_0054,i have to change the fucking coverage of my insurance policy could i get somw help,COVERAGE,change_coverage
54,CONS_0055,i have to see the coverage of my life insurance policy where to do it,COVERAGE,check_coverage
55,CONS_0056,id like to upgrade the coverage of my life insurance policy will ya hlp me,COVERAGE,upgrade_coverage
56,CONS_0057,I'm looking for information about the coverage of my insurance policy how could i get ig,COVERAGE,check_coverage
57,CONS_0058,can ya help me increase the coverage of my life insurance policy,COVERAGE,upgrade_coverage
58,CONS_0059,i need help reducing the fucking coverage of my life insurance policy,COVERAGE,downgrade_coverage
59,CONS_0060,"I'd like to increase the fucking coverage of my health insurance policy, will you help me?",COVERAGE,upgrade_coverage


✅ Bloque COVERAGE presentado para revisión.


### **🧠 4.7 Traducción y revisión del bloque COVERAGE**

Se propondrán traducciones para las diez consultas del bloque,
conservando el significado, el tono y las referencias al
producto asegurador original.

La categoría COVERAGE comprenderá consultas sobre coberturas,
exclusiones y solicitudes de modificación de cobertura.

CONS_0060 menciona explícitamente un seguro de salud.
Se propondrá su exclusión del conjunto de seguros de vida,
sin modificar el producto mencionado para hacerla encajar.

Las consultas que no especifican un producto se conservarán
como candidatas generales, pendientes de revisión.

**Estado:** Traducciones y etiquetas propuestas.
Una consulta requiere exclusión y reemplazo.

In [ ]:
# ============================================================
# 4.7 Traducción propuesta del bloque COVERAGE
# ============================================================

# Conservamos el significado y el tono de las consultas originales
traducciones_cobertura = {
    "CONS_0051": "Necesito revisar mi cobertura, ¿cómo puedo hacerlo?",
    "CONS_0052": "Quisiera saber sobre mi cobertura, ¿podrías darme información?",
    "CONS_0053": "¿Dónde puedo reducir la cobertura de mi póliza de seguro?",
    "CONS_0054": "Tengo que cambiar la cobertura de mi póliza de seguro, ¿me pueden ayudar?",
    "CONS_0055": "Tengo que consultar la cobertura de mi póliza de seguro de vida, ¿dónde puedo hacerlo?",
    "CONS_0056": "Quisiera ampliar la cobertura de mi póliza de seguro de vida, ¿me ayudan?",
    "CONS_0057": "Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",
    "CONS_0058": "¿Puedes ayudarme a aumentar la cobertura de mi póliza de seguro de vida?",
    "CONS_0059": "Necesito ayuda para reducir la cobertura de mi póliza de seguro de vida.",
    "CONS_0060": "Quisiera aumentar la cobertura de mi póliza de seguro de salud, ¿me ayudas?"
}

# Verificamos que los identificadores correspondan al bloque esperado
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_cobertura
)

bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != len(traducciones_cobertura)
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("desarrollo").all()
    or not bloque["categoria_original"].eq("COVERAGE").all()
):
    raise ValueError(
        "Los identificadores no coinciden con el bloque esperado."
    )

# Registramos propuestas y documentamos los límites de pertinencia
for identificador, traduccion in traducciones_cobertura.items():
    mascara = df_muestra_revision["id_consulta"].eq(identificador)

    df_muestra_revision.loc[mascara, "consulta_es"] = traduccion
    df_muestra_revision.loc[
        mascara, "categoria_referencia"
    ] = "COVERAGE"

    if identificador == "CONS_0060":
        estado = "exclusion_propuesta"
        observacion = (
            "Menciona explícitamente un seguro de salud. "
            "COVERAGE describe su motivo, pero la consulta está "
            "fuera del alcance del proyecto. Requiere reemplazo."
        )
    else:
        estado = "propuesta_pendiente_revision"

        if identificador in {
            "CONS_0051", "CONS_0052", "CONS_0053", "CONS_0054"
        }:
            observacion = (
                "Consulta o modificación de cobertura. "
                "No especifica el producto asegurador; "
                "candidata general pendiente de revisión."
            )
        else:
            observacion = (
                "Consulta o modificación de cobertura "
                "de un seguro de vida. Traducción y etiqueta "
                "pendientes de revisión."
            )

    df_muestra_revision.loc[mascara, "estado_revision"] = estado
    df_muestra_revision.loc[mascara, "observaciones"] = observacion

# Mostramos las propuestas y el caso separado para revisión
with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision",
                "observaciones"
            ]
        ]
    )

print("=" * 60)
print("✅ Propuestas del bloque COVERAGE registradas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision,observaciones
50,CONS_0051,"Necesito revisar mi cobertura, ¿cómo puedo hacerlo?",COVERAGE,propuesta_pendiente_revision,Consulta o modificación de cobertura. No especifica el producto asegurador; candidata general pendiente de revisión.
51,CONS_0052,"Quisiera saber sobre mi cobertura, ¿podrías darme información?",COVERAGE,propuesta_pendiente_revision,Consulta o modificación de cobertura. No especifica el producto asegurador; candidata general pendiente de revisión.
52,CONS_0053,¿Dónde puedo reducir la cobertura de mi póliza de seguro?,COVERAGE,propuesta_pendiente_revision,Consulta o modificación de cobertura. No especifica el producto asegurador; candidata general pendiente de revisión.
53,CONS_0054,"Tengo que cambiar la cobertura de mi póliza de seguro, ¿me pueden ayudar?",COVERAGE,propuesta_pendiente_revision,Consulta o modificación de cobertura. No especifica el producto asegurador; candidata general pendiente de revisión.
54,CONS_0055,"Tengo que consultar la cobertura de mi póliza de seguro de vida, ¿dónde puedo hacerlo?",COVERAGE,propuesta_pendiente_revision,Consulta o modificación de cobertura de un seguro de vida. Traducción y etiqueta pendientes de revisión.
55,CONS_0056,"Quisiera ampliar la cobertura de mi póliza de seguro de vida, ¿me ayudan?",COVERAGE,propuesta_pendiente_revision,Consulta o modificación de cobertura de un seguro de vida. Traducción y etiqueta pendientes de revisión.
56,CONS_0057,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",COVERAGE,propuesta_pendiente_revision,Consulta o modificación de cobertura de un seguro de vida. Traducción y etiqueta pendientes de revisión.
57,CONS_0058,¿Puedes ayudarme a aumentar la cobertura de mi póliza de seguro de vida?,COVERAGE,propuesta_pendiente_revision,Consulta o modificación de cobertura de un seguro de vida. Traducción y etiqueta pendientes de revisión.
58,CONS_0059,Necesito ayuda para reducir la cobertura de mi póliza de seguro de vida.,COVERAGE,propuesta_pendiente_revision,Consulta o modificación de cobertura de un seguro de vida. Traducción y etiqueta pendientes de revisión.
59,CONS_0060,"Quisiera aumentar la cobertura de mi póliza de seguro de salud, ¿me ayudas?",COVERAGE,exclusion_propuesta,"Menciona explícitamente un seguro de salud. COVERAGE describe su motivo, pero la consulta está fuera del alcance del proyecto. Requiere reemplazo."


✅ Propuestas del bloque COVERAGE registradas.


### **🧠 4.8 Revisión complementaria del contexto asegurador**

Se buscarán referencias a otros productos aseguradores,
incluyendo variantes con espacios, guiones y algunas
terminaciones incompletas de la palabra insurance.

Las coincidencias se registrarán para revisión.
No se excluirán automáticamente, porque una referencia
a otro producto puede aparecer dentro de una comparación.

Esta búsqueda complementa la revisión manual y no garantiza
detectar todas las variantes o errores de escritura.

In [ ]:
# ============================================================
# 4.8 Revisión complementaria del contexto asegurador
# ============================================================

# Definimos patrones sin grupos de captura
# Incluimos insurance, insuranc e insurace como variantes explícitas
termino_seguro = r"(?:insurance|insuranc|insurace)\b"

patrones_otros_productos = {
    "Automóvil": (
        r"\b(?:auto|car|vehicle)[\s-]*" + termino_seguro
    ),
    "Salud": (
        r"\b(?:health|medical)[\s-]*" + termino_seguro
    ),
    "Hogar": (
        r"\b(?:home|homeowners|homeowner)[\s-]*" + termino_seguro
    ),
    "Mascotas": (
        r"\bpet[\s-]*" + termino_seguro
    ),
    "Viaje": (
        r"\btravel[\s-]*" + termino_seguro
    )
}

# Normalizamos las consultas originales para la búsqueda
textos_revision = (
    df_muestra_revision["consulta_original"]
    .map(normalizar_consulta)
)

# Identificamos las coincidencias por producto
detecciones = pd.DataFrame(
    {
        producto: textos_revision.str.contains(
            patron,
            regex=True,
            na=False
        )
        for producto, patron in patrones_otros_productos.items()
    },
    index=df_muestra_revision.index
)

# Añadimos campos de auditoría sin modificar etiquetas ni estados
df_muestra_revision["otros_productos_detectados"] = (
    detecciones.apply(
        lambda fila: ", ".join(fila.index[fila].tolist()),
        axis=1
    )
)

df_muestra_revision["requiere_revision_contexto"] = (
    detecciones.any(axis=1)
)

# Conservamos los casos detectados para su revisión
df_alertas_contexto = df_muestra_revision[
    df_muestra_revision["requiere_revision_contexto"]
].copy()

print("Consultas examinadas:", len(df_muestra_revision))
print("Consultas con coincidencias:", len(df_alertas_contexto))

# Mostramos las referencias detectadas en su contexto original
if not df_alertas_contexto.empty:
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            df_alertas_contexto[
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "categoria_original",
                    "otros_productos_detectados"
                ]
            ]
        )
else:
    print("No se encontraron coincidencias con estos patrones.")

print("=" * 60)
print("✅ Revisión complementaria del contexto completada.")
print("=" * 60)

Consultas examinadas: 300
Consultas con coincidencias: 5


,id_consulta,conjunto,consulta_original,categoria_original,otros_productos_detectados
59,CONS_0060,desarrollo,"I'd like to increase the fucking coverage of my health insurance policy, will you help me?",COVERAGE,Salud
79,CONS_0080,prueba,I would like to downgrade the coverage of my homeinsurance policy,COVERAGE,Hogar
99,CONS_0100,prueba,need to upgrade the fucking coverage of my petinsurance policy help me,COVERAGE,Mascotas
119,CONS_0120,prueba,i want the fucking auto insuranc rates can uhelp me to see them,QUOTE,Automóvil
133,CONS_0134,prueba,i have to calculate a health insuranc quote how to do it,QUOTE,Salud


✅ Revisión complementaria del contexto completada.


### **🧠 4.9 Selección de reemplazos candidatos**

Se seleccionarán cinco consultas candidatas para reemplazar
los registros que mencionan explícitamente otros productos
aseguradores.

Cada reemplazo conservará la categoría y el conjunto
del registro correspondiente.

Se excluirán de la selección los textos ya presentes en
la muestra y aquellos que coincidan con los patrones
de otros productos aseguradores.

Los reemplazos deberán revisarse antes de incorporarse.
La ausencia de coincidencias no garantiza su pertinencia.

**Estado:** Reemplazos candidatos pendientes de revisión.

In [ ]:
# ============================================================
# 4.9 Selección de reemplazos candidatos
# ============================================================

# Definimos los registros que requieren reemplazo
ids_reemplazar = [
    "CONS_0060",
    "CONS_0080",
    "CONS_0100",
    "CONS_0120",
    "CONS_0134"
]

# Verificamos que cada identificador exista una sola vez
objetivos = df_muestra_revision[
    df_muestra_revision["id_consulta"].isin(ids_reemplazar)
].copy()

if (
    len(objetivos) != len(ids_reemplazar)
    or not objetivos["id_consulta"].is_unique
):
    raise ValueError("No se encontraron los cinco registros esperados.")

# Identificamos otros productos en el conjunto de candidatos
textos_reserva = df_elegibles["instruction"].map(
    normalizar_consulta
)

mascara_otros_productos = pd.Series(
    False,
    index=df_elegibles.index
)

for patron in patrones_otros_productos.values():
    mascara_otros_productos |= textos_reserva.str.contains(
        patron,
        regex=True,
        na=False
    )

# Excluimos todos los textos ya utilizados en la muestra
textos_utilizados = set(
    df_muestra_revision["consulta_normalizada"]
)

df_reserva = df_elegibles[
    ~mascara_otros_productos
    & ~textos_reserva.isin(textos_utilizados)
].copy()

# Comprobamos que existan suficientes candidatos por categoría
necesarios = objetivos["categoria_original"].value_counts()

for categoria, cantidad in necesarios.items():
    disponibles = df_reserva["category"].eq(categoria).sum()

    if disponibles < cantidad:
        raise ValueError(
            f"No hay suficientes reemplazos para {categoria}."
        )

# Seleccionamos candidatos sin repetir textos
reemplazos_propuestos = []

for posicion, identificador in enumerate(ids_reemplazar):
    original = objetivos[
        objetivos["id_consulta"].eq(identificador)
    ].iloc[0]

    candidatos = df_reserva[
        df_reserva["category"].eq(original["categoria_original"])
        & ~df_reserva["consulta_normalizada"].isin(textos_utilizados)
    ].sort_values("consulta_normalizada")

    candidato = candidatos.sample(
        n=1,
        random_state=42 + posicion
    ).iloc[0]

    reemplazos_propuestos.append({
        "id_a_reemplazar": identificador,
        "conjunto": original["conjunto"],
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"]
    })

    textos_utilizados.add(candidato["consulta_normalizada"])

# Conservamos las propuestas separadas de la muestra actual
df_reemplazos_propuestos = pd.DataFrame(reemplazos_propuestos)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_reemplazos_propuestos[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Reemplazos candidatos seleccionados para revisión.")
print("=" * 60)

,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
0,CONS_0060,desarrollo,can ya help me upgrade the fucking coverage of my travel insuracne policy,COVERAGE,upgrade_coverage
1,CONS_0080,prueba,i have to upgrade my coverage i need help,COVERAGE,upgrade_coverage
2,CONS_0100,prueba,wanna know about the coverage of my insurance popicy where could i find more information,COVERAGE,check_coverage
3,CONS_0120,prueba,I'd like to check the fucking rates could i get some hel,QUOTE,check_rates
4,CONS_0134,prueba,"I'm looking for a home insurancxe quote, I need help finding it",QUOTE,calculate_insurance_quote


✅ Reemplazos candidatos seleccionados para revisión.


### **🧠 4.10 Corrección y verificación de reemplazos candidatos**

Se reconstruirán los patrones de otros productos aseguradores
y se comprobará su funcionamiento con ejemplos conocidos.

Se conservarán los tres reemplazos generales propuestos
y se seleccionarán nuevos candidatos para CONS_0060
y CONS_0134.

La muestra original permanecerá sin cambios hasta completar
la revisión de los reemplazos.

In [ ]:
# ============================================================
# 4.10 Corrección y verificación de reemplazos candidatos
# ============================================================

import re

# Reconstruimos los patrones para asegurar su configuración
termino_seguro = r"(?:insurance|insuranc|insurace)\b"

patrones_otros_productos = {
    "Automóvil": r"\b(?:auto|car|vehicle)[\s-]*" + termino_seguro,
    "Salud": r"\b(?:health|medical)[\s-]*" + termino_seguro,
    "Hogar": r"\b(?:home|homeowners|homeowner)[\s-]*" + termino_seguro,
    "Mascotas": r"\bpet[\s-]*" + termino_seguro,
    "Viaje": r"\btravel[\s-]*" + termino_seguro
}

# Centralizamos la detección para usarla en todas las comprobaciones
def detectar_otros_productos(texto):
    texto = normalizar_consulta(texto)

    return [
        producto
        for producto, patron in patrones_otros_productos.items()
        if re.search(patron, texto)
    ]

# Comprobamos expresiones que deben ser detectadas
casos_comprobacion = {
    "travel insurance policy": "Viaje",
    "home insurance quote": "Hogar",
    "homeinsurance policy": "Hogar",
    "petinsurance policy": "Mascotas",
    "health insuranc policy": "Salud",
    "auto insuranc rates": "Automóvil"
}

for texto, producto in casos_comprobacion.items():
    if producto not in detectar_otros_productos(texto):
        raise ValueError(f"No se detectó el caso esperado: {texto}")

# Revisamos nuevamente los reemplazos propuestos
df_reemplazos_propuestos["otros_productos_detectados"] = (
    df_reemplazos_propuestos["consulta_original"]
    .map(lambda texto: ", ".join(detectar_otros_productos(texto)))
)

# Conservamos los candidatos sin coincidencias
df_reemplazos_corregidos = df_reemplazos_propuestos[
    df_reemplazos_propuestos["otros_productos_detectados"].eq("")
].copy()

rechazados = df_reemplazos_propuestos[
    df_reemplazos_propuestos["otros_productos_detectados"].ne("")
].copy()

# Excluimos textos de la muestra y de todas las propuestas anteriores
textos_utilizados = set(
    df_muestra_revision["consulta_normalizada"]
) | set(
    df_reemplazos_propuestos["consulta_normalizada"]
)

# Construimos una reserva revisada con los patrones reconstruidos
reserva = df_elegibles.copy()

reserva["otros_productos_detectados"] = (
    reserva["instruction"]
    .map(lambda texto: ", ".join(detectar_otros_productos(texto)))
)

reserva = reserva[
    reserva["otros_productos_detectados"].eq("")
    & ~reserva["consulta_normalizada"].isin(textos_utilizados)
].copy()

# Seleccionamos nuevos candidatos para las propuestas rechazadas
nuevas_propuestas = []

for posicion, (_, anterior) in enumerate(rechazados.iterrows()):
    candidatos = reserva[
        reserva["category"].eq(anterior["categoria_original"])
        & ~reserva["consulta_normalizada"].isin(textos_utilizados)
    ].sort_values("consulta_normalizada")

    if candidatos.empty:
        raise ValueError(
            f"No hay candidatos para {anterior['id_a_reemplazar']}."
        )

    candidato = candidatos.sample(
        n=1,
        random_state=142 + posicion
    ).iloc[0]

    nuevas_propuestas.append({
        "id_a_reemplazar": anterior["id_a_reemplazar"],
        "conjunto": anterior["conjunto"],
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"],
        "otros_productos_detectados": ""
    })

    textos_utilizados.add(candidato["consulta_normalizada"])

# Reunimos las propuestas conservadas y las nuevas
df_reemplazos_corregidos = pd.concat(
    [
        df_reemplazos_corregidos,
        pd.DataFrame(nuevas_propuestas)
    ],
    ignore_index=True
).sort_values("id_a_reemplazar")

# Verificamos los cinco candidatos antes de mostrarlos
if (
    len(df_reemplazos_corregidos) != 5
    or not df_reemplazos_corregidos["id_a_reemplazar"].is_unique
    or df_reemplazos_corregidos["consulta_normalizada"].duplicated().any()
    or df_reemplazos_corregidos["consulta_original"]
        .map(detectar_otros_productos).map(bool).any()
):
    raise ValueError("La comprobación de reemplazos no fue satisfactoria.")

with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazos_corregidos[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Reemplazos corregidos presentados para revisión.")
print("=" * 60)

,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
0,CONS_0060,desarrollo,can ya help me upgrade the fucking coverage of my travel insuracne policy,COVERAGE,upgrade_coverage
1,CONS_0080,prueba,i have to upgrade my coverage i need help,COVERAGE,upgrade_coverage
2,CONS_0100,prueba,wanna know about the coverage of my insurance popicy where could i find more information,COVERAGE,check_coverage
3,CONS_0120,prueba,I'd like to check the fucking rates could i get some hel,QUOTE,check_rates
4,CONS_0134,prueba,"I'm looking for a home insurancxe quote, I need help finding it",QUOTE,calculate_insurance_quote


✅ Reemplazos corregidos presentados para revisión.


### **🧠 4.11 Diagnóstico de la detección de otros productos**

In [ ]:
# ============================================================
# 4.11 Diagnóstico de la detección de otros productos
# ============================================================

import re
import unicodedata
import pandas as pd

# Comprobamos directamente dos expresiones conocidas
print(
    "Detección directa de viaje:",
    bool(re.search(r"\btravel\s+insurance\b", "travel insurance"))
)
print(
    "Detección directa de hogar:",
    bool(re.search(r"\bhome\s+insurance\b", "home insurance"))
)

# Mostramos los patrones que realmente existen en la sesión
print("\nPatrones actuales:")

for producto, patron in patrones_otros_productos.items():
    print(producto, repr(patron))

# Revisamos los textos con una normalización independiente
diagnostico = []

for _, fila in df_reemplazos_corregidos.iterrows():
    texto_original = str(fila["consulta_original"])

    texto_normalizado = " ".join(
        unicodedata.normalize(
            "NFKC", texto_original
        ).casefold().split()
    )

    diagnostico.append({
        "ID": fila["id_a_reemplazar"],
        "Texto normalizado independiente": texto_normalizado,
        "Salida de normalizar_consulta": repr(
            normalizar_consulta(texto_original)
        ),
        "Viaje detectado directamente": bool(
            re.search(
                r"\btravel[\s-]*insuranc(?:e)?\b",
                texto_normalizado
            )
        ),
        "Hogar detectado directamente": bool(
            re.search(
                r"\bhome[\s-]*insuranc(?:e)?\b",
                texto_normalizado
            )
        ),
        "Salida del detector actual": str(
            detectar_otros_productos(texto_original)
        )
    })

with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(diagnostico))

print("=" * 60)
print("✅ Diagnóstico presentado; muestra sin cambios.")
print("=" * 60)

Detección directa de viaje: True
Detección directa de hogar: True

Patrones actuales:
Automóvil '\\b(?:auto|car|vehicle)[\\s-]*(?:insurance|insuranc|insurace)\\b'
Salud '\\b(?:health|medical)[\\s-]*(?:insurance|insuranc|insurace)\\b'
Hogar '\\b(?:home|homeowners|homeowner)[\\s-]*(?:insurance|insuranc|insurace)\\b'
Mascotas '\\bpet[\\s-]*(?:insurance|insuranc|insurace)\\b'
Viaje '\\btravel[\\s-]*(?:insurance|insuranc|insurace)\\b'


,ID,Texto normalizado independiente,Salida de normalizar_consulta,Viaje detectado directamente,Hogar detectado directamente,Salida del detector actual
0,CONS_0060,can ya help me upgrade the fucking coverage of my travel insuracne policy,'can ya help me upgrade the fucking coverage of my travel insuracne policy',False,False,[]
1,CONS_0080,i have to upgrade my coverage i need help,'i have to upgrade my coverage i need help',False,False,[]
2,CONS_0100,wanna know about the coverage of my insurance popicy where could i find more information,'wanna know about the coverage of my insurance popicy where could i find more information',False,False,[]
3,CONS_0120,i'd like to check the fucking rates could i get some hel,"""i'd like to check the fucking rates could i get some hel""",False,False,[]
4,CONS_0134,"i'm looking for a home insurancxe quote, i need help finding it","""i'm looking for a home insurancxe quote, i need help finding it""",False,False,[]


✅ Diagnóstico presentado; muestra sin cambios.


### **🧠 4.12 Reconstrucción del detector y selección de reemplazos**

Se reconstruirá el detector utilizando patrones sin barras
invertidas para evitar posibles alteraciones al copiar el código.

Se comprobará la detección de productos aseguradores conocidos
antes de seleccionar nuevos candidatos.

Los reemplazos permanecerán separados de la muestra principal
hasta completar su revisión.

In [ ]:
# ============================================================
# 4.12 Reconstrucción del detector y selección de reemplazos
# ============================================================

import re
import unicodedata
import pandas as pd

# Normalizamos con una función independiente
def normalizar_contexto(texto):
    if pd.isna(texto):
        return ""

    return " ".join(
        unicodedata.normalize("NFKC", str(texto)).casefold().split()
    )

# Construimos patrones sin barras invertidas
termino_seguro = "(?:insurance|insuranc|insurace)(?![a-z])"

patrones_otros_productos = {
    "Automóvil": "(?<![a-z])(?:auto|car|vehicle)[ -]*" + termino_seguro,
    "Salud": "(?<![a-z])(?:health|medical)[ -]*" + termino_seguro,
    "Hogar": "(?<![a-z])(?:home|homeowners|homeowner)[ -]*" + termino_seguro,
    "Mascotas": "(?<![a-z])pet[ -]*" + termino_seguro,
    "Viaje": "(?<![a-z])travel[ -]*" + termino_seguro
}

def detectar_otros_productos(texto):
    texto = normalizar_contexto(texto)

    return [
        producto
        for producto, patron in patrones_otros_productos.items()
        if re.search(patron, texto) is not None
    ]

# Detenemos la ejecución si falla algún caso conocido
casos_comprobacion = {
    "travel insurance policy": "Viaje",
    "home insurance quote": "Hogar",
    "homeinsurance policy": "Hogar",
    "petinsurance policy": "Mascotas",
    "health insuranc policy": "Salud",
    "auto insuranc rates": "Automóvil"
}

for texto, esperado in casos_comprobacion.items():
    if esperado not in detectar_otros_productos(texto):
        raise ValueError(f"Falló la detección de: {texto}")

if detectar_otros_productos("life insurance policy"):
    raise ValueError("El detector marcó incorrectamente seguros de vida.")

print("Comprobaciones del detector: satisfactorias.")

# Revisamos las cinco propuestas actuales
propuestas = df_reemplazos_corregidos.copy()

if (
    len(propuestas) != 5
    or not propuestas["id_a_reemplazar"].is_unique
):
    raise ValueError("Se esperaban cinco propuestas con IDs únicos.")

propuestas["otros_productos_detectados"] = (
    propuestas["consulta_original"]
    .map(lambda texto: ", ".join(detectar_otros_productos(texto)))
)

rechazadas = propuestas[
    propuestas["otros_productos_detectados"].ne("")
].copy()

conservadas = propuestas[
    propuestas["otros_productos_detectados"].eq("")
].copy()

print("Propuestas fuera del alcance:", len(rechazadas))

# Excluimos textos de la muestra y de las propuestas actuales
textos_utilizados = set(
    df_muestra_revision["consulta_original"].map(normalizar_contexto)
) | set(
    propuestas["consulta_original"].map(normalizar_contexto)
)

# Construimos la reserva con el detector comprobado
reserva = df_elegibles.copy()
reserva["texto_revision"] = (
    reserva["instruction"].map(normalizar_contexto)
)

reserva = reserva[
    ~reserva["instruction"].map(detectar_otros_productos).map(bool)
    & ~reserva["texto_revision"].isin(textos_utilizados)
].copy()

nuevas_propuestas = []

# Seleccionamos reemplazos conservando categoría y conjunto
for posicion, (_, anterior) in enumerate(
    rechazadas.sort_values("id_a_reemplazar").iterrows()
):
    candidatos = reserva[
        reserva["category"].eq(anterior["categoria_original"])
        & ~reserva["texto_revision"].isin(textos_utilizados)
    ].sort_values("texto_revision")

    if candidatos.empty:
        raise ValueError(
            f"No hay candidatos para {anterior['id_a_reemplazar']}."
        )

    candidato = candidatos.sample(
        n=1,
        random_state=242 + posicion
    ).iloc[0]

    nuevas_propuestas.append({
        "id_a_reemplazar": anterior["id_a_reemplazar"],
        "conjunto": anterior["conjunto"],
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"],
        "otros_productos_detectados": ""
    })

    textos_utilizados.add(candidato["texto_revision"])

# Construimos el resultado antes de actualizar la tabla de propuestas
resultado = pd.concat(
    [conservadas, pd.DataFrame(nuevas_propuestas)],
    ignore_index=True
).sort_values("id_a_reemplazar")

if (
    len(resultado) != 5
    or not resultado["id_a_reemplazar"].is_unique
    or resultado["consulta_original"]
        .map(normalizar_contexto).duplicated().any()
    or resultado["consulta_original"]
        .map(detectar_otros_productos).map(bool).any()
):
    raise ValueError("La verificación final de propuestas falló.")

df_reemplazos_corregidos = resultado.copy()

# Mostramos los candidatos sin modificar la muestra principal
with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazos_corregidos[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Detector comprobado y candidatos presentados para revisión.")
print("=" * 60)

Comprobaciones del detector: satisfactorias.
Propuestas fuera del alcance: 0


,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
0,CONS_0060,desarrollo,can ya help me upgrade the fucking coverage of my travel insuracne policy,COVERAGE,upgrade_coverage
1,CONS_0080,prueba,i have to upgrade my coverage i need help,COVERAGE,upgrade_coverage
2,CONS_0100,prueba,wanna know about the coverage of my insurance popicy where could i find more information,COVERAGE,check_coverage
3,CONS_0120,prueba,I'd like to check the fucking rates could i get some hel,QUOTE,check_rates
4,CONS_0134,prueba,"I'm looking for a home insurancxe quote, I need help finding it",QUOTE,calculate_insurance_quote


✅ Detector comprobado y candidatos presentados para revisión.


### **🧠 4.13 Inspección de caracteres de los casos pendientes**

Se examinarán los caracteres exactos de CONS_0060 y CONS_0134
para identificar diferencias de escritura o caracteres Unicode
que puedan explicar las coincidencias no detectadas.

Se mostrarán los textos originales, su versión normalizada
y los códigos de sus caracteres.

**Estado:** Diagnóstico pendiente. La muestra permanece sin cambios.

In [ ]:
# ============================================================
# 4.13 Inspección de caracteres de los casos pendientes
# ============================================================

# Mostramos los caracteres exactos sin modificar ningún registro
for identificador in ["CONS_0060", "CONS_0134"]:
    texto = df_reemplazos_corregidos.loc[
        df_reemplazos_corregidos["id_a_reemplazar"].eq(identificador),
        "consulta_original"
    ].iloc[0]

    print("ID:", identificador)
    print("Texto original:", ascii(str(texto)))
    print("Texto normalizado:", ascii(normalizar_contexto(texto)))
    print("Productos detectados:", detectar_otros_productos(texto))

    # Mostramos cada palabra con sus códigos de caracteres
    for palabra in str(texto).split():
        print(
            ascii(palabra),
            " ".join(f"U+{ord(caracter):04X}" for caracter in palabra)
        )

    print("=" * 60)

print("✅ Caracteres presentados para diagnóstico.")

ID: CONS_0060
Texto original: 'can ya help me upgrade the fucking coverage of my travel insuracne policy'
Texto normalizado: 'can ya help me upgrade the fucking coverage of my travel insuracne policy'
Productos detectados: []
'can' U+0063 U+0061 U+006E
'ya' U+0079 U+0061
'help' U+0068 U+0065 U+006C U+0070
'me' U+006D U+0065
'upgrade' U+0075 U+0070 U+0067 U+0072 U+0061 U+0064 U+0065
'the' U+0074 U+0068 U+0065
'fucking' U+0066 U+0075 U+0063 U+006B U+0069 U+006E U+0067
'coverage' U+0063 U+006F U+0076 U+0065 U+0072 U+0061 U+0067 U+0065
'of' U+006F U+0066
'my' U+006D U+0079
'travel' U+0074 U+0072 U+0061 U+0076 U+0065 U+006C
'insuracne' U+0069 U+006E U+0073 U+0075 U+0072 U+0061 U+0063 U+006E U+0065
'policy' U+0070 U+006F U+006C U+0069 U+0063 U+0079
ID: CONS_0134
Texto original: "I'm looking for a home insurancxe quote, I need help finding it"
Texto normalizado: "i'm looking for a home insurancxe quote, i need help finding it"
Productos detectados: []
"I'm" U+0049 U+0027 U+006D
'looking' U+00

### **🧠 4.14 Ajuste del detector y renovación de candidatos**

La inspección identificó las variantes insuracne e insurancxe
en los dos reemplazos fuera del alcance.

Se ampliará la detección para reconocer referencias a otros
productos seguidas de palabras que comiencen con insur.

Se comprobarán los casos encontrados y se seleccionarán
nuevos candidatos, conservando categoría y conjunto.

**Estado:** Reemplazos pendientes de revisión.
La muestra principal permanece sin cambios.

In [ ]:
# ============================================================
# 4.14 Ajuste del detector y renovación de candidatos
# ============================================================

import re
import pandas as pd

# Ampliamos la búsqueda para detectar variantes de insurance
termino_seguro = "insur[a-z]*(?![a-z])"

patrones_otros_productos = {
    "Automóvil": "(?<![a-z])(?:auto|car|vehicle)[ -]*" + termino_seguro,
    "Salud": "(?<![a-z])(?:health|medical)[ -]*" + termino_seguro,
    "Hogar": "(?<![a-z])(?:home|homeowners|homeowner)[ -]*" + termino_seguro,
    "Mascotas": "(?<![a-z])pet[ -]*" + termino_seguro,
    "Viaje": "(?<![a-z])travel[ -]*" + termino_seguro
}

# Reconstruimos el detector con los patrones actualizados
def detectar_otros_productos(texto):
    texto = normalizar_contexto(texto)

    return [
        producto
        for producto, patron in patrones_otros_productos.items()
        if re.search(patron, texto) is not None
    ]

# Comprobamos las variantes observadas y las formas habituales
casos_comprobacion = {
    "travel insuracne policy": "Viaje",
    "home insurancxe quote": "Hogar",
    "travel insurance policy": "Viaje",
    "homeinsurance policy": "Hogar",
    "petinsurance policy": "Mascotas",
    "health insuranc policy": "Salud",
    "auto insurance rates": "Automóvil"
}

for texto, esperado in casos_comprobacion.items():
    if esperado not in detectar_otros_productos(texto):
        raise ValueError(f"Falló la detección de: {texto}")

if detectar_otros_productos("life insurance policy"):
    raise ValueError("Se marcó incorrectamente un seguro de vida.")

# Revisamos las propuestas actuales
propuestas = df_reemplazos_corregidos.copy()

if len(propuestas) != 5 or not propuestas["id_a_reemplazar"].is_unique:
    raise ValueError("Se esperaban cinco propuestas con IDs únicos.")

mascara_rechazo = (
    propuestas["consulta_original"]
    .map(detectar_otros_productos)
    .map(bool)
)

rechazadas = propuestas[mascara_rechazo].copy()
conservadas = propuestas[~mascara_rechazo].copy()

print("Comprobaciones del detector: satisfactorias.")
print("Propuestas separadas por otros productos:", len(rechazadas))

# Excluimos los textos ya utilizados o propuestos
textos_utilizados = set(
    df_muestra_revision["consulta_original"].map(normalizar_contexto)
) | set(
    propuestas["consulta_original"].map(normalizar_contexto)
)

# Construimos una reserva con el detector actualizado
reserva = df_elegibles.copy()
reserva["texto_revision"] = (
    reserva["instruction"].map(normalizar_contexto)
)

reserva = reserva[
    ~reserva["instruction"].map(detectar_otros_productos).map(bool)
    & ~reserva["texto_revision"].isin(textos_utilizados)
].copy()

# Seleccionamos nuevos candidatos sin repetir consultas
nuevas_propuestas = []

for posicion, (_, anterior) in enumerate(
    rechazadas.sort_values("id_a_reemplazar").iterrows()
):
    candidatos = reserva[
        reserva["category"].eq(anterior["categoria_original"])
        & ~reserva["texto_revision"].isin(textos_utilizados)
    ].sort_values("texto_revision")

    if candidatos.empty:
        raise ValueError(
            f"No hay candidatos para {anterior['id_a_reemplazar']}."
        )

    candidato = candidatos.sample(
        n=1,
        random_state=342 + posicion
    ).iloc[0]

    nuevas_propuestas.append({
        "id_a_reemplazar": anterior["id_a_reemplazar"],
        "conjunto": anterior["conjunto"],
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"]
    })

    textos_utilizados.add(candidato["texto_revision"])

# Reunimos y verificamos las cinco propuestas
resultado = pd.concat(
    [conservadas, pd.DataFrame(nuevas_propuestas)],
    ignore_index=True
).sort_values("id_a_reemplazar")

resultado["otros_productos_detectados"] = (
    resultado["consulta_original"]
    .map(lambda texto: ", ".join(detectar_otros_productos(texto)))
)

if (
    len(resultado) != 5
    or not resultado["id_a_reemplazar"].is_unique
    or resultado["consulta_original"]
        .map(normalizar_contexto).duplicated().any()
    or resultado["otros_productos_detectados"].ne("").any()
):
    raise ValueError("La verificación final de propuestas falló.")

df_reemplazos_corregidos = resultado.copy()

# Mostramos los candidatos para revisar su pertinencia
with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazos_corregidos[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Detector ajustado y nuevos candidatos presentados.")
print("=" * 60)

Comprobaciones del detector: satisfactorias.
Propuestas separadas por otros productos: 2


,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
3,CONS_0060,desarrollo,can I modify the coverage of my insurance policy ?,COVERAGE,change_coverage
0,CONS_0080,prueba,i have to upgrade my coverage i need help,COVERAGE,upgrade_coverage
1,CONS_0100,prueba,wanna know about the coverage of my insurance popicy where could i find more information,COVERAGE,check_coverage
2,CONS_0120,prueba,I'd like to check the fucking rates could i get some hel,QUOTE,check_rates
4,CONS_0134,prueba,"I'd like to estimate a home inaurance quote, could you help me?",QUOTE,calculate_insurance_quote


✅ Detector ajustado y nuevos candidatos presentados.


### **🧠 4.15 Revisión por referencias al producto asegurador**

Se buscarán menciones a otros productos sin depender
de la escritura correcta de la palabra insurance.

Las coincidencias serán alertas de revisión y no exclusiones
automáticas, porque algunas palabras pueden tener otros usos.

Se seleccionará una nueva propuesta para CONS_0134.
La muestra principal permanecerá sin cambios.

In [ ]:
# ============================================================
# 4.15 Revisión por referencias al producto asegurador
# ============================================================

# Definimos referencias conservadoras a otros productos
patrones_referencia_producto = {
    "Automóvil": "(?<![a-z])(?:auto|car|vehicle|automobile)(?![a-z])",
    "Salud": "(?<![a-z])(?:health|medical)(?![a-z])",
    "Hogar": "(?<![a-z])(?:home|homeowner|homeowners)(?![a-z])",
    "Mascotas": "(?<![a-z])(?:pet|pets)(?![a-z])",
    "Viaje": "(?<![a-z])travel(?![a-z])"
}

# Combinamos referencias al producto con el detector anterior
def detectar_referencias_producto(texto):
    normalizado = normalizar_contexto(texto)
    productos = set(detectar_otros_productos(texto))

    for producto, patron in patrones_referencia_producto.items():
        if re.search(patron, normalizado):
            productos.add(producto)

    return sorted(productos)

# Comprobamos directamente el candidato rechazado
anterior = df_reemplazos_corregidos.loc[
    df_reemplazos_corregidos["id_a_reemplazar"].eq("CONS_0134")
].iloc[0]

if "Hogar" not in detectar_referencias_producto(
    anterior["consulta_original"]
):
    raise ValueError("No se detectó la referencia a hogar.")

# Excluimos textos ya presentes en la muestra o en las propuestas
textos_utilizados = set(
    df_muestra_revision["consulta_original"].map(normalizar_contexto)
) | set(
    df_reemplazos_corregidos["consulta_original"].map(normalizar_contexto)
)

# Buscamos una nueva consulta de la misma categoría
reserva = df_elegibles.copy()
reserva["texto_revision"] = (
    reserva["instruction"].map(normalizar_contexto)
)

candidatos = reserva[
    reserva["category"].eq(anterior["categoria_original"])
    & ~reserva["texto_revision"].isin(textos_utilizados)
    & ~reserva["instruction"].map(
        detectar_referencias_producto
    ).map(bool)
].sort_values("texto_revision")

if candidatos.empty:
    raise ValueError("No hay candidatos disponibles para CONS_0134.")

candidato = candidatos.sample(
    n=1,
    random_state=442
).iloc[0]

# Actualizamos únicamente la tabla de propuestas
resultado = df_reemplazos_corregidos.copy()
mascara = resultado["id_a_reemplazar"].eq("CONS_0134")

campos_nuevos = {
    "indice_origen": candidato["indice_origen"],
    "consulta_original": candidato["instruction"],
    "categoria_original": candidato["category"],
    "intencion_original": candidato["intent"],
    "consulta_normalizada": candidato["consulta_normalizada"],
    "otros_productos_detectados": ""
}

for campo, valor in campos_nuevos.items():
    resultado.loc[mascara, campo] = valor

# Mostramos alertas de todos los reemplazos antes de incorporarlos
resultado["alertas_producto"] = (
    resultado["consulta_original"]
    .map(lambda texto: ", ".join(detectar_referencias_producto(texto)))
)

df_reemplazos_corregidos = resultado.copy()

with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazos_corregidos[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original",
                "alertas_producto"
            ]
        ]
    )

print("=" * 60)
print("✅ Propuestas revisadas por referencias al producto.")
print("=" * 60)

,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original,alertas_producto
3,CONS_0060,desarrollo,can I modify the coverage of my insurance policy ?,COVERAGE,change_coverage,
0,CONS_0080,prueba,i have to upgrade my coverage i need help,COVERAGE,upgrade_coverage,
1,CONS_0100,prueba,wanna know about the coverage of my insurance popicy where could i find more information,COVERAGE,check_coverage,
2,CONS_0120,prueba,I'd like to check the fucking rates could i get some hel,QUOTE,check_rates,
4,CONS_0134,prueba,im trying to find the fucking insurance rates could i get some help,QUOTE,check_rates,


✅ Propuestas revisadas por referencias al producto.


### **🧠 4.16 Incorporación de reemplazos y auditoría de la muestra**

Se incorporarán los cinco reemplazos revisados, conservando
la categoría, el conjunto y el identificador de posición
de cada consulta sustituida.

Los registros retirados se conservarán en una tabla de auditoría.
El índice de origen permitirá distinguir el registro anterior
del nuevo.

Las traducciones y etiquetas anteriores de las posiciones
reemplazadas se borrarán para evitar asociarlas con textos nuevos.

Se verificará la distribución de la muestra y se buscarán
referencias a otros productos con el detector actualizado.

**Estado:** Reemplazos incorporados.
Traducción y validación pendientes.

In [ ]:
# ============================================================
# 4.16 Incorporación de reemplazos y auditoría de la muestra
# ============================================================

# Evitamos repetir la sustitución y perder la auditoría original
if "df_registros_retirados" in globals():
    raise ValueError(
        "La auditoría ya existe. Revisa si esta celda fue ejecutada."
    )

ids_reemplazo = df_reemplazos_corregidos["id_a_reemplazar"].tolist()

if len(ids_reemplazo) != 5 or len(set(ids_reemplazo)) != 5:
    raise ValueError("Se esperaban cinco reemplazos distintos.")

# Trabajamos sobre una copia antes de actualizar la muestra
muestra_actualizada = df_muestra_revision.copy()

retirados = muestra_actualizada[
    muestra_actualizada["id_consulta"].isin(ids_reemplazo)
].copy()

if len(retirados) != 5 or not retirados["id_consulta"].is_unique:
    raise ValueError("No se encontraron cinco posiciones únicas.")

# Verificamos categoría, conjunto y ausencia de alertas
for _, reemplazo in df_reemplazos_corregidos.iterrows():
    original = retirados[
        retirados["id_consulta"].eq(reemplazo["id_a_reemplazar"])
    ].iloc[0]

    if (
        original["conjunto"] != reemplazo["conjunto"]
        or original["categoria_original"]
        != reemplazo["categoria_original"]
        or detectar_referencias_producto(
            reemplazo["consulta_original"]
        )
    ):
        raise ValueError("Un reemplazo no cumple las condiciones.")

    mascara = muestra_actualizada["id_consulta"].eq(
        reemplazo["id_a_reemplazar"]
    )

    # Actualizamos los datos de origen de la consulta
    for campo in [
        "indice_origen",
        "consulta_original",
        "categoria_original",
        "intencion_original",
        "consulta_normalizada"
    ]:
        muestra_actualizada.loc[mascara, campo] = reemplazo[campo]

    # Reiniciamos la revisión para el nuevo texto
    muestra_actualizada.loc[mascara, "consulta_es"] = ""
    muestra_actualizada.loc[mascara, "categoria_referencia"] = ""
    muestra_actualizada.loc[mascara, "estado_revision"] = "pendiente"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Reemplazo por consulta fuera del alcance. "
        "Pendiente de traducción y validación."
    )

# Verificamos que no existan textos repetidos
if (
    muestra_actualizada["consulta_original"]
    .map(normalizar_contexto).duplicated().any()
):
    raise ValueError("Se detectaron textos repetidos tras el reemplazo.")

# Comprobamos la distribución por categoría y conjunto
resumen = pd.crosstab(
    muestra_actualizada["categoria_original"],
    muestra_actualizada["conjunto"]
).reindex(
    index=categorias_candidatas,
    columns=["desarrollo", "prueba"],
    fill_value=0
)

if (
    len(muestra_actualizada) != 300
    or not resumen["desarrollo"].eq(10).all()
    or not resumen["prueba"].eq(40).all()
):
    raise ValueError("La distribución de la muestra cambió.")

# Actualizamos las alertas con el detector más reciente
muestra_actualizada["otros_productos_detectados"] = (
    muestra_actualizada["consulta_original"]
    .map(lambda texto: ", ".join(detectar_otros_productos(texto)))
)

muestra_actualizada["alertas_producto"] = (
    muestra_actualizada["consulta_original"]
    .map(lambda texto: ", ".join(detectar_referencias_producto(texto)))
)

muestra_actualizada["requiere_revision_contexto"] = (
    muestra_actualizada["alertas_producto"].ne("")
)

# Conservamos la auditoría y confirmamos la actualización
retirados["motivo_retiro"] = "Referencia explícita a otro producto"
df_registros_retirados = retirados.copy()
df_muestra_revision = muestra_actualizada.copy()

# Presentamos la distribución y las alertas pendientes
resumen["Total"] = resumen.sum(axis=1)
resumen.index.name = "Categoría"
display(resumen.reset_index())

df_alertas_contexto = df_muestra_revision[
    df_muestra_revision["requiere_revision_contexto"]
].copy()

print("Registros retirados conservados:", len(df_registros_retirados))
print("Consultas con alertas de producto:", len(df_alertas_contexto))

if not df_alertas_contexto.empty:
    with pd.option_context("display.max_colwidth", None):
        display(
            df_alertas_contexto[
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "alertas_producto"
                ]
            ]
        )

print("=" * 60)
print("✅ Reemplazos incorporados y muestra auditada.")
print("=" * 60)

conjunto,Categoría,desarrollo,prueba,Total
0,LIFE_INSURANCE,10,40,50
1,COVERAGE,10,40,50
2,QUOTE,10,40,50
3,PAYMENT,10,40,50
4,ENROLLMENT,10,40,50
5,CONTACT,10,40,50


Registros retirados conservados: 5
Consultas con alertas de producto: 13


,id_consulta,conjunto,consulta_original,alertas_producto
61,CONS_0062,prueba,id like to change the coverage of my travel insuance policy where can i do it,Viaje
120,CONS_0121,prueba,wanna estimate a pet insuarnce quote how can i do it,Mascotas
122,CONS_0123,prueba,"I need a pet insursnce quote, will you help me calculate it?",Mascotas
129,CONS_0130,prueba,is it possible to estimate an auto imsurance quote,Automóvil
137,CONS_0138,prueba,"I am looking for a pet inusrance quote, can you help me finding it?",Mascotas
140,CONS_0141,prueba,could i calculate a travel insurande quote,Viaje
144,CONS_0145,prueba,i need the fucking auto isurance rates help me see them,Automóvil
201,CONS_0202,desarrollo,I want help checking the different pet insuraznce policies,Mascotas
202,CONS_0203,desarrollo,i got to purchase a fucking pet insuracne policy could uhelp me,Mascotas
203,CONS_0204,desarrollo,i got to see the fucking different auto iinsurance policies how could i do it,Automóvil


✅ Reemplazos incorporados y muestra auditada.


### **🧠 4.17 Selección de reemplazos para las alertas adicionales**

La auditoría identificó 13 consultas adicionales que mencionan
seguros de automóvil, mascotas o viaje.

Se seleccionarán reemplazos de la misma categoría y conjunto,
excluyendo textos ya utilizados, registros retirados y consultas
con referencias detectadas a otros productos.

Los candidatos se revisarán antes de incorporarse.
La muestra principal permanecerá sin cambios en este paso.

**Estado:** Trece reemplazos pendientes de revisión.

In [ ]:
# ============================================================
# 4.17 Selección de reemplazos para las alertas adicionales
# ============================================================

# Identificamos los casos revisados como fuera del alcance
ids_alertas = [
    "CONS_0062",
    "CONS_0121",
    "CONS_0123",
    "CONS_0130",
    "CONS_0138",
    "CONS_0141",
    "CONS_0145",
    "CONS_0202",
    "CONS_0203",
    "CONS_0204",
    "CONS_0219",
    "CONS_0235",
    "CONS_0249"
]

objetivos = df_muestra_revision[
    df_muestra_revision["id_consulta"].isin(ids_alertas)
].copy()

if (
    len(objetivos) != 13
    or not objetivos["id_consulta"].is_unique
    or not objetivos["consulta_original"]
        .map(detectar_referencias_producto).map(bool).all()
):
    raise ValueError("Los registros no coinciden con las alertas esperadas.")

# Excluimos consultas actuales, retiradas y propuestas anteriores
textos_utilizados = set(
    df_muestra_revision["consulta_original"].map(normalizar_contexto)
) | set(
    df_registros_retirados["consulta_original"].map(normalizar_contexto)
) | set(
    df_reemplazos_corregidos["consulta_original"].map(normalizar_contexto)
)

# Construimos la reserva con el detector actualizado
reserva = df_elegibles.copy()

reserva["texto_revision"] = (
    reserva["instruction"].map(normalizar_contexto)
)

reserva = reserva[
    reserva["texto_revision"].ne("")
    & ~reserva["texto_revision"].isin(textos_utilizados)
    & ~reserva["instruction"]
        .map(detectar_referencias_producto).map(bool)
].copy()

# Conservamos un candidato por texto normalizado
reserva = (
    reserva.sort_values("texto_revision")
    .drop_duplicates("texto_revision")
)

# Comprobamos disponibilidad por categoría
necesarios = objetivos["categoria_original"].value_counts()

for categoria, cantidad in necesarios.items():
    if reserva["category"].eq(categoria).sum() < cantidad:
        raise ValueError(
            f"No hay suficientes reemplazos para {categoria}."
        )

# Seleccionamos candidatos conservando categoría y conjunto
propuestas_adicionales = []

for posicion, (_, original) in enumerate(
    objetivos.sort_values("id_consulta").iterrows()
):
    candidatos = reserva[
        reserva["category"].eq(original["categoria_original"])
        & ~reserva["texto_revision"].isin(textos_utilizados)
    ]

    candidato = candidatos.sample(
        n=1,
        random_state=542 + posicion
    ).iloc[0]

    propuestas_adicionales.append({
        "id_a_reemplazar": original["id_consulta"],
        "conjunto": original["conjunto"],
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"]
    })

    textos_utilizados.add(candidato["texto_revision"])

# Conservamos las propuestas separadas de la muestra principal
df_reemplazos_adicionales = pd.DataFrame(propuestas_adicionales)

# Verificamos que no existan repeticiones ni alertas detectadas
if (
    len(df_reemplazos_adicionales) != 13
    or df_reemplazos_adicionales["consulta_original"]
        .map(normalizar_contexto).duplicated().any()
    or df_reemplazos_adicionales["consulta_original"]
        .map(detectar_referencias_producto).map(bool).any()
):
    raise ValueError("La comprobación de candidatos falló.")

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_reemplazos_adicionales[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Trece reemplazos candidatos presentados para revisión.")
print("=" * 60)

,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
0,CONS_0062,prueba,could you help me to upgrade my fucking coverage?,COVERAGE,upgrade_coverage
1,CONS_0121,prueba,"I need the rates, how can I see them?",QUOTE,check_rates
2,CONS_0123,prueba,"I want to know more about the fucking life insurance rates, give m some information",QUOTE,check_rates
3,CONS_0130,prueba,im lioking for the rates could ya help me to find them,QUOTE,check_rates
4,CONS_0138,prueba,i need a fucking insurance quote how can i estimate it,QUOTE,calculate_insurance_quote
5,CONS_0141,prueba,"I want to see the fucking insurance rates, how do I do it ?",QUOTE,check_rates
6,CONS_0145,prueba,"I want to estimate an insurance quote, help me",QUOTE,calculate_insurance_quote
7,CONS_0202,desarrollo,i do not want my fucking life insurance policy to be renewed I'd like to cancel it,ENROLLMENT,cancel_insurance_policy
8,CONS_0203,desarrollo,"I have to cancel my fucking life insurance policy , where could I do it?",ENROLLMENT,cancel_insurance_policy
9,CONS_0204,desarrollo,"I don't want my fucking insurance policy to be renewed, I want to cancel it",ENROLLMENT,cancel_insurance_policy


✅ Trece reemplazos candidatos presentados para revisión.


### **🧠 4.18 Revisión de candidatos y ajuste de ENROLLMENT**

Los trece reemplazos propuestos no presentan referencias
explícitas a otros productos aseguradores.

Las consultas de COVERAGE tratan sobre modificación de cobertura.
Las consultas de QUOTE solicitan tarifas o cotizaciones.

En ENROLLMENT se identificaron consultas sobre cancelación,
no renovación, cargos de terminación y comparación de pólizas.

Se ampliará el criterio de ENROLLMENT para incluir la selección
y comparación de pólizas, contratación, renovación y cancelación,
incluidos cargos asociados a la terminación.

Las etiquetas propuestas deberán revisarse antes de considerarse
validadas. Las descripciones de clasificación se actualizarán
de manera uniforme para los tres modelos.

**Estado:** Reemplazos revisados como candidatos.
Incorporación y validación pendientes.

In [ ]:
# ============================================================
# 4.18 Revisión de candidatos y ajuste de ENROLLMENT
# ============================================================

# Registramos el resultado de la revisión de los candidatos
df_reemplazos_adicionales["estado_revision"] = (
    "candidato_revisado_pendiente_validacion"
)

df_reemplazos_adicionales["observaciones"] = (
    "Sin referencia explícita a otros productos. "
    "Pendiente de traducción y validación de etiqueta."
)

mascara_enrollment = (
    df_reemplazos_adicionales["categoria_original"].eq("ENROLLMENT")
)

df_reemplazos_adicionales.loc[
    mascara_enrollment, "observaciones"
] = (
    "Aplicar el criterio ampliado de ENROLLMENT: comparación, "
    "contratación, renovación y cancelación de pólizas, "
    "incluidos cargos de terminación. "
    "Pendiente de traducción y validación."
)

# Conservamos una configuración nueva para la evaluación posterior
descripciones_categorias_v2 = {
    "LIFE_INSURANCE": (
        "información general o consulta de información "
        "sobre un seguro de vida"
    ),
    "COVERAGE": (
        "consulta de coberturas, exclusiones o modificación "
        "de la cobertura del seguro"
    ),
    "QUOTE": (
        "consulta de tarifas, estimación de costo "
        "o solicitud de cotización del seguro"
    ),
    "PAYMENT": (
        "métodos, fechas o problemas de pago del seguro"
    ),
    "ENROLLMENT": (
        "comparación, contratación, renovación o cancelación "
        "de pólizas, incluidos requisitos y cargos de terminación"
    ),
    "CONTACT": "contacto con un asesor o canal de atención"
}

# Mostramos los criterios que se utilizarán en pruebas posteriores
display(
    pd.DataFrame(
        descripciones_categorias_v2.items(),
        columns=["Categoría", "Descripción propuesta"]
    )
)

print("=" * 60)
print("✅ Revisión y descripciones actualizadas registradas.")
print("=" * 60)

,Categoría,Descripción propuesta
0,LIFE_INSURANCE,información general o consulta de información ...
1,COVERAGE,"consulta de coberturas, exclusiones o modifica..."
2,QUOTE,"consulta de tarifas, estimación de costo o sol..."
3,PAYMENT,"métodos, fechas o problemas de pago del seguro"
4,ENROLLMENT,"comparación, contratación, renovación o cancel..."
5,CONTACT,contacto con un asesor o canal de atención


✅ Revisión y descripciones actualizadas registradas.


### **🧠 4.19 Incorporación de reemplazos adicionales y guardado**

Se incorporarán los trece candidatos revisados, conservando
la categoría, el conjunto y el identificador de posición.

Los registros sustituidos se añadirán a la auditoría existente.
Los nuevos textos quedarán pendientes de traducción y validación.

Se comprobarán la distribución, los duplicados textuales
y las alertas de otros productos.

Se guardarán la muestra actualizada, la auditoría y las
descripciones propuestas en archivos nuevos.

**Estado:** Depuración por referencias detectadas completada.
Revisión semántica, traducción y validación pendientes.

In [ ]:
# ============================================================
# 4.19 Incorporación de reemplazos adicionales y guardado
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

# Evitamos incorporar dos veces los mismos reemplazos
ids_nuevos = df_reemplazos_adicionales["id_a_reemplazar"].tolist()

if len(ids_nuevos) != 13 or len(set(ids_nuevos)) != 13:
    raise ValueError("Se esperaban trece reemplazos distintos.")

if df_registros_retirados["id_consulta"].isin(ids_nuevos).any():
    raise ValueError("Estos reemplazos ya aparecen en la auditoría.")

# Preparamos copias antes de actualizar las tablas principales
muestra_actualizada = df_muestra_revision.copy()

retirados_adicionales = muestra_actualizada[
    muestra_actualizada["id_consulta"].isin(ids_nuevos)
].copy()

if (
    len(retirados_adicionales) != 13
    or not retirados_adicionales["id_consulta"].is_unique
):
    raise ValueError("No se encontraron trece posiciones únicas.")

# Verificamos e incorporamos cada reemplazo
for _, reemplazo in df_reemplazos_adicionales.iterrows():
    original = retirados_adicionales[
        retirados_adicionales["id_consulta"].eq(
            reemplazo["id_a_reemplazar"]
        )
    ].iloc[0]

    if (
        original["conjunto"] != reemplazo["conjunto"]
        or original["categoria_original"]
        != reemplazo["categoria_original"]
        or detectar_referencias_producto(
            reemplazo["consulta_original"]
        )
    ):
        raise ValueError("Un reemplazo no cumple las condiciones.")

    mascara = muestra_actualizada["id_consulta"].eq(
        reemplazo["id_a_reemplazar"]
    )

    # Conservamos el origen del nuevo registro
    for campo in [
        "indice_origen",
        "consulta_original",
        "categoria_original",
        "intencion_original",
        "consulta_normalizada"
    ]:
        muestra_actualizada.loc[mascara, campo] = reemplazo[campo]

    # Reiniciamos los campos de revisión del texto sustituido
    muestra_actualizada.loc[mascara, "consulta_es"] = ""
    muestra_actualizada.loc[mascara, "categoria_referencia"] = ""
    muestra_actualizada.loc[mascara, "estado_revision"] = "pendiente"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        reemplazo["observaciones"]
    )

# Verificamos textos únicos y distribución
if (
    muestra_actualizada["consulta_original"]
    .map(normalizar_contexto).duplicated().any()
):
    raise ValueError("Se detectaron consultas repetidas.")

resumen = pd.crosstab(
    muestra_actualizada["categoria_original"],
    muestra_actualizada["conjunto"]
).reindex(
    index=categorias_candidatas,
    columns=["desarrollo", "prueba"],
    fill_value=0
)

if (
    len(muestra_actualizada) != 300
    or not resumen["desarrollo"].eq(10).all()
    or not resumen["prueba"].eq(40).all()
):
    raise ValueError("La distribución no coincide con lo previsto.")

# Recalculamos las alertas con los detectores actuales
muestra_actualizada["otros_productos_detectados"] = (
    muestra_actualizada["consulta_original"]
    .map(lambda texto: ", ".join(detectar_otros_productos(texto)))
)

muestra_actualizada["alertas_producto"] = (
    muestra_actualizada["consulta_original"]
    .map(lambda texto: ", ".join(detectar_referencias_producto(texto)))
)

muestra_actualizada["requiere_revision_contexto"] = (
    muestra_actualizada["alertas_producto"].ne("")
)

if muestra_actualizada["requiere_revision_contexto"].any():
    raise ValueError(
        "Quedan alertas de producto. Revisa antes de guardar."
    )

# Añadimos los registros retirados a la auditoría
retirados_adicionales["motivo_retiro"] = (
    "Referencia explícita a otro producto"
)

auditoria_actualizada = pd.concat(
    [df_registros_retirados, retirados_adicionales],
    ignore_index=True
)

# Creamos una carpeta nueva para conservar esta versión
marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_version = (
    Path("/content/revision_evaluacion")
    / f"version_revision_{marca_version}"
)

carpeta_version.mkdir(parents=True, exist_ok=False)

# Guardamos la muestra y la auditoría
muestra_actualizada.to_csv(
    carpeta_version / "muestra_actualizada.csv",
    index=False,
    encoding="utf-8-sig"
)

auditoria_actualizada.to_csv(
    carpeta_version / "registros_retirados.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardamos las descripciones para mantener la trazabilidad
with (carpeta_version / "descripciones_v2.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(
        descripciones_categorias_v2,
        archivo,
        ensure_ascii=False,
        indent=2
    )

# Confirmamos la actualización después de guardar los archivos
df_muestra_revision = muestra_actualizada.copy()
df_registros_retirados = auditoria_actualizada.copy()

df_alertas_contexto = df_muestra_revision[
    df_muestra_revision["requiere_revision_contexto"]
].copy()

resumen["Total"] = resumen.sum(axis=1)
resumen.index.name = "Categoría"
display(resumen.reset_index())

print("Registros retirados conservados:", len(df_registros_retirados))
print("Consultas con alertas:", len(df_alertas_contexto))
print("Versión guardada en:", carpeta_version)

print("=" * 60)
print("✅ Reemplazos incorporados y versión de revisión guardada.")
print("=" * 60)

conjunto,Categoría,desarrollo,prueba,Total
0,LIFE_INSURANCE,10,40,50
1,COVERAGE,10,40,50
2,QUOTE,10,40,50
3,PAYMENT,10,40,50
4,ENROLLMENT,10,40,50
5,CONTACT,10,40,50


Registros retirados conservados: 18
Consultas con alertas: 0
Versión guardada en: /content/revision_evaluacion/version_revision_20261010T185845608801Z
✅ Reemplazos incorporados y versión de revisión guardada.


### **🧠 4.20 Revisión del bloque COVERAGE actualizado**

Se mostrará el bloque de desarrollo de COVERAGE después
de incorporar los reemplazos.

Se comprobarán las traducciones y etiquetas propuestas
para las nueve consultas conservadas y se preparará
la traducción del nuevo registro CONS_0060.

**Estado:** Revisión y validación del bloque pendientes.

In [ ]:
# ============================================================
# 4.20 Revisión del bloque COVERAGE actualizado
# ============================================================

# Seleccionamos las diez consultas de desarrollo de COVERAGE
df_bloque_revision = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
    & df_muestra_revision["categoria_original"].eq("COVERAGE")
].copy()

# Mostramos el texto original y los campos actuales de revisión
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_revision[
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("=" * 60)
print("✅ Bloque COVERAGE actualizado presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
50,CONS_0051,"I need to check my fucking coverage , how can I do it?","Necesito revisar mi cobertura, ¿cómo puedo hacerlo?",COVERAGE,propuesta_pendiente_revision
51,CONS_0052,"I'd like to know about my coverage, could you give me some information ?","Quisiera saber sobre mi cobertura, ¿podrías darme información?",COVERAGE,propuesta_pendiente_revision
52,CONS_0053,where can i downgrade the fucking coverage of my insurance polikcy,¿Dónde puedo reducir la cobertura de mi póliza de seguro?,COVERAGE,propuesta_pendiente_revision
53,CONS_0054,i have to change the fucking coverage of my insurance policy could i get somw help,"Tengo que cambiar la cobertura de mi póliza de seguro, ¿me pueden ayudar?",COVERAGE,propuesta_pendiente_revision
54,CONS_0055,i have to see the coverage of my life insurance policy where to do it,"Tengo que consultar la cobertura de mi póliza de seguro de vida, ¿dónde puedo hacerlo?",COVERAGE,propuesta_pendiente_revision
55,CONS_0056,id like to upgrade the coverage of my life insurance policy will ya hlp me,"Quisiera ampliar la cobertura de mi póliza de seguro de vida, ¿me ayudan?",COVERAGE,propuesta_pendiente_revision
56,CONS_0057,I'm looking for information about the coverage of my insurance policy how could i get ig,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",COVERAGE,propuesta_pendiente_revision
57,CONS_0058,can ya help me increase the coverage of my life insurance policy,¿Puedes ayudarme a aumentar la cobertura de mi póliza de seguro de vida?,COVERAGE,propuesta_pendiente_revision
58,CONS_0059,i need help reducing the fucking coverage of my life insurance policy,Necesito ayuda para reducir la cobertura de mi póliza de seguro de vida.,COVERAGE,propuesta_pendiente_revision
59,CONS_0060,can I modify the coverage of my insurance policy ?,,,pendiente


✅ Bloque COVERAGE actualizado presentado para revisión.


### **🧠 4.21 Traducción del reemplazo de COVERAGE**

Se registrará la traducción del nuevo CONS_0060,
conservando la ausencia de un producto asegurador específico.

Se propondrá COVERAGE como categoría de referencia,
porque la consulta solicita modificar la cobertura.

**Estado:** Diez consultas del bloque con traducción y etiqueta
propuestas. Validación final pendiente.

In [ ]:
# ============================================================
# 4.21 Traducción del reemplazo de COVERAGE
# ============================================================

# Localizamos el reemplazo en el conjunto de desarrollo
mascara = df_muestra_revision["id_consulta"].eq("CONS_0060")
registro = df_muestra_revision.loc[mascara]

if (
    len(registro) != 1
    or not registro["conjunto"].eq("desarrollo").all()
    or not registro["categoria_original"].eq("COVERAGE").all()
):
    raise ValueError("El registro no corresponde al bloque esperado.")

# Verificamos el texto antes de asociarle la traducción
texto_esperado = "can I modify the coverage of my insurance policy ?"

if (
    normalizar_contexto(registro.iloc[0]["consulta_original"])
    != normalizar_contexto(texto_esperado)
):
    raise ValueError("El texto de CONS_0060 cambió; revisa la traducción.")

# Registramos la propuesta sin marcarla como validada
df_muestra_revision.loc[mascara, "consulta_es"] = (
    "¿Puedo modificar la cobertura de mi póliza de seguro?"
)

df_muestra_revision.loc[
    mascara, "categoria_referencia"
] = "COVERAGE"

df_muestra_revision.loc[
    mascara, "estado_revision"
] = "propuesta_pendiente_revision"

df_muestra_revision.loc[mascara, "observaciones"] = (
    "Reemplazo de una consulta fuera del alcance. "
    "Solicita modificar cobertura sin especificar producto. "
    "Traducción y etiqueta propuestas, pendientes de validación."
)

# Mostramos la propuesta registrada
with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            mascara,
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("=" * 60)
print("✅ Traducción propuesta del reemplazo registrada.")
print("=" * 60)

,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
59,CONS_0060,can I modify the coverage of my insurance policy ?,¿Puedo modificar la cobertura de mi póliza de seguro?,COVERAGE,propuesta_pendiente_revision


✅ Traducción propuesta del reemplazo registrada.


### **🧠 4.22 Revisión del bloque QUOTE**

Se examinarán las diez consultas de desarrollo cuya categoría
original es QUOTE.

Se comprobará si solicitan tarifas, estimaciones de costo
o cotizaciones de seguros, antes de preparar las traducciones
y proponer las etiquetas de referencia.

**Estado:** Bloque pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 4.22 Revisión del bloque QUOTE
# ============================================================

# Seleccionamos las consultas de desarrollo de QUOTE
df_bloque_revision = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
    & df_muestra_revision["categoria_original"].eq("QUOTE")
].copy()

# Comprobamos el tamaño previsto del bloque
if len(df_bloque_revision) != 10:
    raise ValueError("Se esperaban diez consultas de desarrollo.")

# Mostramos los textos completos y sus datos de origen
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_revision[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Bloque QUOTE presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
100,CONS_0101,I'm trying to find a fucking insurance quote i need assistane,QUOTE,calculate_insurance_quote
101,CONS_0102,i want an insurance quote help me calculating it,QUOTE,calculate_insurance_quote
102,CONS_0103,"I am looking for information about the fuckinv insurance rates, where could I find it?",QUOTE,check_rates
103,CONS_0104,id like to know more about the fucking life insurance rates give me some information,QUOTE,check_rates
104,CONS_0105,i want the life inurance rates could i see them,QUOTE,check_rates
105,CONS_0106,i need anauto insurance quote where do i estimate it,QUOTE,calculate_insurance_quote
106,CONS_0107,where do I see the life insurance rates?,QUOTE,check_rates
107,CONS_0108,"I am trying to find an insurance quote , I need help",QUOTE,calculate_insurance_quote
108,CONS_0109,I'd like to know about hte rates could ya give me some information,QUOTE,check_rates
109,CONS_0110,im looking for information about the insurance rates can i get some help,QUOTE,check_rates


✅ Bloque QUOTE presentado para revisión.


### **🧠 4.23 Ajuste por palabras unidas y actualización de alertas**

Se ampliará la detección para reconocer auto precedido
por los artículos a o an cuando aparezcan unidos y seguidos
de una variante de insurance.

Se comprobará el caso real CONS_0106 y se actualizarán
las alertas de la muestra.

Las coincidencias se conservarán para revisión manual.
Los textos originales permanecerán sin cambios.

In [ ]:
# ============================================================
# 4.23 Ajuste por palabras unidas y actualización de alertas
# ============================================================

# Añadimos la variante observada: anauto insurance
patrones_referencia_producto["Automóvil"] = (
    "(?<![a-z])(?:auto|car|vehicle|automobile)(?![a-z])"
    "|(?<![a-z])(?:an|a)auto[ -]*insur[a-z]*(?![a-z])"
)

# Comprobamos el caso real antes de actualizar las alertas
registro = df_muestra_revision.loc[
    df_muestra_revision["id_consulta"].eq("CONS_0106")
]

if len(registro) != 1:
    raise ValueError("Se esperaba un único registro CONS_0106.")

texto_actual = registro.iloc[0]["consulta_original"]
productos = detectar_referencias_producto(texto_actual)

print("Texto actual:", ascii(str(texto_actual)))
print("Productos detectados:", productos)

if "Automóvil" not in productos:
    raise ValueError("La corrección no detectó el caso real.")

# Verificamos que no se marque una consulta de seguro de vida
if detectar_referencias_producto("I need a life insurance quote"):
    raise ValueError("Se detectó incorrectamente otro producto.")

# Actualizamos las alertas sobre los 300 registros actuales
df_muestra_revision["alertas_producto"] = (
    df_muestra_revision["consulta_original"]
    .map(lambda texto: ", ".join(detectar_referencias_producto(texto)))
)

df_muestra_revision["requiere_revision_contexto"] = (
    df_muestra_revision["alertas_producto"].ne("")
)

df_alertas_contexto = df_muestra_revision[
    df_muestra_revision["requiere_revision_contexto"]
].copy()

print("Consultas con alertas actuales:", len(df_alertas_contexto))

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_alertas_contexto[
            [
                "id_consulta",
                "conjunto",
                "consulta_original",
                "alertas_producto"
            ]
        ]
    )

print("=" * 60)
print("✅ Variante de palabras unidas detectada y alertas actualizadas.")
print("=" * 60)

Texto actual: 'i need anauto insurance quote where do i estimate it'
Productos detectados: ['Automóvil']
Consultas con alertas actuales: 1


,id_consulta,conjunto,consulta_original,alertas_producto
105,CONS_0106,desarrollo,i need anauto insurance quote where do i estimate it,Automóvil


✅ Variante de palabras unidas detectada y alertas actualizadas.


### **🧠 4.24 Traducción y revisión del bloque QUOTE**

Se registrarán traducciones y etiquetas propuestas para nueve
consultas sobre tarifas o cotizaciones.

CONS_0106 menciona un seguro de automóvil y quedará con
exclusión propuesta, pendiente de reemplazo.

Las traducciones conservarán el significado y el tono original,
sin añadir referencias a seguros de vida cuando no existan.

**Estado:** Nueve propuestas pendientes de validación.
Una consulta pendiente de reemplazo.

In [ ]:
# ============================================================
# 4.24 Traducción y revisión del bloque QUOTE
# ============================================================

# Definimos las traducciones de las nueve consultas candidatas
traducciones_quote = {
    "CONS_0101": (
        "Estoy intentando conseguir una cotización "
        "de seguro, necesito ayuda."
    ),
    "CONS_0102": (
        "Quiero una cotización de seguro, ayúdame a calcularla."
    ),
    "CONS_0103": (
        "Busco información sobre las tarifas del seguro, "
        "¿dónde puedo encontrarla?"
    ),
    "CONS_0104": (
        "Quisiera saber más sobre las tarifas del seguro "
        "de vida, dame información."
    ),
    "CONS_0105": (
        "Quiero consultar las tarifas del seguro de vida, "
        "¿puedo verlas?"
    ),
    "CONS_0107": (
        "¿Dónde puedo consultar las tarifas del seguro de vida?"
    ),
    "CONS_0108": (
        "Estoy intentando conseguir una cotización de seguro, "
        "necesito ayuda."
    ),
    "CONS_0109": (
        "Quisiera saber sobre las tarifas, "
        "¿podrías darme información?"
    ),
    "CONS_0110": (
        "Busco información sobre las tarifas del seguro, "
        "¿me pueden ayudar?"
    )
}

# Comprobamos los diez registros antes de modificar la tabla
ids_bloque = list(traducciones_quote) + ["CONS_0106"]

bloque = df_muestra_revision[
    df_muestra_revision["id_consulta"].isin(ids_bloque)
]

if (
    len(bloque) != 10
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("desarrollo").all()
    or not bloque["categoria_original"].eq("QUOTE").all()
):
    raise ValueError("Los registros no coinciden con el bloque esperado.")

registro_auto = bloque[
    bloque["id_consulta"].eq("CONS_0106")
].iloc[0]

if "Automóvil" not in detectar_referencias_producto(
    registro_auto["consulta_original"]
):
    raise ValueError("CONS_0106 requiere una nueva revisión.")

# Trabajamos sobre una copia para registrar las propuestas
muestra_actualizada = df_muestra_revision.copy()

for identificador, traduccion in traducciones_quote.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[mascara, "categoria_referencia"] = "QUOTE"
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Solicitud de tarifas o cotización. "
        "Traducción y etiqueta propuestas; validación pendiente."
    )

# Conservamos el caso fuera del alcance para su posterior reemplazo
mascara_auto = muestra_actualizada["id_consulta"].eq("CONS_0106")

muestra_actualizada.loc[mascara_auto, "consulta_es"] = (
    "Necesito una cotización de seguro de automóvil, "
    "¿dónde puedo calcularla?"
)
muestra_actualizada.loc[
    mascara_auto, "categoria_referencia"
] = "QUOTE"
muestra_actualizada.loc[
    mascara_auto, "estado_revision"
] = "exclusion_propuesta"
muestra_actualizada.loc[mascara_auto, "observaciones"] = (
    "Menciona un seguro de automóvil con las palabras an y auto "
    "unidas en el original. Fuera del alcance; requiere reemplazo."
)

# Confirmamos la actualización y mostramos las propuestas
df_muestra_revision = muestra_actualizada.copy()

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(ids_bloque),
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("=" * 60)
print("✅ Propuestas del bloque QUOTE registradas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
100,CONS_0101,"Estoy intentando conseguir una cotización de seguro, necesito ayuda.",QUOTE,propuesta_pendiente_revision
101,CONS_0102,"Quiero una cotización de seguro, ayúdame a calcularla.",QUOTE,propuesta_pendiente_revision
102,CONS_0103,"Busco información sobre las tarifas del seguro, ¿dónde puedo encontrarla?",QUOTE,propuesta_pendiente_revision
103,CONS_0104,"Quisiera saber más sobre las tarifas del seguro de vida, dame información.",QUOTE,propuesta_pendiente_revision
104,CONS_0105,"Quiero consultar las tarifas del seguro de vida, ¿puedo verlas?",QUOTE,propuesta_pendiente_revision
105,CONS_0106,"Necesito una cotización de seguro de automóvil, ¿dónde puedo calcularla?",QUOTE,exclusion_propuesta
106,CONS_0107,¿Dónde puedo consultar las tarifas del seguro de vida?,QUOTE,propuesta_pendiente_revision
107,CONS_0108,"Estoy intentando conseguir una cotización de seguro, necesito ayuda.",QUOTE,propuesta_pendiente_revision
108,CONS_0109,"Quisiera saber sobre las tarifas, ¿podrías darme información?",QUOTE,propuesta_pendiente_revision
109,CONS_0110,"Busco información sobre las tarifas del seguro, ¿me pueden ayudar?",QUOTE,propuesta_pendiente_revision


✅ Propuestas del bloque QUOTE registradas.


### **🧠 4.25 Selección del reemplazo de CONS_0106**

Se seleccionará una consulta candidata de QUOTE para reemplazar
el registro de automóvil del conjunto de desarrollo.

Se excluirán textos presentes en la muestra, registros retirados
y consultas con referencias detectadas a otros productos.

Se conservará la intención calculate_insurance_quote del registro
original para mantener la composición de este bloque.

**Estado:** Reemplazo candidato pendiente de revisión.

In [ ]:
# ============================================================
# 4.25 Selección del reemplazo de CONS_0106
# ============================================================

# Verificamos la posición que requiere reemplazo
original = df_muestra_revision.loc[
    df_muestra_revision["id_consulta"].eq("CONS_0106")
]

if (
    len(original) != 1
    or not original["conjunto"].eq("desarrollo").all()
    or not original["categoria_original"].eq("QUOTE").all()
    or not original["estado_revision"].eq("exclusion_propuesta").all()
):
    raise ValueError("CONS_0106 no coincide con el registro esperado.")

original = original.iloc[0]

# Excluimos consultas actuales y registros retirados
textos_utilizados = set(
    df_muestra_revision["consulta_original"].map(normalizar_contexto)
) | set(
    df_registros_retirados["consulta_original"].map(normalizar_contexto)
)

# Preparamos la reserva con las mismas categoría e intención
reserva = df_elegibles.copy()

reserva["texto_revision"] = (
    reserva["instruction"].map(normalizar_contexto)
)

candidatos = reserva[
    reserva["category"].eq(original["categoria_original"])
    & reserva["intent"].eq(original["intencion_original"])
    & reserva["texto_revision"].ne("")
    & ~reserva["texto_revision"].isin(textos_utilizados)
    & ~reserva["instruction"]
        .map(detectar_referencias_producto).map(bool)
].sort_values("texto_revision").drop_duplicates("texto_revision")

if candidatos.empty:
    raise ValueError(
        "No hay candidatos con la misma categoría e intención."
    )

# Seleccionamos un candidato reproducible
candidato = candidatos.sample(
    n=1,
    random_state=642
).iloc[0]

# Conservamos la propuesta separada de la muestra principal
df_reemplazo_quote = pd.DataFrame([{
    "id_a_reemplazar": "CONS_0106",
    "conjunto": original["conjunto"],
    "indice_origen": candidato["indice_origen"],
    "consulta_original": candidato["instruction"],
    "categoria_original": candidato["category"],
    "intencion_original": candidato["intent"],
    "consulta_normalizada": candidato["consulta_normalizada"]
}])

with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazo_quote[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Reemplazo de CONS_0106 presentado para revisión.")
print("=" * 60)

,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
0,CONS_0106,desarrollo,i got to calculate a fucking insurance quote will ya help em,QUOTE,calculate_insurance_quote


✅ Reemplazo de CONS_0106 presentado para revisión.


### **🧠 4.26 Incorporación y traducción del reemplazo de QUOTE**

Se incorporará el candidato revisado en la posición CONS_0106,
conservando su conjunto, categoría e intención original.

El registro de automóvil retirado se añadirá a la auditoría.
Se registrarán la traducción y la etiqueta propuestas
para la nueva consulta.

**Estado:** Bloque QUOTE con diez traducciones propuestas.
Validación final pendiente.

In [ ]:
# ============================================================
# 4.26 Incorporación y traducción del reemplazo de QUOTE
# ============================================================

# Verificamos que el reemplazo no haya sido incorporado
if df_registros_retirados["id_consulta"].eq("CONS_0106").any():
    raise ValueError("CONS_0106 ya fue reemplazado y auditado.")

if (
    len(df_reemplazo_quote) != 1
    or df_reemplazo_quote.iloc[0]["id_a_reemplazar"] != "CONS_0106"
):
    raise ValueError("No se encontró la propuesta esperada.")

reemplazo = df_reemplazo_quote.iloc[0]

# Comprobamos el texto revisado antes de asociarle la traducción
texto_esperado = (
    "i got to calculate a fucking insurance quote will ya help em"
)

if (
    normalizar_contexto(reemplazo["consulta_original"])
    != normalizar_contexto(texto_esperado)
    or detectar_referencias_producto(reemplazo["consulta_original"])
):
    raise ValueError("El candidato cambió o requiere revisión.")

# Trabajamos sobre una copia y conservamos el registro anterior
muestra_actualizada = df_muestra_revision.copy()
mascara = muestra_actualizada["id_consulta"].eq("CONS_0106")
retirado = muestra_actualizada.loc[mascara].copy()

if (
    len(retirado) != 1
    or not retirado["estado_revision"].eq("exclusion_propuesta").all()
    or retirado.iloc[0]["conjunto"] != reemplazo["conjunto"]
    or retirado.iloc[0]["categoria_original"]
    != reemplazo["categoria_original"]
    or retirado.iloc[0]["intencion_original"]
    != reemplazo["intencion_original"]
):
    raise ValueError("El registro original no coincide con lo esperado.")

# Actualizamos los datos de origen de la nueva consulta
for campo in [
    "indice_origen",
    "consulta_original",
    "categoria_original",
    "intencion_original",
    "consulta_normalizada"
]:
    muestra_actualizada.loc[mascara, campo] = reemplazo[campo]

# Registramos la traducción y etiqueta propuestas
muestra_actualizada.loc[mascara, "consulta_es"] = (
    "Tengo que calcular una pinche cotización de seguro, ¿me ayudas?"
)
muestra_actualizada.loc[mascara, "categoria_referencia"] = "QUOTE"
muestra_actualizada.loc[
    mascara, "estado_revision"
] = "propuesta_pendiente_revision"
muestra_actualizada.loc[mascara, "observaciones"] = (
    "Reemplazo de una consulta de automóvil. "
    "Solicitud general de cotización; conserva el tono informal. "
    "Traducción y etiqueta pendientes de validación."
)

# Actualizamos los campos de detección
muestra_actualizada["otros_productos_detectados"] = (
    muestra_actualizada["consulta_original"]
    .map(lambda texto: ", ".join(detectar_otros_productos(texto)))
)
muestra_actualizada["alertas_producto"] = (
    muestra_actualizada["consulta_original"]
    .map(lambda texto: ", ".join(detectar_referencias_producto(texto)))
)
muestra_actualizada["requiere_revision_contexto"] = (
    muestra_actualizada["alertas_producto"].ne("")
)

# Verificamos duplicados y distribución
if (
    muestra_actualizada["consulta_original"]
    .map(normalizar_contexto).duplicated().any()
):
    raise ValueError("El reemplazo genera un texto duplicado.")

resumen = pd.crosstab(
    muestra_actualizada["categoria_original"],
    muestra_actualizada["conjunto"]
).reindex(
    index=categorias_candidatas,
    columns=["desarrollo", "prueba"],
    fill_value=0
)

if (
    len(muestra_actualizada) != 300
    or not resumen["desarrollo"].eq(10).all()
    or not resumen["prueba"].eq(40).all()
):
    raise ValueError("La distribución de la muestra cambió.")

# Añadimos el registro retirado y confirmamos la actualización
retirado["motivo_retiro"] = "Referencia explícita a seguro de automóvil"

df_registros_retirados = pd.concat(
    [df_registros_retirados, retirado],
    ignore_index=True
)
df_muestra_revision = muestra_actualizada.copy()

df_alertas_contexto = df_muestra_revision[
    df_muestra_revision["requiere_revision_contexto"]
].copy()

# Mostramos el reemplazo y el resultado de la auditoría
with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            mascara,
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("Registros retirados conservados:", len(df_registros_retirados))
print("Consultas con alertas actuales:", len(df_alertas_contexto))

print("=" * 60)
print("✅ Reemplazo de QUOTE incorporado y traducción propuesta registrada.")
print("=" * 60)

,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
105,CONS_0106,i got to calculate a fucking insurance quote will ya help em,"Tengo que calcular una pinche cotización de seguro, ¿me ayudas?",QUOTE,propuesta_pendiente_revision


Registros retirados conservados: 19
Consultas con alertas actuales: 0
✅ Reemplazo de QUOTE incorporado y traducción propuesta registrada.


### **🧠 4.27 Revisión del bloque PAYMENT**

Se examinarán las diez consultas de desarrollo cuya categoría
original es PAYMENT.

Se revisará si tratan sobre métodos, fechas, periodicidad
o problemas de pago del seguro, antes de preparar
las traducciones y etiquetas propuestas.

**Estado:** Bloque pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 4.27 Revisión del bloque PAYMENT
# ============================================================

# Seleccionamos las consultas de desarrollo de PAYMENT
df_bloque_revision = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
    & df_muestra_revision["categoria_original"].eq("PAYMENT")
].copy()

# Comprobamos el tamaño previsto del bloque
if len(df_bloque_revision) != 10:
    raise ValueError("Se esperaban diez consultas de desarrollo.")

# Mostramos los textos completos y sus intenciones originales
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_revision[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Bloque PAYMENT presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
150,CONS_0151,i have to schedule the payments could yahelp me,PAYMENT,schedule_payments
151,CONS_0152,"I cannot make payments, can youj help me to notify of payment issues?",PAYMENT,report_payment_issue
152,CONS_0153,i cant make a transfer how do i notify of fucking payment problems,PAYMENT,report_payment_issue
153,CONS_0154,I'd like to pay by fucking credit card can i get some help,PAYMENT,payment_methods
154,CONS_0155,i have to see my fucking payments to the life insurance company can uhelp me,PAYMENT,check_payments
155,CONS_0156,"my card didn't work, can I report a fucking payment issue?",PAYMENT,report_payment_issue
156,CONS_0157,"I have to pay, how to do it?",PAYMENT,pay
157,CONS_0158,i got to scehdule the payments could uhelp me,PAYMENT,schedule_payments
158,CONS_0159,"I want to see your payment methods, how could I do it?",PAYMENT,payment_methods
159,CONS_0160,wanna pay with a fucking debit card how do i do it,PAYMENT,payment_methods


✅ Bloque PAYMENT presentado para revisión.


### **🧠 4.28 Traducción y revisión del bloque PAYMENT**

Se propondrán traducciones para diez consultas relacionadas
con programación y realización de pagos, consulta de pagos,
métodos disponibles y reporte de problemas.

Se conservarán el significado y el tono de los textos originales,
sin añadir un producto asegurador cuando no se especifique.

La descripción de PAYMENT incluirá explícitamente
la consulta de pagos realizados.

**Estado:** Traducciones y etiquetas propuestas.
Validación final pendiente.

In [ ]:
# ============================================================
# 4.28 Traducción y revisión del bloque PAYMENT
# ============================================================

# Definimos traducciones que conservan el significado y el tono
traducciones_payment = {
    "CONS_0151": (
        "Tengo que programar los pagos, ¿puedes ayudarme?"
    ),
    "CONS_0152": (
        "No puedo realizar pagos, ¿puedes ayudarme "
        "a reportar problemas de pago?"
    ),
    "CONS_0153": (
        "No puedo hacer una transferencia, "
        "¿cómo reporto los pinches problemas de pago?"
    ),
    "CONS_0154": (
        "Quisiera pagar con mi tarjeta de crédito, "
        "¿me pueden ayudar?"
    ),
    "CONS_0155": (
        "Tengo que consultar mis pagos a la compañía "
        "de seguros de vida, ¿puedes ayudarme?"
    ),
    "CONS_0156": (
        "Mi tarjeta no funcionó, "
        "¿puedo reportar un problema de pago?"
    ),
    "CONS_0157": (
        "Tengo que pagar, ¿cómo lo hago?"
    ),
    "CONS_0158": (
        "Tengo que programar los pagos, ¿puedes ayudarme?"
    ),
    "CONS_0159": (
        "Quiero consultar sus métodos de pago, ¿cómo puedo hacerlo?"
    ),
    "CONS_0160": (
        "Quiero pagar con una tarjeta de débito, ¿cómo lo hago?"
    )
}

# Verificamos los registros antes de modificar la tabla
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_payment
)

bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != 10
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("desarrollo").all()
    or not bloque["categoria_original"].eq("PAYMENT").all()
    or bloque["consulta_original"]
        .map(detectar_referencias_producto).map(bool).any()
):
    raise ValueError("El bloque PAYMENT requiere revisión.")

# Registramos las propuestas sobre una copia
muestra_actualizada = df_muestra_revision.copy()

for identificador, traduccion in traducciones_payment.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[mascara, "categoria_referencia"] = "PAYMENT"
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Consulta sobre pagos. Traducción y etiqueta propuestas; "
        "validación pendiente."
    )

# Documentamos un duplicado generado por la traducción
ids_traduccion_coincidente = ["CONS_0151", "CONS_0158"]

muestra_actualizada.loc[
    muestra_actualizada["id_consulta"].isin(ids_traduccion_coincidente),
    "observaciones"
] = (
    "Programación de pagos. Dos originales distintos producen "
    "la misma traducción: CONS_0151 y CONS_0158. "
    "Revisar duplicados en español antes de evaluar."
)

# Actualizamos la descripción común para la evaluación posterior
descripciones_categorias_v2["PAYMENT"] = (
    "realización y consulta de pagos del seguro, "
    "programación, métodos y problemas de pago"
)

# Confirmamos la actualización y mostramos el bloque
df_muestra_revision = muestra_actualizada.copy()

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("=" * 60)
print("✅ Propuestas del bloque PAYMENT registradas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
150,CONS_0151,"Tengo que programar los pagos, ¿puedes ayudarme?",PAYMENT,propuesta_pendiente_revision
151,CONS_0152,"No puedo realizar pagos, ¿puedes ayudarme a reportar problemas de pago?",PAYMENT,propuesta_pendiente_revision
152,CONS_0153,"No puedo hacer una transferencia, ¿cómo reporto los pinches problemas de pago?",PAYMENT,propuesta_pendiente_revision
153,CONS_0154,"Quisiera pagar con mi tarjeta de crédito, ¿me pueden ayudar?",PAYMENT,propuesta_pendiente_revision
154,CONS_0155,"Tengo que consultar mis pagos a la compañía de seguros de vida, ¿puedes ayudarme?",PAYMENT,propuesta_pendiente_revision
155,CONS_0156,"Mi tarjeta no funcionó, ¿puedo reportar un problema de pago?",PAYMENT,propuesta_pendiente_revision
156,CONS_0157,"Tengo que pagar, ¿cómo lo hago?",PAYMENT,propuesta_pendiente_revision
157,CONS_0158,"Tengo que programar los pagos, ¿puedes ayudarme?",PAYMENT,propuesta_pendiente_revision
158,CONS_0159,"Quiero consultar sus métodos de pago, ¿cómo puedo hacerlo?",PAYMENT,propuesta_pendiente_revision
159,CONS_0160,"Quiero pagar con una tarjeta de débito, ¿cómo lo hago?",PAYMENT,propuesta_pendiente_revision


✅ Propuestas del bloque PAYMENT registradas.


### **🧠 4.29 Revisión del bloque ENROLLMENT**

Se examinarán las diez consultas de desarrollo de ENROLLMENT
después de incorporar los reemplazos.

Se aplicará el criterio ampliado de la categoría: comparación
y selección de pólizas, contratación, renovación, no renovación
y cancelación, incluidos cargos asociados a la terminación.

La revisión precederá a la traducción y a la propuesta
de etiquetas de referencia.

**Estado:** Bloque pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 4.29 Revisión del bloque ENROLLMENT
# ============================================================

# Seleccionamos las consultas de desarrollo de ENROLLMENT
df_bloque_revision = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
    & df_muestra_revision["categoria_original"].eq("ENROLLMENT")
].copy()

# Comprobamos el tamaño y los identificadores del bloque
if (
    len(df_bloque_revision) != 10
    or not df_bloque_revision["id_consulta"].is_unique
):
    raise ValueError("Se esperaban diez consultas con IDs únicos.")

# Mostramos los textos completos y las intenciones originales
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_revision[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Bloque ENROLLMENT presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
200,CONS_0201,"I have to compare fucking life insurance policies, can I get some help?",ENROLLMENT,compare_insurance_policies
201,CONS_0202,i do not want my fucking life insurance policy to be renewed I'd like to cancel it,ENROLLMENT,cancel_insurance_policy
202,CONS_0203,"I have to cancel my fucking life insurance policy , where could I do it?",ENROLLMENT,cancel_insurance_policy
203,CONS_0204,"I don't want my fucking insurance policy to be renewed, I want to cancel it",ENROLLMENT,cancel_insurance_policy
204,CONS_0205,wanna see the fuckinmg early termination fees where can i do it,ENROLLMENT,cancellation_fees
205,CONS_0206,i want an fucking insurance policy will uhelp me buying it,ENROLLMENT,buy_insurance_policy
206,CONS_0207,"I have to check the fucking cancellation charges, how to do it?",ENROLLMENT,cancellation_fees
207,CONS_0208,"I don't want to reenw my life insurance policy, I need assistance to cancel it",ENROLLMENT,cancel_insurance_policy
208,CONS_0209,"I would like to see the cancellation charhes, I need help",ENROLLMENT,cancellation_fees
209,CONS_0210,I'd like to know about the fucking cancellation charges find some information,ENROLLMENT,cancellation_fees


✅ Bloque ENROLLMENT presentado para revisión.


### **🧠 4.30 Traducción y revisión del bloque ENROLLMENT**

Se propondrán traducciones para diez consultas sobre comparación,
contratación y cancelación de pólizas, incluidos cargos
por cancelación o terminación anticipada.

Se conservarán las referencias explícitas a seguros de vida
y el tono de los originales, sin añadir productos no mencionados.

Los cargos asociados a terminar la póliza se clasificarán
como ENROLLMENT según el criterio ampliado.

**Hallazgo:** Ocho consultas del bloque tratan sobre cancelación,
no renovación o cargos de terminación. Esta concentración
se considerará al interpretar los resultados.

**Estado:** Traducciones y etiquetas propuestas.
Validación final pendiente.

In [ ]:
# ============================================================
# 4.30 Traducción y revisión del bloque ENROLLMENT
# ============================================================

# Definimos las traducciones conservando significado y tono
traducciones_enrollment = {
    "CONS_0201": (
        "Tengo que comparar pólizas de seguro de vida, "
        "¿me pueden ayudar?"
    ),
    "CONS_0202": (
        "No quiero que se renueve mi póliza de seguro "
        "de vida, quisiera cancelarla."
    ),
    "CONS_0203": (
        "Tengo que cancelar mi póliza de seguro de vida, "
        "¿dónde puedo hacerlo?"
    ),
    "CONS_0204": (
        "No quiero que se renueve mi póliza de seguro, "
        "quiero cancelarla."
    ),
    "CONS_0205": (
        "Quiero consultar los cargos por terminación "
        "anticipada, ¿dónde puedo hacerlo?"
    ),
    "CONS_0206": (
        "Quiero una póliza de seguro, "
        "¿me ayudas a contratarla?"
    ),
    "CONS_0207": (
        "Tengo que consultar los cargos por cancelación, "
        "¿cómo lo hago?"
    ),
    "CONS_0208": (
        "No quiero renovar mi póliza de seguro de vida, "
        "necesito ayuda para cancelarla."
    ),
    "CONS_0209": (
        "Quisiera consultar los cargos por cancelación, "
        "necesito ayuda."
    ),
    "CONS_0210": (
        "Quisiera saber sobre los cargos por cancelación, "
        "dame información."
    )
}

# Verificamos el bloque antes de registrar las propuestas
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_enrollment
)

bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != 10
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("desarrollo").all()
    or not bloque["categoria_original"].eq("ENROLLMENT").all()
    or bloque["consulta_original"]
        .map(detectar_referencias_producto).map(bool).any()
):
    raise ValueError("El bloque ENROLLMENT requiere revisión.")

# Registramos las propuestas sobre una copia
muestra_actualizada = df_muestra_revision.copy()

for identificador, traduccion in traducciones_enrollment.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[
        mascara, "categoria_referencia"
    ] = "ENROLLMENT"
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"

    intencion = muestra_actualizada.loc[
        mascara, "intencion_original"
    ].iloc[0]

    muestra_actualizada.loc[mascara, "observaciones"] = (
        f"Intención original: {intencion}. "
        "Aplicar el criterio ampliado de ENROLLMENT. "
        "Traducción y etiqueta propuestas; validación pendiente."
    )

# Confirmamos la actualización y mostramos el resultado
df_muestra_revision = muestra_actualizada.copy()

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("=" * 60)
print("✅ Propuestas del bloque ENROLLMENT registradas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
200,CONS_0201,"Tengo que comparar pólizas de seguro de vida, ¿me pueden ayudar?",ENROLLMENT,propuesta_pendiente_revision
201,CONS_0202,"No quiero que se renueve mi póliza de seguro de vida, quisiera cancelarla.",ENROLLMENT,propuesta_pendiente_revision
202,CONS_0203,"Tengo que cancelar mi póliza de seguro de vida, ¿dónde puedo hacerlo?",ENROLLMENT,propuesta_pendiente_revision
203,CONS_0204,"No quiero que se renueve mi póliza de seguro, quiero cancelarla.",ENROLLMENT,propuesta_pendiente_revision
204,CONS_0205,"Quiero consultar los cargos por terminación anticipada, ¿dónde puedo hacerlo?",ENROLLMENT,propuesta_pendiente_revision
205,CONS_0206,"Quiero una póliza de seguro, ¿me ayudas a contratarla?",ENROLLMENT,propuesta_pendiente_revision
206,CONS_0207,"Tengo que consultar los cargos por cancelación, ¿cómo lo hago?",ENROLLMENT,propuesta_pendiente_revision
207,CONS_0208,"No quiero renovar mi póliza de seguro de vida, necesito ayuda para cancelarla.",ENROLLMENT,propuesta_pendiente_revision
208,CONS_0209,"Quisiera consultar los cargos por cancelación, necesito ayuda.",ENROLLMENT,propuesta_pendiente_revision
209,CONS_0210,"Quisiera saber sobre los cargos por cancelación, dame información.",ENROLLMENT,propuesta_pendiente_revision


✅ Propuestas del bloque ENROLLMENT registradas.


### **🧠 4.31 Revisión del bloque CONTACT**

Se examinarán las diez consultas de desarrollo cuya categoría
original es CONTACT.

Se revisará si solicitan comunicación con un asesor,
atención al cliente o información sobre canales de contacto,
antes de preparar las traducciones y etiquetas propuestas.

**Estado:** Bloque pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 4.31 Revisión del bloque CONTACT
# ============================================================

# Seleccionamos las consultas de desarrollo de CONTACT
df_bloque_revision = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
    & df_muestra_revision["categoria_original"].eq("CONTACT")
].copy()

# Comprobamos el tamaño y los identificadores del bloque
if (
    len(df_bloque_revision) != 10
    or not df_bloque_revision["id_consulta"].is_unique
):
    raise ValueError("Se esperaban diez consultas con IDs únicos.")

# Mostramos los textos completos y sus intenciones originales
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_revision[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Bloque CONTACT presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
250,CONS_0251,id like to esnd an email to customer service where can i do it,CONTACT,customer_service
251,CONS_0252,"I would like to call client service, where could I do it?",CONTACT,customer_service
252,CONS_0253,would it be possible to speak with my fucking insurance representative,CONTACT,insurance_representative
253,CONS_0254,"I got to talk to the imsurer, I need assistance",CONTACT,agent
254,CONS_0255,how could I speak with the insurer?,CONTACT,agent
255,CONS_0256,"I have to speak with the insurance company, could I get some help?",CONTACT,agent
256,CONS_0257,how do i talk to somebdy,CONTACT,human_agent
257,CONS_0258,could uhelp me to contact fuckimg client service,CONTACT,customer_service
258,CONS_0259,"I want to talk to an person, how do I do it?",CONTACT,human_agent
259,CONS_0260,i do not understand ya i want assistance speaking with a fucking human agent,CONTACT,human_agent


✅ Bloque CONTACT presentado para revisión.


### **🧠 4.32 Traducción y revisión del bloque CONTACT**

Se propondrán traducciones para diez consultas que solicitan
contacto con atención al cliente, representantes, la aseguradora
o una persona.

Se conservarán el significado y el tono original.
Los errores de escritura se interpretarán según el contexto,
sin añadir información ausente.

**Estado:** Diez traducciones y etiquetas propuestas.
Validación final pendiente.

In [ ]:
# ============================================================
# 4.32 Traducción y revisión del bloque CONTACT
# ============================================================

# Definimos las traducciones conservando significado y tono
traducciones_contact = {
    "CONS_0251": (
        "Quisiera enviar un correo a atención al cliente, "
        "¿dónde puedo hacerlo?"
    ),
    "CONS_0252": (
        "Quisiera llamar a atención al cliente, "
        "¿dónde puedo hacerlo?"
    ),
    "CONS_0253": (
        "¿Sería posible hablar con mi representante de seguros?"
    ),
    "CONS_0254": (
        "Tengo que hablar con la aseguradora, necesito ayuda."
    ),
    "CONS_0255": (
        "¿Cómo puedo hablar con la aseguradora?"
    ),
    "CONS_0256": (
        "Tengo que hablar con la compañía de seguros, "
        "¿me pueden ayudar?"
    ),
    "CONS_0257": (
        "¿Cómo puedo hablar con alguien?"
    ),
    "CONS_0258": (
        "¿Podrías ayudarme a contactar al servicio "
        "de atención al cliente?"
    ),
    "CONS_0259": (
        "Quiero hablar con una persona, ¿cómo puedo hacerlo?"
    ),
    "CONS_0260": (
        "No entiendo, quiero ayuda para hablar con "
        "una persona."
    )
}

# Verificamos los registros antes de modificar la tabla
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_contact
)

bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != 10
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("desarrollo").all()
    or not bloque["categoria_original"].eq("CONTACT").all()
    or bloque["consulta_original"]
        .map(detectar_referencias_producto).map(bool).any()
):
    raise ValueError("El bloque CONTACT requiere revisión.")

# Registramos las propuestas sobre una copia
muestra_actualizada = df_muestra_revision.copy()

for identificador, traduccion in traducciones_contact.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[
        mascara, "categoria_referencia"
    ] = "CONTACT"
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Solicitud de contacto o atención humana. "
        "Traducción y etiqueta propuestas; validación pendiente."
    )

# Precisamos la descripción común de CONTACT
descripciones_categorias_v2["CONTACT"] = (
    "contacto con atención al cliente, un representante, "
    "la aseguradora o una persona"
)

# Confirmamos la actualización y mostramos las propuestas
df_muestra_revision = muestra_actualizada.copy()

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("=" * 60)
print("✅ Propuestas del bloque CONTACT registradas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
250,CONS_0251,"Quisiera enviar un correo a atención al cliente, ¿dónde puedo hacerlo?",CONTACT,propuesta_pendiente_revision
251,CONS_0252,"Quisiera llamar a atención al cliente, ¿dónde puedo hacerlo?",CONTACT,propuesta_pendiente_revision
252,CONS_0253,¿Sería posible hablar con mi representante de seguros?,CONTACT,propuesta_pendiente_revision
253,CONS_0254,"Tengo que hablar con la aseguradora, necesito ayuda.",CONTACT,propuesta_pendiente_revision
254,CONS_0255,¿Cómo puedo hablar con la aseguradora?,CONTACT,propuesta_pendiente_revision
255,CONS_0256,"Tengo que hablar con la compañía de seguros, ¿me pueden ayudar?",CONTACT,propuesta_pendiente_revision
256,CONS_0257,¿Cómo puedo hablar con alguien?,CONTACT,propuesta_pendiente_revision
257,CONS_0258,¿Podrías ayudarme a contactar al servicio de atención al cliente?,CONTACT,propuesta_pendiente_revision
258,CONS_0259,"Quiero hablar con una persona, ¿cómo puedo hacerlo?",CONTACT,propuesta_pendiente_revision
259,CONS_0260,"No entiendo, quiero ayuda para hablar con una persona.",CONTACT,propuesta_pendiente_revision


✅ Propuestas del bloque CONTACT registradas.


### **🧠 4.33 Comprobación del conjunto de desarrollo**

Se verificará que las 60 consultas de desarrollo tengan
traducción al español y una categoría de referencia reconocida.

Se identificarán traducciones repetidas mediante normalización
de mayúsculas, minúsculas y espacios. Se mostrarán todos
los registros de cada grupo repetido.

Esta comprobación no sustituye la revisión semántica ni detecta
necesariamente consultas con significado equivalente.

**Estado:** Comprobación de campos y duplicados.
Validación final pendiente.

In [ ]:
# ============================================================
# 4.33 Comprobación del conjunto de desarrollo
# ============================================================

# Creamos una copia sin modificar estados ni etiquetas
df_desarrollo_revision = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
].copy()

if (
    len(df_desarrollo_revision) != 60
    or not df_desarrollo_revision["id_consulta"].is_unique
):
    raise ValueError("Se esperaban 60 consultas con IDs únicos.")

# Comprobamos traducciones y etiquetas
df_desarrollo_revision["texto_es_normalizado"] = (
    df_desarrollo_revision["consulta_es"].map(normalizar_contexto)
)

mascara_sin_traduccion = (
    df_desarrollo_revision["texto_es_normalizado"].eq("")
)

mascara_etiqueta_invalida = (
    ~df_desarrollo_revision["categoria_referencia"]
    .isin(descripciones_categorias_v2)
)

# Identificamos todos los miembros de grupos repetidos
mascara_duplicados = (
    ~mascara_sin_traduccion
    & df_desarrollo_revision["texto_es_normalizado"].duplicated(
        keep=False
    )
)

df_duplicados_es_desarrollo = (
    df_desarrollo_revision[mascara_duplicados]
    .sort_values(["texto_es_normalizado", "id_consulta"])
    .copy()
)

# Mostramos el estado de los campos por categoría
resumen_desarrollo = (
    df_desarrollo_revision.assign(
        con_traduccion=~mascara_sin_traduccion,
        con_etiqueta_valida=~mascara_etiqueta_invalida
    )
    .groupby("categoria_original")
    .agg(
        Consultas=("id_consulta", "size"),
        Con_traduccion=("con_traduccion", "sum"),
        Con_etiqueta_reconocida=("con_etiqueta_valida", "sum")
    )
    .reindex(categorias_candidatas)
)

display(resumen_desarrollo.reset_index())

print("Consultas sin traducción:", int(mascara_sin_traduccion.sum()))
print("Etiquetas vacías o no reconocidas:", int(mascara_etiqueta_invalida.sum()))
print(
    "Grupos de traducciones repetidas:",
    df_duplicados_es_desarrollo["texto_es_normalizado"].nunique()
)
print(
    "Consultas en grupos repetidos:",
    len(df_duplicados_es_desarrollo)
)

# Mostramos los casos que requieren revisión
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    if not df_duplicados_es_desarrollo.empty:
        display(
            df_duplicados_es_desarrollo[
                [
                    "id_consulta",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ]
        )

    pendientes_campos = df_desarrollo_revision[
        mascara_sin_traduccion | mascara_etiqueta_invalida
    ]

    if not pendientes_campos.empty:
        display(
            pendientes_campos[
                [
                    "id_consulta",
                    "consulta_es",
                    "categoria_referencia",
                    "estado_revision"
                ]
            ]
        )

print("=" * 60)
print("✅ Campos y duplicados del desarrollo comprobados.")
print("=" * 60)

,categoria_original,Consultas,Con_traduccion,Con_etiqueta_reconocida
0,LIFE_INSURANCE,10,10,10
1,COVERAGE,10,10,10
2,QUOTE,10,10,10
3,PAYMENT,10,10,10
4,ENROLLMENT,10,10,10
5,CONTACT,10,10,10


Consultas sin traducción: 0
Etiquetas vacías o no reconocidas: 0
Grupos de traducciones repetidas: 2
Consultas en grupos repetidos: 4


,id_consulta,consulta_original,consulta_es,categoria_referencia
100,CONS_0101,I'm trying to find a fucking insurance quote i need assistane,"Estoy intentando conseguir una cotización de seguro, necesito ayuda.",QUOTE
107,CONS_0108,"I am trying to find an insurance quote , I need help","Estoy intentando conseguir una cotización de seguro, necesito ayuda.",QUOTE
150,CONS_0151,i have to schedule the payments could yahelp me,"Tengo que programar los pagos, ¿puedes ayudarme?",PAYMENT
157,CONS_0158,i got to scehdule the payments could uhelp me,"Tengo que programar los pagos, ¿puedes ayudarme?",PAYMENT


✅ Campos y duplicados del desarrollo comprobados.


### **🧠 4.34 Selección de reemplazo por duplicado en español**

Se conservará CONS_0151 y se seleccionará un candidato
para reemplazar CONS_0158, cuya traducción es idéntica.

El candidato conservará la categoría PAYMENT, el conjunto
de desarrollo y la intención schedule_payments.

Se excluirán textos utilizados, registros retirados y consultas
con referencias detectadas a otros productos.

La traducción del candidato deberá revisarse para comprobar
que no genera otro duplicado en español.

**Estado:** Reemplazo candidato pendiente de revisión.

In [ ]:
# ============================================================
# 4.34 Selección de reemplazo por duplicado en español
# ============================================================

# Verificamos los dos registros y la traducción coincidente
par = df_muestra_revision[
    df_muestra_revision["id_consulta"].isin(
        ["CONS_0151", "CONS_0158"]
    )
].copy()

if (
    len(par) != 2
    or not par["id_consulta"].is_unique
    or not par["conjunto"].eq("desarrollo").all()
    or not par["categoria_original"].eq("PAYMENT").all()
    or par["consulta_es"].map(normalizar_contexto).eq("").any()
    or par["consulta_es"].map(normalizar_contexto).nunique() != 1
):
    raise ValueError("El par no coincide con el duplicado esperado.")

original = par[par["id_consulta"].eq("CONS_0158")].iloc[0]

# Excluimos consultas actuales y registros retirados
textos_utilizados = set(
    df_muestra_revision["consulta_original"].map(normalizar_contexto)
) | set(
    df_registros_retirados["consulta_original"].map(normalizar_contexto)
)

# Preparamos candidatos de la misma categoría e intención
reserva = df_elegibles.copy()
reserva["texto_revision"] = (
    reserva["instruction"].map(normalizar_contexto)
)

candidatos = reserva[
    reserva["category"].eq(original["categoria_original"])
    & reserva["intent"].eq(original["intencion_original"])
    & reserva["texto_revision"].ne("")
    & ~reserva["texto_revision"].isin(textos_utilizados)
    & ~reserva["instruction"]
        .map(detectar_referencias_producto).map(bool)
].sort_values("texto_revision").drop_duplicates("texto_revision")

if candidatos.empty:
    raise ValueError("No hay candidatos con la misma categoría e intención.")

# Seleccionamos una propuesta reproducible
candidato = candidatos.sample(
    n=1,
    random_state=742
).iloc[0]

# Conservamos la propuesta separada de la muestra principal
df_reemplazo_payment = pd.DataFrame([{
    "id_a_reemplazar": "CONS_0158",
    "conjunto": original["conjunto"],
    "indice_origen": candidato["indice_origen"],
    "consulta_original": candidato["instruction"],
    "categoria_original": candidato["category"],
    "intencion_original": candidato["intent"],
    "consulta_normalizada": candidato["consulta_normalizada"]
}])

with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazo_payment[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Candidato para reemplazar el duplicado presentado.")
print("=" * 60)

,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
0,CONS_0158,desarrollo,need to schedule the paayments,PAYMENT,schedule_payments


✅ Candidato para reemplazar el duplicado presentado.


In [ ]:
# ============================================================
# Diagnóstico de traducciones repetidas en desarrollo
# ============================================================

revision = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
].copy()

revision["texto_normalizado"] = (
    revision["consulta_es"].map(normalizar_contexto)
)

duplicados = revision[
    revision["texto_normalizado"].ne("")
    & revision["texto_normalizado"].duplicated(keep=False)
].sort_values(["texto_normalizado", "id_consulta"])

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        duplicados[
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia"
            ]
        ]
    )

print("=" * 60)
print("✅ Traducciones repetidas presentadas para revisión.")
print("=" * 60)

,id_consulta,consulta_original,consulta_es,categoria_referencia
100,CONS_0101,I'm trying to find a fucking insurance quote i need assistane,"Estoy intentando conseguir una cotización de seguro, necesito ayuda.",QUOTE
107,CONS_0108,"I am trying to find an insurance quote , I need help","Estoy intentando conseguir una cotización de seguro, necesito ayuda.",QUOTE
150,CONS_0151,i have to schedule the payments could yahelp me,"Tengo que programar los pagos, ¿puedes ayudarme?",PAYMENT
157,CONS_0158,i got to scehdule the payments could uhelp me,"Tengo que programar los pagos, ¿puedes ayudarme?",PAYMENT


✅ Traducciones repetidas presentadas para revisión.


### **🧠 4.34.1 Selección de reemplazo del duplicado de QUOTE**

La neutralización de expresiones ofensivas produjo traducciones
idénticas para CONS_0101 y CONS_0108.

Se conservará CONS_0101 y se seleccionará un candidato
para CONS_0108, manteniendo categoría, intención y conjunto.

El candidato se traducirá y revisará antes de incorporarse.

**Estado:** Reemplazo de QUOTE pendiente de revisión.

In [ ]:
# ============================================================
# 4.34.1 Selección de reemplazo del duplicado de QUOTE
# ============================================================

# Comprobamos el par de traducciones repetidas
par = df_muestra_revision[
    df_muestra_revision["id_consulta"].isin(
        ["CONS_0101", "CONS_0108"]
    )
].copy()

if (
    len(par) != 2
    or not par["id_consulta"].is_unique
    or not par["conjunto"].eq("desarrollo").all()
    or not par["categoria_original"].eq("QUOTE").all()
    or par["consulta_es"].map(normalizar_contexto).eq("").any()
    or par["consulta_es"].map(normalizar_contexto).nunique() != 1
):
    raise ValueError("El par no coincide con el duplicado esperado.")

original = par[par["id_consulta"].eq("CONS_0108")].iloc[0]

# Excluimos consultas actuales y registros retirados
textos_utilizados = set(
    df_muestra_revision["consulta_original"].map(normalizar_contexto)
) | set(
    df_registros_retirados["consulta_original"].map(normalizar_contexto)
)

# Buscamos candidatos con la misma categoría e intención
reserva = df_elegibles.copy()
reserva["texto_revision"] = (
    reserva["instruction"].map(normalizar_contexto)
)

candidatos = (
    reserva[
        reserva["category"].eq(original["categoria_original"])
        & reserva["intent"].eq(original["intencion_original"])
        & reserva["texto_revision"].ne("")
        & ~reserva["texto_revision"].isin(textos_utilizados)
        & ~reserva["instruction"]
            .map(detectar_referencias_producto).map(bool)
    ]
    .sort_values("texto_revision")
    .drop_duplicates("texto_revision")
)

if candidatos.empty:
    raise ValueError("No hay candidatos con la misma categoría e intención.")

# Seleccionamos una propuesta reproducible
candidato = candidatos.sample(
    n=1,
    random_state=942
).iloc[0]

df_reemplazo_quote_duplicado = pd.DataFrame([{
    "id_a_reemplazar": "CONS_0108",
    "conjunto": original["conjunto"],
    "indice_origen": candidato["indice_origen"],
    "consulta_original": candidato["instruction"],
    "categoria_original": candidato["category"],
    "intencion_original": candidato["intent"],
    "consulta_normalizada": candidato["consulta_normalizada"]
}])

with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazo_quote_duplicado[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Candidato de QUOTE presentado para revisión.")
print("=" * 60)

,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
0,CONS_0108,desarrollo,I'm looking for a life insurance quote i need help,QUOTE,calculate_insurance_quote


✅ Candidato de QUOTE presentado para revisión.


### **🧠 4.35 Incorporación de reemplazos y revisión de duplicados**

Se reemplazarán CONS_0108 y CONS_0158 por los candidatos
revisados, conservando categoría, intención y conjunto.

Los registros retirados se conservarán en la auditoría.
Se comprobarán las traducciones neutralizadas, los textos
originales y la distribución antes de confirmar los cambios.

**Estado:** Reemplazos con traducciones y etiquetas propuestas.
Validación final pendiente.

In [ ]:
# ============================================================
# 4.35 Incorporación de reemplazos y revisión de duplicados
# ============================================================

import pandas as pd

# Reunimos los candidatos seleccionados previamente
candidatos = pd.concat(
    [df_reemplazo_quote_duplicado, df_reemplazo_payment],
    ignore_index=True
)

traducciones_reemplazo = {
    "CONS_0108": "Busco una cotización de seguro de vida, necesito ayuda.",
    "CONS_0158": "Necesito programar los pagos."
}

ids_reemplazo = list(traducciones_reemplazo)

# Comprobamos las propuestas y evitamos repetir la incorporación
if (
    len(candidatos) != 2
    or not candidatos["id_a_reemplazar"].is_unique
    or set(candidatos["id_a_reemplazar"]) != set(ids_reemplazo)
):
    raise ValueError("Se esperaban los dos candidatos revisados.")

if df_registros_retirados["id_consulta"].isin(ids_reemplazo).any():
    raise ValueError("Alguno de estos reemplazos ya fue auditado.")

# Trabajamos sobre una copia
muestra_actualizada = df_muestra_revision.copy()

retirados = muestra_actualizada[
    muestra_actualizada["id_consulta"].isin(ids_reemplazo)
].copy()

if len(retirados) != 2 or not retirados["id_consulta"].is_unique:
    raise ValueError("No se encontraron dos posiciones únicas.")

# Comprobamos las traducciones frente a los registros conservados
textos_conservados = set(
    muestra_actualizada.loc[
        ~muestra_actualizada["id_consulta"].isin(ids_reemplazo),
        "consulta_es"
    ].map(normalizar_contexto)
)

for _, candidato in candidatos.iterrows():
    identificador = candidato["id_a_reemplazar"]

    original = retirados[
        retirados["id_consulta"].eq(identificador)
    ].iloc[0]

    if (
        original["conjunto"] != "desarrollo"
        or candidato["conjunto"] != "desarrollo"
        or original["categoria_original"] != candidato["categoria_original"]
        or original["intencion_original"] != candidato["intencion_original"]
        or not normalizar_contexto(candidato["consulta_original"])
        or detectar_referencias_producto(candidato["consulta_original"])
    ):
        raise ValueError(f"El candidato de {identificador} requiere revisión.")

    traduccion = traducciones_reemplazo[identificador]
    texto_es = normalizar_contexto(traduccion)

    if texto_es in textos_conservados:
        raise ValueError(f"La traducción de {identificador} está repetida.")

    textos_conservados.add(texto_es)
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    # Actualizamos los datos del nuevo registro
    for campo in [
        "indice_origen",
        "consulta_original",
        "categoria_original",
        "intencion_original",
        "consulta_normalizada"
    ]:
        muestra_actualizada.loc[mascara, campo] = candidato[campo]

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[
        mascara, "categoria_referencia"
    ] = candidato["categoria_original"]
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Reemplazo por traducción duplicada en español. "
        "Traducción neutralizada y etiqueta propuestas; "
        "validación pendiente."
    )

# Corregimos las dos expresiones que aún permanecían en desarrollo
correcciones_neutralizacion = {
    "CONS_0106": "Tengo que calcular una cotización de seguro, ¿me ayudas?",
    "CONS_0153": (
        "No puedo hacer una transferencia, "
        "¿cómo reporto los problemas de pago?"
    )
}

for identificador, traduccion in correcciones_neutralizacion.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    if mascara.sum() != 1:
        raise ValueError(f"No se encontró un único {identificador}.")

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion

# Actualizamos las observaciones de los registros conservados
for identificador in ["CONS_0101", "CONS_0151"]:
    muestra_actualizada.loc[
        muestra_actualizada["id_consulta"].eq(identificador),
        "observaciones"
    ] = (
        "Registro conservado frente a una traducción duplicada. "
        "Traducción neutralizada; validación pendiente."
    )

# Comprobamos originales únicos
originales = (
    muestra_actualizada["consulta_original"].map(normalizar_contexto)
)

if originales.eq("").any() or originales.duplicated().any():
    raise ValueError("Se detectaron originales vacíos o repetidos.")

# Comprobamos las 60 traducciones de desarrollo
desarrollo = muestra_actualizada[
    muestra_actualizada["conjunto"].eq("desarrollo")
].copy()

textos_es = desarrollo["consulta_es"].map(normalizar_contexto)
mascara_duplicados = textos_es.duplicated(keep=False)

if textos_es.eq("").any() or mascara_duplicados.any():
    with pd.option_context("display.max_colwidth", None):
        display(
            desarrollo.loc[
                textos_es.eq("") | mascara_duplicados,
                ["id_consulta", "consulta_es", "categoria_referencia"]
            ]
        )
    raise ValueError(
        "Quedan traducciones vacías o repetidas. "
        "La muestra principal permanece sin cambios."
    )

# Verificamos distribución y etiquetas
resumen = pd.crosstab(
    muestra_actualizada["categoria_original"],
    muestra_actualizada["conjunto"]
).reindex(
    index=categorias_candidatas,
    columns=["desarrollo", "prueba"],
    fill_value=0
)

if (
    len(muestra_actualizada) != 300
    or not muestra_actualizada["id_consulta"].is_unique
    or not resumen["desarrollo"].eq(10).all()
    or not resumen["prueba"].eq(40).all()
    or not desarrollo["categoria_referencia"]
        .isin(categorias_candidatas).all()
):
    raise ValueError("La distribución o las etiquetas requieren revisión.")

# Actualizamos las alertas de contexto
muestra_actualizada["otros_productos_detectados"] = (
    muestra_actualizada["consulta_original"]
    .map(lambda texto: ", ".join(detectar_otros_productos(texto)))
)
muestra_actualizada["alertas_producto"] = (
    muestra_actualizada["consulta_original"]
    .map(lambda texto: ", ".join(detectar_referencias_producto(texto)))
)
muestra_actualizada["requiere_revision_contexto"] = (
    muestra_actualizada["alertas_producto"].ne("")
)

# Conservamos los registros retirados
retirados["motivo_retiro"] = "Traducción duplicada en español"

auditoria_actualizada = pd.concat(
    [df_registros_retirados, retirados],
    ignore_index=True
)

# Confirmamos los cambios después de las comprobaciones
df_muestra_revision = muestra_actualizada.copy()
df_registros_retirados = auditoria_actualizada.copy()

df_alertas_contexto = df_muestra_revision[
    df_muestra_revision["requiere_revision_contexto"]
].copy()

with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(ids_reemplazo),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("Consultas de desarrollo con traducción:", len(desarrollo))
print("Traducciones repetidas en desarrollo:", int(textos_es.duplicated().sum()))
print("Registros retirados conservados:", len(df_registros_retirados))
print("Consultas con alertas actuales:", len(df_alertas_contexto))

print("=" * 60)
print("✅ Ambos reemplazos incorporados y duplicados comprobados.")
print("=" * 60)

,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
107,CONS_0108,I'm looking for a life insurance quote i need help,"Busco una cotización de seguro de vida, necesito ayuda.",QUOTE,propuesta_pendiente_revision
157,CONS_0158,need to schedule the paayments,Necesito programar los pagos.,PAYMENT,propuesta_pendiente_revision


Consultas de desarrollo con traducción: 60
Traducciones repetidas en desarrollo: 0
Registros retirados conservados: 21
Consultas con alertas actuales: 0
✅ Ambos reemplazos incorporados y duplicados comprobados.


### **🧠 4.36 Guardado del conjunto de desarrollo propuesto**

Se guardará una nueva versión de la muestra completa,
las 60 consultas de desarrollo traducidas, la auditoría
de registros retirados y las descripciones de categorías.

Las traducciones y etiquetas conservarán su estado de propuesta.
El conjunto de prueba permanecerá pendiente de traducción
y revisión.

**Estado:** Propuestas de desarrollo completas y respaldadas.
Validación final pendiente.

In [ ]:
# ============================================================
# 4.36 Guardado del conjunto de desarrollo propuesto
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
from google.colab import files

# Recuperamos el conjunto de desarrollo actualizado
df_desarrollo_propuesto = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
].copy()

# Comprobamos campos, etiquetas y duplicados antes de guardar
textos_es = df_desarrollo_propuesto["consulta_es"].map(
    normalizar_contexto
)

if (
    len(df_desarrollo_propuesto) != 60
    or not df_desarrollo_propuesto["id_consulta"].is_unique
    or textos_es.eq("").any()
    or textos_es.duplicated().any()
    or not df_desarrollo_propuesto["categoria_referencia"]
        .isin(descripciones_categorias_v2).all()
):
    raise ValueError("El desarrollo no cumple las condiciones de guardado.")

# Creamos una carpeta nueva sin sobrescribir versiones anteriores
marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_respaldo = (
    Path("/content/revision_evaluacion")
    / f"desarrollo_propuesto_{marca_version}"
)

carpeta_respaldo.mkdir(parents=True, exist_ok=False)

# Guardamos las tablas completas para mantener la trazabilidad
df_muestra_revision.to_csv(
    carpeta_respaldo / "muestra_completa.csv",
    index=False,
    encoding="utf-8-sig"
)

df_desarrollo_propuesto.to_csv(
    carpeta_respaldo / "desarrollo_propuesto.csv",
    index=False,
    encoding="utf-8-sig"
)

df_registros_retirados.to_csv(
    carpeta_respaldo / "registros_retirados.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardamos las descripciones actuales y el estado del avance
configuracion = {
    "descripciones_categorias": descripciones_categorias_v2,
    "plantilla_hipotesis": "Esta consulta trata sobre {}.",
    "estado_desarrollo": "propuestas pendientes de validación",
    "estado_prueba": "pendiente de traducción y revisión",
    "consultas_desarrollo": len(df_desarrollo_propuesto),
    "registros_retirados": len(df_registros_retirados)
}

with (carpeta_respaldo / "configuracion.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(
        configuracion,
        archivo,
        ensure_ascii=False,
        indent=2
    )

# Comprimimos los archivos para descargarlos juntos
ruta_respaldo_zip = shutil.make_archive(
    str(carpeta_respaldo),
    "zip",
    root_dir=str(carpeta_respaldo)
)

print("Consultas de desarrollo guardadas:", len(df_desarrollo_propuesto))
print("Registros retirados guardados:", len(df_registros_retirados))
print("Estado: propuestas pendientes de validación.")

print("=" * 60)
print("✅ Versión de desarrollo propuesta guardada.")
print("=" * 60)

files.download(ruta_respaldo_zip)

Consultas de desarrollo guardadas: 60
Registros retirados guardados: 21
Estado: propuestas pendientes de validación.
✅ Versión de desarrollo propuesta guardada.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **🧠 4.37 Revisión final de las propuestas de desarrollo**

Se presentarán las 60 consultas de desarrollo con su texto
original, traducción, intención y categoría de referencia propuesta.

Se comprobarán la fidelidad de las traducciones y la coherencia
de las etiquetas con los criterios definidos.

Tener campos completos y no presentar duplicados textuales
no implica que las consultas estén validadas.

**Estado:** Propuestas pendientes de revisión final.

In [ ]:
# ============================================================
# 4.37 Revisión final de las propuestas de desarrollo
# ============================================================

# Recuperamos las propuestas actuales sin modificar sus estados
df_revision_final = df_muestra_revision[
    df_muestra_revision["conjunto"].eq("desarrollo")
].copy().sort_values("id_consulta")

# Presentamos cada categoría por separado para facilitar la revisión
for categoria in categorias_candidatas:
    print("=" * 60)
    print("Categoría:", categoria)

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            df_revision_final[
                df_revision_final["categoria_original"].eq(categoria)
            ][
                [
                    "id_consulta",
                    "consulta_original",
                    "consulta_es",
                    "intencion_original",
                    "categoria_referencia"
                ]
            ]
        )

print("=" * 60)
print("✅ Propuestas de desarrollo presentadas para revisión final.")
print("=" * 60)

Categoría: LIFE_INSURANCE


,id_consulta,consulta_original,consulta_es,intencion_original,categoria_referencia
0,CONS_0001,i have to check my fucking life insurance can i get some help,"Tengo que revisar mi seguro de vida, ¿me pueden ayudar?",information_life_insurance,LIFE_INSURANCE
1,CONS_0002,I'd like information about my fucking life insuarnce where could i get it,"Quisiera información sobre mi seguro de vida, ¿dónde puedo conseguirla?",information_life_insurance,LIFE_INSURANCE
2,CONS_0003,"I got to see my fuckinglife insurance, can you help me?","Tengo que consultar mi seguro de vida, ¿puedes ayudarme?",information_life_insurance,LIFE_INSURANCE
3,CONS_0004,"I would like to check my life insurance, where can I do it?","Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",information_life_insurance,LIFE_INSURANCE
4,CONS_0005,how can I find information about my life insurance?,¿Cómo puedo encontrar información sobre mi seguro de vida?,information_life_insurance,LIFE_INSURANCE
5,CONS_0006,"I would like to know about my life insurance, how do I find more information?","Quisiera saber sobre mi seguro de vida, ¿cómo encuentro más información?",information_life_insurance,LIFE_INSURANCE
6,CONS_0007,need to see my life insurance where can i do it,"Necesito consultar mi seguro de vida, ¿dónde puedo hacerlo?",information_life_insurance,LIFE_INSURANCE
7,CONS_0008,"I'd like to knoe about my fucking life insurance, will you find some information?","Quisiera saber sobre mi seguro de vida, ¿me puedes conseguir información?",information_life_insurance,LIFE_INSURANCE
8,CONS_0009,wana check my life insurance where can i do it,"Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",information_life_insurance,LIFE_INSURANCE
9,CONS_0010,i got to se my fucking life insurance,Tengo que consultar mi seguro de vida.,information_life_insurance,LIFE_INSURANCE


Categoría: COVERAGE


,id_consulta,consulta_original,consulta_es,intencion_original,categoria_referencia
50,CONS_0051,"I need to check my fucking coverage , how can I do it?","Necesito revisar mi cobertura, ¿cómo puedo hacerlo?",check_coverage,COVERAGE
51,CONS_0052,"I'd like to know about my coverage, could you give me some information ?","Quisiera saber sobre mi cobertura, ¿podrías darme información?",check_coverage,COVERAGE
52,CONS_0053,where can i downgrade the fucking coverage of my insurance polikcy,¿Dónde puedo reducir la cobertura de mi póliza de seguro?,downgrade_coverage,COVERAGE
53,CONS_0054,i have to change the fucking coverage of my insurance policy could i get somw help,"Tengo que cambiar la cobertura de mi póliza de seguro, ¿me pueden ayudar?",change_coverage,COVERAGE
54,CONS_0055,i have to see the coverage of my life insurance policy where to do it,"Tengo que consultar la cobertura de mi póliza de seguro de vida, ¿dónde puedo hacerlo?",check_coverage,COVERAGE
55,CONS_0056,id like to upgrade the coverage of my life insurance policy will ya hlp me,"Quisiera ampliar la cobertura de mi póliza de seguro de vida, ¿me ayudan?",upgrade_coverage,COVERAGE
56,CONS_0057,I'm looking for information about the coverage of my insurance policy how could i get ig,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",check_coverage,COVERAGE
57,CONS_0058,can ya help me increase the coverage of my life insurance policy,¿Puedes ayudarme a aumentar la cobertura de mi póliza de seguro de vida?,upgrade_coverage,COVERAGE
58,CONS_0059,i need help reducing the fucking coverage of my life insurance policy,Necesito ayuda para reducir la cobertura de mi póliza de seguro de vida.,downgrade_coverage,COVERAGE
59,CONS_0060,can I modify the coverage of my insurance policy ?,¿Puedo modificar la cobertura de mi póliza de seguro?,change_coverage,COVERAGE


Categoría: QUOTE


...

Categoría: ENROLLMENT


,id_consulta,consulta_original,consulta_es,intencion_original,categoria_referencia
200,CONS_0201,"I have to compare fucking life insurance policies, can I get some help?","Tengo que comparar pólizas de seguro de vida, ¿me pueden ayudar?",compare_insurance_policies,ENROLLMENT
201,CONS_0202,i do not want my fucking life insurance policy to be renewed I'd like to cancel it,"No quiero que se renueve mi póliza de seguro de vida, quisiera cancelarla.",cancel_insurance_policy,ENROLLMENT
202,CONS_0203,"I have to cancel my fucking life insurance policy , where could I do it?","Tengo que cancelar mi póliza de seguro de vida, ¿dónde puedo hacerlo?",cancel_insurance_policy,ENROLLMENT
203,CONS_0204,"I don't want my fucking insurance policy to be renewed, I want to cancel it","No quiero que se renueve mi póliza de seguro, quiero cancelarla.",cancel_insurance_policy,ENROLLMENT
204,CONS_0205,wanna see the fuckinmg early termination fees where can i do it,"Quiero consultar los cargos por terminación anticipada, ¿dónde puedo hacerlo?",cancellation_fees,ENROLLMENT
205,CONS_0206,i want an fucking insurance policy will uhelp me buying it,"Quiero una póliza de seguro, ¿me ayudas a contratarla?",buy_insurance_policy,ENROLLMENT
206,CONS_0207,"I have to check the fucking cancellation charges, how to do it?","Tengo que consultar los cargos por cancelación, ¿cómo lo hago?",cancellation_fees,ENROLLMENT
207,CONS_0208,"I don't want to reenw my life insurance policy, I need assistance to cancel it","No quiero renovar mi póliza de seguro de vida, necesito ayuda para cancelarla.",cancel_insurance_policy,ENROLLMENT
208,CONS_0209,"I would like to see the cancellation charhes, I need help","Quisiera consultar los cargos por cancelación, necesito ayuda.",cancellation_fees,ENROLLMENT
209,CONS_0210,I'd like to know about the fucking cancellation charges find some information,"Quisiera saber sobre los cargos por cancelación, dame información.",cancellation_fees,ENROLLMENT


Categoría: CONTACT


,id_consulta,consulta_original,consulta_es,intencion_original,categoria_referencia
250,CONS_0251,id like to esnd an email to customer service where can i do it,"Quisiera enviar un correo a atención al cliente, ¿dónde puedo hacerlo?",customer_service,CONTACT
251,CONS_0252,"I would like to call client service, where could I do it?","Quisiera llamar a atención al cliente, ¿dónde puedo hacerlo?",customer_service,CONTACT
252,CONS_0253,would it be possible to speak with my fucking insurance representative,¿Sería posible hablar con mi representante de seguros?,insurance_representative,CONTACT
253,CONS_0254,"I got to talk to the imsurer, I need assistance","Tengo que hablar con la aseguradora, necesito ayuda.",agent,CONTACT
254,CONS_0255,how could I speak with the insurer?,¿Cómo puedo hablar con la aseguradora?,agent,CONTACT
255,CONS_0256,"I have to speak with the insurance company, could I get some help?","Tengo que hablar con la compañía de seguros, ¿me pueden ayudar?",agent,CONTACT
256,CONS_0257,how do i talk to somebdy,¿Cómo puedo hablar con alguien?,human_agent,CONTACT
257,CONS_0258,could uhelp me to contact fuckimg client service,¿Podrías ayudarme a contactar al servicio de atención al cliente?,customer_service,CONTACT
258,CONS_0259,"I want to talk to an person, how do I do it?","Quiero hablar con una persona, ¿cómo puedo hacerlo?",human_agent,CONTACT
259,CONS_0260,i do not understand ya i want assistance speaking with a fucking human agent,"No entiendo, quiero ayuda para hablar con una persona.",human_agent,CONTACT


✅ Propuestas de desarrollo presentadas para revisión final.


### **🧠 4.38 Resultado de la revisión semántica de desarrollo**

Se revisaron las 60 consultas de desarrollo comparando
el texto original, la traducción y la categoría propuesta.

Las etiquetas son coherentes con los criterios definidos.
Se precisará CONS_0260 para conservar la referencia
a un agente humano.

**Limitaciones observadas:**

- LIFE_INSURANCE contiene consultas muy similares entre sí.
- ENROLLMENT concentra consultas relacionadas con cancelación.
- Las traducciones corrigen errores ortográficos del original;
  la evaluación representará esta versión revisada en español.

**Estado:** Revisión del asistente completada.
Confirmación del usuario pendiente.

In [ ]:
# ============================================================
# 4.38 Resultado de la revisión semántica de desarrollo
# ============================================================

# Definimos las traducciones corregidas y sus categorías
correcciones = {
    "CONS_0057": {
        "categoria": "COVERAGE",
        "traduccion": (
            "Busco información sobre la cobertura de mi póliza "
            "de seguro, ¿cómo puedo obtenerla?"
        ),
        "observacion": (
            "Se elimina la referencia añadida a seguros de vida, "
            "porque el original no especifica el producto."
        )
    },
    "CONS_0260": {
        "categoria": "CONTACT",
        "traduccion": (
            "No entiendo, quiero ayuda para hablar con un agente humano."
        ),
        "observacion": (
            "Se conserva explícitamente la referencia a human agent "
            "y se neutraliza la expresión ofensiva."
        )
    }
}

# Trabajamos sobre una copia antes de confirmar los cambios
muestra_actualizada = df_muestra_revision.copy()

# Verificamos los registros y corregimos sus traducciones
for identificador, correccion in correcciones.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)
    registro = muestra_actualizada.loc[mascara]

    if (
        len(registro) != 1
        or not registro["conjunto"].eq("desarrollo").all()
        or not registro["categoria_referencia"]
            .eq(correccion["categoria"]).all()
    ):
        raise ValueError(
            f"{identificador} no coincide con el registro esperado."
        )

    muestra_actualizada.loc[
        mascara, "consulta_es"
    ] = correccion["traduccion"]

    muestra_actualizada.loc[mascara, "observaciones"] = (
        correccion["observacion"]
        + " Revisión semántica del asistente completada; "
        "confirmación del usuario pendiente."
    )

# Comprobamos las traducciones de desarrollo después de corregir
mascara_desarrollo = muestra_actualizada["conjunto"].eq("desarrollo")
desarrollo = muestra_actualizada.loc[mascara_desarrollo].copy()

textos_es = desarrollo["consulta_es"].map(normalizar_contexto)

if (
    len(desarrollo) != 60
    or not desarrollo["id_consulta"].is_unique
    or textos_es.eq("").any()
    or textos_es.duplicated().any()
):
    raise ValueError(
        "El desarrollo contiene registros faltantes, "
        "traducciones vacías o repetidas. No se confirmaron los cambios."
    )

# Registramos la revisión sin cambiar el estado de validación
muestra_actualizada.loc[
    mascara_desarrollo, "revision_asistente"
] = "completada"

# Confirmamos los cambios después de las comprobaciones
df_muestra_revision = muestra_actualizada.copy()

# Mostramos las dos traducciones corregidas
with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(correcciones),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

print("Consultas de desarrollo revisadas:", len(desarrollo))
print("Traducciones repetidas:", int(textos_es.duplicated().sum()))

print("=" * 60)
print("✅ Revisión semántica registrada y traducciones precisadas.")
print("=" * 60)

,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
56,CONS_0057,I'm looking for information about the coverage of my insurance policy how could i get ig,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",COVERAGE,propuesta_pendiente_revision
259,CONS_0260,i do not understand ya i want assistance speaking with a fucking human agent,"No entiendo, quiero ayuda para hablar con un agente humano.",CONTACT,propuesta_pendiente_revision


Consultas de desarrollo revisadas: 60
Traducciones repetidas: 0
✅ Revisión semántica registrada y traducciones precisadas.


### **🧠 4.39 Validación del conjunto de desarrollo y respaldo**

El usuario confirmó las traducciones y etiquetas de las
60 consultas de desarrollo después de la revisión.

Se registrará su validación y se guardará una nueva versión
de la muestra, el desarrollo validado, la auditoría
y las descripciones de categorías.

Esta validación corresponde al proyecto y no constituye
una revisión independiente por especialistas.

**Estado:** Desarrollo validado para la comparación exploratoria.
Conjunto de prueba pendiente de traducción y validación.

In [ ]:
# ============================================================
# 4.39 Validación del conjunto de desarrollo y respaldo
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
from google.colab import files

# Recuperamos las 60 consultas revisadas
mascara_desarrollo = df_muestra_revision["conjunto"].eq("desarrollo")
desarrollo = df_muestra_revision.loc[mascara_desarrollo].copy()

textos_es = desarrollo["consulta_es"].map(normalizar_contexto)

# Comprobamos las condiciones antes de registrar la confirmación
if (
    len(desarrollo) != 60
    or not desarrollo["id_consulta"].is_unique
    or textos_es.eq("").any()
    or textos_es.duplicated().any()
    or not desarrollo["categoria_referencia"]
        .isin(descripciones_categorias_v2).all()
    or not desarrollo["revision_asistente"].eq("completada").all()
):
    raise ValueError("El desarrollo requiere revisión antes de validarse.")

# Registramos la confirmación explícita del usuario
fecha_validacion = datetime.now(timezone.utc).isoformat()

muestra_validada = df_muestra_revision.copy()
muestra_validada.loc[
    mascara_desarrollo, "estado_revision"
] = "validado_usuario"
muestra_validada.loc[
    mascara_desarrollo, "validado_por"
] = "usuario"
muestra_validada.loc[
    mascara_desarrollo, "fecha_validacion_utc"
] = fecha_validacion

# Conservamos las observaciones anteriores como historial
muestra_validada.loc[
    mascara_desarrollo, "observaciones"
] = (
    muestra_validada.loc[mascara_desarrollo, "observaciones"]
    .fillna("")
    .astype(str)
    + " | Confirmación explícita del usuario registrada."
)

df_desarrollo_validado = muestra_validada.loc[
    mascara_desarrollo
].copy()

# Creamos una nueva carpeta sin sobrescribir respaldos
marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_validada = (
    Path("/content/revision_evaluacion")
    / f"desarrollo_validado_{marca_version}"
)
carpeta_validada.mkdir(parents=True, exist_ok=False)

# Guardamos los datos y la auditoría
muestra_validada.to_csv(
    carpeta_validada / "muestra_completa.csv",
    index=False,
    encoding="utf-8-sig"
)
df_desarrollo_validado.to_csv(
    carpeta_validada / "desarrollo_validado.csv",
    index=False,
    encoding="utf-8-sig"
)
df_registros_retirados.to_csv(
    carpeta_validada / "registros_retirados.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardamos la configuración que compararemos sobre desarrollo
configuracion = {
    "descripciones_categorias": descripciones_categorias_v2,
    "plantilla_hipotesis": "Esta consulta trata sobre {}.",
    "multi_label": False,
    "estado_desarrollo": "validado_usuario",
    "estado_prueba": "pendiente de traducción y validación",
    "fecha_validacion_utc": fecha_validacion
}

with (carpeta_validada / "configuracion.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(
        configuracion,
        archivo,
        ensure_ascii=False,
        indent=2
    )

# Confirmamos la actualización después de guardar los archivos
df_muestra_revision = muestra_validada.copy()

ruta_zip_validado = shutil.make_archive(
    str(carpeta_validada),
    "zip",
    root_dir=str(carpeta_validada)
)

print("Consultas de desarrollo validadas:", len(df_desarrollo_validado))
print("Registros retirados conservados:", len(df_registros_retirados))

print("=" * 60)
print("✅ Validación del usuario registrada y respaldo guardado.")
print("=" * 60)

files.download(ruta_zip_validado)

Consultas de desarrollo validadas: 60
Registros retirados conservados: 21
✅ Validación del usuario registrada y respaldo guardado.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **💻 5. Comparación de modelos sobre desarrollo**

Se compararán los tres modelos utilizando las mismas
60 consultas de desarrollo validadas.

Se calcularán exactitud, precisión macro, exhaustividad macro
y F1 macro. También se medirán tiempos de inferencia después
de un calentamiento y mediante ejecuciones repetidas.

Los resultados permitirán revisar la configuración.
La evaluación final se realizará posteriormente sobre
el conjunto de prueba, pendiente de preparación.

### **🧠 5.1 Preparación de datos y configuración común**

Se fijarán el orden de las consultas, las descripciones
de categorías y la plantilla utilizada por los tres modelos.

Se utilizará una categoría por consulta, precisión de 32 bits
y tamaño de lote igual a uno. Cada modelo se cargará
por separado para reducir el consumo simultáneo de memoria.

**Estado:** Configuración preparada para comparar sobre desarrollo.

In [ ]:
# ============================================================
# 5.1 Preparación de datos y configuración común
# ============================================================

import copy
import torch

# Recuperamos los datos actuales y fijamos su orden
df_comparacion_desarrollo = (
    df_muestra_revision[
        df_muestra_revision["conjunto"].eq("desarrollo")
    ]
    .sort_values("id_consulta")
    .copy()
)

# Verificamos validación, textos y etiquetas antes de evaluar
textos_normalizados = (
    df_comparacion_desarrollo["consulta_es"].map(normalizar_contexto)
)

if (
    len(df_comparacion_desarrollo) != 60
    or not df_comparacion_desarrollo["id_consulta"].is_unique
    or not df_comparacion_desarrollo["estado_revision"]
        .eq("validado_usuario").all()
    or textos_normalizados.eq("").any()
    or textos_normalizados.duplicated().any()
    or not df_comparacion_desarrollo["categoria_referencia"]
        .isin(descripciones_categorias_v2).all()
):
    raise ValueError("Los datos no cumplen las condiciones de comparación.")

# Copiamos las descripciones para conservar esta configuración
configuracion_comparacion = {
    "descripciones": copy.deepcopy(descripciones_categorias_v2),
    "plantilla": "Esta consulta trata sobre {}.",
    "multi_label": False,
    "batch_size": 1,
    "repeticiones": 3,
    "precision": "float32"
}

# Preparamos los textos y etiquetas en el mismo orden
consultas_desarrollo = (
    df_comparacion_desarrollo["consulta_es"].tolist()
)
etiquetas_desarrollo = (
    df_comparacion_desarrollo["categoria_referencia"].tolist()
)

# Registramos los modelos con sus identificadores completos
modelos_comparacion = {
    "MiniLMv2 multilingüe": (
        "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
    ),
    "mDeBERTa-v3-base": (
        "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
    ),
    "XLM-RoBERTa-large": (
        "joeddav/xlm-roberta-large-xnli"
    )
}

dispositivo_comparacion = 0 if torch.cuda.is_available() else -1

# Mostramos la configuración y distribución de referencia
print("Consultas:", len(consultas_desarrollo))
print(
    "Dispositivo:",
    torch.cuda.get_device_name(0)
    if dispositivo_comparacion == 0
    else "CPU"
)
print("Tamaño de lote:", configuracion_comparacion["batch_size"])
print("Repeticiones:", configuracion_comparacion["repeticiones"])

display(
    df_comparacion_desarrollo["categoria_referencia"]
    .value_counts()
    .reindex(configuracion_comparacion["descripciones"], fill_value=0)
    .rename_axis("Categoría")
    .reset_index(name="Consultas")
)

print("=" * 60)
print("✅ Datos y configuración común preparados.")
print("=" * 60)

Consultas: 60
Dispositivo: Tesla T4
Tamaño de lote: 1
Repeticiones: 3


,Categoría,Consultas
0,LIFE_INSURANCE,10
1,COVERAGE,10
2,QUOTE,10
3,PAYMENT,10
4,ENROLLMENT,10
5,CONTACT,10


✅ Datos y configuración común preparados.


### **🧠 5.2 Evaluación de MiniLMv2 sobre desarrollo**

Se evaluará MiniLMv2 utilizando las 60 consultas validadas
y la configuración común.

Se realizará un calentamiento antes de medir tres ejecuciones
completas del conjunto.

Se calcularán exactitud, precisión macro, exhaustividad macro
y F1 macro. Se presentarán métricas por categoría
y los casos clasificados incorrectamente.

Los tiempos comprenderán el procesamiento completo del pipeline,
incluida la preparación de textos y la inferencia.

**Estado:** Evaluación exploratoria sobre desarrollo.
Los resultados no corresponden al conjunto de prueba.

In [ ]:
# ============================================================
# 5.2 Evaluación de MiniLMv2 sobre desarrollo
# ============================================================

import gc
import numpy as np
import pandas as pd
import torch

from time import perf_counter
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# Liberamos cualquier modelo cargado en las pruebas anteriores
for nombre_variable in ["clasificador", "clasificador_comparacion"]:
    modelo_anterior = globals().pop(nombre_variable, None)
    if modelo_anterior is not None:
        del modelo_anterior

gc.collect()

if dispositivo_comparacion == 0:
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# Preparamos las descripciones y su correspondencia con categorías
descripciones = configuracion_comparacion["descripciones"]
categorias = list(descripciones)
categoria_por_descripcion = {
    descripcion: categoria
    for categoria, descripcion in descripciones.items()
}

nombre_modelo = "MiniLMv2 multilingüe"

# Medimos la carga por separado de la evaluación
inicio_carga = perf_counter()

clasificador_comparacion = pipeline(
    task="zero-shot-classification",
    model=modelos_comparacion[nombre_modelo],
    device=dispositivo_comparacion,
    torch_dtype=torch.float32
)

if dispositivo_comparacion == 0:
    torch.cuda.synchronize()

tiempo_carga = perf_counter() - inicio_carga

# Definimos los parámetros comunes para todas las ejecuciones
parametros = {
    "candidate_labels": list(descripciones.values()),
    "hypothesis_template": configuracion_comparacion["plantilla"],
    "multi_label": configuracion_comparacion["multi_label"],
    "batch_size": configuracion_comparacion["batch_size"]
}

# Realizamos un calentamiento sin incluirlo en la medición
_ = clasificador_comparacion(
    consultas_desarrollo[:2],
    **parametros
)

if dispositivo_comparacion == 0:
    torch.cuda.synchronize()

# Medimos tres ejecuciones completas sobre las mismas consultas
tiempos_minilm = []
etiquetas_por_repeticion = []

for repeticion in range(configuracion_comparacion["repeticiones"]):
    if dispositivo_comparacion == 0:
        torch.cuda.synchronize()

    inicio = perf_counter()

    salidas = clasificador_comparacion(
        consultas_desarrollo,
        **parametros
    )

    if dispositivo_comparacion == 0:
        torch.cuda.synchronize()

    duracion = perf_counter() - inicio
    tiempos_minilm.append(duracion)

    etiquetas_predichas = [
        categoria_por_descripcion[salida["labels"][0]]
        for salida in salidas
    ]
    etiquetas_por_repeticion.append(etiquetas_predichas)

    # Conservamos las predicciones de la primera ejecución
    if repeticion == 0:
        predicciones_minilm = etiquetas_predichas.copy()
        puntuaciones_minilm = [
            salida["scores"][0] for salida in salidas
        ]

    print(
        f"Repetición {repeticion + 1}: "
        f"{duracion:.2f} segundos"
    )

# Comprobamos si las etiquetas coinciden entre ejecuciones
predicciones_estables = all(
    etiquetas == predicciones_minilm
    for etiquetas in etiquetas_por_repeticion
)

# Calculamos las métricas macro sobre las seis categorías
precision, exhaustividad, f1, _ = precision_recall_fscore_support(
    etiquetas_desarrollo,
    predicciones_minilm,
    labels=categorias,
    average="macro",
    zero_division=0
)

metricas_minilm = {
    "Modelo": nombre_modelo,
    "Exactitud": accuracy_score(
        etiquetas_desarrollo, predicciones_minilm
    ),
    "Precisión macro": precision,
    "Exhaustividad macro": exhaustividad,
    "F1 macro": f1,
    "Tiempo medio total (s)": float(np.mean(tiempos_minilm)),
    "Desviación de tiempo (s)": float(
        np.std(tiempos_minilm, ddof=1)
    ),
    "Tiempo medio por consulta (s)": float(
        np.mean(tiempos_minilm) / len(consultas_desarrollo)
    )
}

display(pd.DataFrame([metricas_minilm]).round(4))

# Calculamos métricas individuales por categoría
p_clase, r_clase, f1_clase, soporte = (
    precision_recall_fscore_support(
        etiquetas_desarrollo,
        predicciones_minilm,
        labels=categorias,
        average=None,
        zero_division=0
    )
)

tabla_metricas_minilm = pd.DataFrame({
    "Categoría": categorias,
    "Precisión": p_clase,
    "Exhaustividad": r_clase,
    "F1": f1_clase,
    "Consultas": soporte
})

display(tabla_metricas_minilm.round(4))

# Conservamos cada predicción vinculada con su consulta
df_resultados_minilm = df_comparacion_desarrollo[
    ["id_consulta", "consulta_es", "categoria_referencia"]
].copy()

df_resultados_minilm["categoria_predicha"] = predicciones_minilm
df_resultados_minilm["puntuacion"] = puntuaciones_minilm
df_resultados_minilm["acierto"] = (
    df_resultados_minilm["categoria_referencia"]
    .eq(df_resultados_minilm["categoria_predicha"])
)

# Mostramos los errores para interpretarlos
df_errores_minilm = df_resultados_minilm[
    ~df_resultados_minilm["acierto"]
].copy()

print(f"\nTiempo de carga: {tiempo_carga:.2f} segundos")
print(
    "Aciertos:",
    int(df_resultados_minilm["acierto"].sum()),
    "de",
    len(df_resultados_minilm)
)
print("Etiquetas iguales entre repeticiones:", predicciones_estables)

if not df_errores_minilm.empty:
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(df_errores_minilm)
else:
    print("No se observaron errores en desarrollo.")

print("=" * 60)
print("✅ Evaluación de MiniLMv2 sobre desarrollo completada.")
print("=" * 60)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Repetición 1: 9.09 segundos
Repetición 2: 7.75 segundos
Repetición 3: 8.12 segundos


,Modelo,Exactitud,Precisión macro,Exhaustividad macro,F1 macro,Tiempo medio total (s),Desviación de tiempo (s),Tiempo medio por consulta (s)
0,MiniLMv2 multilingüe,0.3833,0.324,0.3833,0.3437,8.3177,0.6939,0.1386


,Categoría,Precisión,Exhaustividad,F1,Consultas
0,LIFE_INSURANCE,0.0000,0.0,0.0000,10
1,COVERAGE,0.3333,0.4,0.3636,10
2,QUOTE,0.3333,0.2,0.2500,10
3,PAYMENT,0.0909,0.1,0.0952,10
4,ENROLLMENT,0.3529,0.6,0.4444,10
5,CONTACT,0.8333,1.0,0.9091,10



Tiempo de carga: 6.62 segundos
Aciertos: 23 de 60
Etiquetas iguales entre repeticiones: True


,id_consulta,consulta_es,categoria_referencia,categoria_predicha,puntuacion,acierto
0,CONS_0001,"Tengo que revisar mi seguro de vida, ¿me pueden ayudar?",LIFE_INSURANCE,PAYMENT,0.280317,False
1,CONS_0002,"Quisiera información sobre mi seguro de vida, ¿dónde puedo conseguirla?",LIFE_INSURANCE,PAYMENT,0.270239,False
2,CONS_0003,"Tengo que consultar mi seguro de vida, ¿puedes ayudarme?",LIFE_INSURANCE,COVERAGE,0.272416,False
3,CONS_0004,"Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,PAYMENT,0.314554,False
4,CONS_0005,¿Cómo puedo encontrar información sobre mi seguro de vida?,LIFE_INSURANCE,CONTACT,0.240769,False
5,CONS_0006,"Quisiera saber sobre mi seguro de vida, ¿cómo encuentro más información?",LIFE_INSURANCE,PAYMENT,0.272890,False
6,CONS_0007,"Necesito consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,COVERAGE,0.313442,False
7,CONS_0008,"Quisiera saber sobre mi seguro de vida, ¿me puedes conseguir información?",LIFE_INSURANCE,PAYMENT,0.292131,False
8,CONS_0009,"Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,PAYMENT,0.296170,False
9,CONS_0010,Tengo que consultar mi seguro de vida.,LIFE_INSURANCE,COVERAGE,0.349212,False


✅ Evaluación de MiniLMv2 sobre desarrollo completada.


### **📝 Interpretación de MiniLMv2 sobre desarrollo**

MiniLMv2 obtuvo 23 aciertos en 60 consultas traducidas
y neutralizadas, con exactitud de 38.33%, precisión macro
de 0.3240 y F1 macro de 0.3437.

Los aciertos por categoría fueron:

- LIFE_INSURANCE: 0 de 10.
- COVERAGE: 4 de 10.
- QUOTE: 2 de 10.
- PAYMENT: 1 de 10.
- ENROLLMENT: 6 de 10.
- CONTACT: 10 de 10.

El modelo no identificó correctamente las consultas
de LIFE_INSURANCE. Además, ocho consultas de PAYMENT
fueron clasificadas como ENROLLMENT.

El tiempo medio fue de 8.3177 segundos para las 60 consultas,
con desviación de 0.6939 segundos y promedio de 0.1386
segundos por consulta. Las etiquetas coincidieron
en las tres repeticiones.

**Limitación:** Los resultados corresponden a desarrollo.
La reconstrucción incluyó neutralización, correcciones
y reemplazos; los cambios frente al experimento anterior
no pueden atribuirse únicamente a quitar expresiones ofensivas.

**Resultados disponibles en memoria:**

- metricas_minilm: métricas generales.
- tabla_metricas_minilm: métricas por categoría.
- df_resultados_minilm: predicciones por consulta.
- df_errores_minilm: consultas clasificadas incorrectamente.
- tiempos_minilm: duración de cada repetición.

Estos resultados se guardarán en el respaldo de la comparación.

**Estado:** MiniLMv2 evaluado sobre desarrollo.
Comparación con los otros dos modelos pendiente.

### **🧠 5.3 Evaluación de mDeBERTa sobre desarrollo**

Se evaluará mDeBERTa sobre las mismas 60 consultas validadas
y con la configuración utilizada para MiniLMv2.

Se realizará un calentamiento y se medirán tres ejecuciones.
Las métricas se calcularán con las predicciones de la primera
repetición, comprobando su coincidencia con las otras dos.

Se mostrarán métricas generales, resultados por categoría
y consultas clasificadas incorrectamente.

**Estado:** Evaluación exploratoria sobre desarrollo.

In [ ]:
# ============================================================
# 5.3 Evaluación de mDeBERTa sobre desarrollo
# ============================================================

import gc
import numpy as np
import pandas as pd
import torch

from time import perf_counter
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# Liberamos el modelo anterior
for nombre_variable in ["clasificador", "clasificador_comparacion"]:
    modelo_anterior = globals().pop(nombre_variable, None)
    if modelo_anterior is not None:
        del modelo_anterior

gc.collect()

if dispositivo_comparacion == 0:
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# Recuperamos la configuración común sin modificarla
descripciones = configuracion_comparacion["descripciones"]
categorias = list(descripciones)

categoria_por_descripcion = {
    descripcion: categoria
    for categoria, descripcion in descripciones.items()
}

nombre_modelo = "mDeBERTa-v3-base"

# Medimos la carga por separado
inicio_carga = perf_counter()

clasificador_comparacion = pipeline(
    task="zero-shot-classification",
    model=modelos_comparacion[nombre_modelo],
    device=dispositivo_comparacion,
    torch_dtype=torch.float32
)

if dispositivo_comparacion == 0:
    torch.cuda.synchronize()

tiempo_carga_mdeberta = perf_counter() - inicio_carga

parametros = {
    "candidate_labels": list(descripciones.values()),
    "hypothesis_template": configuracion_comparacion["plantilla"],
    "multi_label": configuracion_comparacion["multi_label"],
    "batch_size": configuracion_comparacion["batch_size"]
}

# Realizamos el mismo calentamiento que con MiniLMv2
_ = clasificador_comparacion(
    consultas_desarrollo[:2],
    **parametros
)

if dispositivo_comparacion == 0:
    torch.cuda.synchronize()

# Medimos tres ejecuciones completas
tiempos_mdeberta = []
etiquetas_por_repeticion = []

for repeticion in range(configuracion_comparacion["repeticiones"]):
    if dispositivo_comparacion == 0:
        torch.cuda.synchronize()

    inicio = perf_counter()

    salidas = clasificador_comparacion(
        consultas_desarrollo,
        **parametros
    )

    if dispositivo_comparacion == 0:
        torch.cuda.synchronize()

    duracion = perf_counter() - inicio
    tiempos_mdeberta.append(duracion)

    etiquetas_predichas = [
        categoria_por_descripcion[salida["labels"][0]]
        for salida in salidas
    ]
    etiquetas_por_repeticion.append(etiquetas_predichas)

    if repeticion == 0:
        predicciones_mdeberta = etiquetas_predichas.copy()
        puntuaciones_mdeberta = [
            salida["scores"][0] for salida in salidas
        ]

    print(
        f"Repetición {repeticion + 1}: "
        f"{duracion:.2f} segundos"
    )

predicciones_estables = all(
    etiquetas == predicciones_mdeberta
    for etiquetas in etiquetas_por_repeticion
)

# Calculamos métricas generales
precision, exhaustividad, f1, _ = precision_recall_fscore_support(
    etiquetas_desarrollo,
    predicciones_mdeberta,
    labels=categorias,
    average="macro",
    zero_division=0
)

metricas_mdeberta = {
    "Modelo": nombre_modelo,
    "Exactitud": accuracy_score(
        etiquetas_desarrollo, predicciones_mdeberta
    ),
    "Precisión macro": precision,
    "Exhaustividad macro": exhaustividad,
    "F1 macro": f1,
    "Tiempo medio total (s)": float(np.mean(tiempos_mdeberta)),
    "Desviación de tiempo (s)": float(
        np.std(tiempos_mdeberta, ddof=1)
    ),
    "Tiempo medio por consulta (s)": float(
        np.mean(tiempos_mdeberta) / len(consultas_desarrollo)
    )
}

display(pd.DataFrame([metricas_mdeberta]).round(4))

# Calculamos métricas por categoría
p_clase, r_clase, f1_clase, soporte = (
    precision_recall_fscore_support(
        etiquetas_desarrollo,
        predicciones_mdeberta,
        labels=categorias,
        average=None,
        zero_division=0
    )
)

tabla_metricas_mdeberta = pd.DataFrame({
    "Categoría": categorias,
    "Precisión": p_clase,
    "Exhaustividad": r_clase,
    "F1": f1_clase,
    "Consultas": soporte
})

display(tabla_metricas_mdeberta.round(4))

# Conservamos los resultados vinculados con sus identificadores
df_resultados_mdeberta = df_comparacion_desarrollo[
    ["id_consulta", "consulta_es", "categoria_referencia"]
].copy()

df_resultados_mdeberta["categoria_predicha"] = predicciones_mdeberta
df_resultados_mdeberta["puntuacion"] = puntuaciones_mdeberta
df_resultados_mdeberta["acierto"] = (
    df_resultados_mdeberta["categoria_referencia"]
    .eq(df_resultados_mdeberta["categoria_predicha"])
)

df_errores_mdeberta = df_resultados_mdeberta[
    ~df_resultados_mdeberta["acierto"]
].copy()

print(f"\nTiempo de carga: {tiempo_carga_mdeberta:.2f} segundos")
print(
    "Aciertos:",
    int(df_resultados_mdeberta["acierto"].sum()),
    "de",
    len(df_resultados_mdeberta)
)
print("Etiquetas iguales entre repeticiones:", predicciones_estables)

if not df_errores_mdeberta.empty:
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(df_errores_mdeberta)
else:
    print("No se observaron errores en desarrollo.")

print("=" * 60)
print("✅ Evaluación de mDeBERTa sobre desarrollo completada.")
print("=" * 60)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Repetición 1: 17.77 segundos
Repetición 2: 10.17 segundos
Repetición 3: 9.80 segundos


,Modelo,Exactitud,Precisión macro,Exhaustividad macro,F1 macro,Tiempo medio total (s),Desviación de tiempo (s),Tiempo medio por consulta (s)
0,mDeBERTa-v3-base,0.2333,0.3522,0.2333,0.1944,12.5794,4.4972,0.2097


,Categoría,Precisión,Exhaustividad,F1,Consultas
0,LIFE_INSURANCE,0.0000,0.0,0.0000,10
1,COVERAGE,1.0000,0.2,0.3333,10
2,QUOTE,0.7500,0.3,0.4286,10
3,PAYMENT,0.2000,0.1,0.1333,10
4,ENROLLMENT,0.0000,0.0,0.0000,10
5,CONTACT,0.1633,0.8,0.2712,10



Tiempo de carga: 12.15 segundos
Aciertos: 14 de 60
Etiquetas iguales entre repeticiones: True


,id_consulta,consulta_es,categoria_referencia,categoria_predicha,puntuacion,acierto
0,CONS_0001,"Tengo que revisar mi seguro de vida, ¿me pueden ayudar?",LIFE_INSURANCE,CONTACT,0.981366,False
1,CONS_0002,"Quisiera información sobre mi seguro de vida, ¿dónde puedo conseguirla?",LIFE_INSURANCE,CONTACT,0.967272,False
2,CONS_0003,"Tengo que consultar mi seguro de vida, ¿puedes ayudarme?",LIFE_INSURANCE,CONTACT,0.990754,False
3,CONS_0004,"Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,CONTACT,0.977961,False
4,CONS_0005,¿Cómo puedo encontrar información sobre mi seguro de vida?,LIFE_INSURANCE,CONTACT,0.921393,False
5,CONS_0006,"Quisiera saber sobre mi seguro de vida, ¿cómo encuentro más información?",LIFE_INSURANCE,CONTACT,0.864493,False
6,CONS_0007,"Necesito consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,CONTACT,0.979355,False
7,CONS_0008,"Quisiera saber sobre mi seguro de vida, ¿me puedes conseguir información?",LIFE_INSURANCE,CONTACT,0.937374,False
8,CONS_0009,"Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,CONTACT,0.971779,False
9,CONS_0010,Tengo que consultar mi seguro de vida.,LIFE_INSURANCE,CONTACT,0.992857,False


✅ Evaluación de mDeBERTa sobre desarrollo completada.


### **📝 Interpretación de mDeBERTa sobre desarrollo**

mDeBERTa obtuvo 14 aciertos en 60 consultas traducidas
y neutralizadas, con exactitud de 23.33%, precisión macro
de 0.3522 y F1 macro de 0.1944.

Los aciertos por categoría fueron:

- LIFE_INSURANCE: 0 de 10.
- COVERAGE: 2 de 10.
- QUOTE: 3 de 10.
- PAYMENT: 1 de 10.
- ENROLLMENT: 0 de 10.
- CONTACT: 8 de 10.

El modelo asignó CONTACT a 49 consultas, con ocho
asignaciones correctas. No acertó consultas de LIFE_INSURANCE
ni de ENROLLMENT.

El solapamiento entre la descripción de CONTACT y las
solicitudes de ayuda es una posible explicación que deberá
contrastarse mediante el experimento de descripciones alternativas.

El tiempo medio fue de 12.5794 segundos para las 60 consultas,
con desviación de 4.4972 segundos.

Las repeticiones duraron aproximadamente 17.77, 10.17
y 9.80 segundos. La variación limita la precisión de
la comparación temporal. Las etiquetas predichas
coincidieron en las tres ejecuciones.

**Comparación actual:**

Con la configuración inicial y este conjunto de desarrollo,
mDeBERTa obtuvo menor exactitud y F1 macro que MiniLMv2.

**Resultados disponibles en memoria:**

- metricas_mdeberta: métricas generales.
- tabla_metricas_mdeberta: métricas por categoría.
- df_resultados_mdeberta: predicciones por consulta.
- df_errores_mdeberta: consultas clasificadas incorrectamente.
- tiempos_mdeberta: duración de cada repetición.

Estos resultados se guardarán en el respaldo de la comparación.

**Limitación:** Los resultados corresponden a desarrollo,
no a la evaluación final sobre prueba.

**Estado:** mDeBERTa evaluado sobre desarrollo.
Evaluación de XLM-RoBERTa pendiente.

### **🧠 5.4 Evaluación de XLM-RoBERTa sobre desarrollo**

Se evaluará XLM-RoBERTa-large sobre las mismas 60 consultas
utilizadas con MiniLMv2 y mDeBERTa.

Se conservarán las descripciones, la plantilla, la precisión
de 32 bits y el tamaño de lote igual a uno.

Después del calentamiento se medirán tres ejecuciones.
Se calcularán métricas generales y por categoría,
y se mostrarán las consultas clasificadas incorrectamente.

**Estado:** Evaluación exploratoria sobre desarrollo.

In [ ]:
# ============================================================
# 5.4 Evaluación de XLM-RoBERTa sobre desarrollo
# ============================================================

import gc
import numpy as np
import pandas as pd
import torch

from time import perf_counter
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# Liberamos el modelo anterior
for nombre_variable in ["clasificador", "clasificador_comparacion"]:
    modelo_anterior = globals().pop(nombre_variable, None)
    if modelo_anterior is not None:
        del modelo_anterior

gc.collect()

if dispositivo_comparacion == 0:
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# Recuperamos la configuración común sin modificarla
descripciones = configuracion_comparacion["descripciones"]
categorias = list(descripciones)

categoria_por_descripcion = {
    descripcion: categoria
    for categoria, descripcion in descripciones.items()
}

nombre_modelo = "XLM-RoBERTa-large"

# Medimos la carga por separado
inicio_carga = perf_counter()

clasificador_comparacion = pipeline(
    task="zero-shot-classification",
    model=modelos_comparacion[nombre_modelo],
    device=dispositivo_comparacion,
    torch_dtype=torch.float32
)

if dispositivo_comparacion == 0:
    torch.cuda.synchronize()

tiempo_carga_xlm_roberta = perf_counter() - inicio_carga

parametros = {
    "candidate_labels": list(descripciones.values()),
    "hypothesis_template": configuracion_comparacion["plantilla"],
    "multi_label": configuracion_comparacion["multi_label"],
    "batch_size": configuracion_comparacion["batch_size"]
}

# Realizamos el mismo calentamiento que con los otros modelos
_ = clasificador_comparacion(
    consultas_desarrollo[:2],
    **parametros
)

if dispositivo_comparacion == 0:
    torch.cuda.synchronize()

# Medimos tres ejecuciones completas
tiempos_xlm_roberta = []
etiquetas_por_repeticion = []

for repeticion in range(configuracion_comparacion["repeticiones"]):
    if dispositivo_comparacion == 0:
        torch.cuda.synchronize()

    inicio = perf_counter()

    salidas = clasificador_comparacion(
        consultas_desarrollo,
        **parametros
    )

    if dispositivo_comparacion == 0:
        torch.cuda.synchronize()

    duracion = perf_counter() - inicio
    tiempos_xlm_roberta.append(duracion)

    etiquetas_predichas = [
        categoria_por_descripcion[salida["labels"][0]]
        for salida in salidas
    ]
    etiquetas_por_repeticion.append(etiquetas_predichas)

    if repeticion == 0:
        predicciones_xlm_roberta = etiquetas_predichas.copy()
        puntuaciones_xlm_roberta = [
            salida["scores"][0] for salida in salidas
        ]

    print(
        f"Repetición {repeticion + 1}: "
        f"{duracion:.2f} segundos"
    )

# Comprobamos la estabilidad de las etiquetas entre repeticiones
predicciones_estables = all(
    etiquetas == predicciones_xlm_roberta
    for etiquetas in etiquetas_por_repeticion
)

# Calculamos métricas generales
precision, exhaustividad, f1, _ = precision_recall_fscore_support(
    etiquetas_desarrollo,
    predicciones_xlm_roberta,
    labels=categorias,
    average="macro",
    zero_division=0
)

metricas_xlm_roberta = {
    "Modelo": nombre_modelo,
    "Exactitud": accuracy_score(
        etiquetas_desarrollo, predicciones_xlm_roberta
    ),
    "Precisión macro": precision,
    "Exhaustividad macro": exhaustividad,
    "F1 macro": f1,
    "Tiempo medio total (s)": float(np.mean(tiempos_xlm_roberta)),
    "Desviación de tiempo (s)": float(
        np.std(tiempos_xlm_roberta, ddof=1)
    ),
    "Tiempo medio por consulta (s)": float(
        np.mean(tiempos_xlm_roberta) / len(consultas_desarrollo)
    )
}

display(pd.DataFrame([metricas_xlm_roberta]).round(4))

# Calculamos métricas por categoría
p_clase, r_clase, f1_clase, soporte = (
    precision_recall_fscore_support(
        etiquetas_desarrollo,
        predicciones_xlm_roberta,
        labels=categorias,
        average=None,
        zero_division=0
    )
)

tabla_metricas_xlm_roberta = pd.DataFrame({
    "Categoría": categorias,
    "Precisión": p_clase,
    "Exhaustividad": r_clase,
    "F1": f1_clase,
    "Consultas": soporte
})

display(tabla_metricas_xlm_roberta.round(4))

# Conservamos las predicciones vinculadas con cada consulta
df_resultados_xlm_roberta = df_comparacion_desarrollo[
    ["id_consulta", "consulta_es", "categoria_referencia"]
].copy()

df_resultados_xlm_roberta["categoria_predicha"] = (
    predicciones_xlm_roberta
)
df_resultados_xlm_roberta["puntuacion"] = puntuaciones_xlm_roberta
df_resultados_xlm_roberta["acierto"] = (
    df_resultados_xlm_roberta["categoria_referencia"]
    .eq(df_resultados_xlm_roberta["categoria_predicha"])
)

# Mostramos los errores para interpretarlos
df_errores_xlm_roberta = df_resultados_xlm_roberta[
    ~df_resultados_xlm_roberta["acierto"]
].copy()

print(f"\nTiempo de carga: {tiempo_carga_xlm_roberta:.2f} segundos")
print(
    "Aciertos:",
    int(df_resultados_xlm_roberta["acierto"].sum()),
    "de",
    len(df_resultados_xlm_roberta)
)
print("Etiquetas iguales entre repeticiones:", predicciones_estables)

if not df_errores_xlm_roberta.empty:
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(df_errores_xlm_roberta)
else:
    print("No se observaron errores en desarrollo.")

print("=" * 60)
print("✅ Evaluación de XLM-RoBERTa sobre desarrollo completada.")
print("=" * 60)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Repetición 1: 7.43 segundos
Repetición 2: 6.60 segundos
Repetición 3: 7.47 segundos


,Modelo,Exactitud,Precisión macro,Exhaustividad macro,F1 macro,Tiempo medio total (s),Desviación de tiempo (s),Tiempo medio por consulta (s)
0,XLM-RoBERTa-large,0.6167,0.6773,0.6167,0.5988,7.1638,0.4925,0.1194


,Categoría,Precisión,Exhaustividad,F1,Consultas
0,LIFE_INSURANCE,0.4545,1.0,0.6250,10
1,COVERAGE,0.6667,0.6,0.6316,10
2,QUOTE,0.6667,0.4,0.5000,10
3,PAYMENT,0.8333,0.5,0.6250,10
4,ENROLLMENT,0.7500,0.3,0.4286,10
5,CONTACT,0.6923,0.9,0.7826,10



Tiempo de carga: 4.19 segundos
Aciertos: 37 de 60
Etiquetas iguales entre repeticiones: True


,id_consulta,consulta_es,categoria_referencia,categoria_predicha,puntuacion,acierto
50,CONS_0051,"Necesito revisar mi cobertura, ¿cómo puedo hacerlo?",COVERAGE,LIFE_INSURANCE,0.522498,False
51,CONS_0052,"Quisiera saber sobre mi cobertura, ¿podrías darme información?",COVERAGE,LIFE_INSURANCE,0.822364,False
54,CONS_0055,"Tengo que consultar la cobertura de mi póliza de seguro de vida, ¿dónde puedo hacerlo?",COVERAGE,LIFE_INSURANCE,0.789484,False
56,CONS_0057,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",COVERAGE,LIFE_INSURANCE,0.856157,False
102,CONS_0103,"Busco información sobre las tarifas del seguro, ¿dónde puedo encontrarla?",QUOTE,LIFE_INSURANCE,0.614203,False
103,CONS_0104,"Quisiera saber más sobre las tarifas del seguro de vida, dame información.",QUOTE,LIFE_INSURANCE,0.469002,False
104,CONS_0105,"Quiero consultar las tarifas del seguro de vida, ¿puedo verlas?",QUOTE,LIFE_INSURANCE,0.742689,False
106,CONS_0107,¿Dónde puedo consultar las tarifas del seguro de vida?,QUOTE,LIFE_INSURANCE,0.707245,False
108,CONS_0109,"Quisiera saber sobre las tarifas, ¿podrías darme información?",QUOTE,LIFE_INSURANCE,0.519313,False
109,CONS_0110,"Busco información sobre las tarifas del seguro, ¿me pueden ayudar?",QUOTE,LIFE_INSURANCE,0.464718,False


✅ Evaluación de XLM-RoBERTa sobre desarrollo completada.


### **📝 Interpretación de los resultados de XLM-RoBERTa**

XLM-RoBERTa-large clasificó correctamente 37 de las 60 consultas
de desarrollo. Alcanzó una exactitud de 0.6167 y un F1 macro de
0.5988, el mayor de los tres modelos con la configuración inicial.

Los aciertos por categoría fueron:

- LIFE_INSURANCE: 10 de 10.
- COVERAGE: 6 de 10.
- QUOTE: 4 de 10.
- PAYMENT: 5 de 10.
- ENROLLMENT: 3 de 10.
- CONTACT: 9 de 10.

Aunque identificó todas las consultas de LIFE_INSURANCE, también
asignó esta categoría a 12 consultas de otras categorías.
Su precisión en LIFE_INSURANCE fue de 0.4545. Esto muestra
confusión entre la información general y las solicitudes específicas,
especialmente las relacionadas con tarifas y coberturas.

ENROLLMENT presentó la menor exhaustividad: 0.3. Persistieron
errores en consultas sobre contratación, cancelación y renovación.

El tiempo medio para procesar las 60 consultas fue de 7.1638
segundos, con una desviación de 0.4925 segundos. El promedio
por consulta fue de 0.1194 segundos. Las etiquetas predichas
coincidieron en las tres repeticiones.

Estos resultados corresponden al conjunto de desarrollo.
La selección continúa siendo provisional y deberá contrastarse
con la evaluación del conjunto de prueba.

### **🧠 5.5 Comparación e interpretación conjunta**

Se reunirán las métricas de los tres modelos evaluados
sobre las mismas 60 consultas de desarrollo.

XLM-RoBERTa obtuvo la mayor exactitud y F1 macro.
MiniLMv2 presentó el menor tiempo medio de procesamiento.
mDeBERTa obtuvo los resultados más bajos de clasificación
y el mayor tiempo medio en estas ejecuciones.

Se observaron patrones de confusión diferentes:

- MiniLMv2 no acertó consultas de LIFE_INSURANCE.
- mDeBERTa concentró 49 predicciones en CONTACT.
- XLM-RoBERTa asignó LIFE_INSURANCE a 24 consultas:
  diez correctas y catorce de otras categorías.

**Decisión preliminar:**

XLM-RoBERTa es el candidato con mejores resultados de
clasificación en esta comparación de desarrollo.

Antes de seleccionar el modelo final, se revisará el solapamiento
de las descripciones mediante experimentos sobre desarrollo.
Los cambios se documentarán y se aplicarán de manera uniforme
a los tres modelos.

**Limitación:** Estos resultados no corresponden al conjunto
de prueba y no permiten estimar todavía el desempeño final.

In [ ]:
# ============================================================
# 5.5 Comparación e interpretación conjunta
# ============================================================

# Reunimos las métricas calculadas, sin introducir valores manuales
df_comparacion_modelos = pd.DataFrame([
    metricas_minilm,
    metricas_mdeberta,
    metricas_xlm_roberta
]).sort_values(
    "F1 macro",
    ascending=False
).reset_index(drop=True)

display(df_comparacion_modelos.round(4))

# Comparamos los aciertos por categoría
tablas_resultados = {
    "MiniLMv2": df_resultados_minilm,
    "mDeBERTa": df_resultados_mdeberta,
    "XLM-RoBERTa": df_resultados_xlm_roberta
}

df_aciertos_categoria = pd.DataFrame({
    nombre: (
        resultados.groupby("categoria_referencia")["acierto"]
        .sum()
        .reindex(configuracion_comparacion["descripciones"], fill_value=0)
        .astype(int)
    )
    for nombre, resultados in tablas_resultados.items()
})

df_aciertos_categoria.index.name = "Categoría"
display(df_aciertos_categoria.reset_index())

print("=" * 60)
print("✅ Comparación inicial sobre desarrollo completada.")
print("=" * 60)

,Modelo,Exactitud,Precisión macro,Exhaustividad macro,F1 macro,Tiempo medio total (s),Desviación de tiempo (s),Tiempo medio por consulta (s)
0,XLM-RoBERTa-large,0.6167,0.6773,0.6167,0.5988,7.1638,0.4925,0.1194
1,MiniLMv2 multilingüe,0.3833,0.3240,0.3833,0.3437,8.3177,0.6939,0.1386
2,mDeBERTa-v3-base,0.2333,0.3522,0.2333,0.1944,12.5794,4.4972,0.2097


,Categoría,MiniLMv2,mDeBERTa,XLM-RoBERTa
0,LIFE_INSURANCE,0,0,10
1,COVERAGE,4,2,6
2,QUOTE,2,3,4
3,PAYMENT,1,1,5
4,ENROLLMENT,6,0,3
5,CONTACT,10,8,9


✅ Comparación inicial sobre desarrollo completada.


### **🧠 5.6 Respaldo de la comparación inicial**

Se guardarán las métricas generales, los aciertos por categoría,
las predicciones individuales y los tiempos de las tres
repeticiones de cada modelo.

También se conservarán las consultas de desarrollo utilizadas
y la configuración exacta de esta comparación.

Este respaldo permitirá contrastar los resultados iniciales
con experimentos posteriores sobre desarrollo.

**Estado:** Comparación inicial documentada.
Selección final pendiente.

In [ ]:
# ============================================================
# 5.6 Respaldo de la comparación inicial
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import transformers
import torch
import sklearn
from google.colab import files

# Creamos una carpeta nueva sin sobrescribir respaldos anteriores
marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_comparacion = (
    Path("/content/revision_evaluacion")
    / f"comparacion_inicial_{marca_version}"
)
carpeta_comparacion.mkdir(parents=True, exist_ok=False)

# Guardamos los resultados con su precisión original
df_comparacion_modelos.to_csv(
    carpeta_comparacion / "metricas_generales.csv",
    index=False,
    encoding="utf-8-sig"
)

df_aciertos_categoria.reset_index().to_csv(
    carpeta_comparacion / "aciertos_por_categoria.csv",
    index=False,
    encoding="utf-8-sig"
)

df_comparacion_desarrollo.to_csv(
    carpeta_comparacion / "desarrollo_utilizado.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardamos las predicciones de cada modelo
resultados_por_modelo = {
    "minilm": df_resultados_minilm,
    "mdeberta": df_resultados_mdeberta,
    "xlm_roberta": df_resultados_xlm_roberta
}

for nombre, resultados in resultados_por_modelo.items():
    resultados.to_csv(
        carpeta_comparacion / f"predicciones_{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# Guardamos las métricas individuales por categoría
metricas_por_categoria = {
    "minilm": tabla_metricas_minilm,
    "mdeberta": tabla_metricas_mdeberta,
    "xlm_roberta": tabla_metricas_xlm_roberta
}

for nombre, tabla in metricas_por_categoria.items():
    tabla.to_csv(
        carpeta_comparacion / f"metricas_categoria_{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# Conservamos los tiempos de cada repetición
tiempos_registrados = {
    "MiniLMv2 multilingüe": tiempos_minilm,
    "mDeBERTa-v3-base": tiempos_mdeberta,
    "XLM-RoBERTa-large": tiempos_xlm_roberta
}

filas_tiempos = [
    {
        "Modelo": modelo,
        "Repetición": numero,
        "Tiempo total (s)": float(tiempo)
    }
    for modelo, tiempos in tiempos_registrados.items()
    for numero, tiempo in enumerate(tiempos, start=1)
]

pd.DataFrame(filas_tiempos).to_csv(
    carpeta_comparacion / "tiempos_repeticiones.csv",
    index=False,
    encoding="utf-8-sig"
)

# Conservamos la configuración utilizada y versiones de bibliotecas
registro_experimento = {
    "tipo": "comparación inicial sobre desarrollo",
    "configuracion": configuracion_comparacion,
    "modelos": modelos_comparacion,
    "consultas": len(df_comparacion_desarrollo),
    "versiones": {
        "transformers": transformers.__version__,
        "torch": torch.__version__,
        "scikit_learn": sklearn.__version__
    },
    "nota": (
        "Resultados de desarrollo. "
        "El conjunto de prueba no fue evaluado."
    )
}

with (carpeta_comparacion / "experimento.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(
        registro_experimento,
        archivo,
        ensure_ascii=False,
        indent=2
    )

# Comprimimos y descargamos el respaldo completo
ruta_zip_comparacion = shutil.make_archive(
    str(carpeta_comparacion),
    "zip",
    root_dir=str(carpeta_comparacion)
)

print("Comparación guardada en:", carpeta_comparacion)

print("=" * 60)
print("✅ Comparación inicial respaldada.")
print("=" * 60)

files.download(ruta_zip_comparacion)

Comparación guardada en: /content/revision_evaluacion/comparacion_inicial_20261010T190858311821Z
✅ Comparación inicial respaldada.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **🧠 5.7 Configuración alternativa de descripciones**

Se preparará una segunda configuración para comprobar
si descripciones más específicas reducen las confusiones
observadas en desarrollo.

LIFE_INSURANCE se centrará en información del producto
o consulta general de una póliza de vida.

CONTACT se centrará en solicitudes explícitas de comunicación,
evitando tratar cualquier petición de ayuda como contacto.

Las demás categorías describirán su motivo específico.
Se conservarán la plantilla, los datos, las etiquetas
y las condiciones de ejecución.

Esta configuración se aplicará a los tres modelos.
Su utilidad se determinará mediante los resultados,
sin asumir que mejorará el desempeño.

**Estado:** Configuración alternativa preparada.
Evaluación sobre desarrollo pendiente.

In [ ]:
# ============================================================
# 5.7 Configuración alternativa de descripciones
# ============================================================

import copy

# Definimos una alternativa centrada en el motivo de consulta
descripciones_alternativas = {
    "LIFE_INSURANCE": (
        "qué es un seguro de vida, para qué sirve "
        "o cómo consultar información general de mi póliza de vida"
    ),
    "COVERAGE": (
        "qué cubre o excluye mi póliza "
        "y cómo ampliar, reducir o cambiar su cobertura"
    ),
    "QUOTE": (
        "cuánto cuesta el seguro, sus tarifas "
        "o cómo obtener y calcular una cotización"
    ),
    "PAYMENT": (
        "cómo pagar el seguro, consultar pagos realizados, "
        "programar pagos o reportar un problema de pago"
    ),
    "ENROLLMENT": (
        "cómo comparar o contratar pólizas, renovarlas "
        "o cancelarlas y conocer los cargos de cancelación"
    ),
    "CONTACT": (
        "cómo llamar, enviar un correo o comunicarme "
        "con atención al cliente, la aseguradora o un agente humano"
    )
}

# Verificamos que conservemos las mismas categorías y su orden
if list(descripciones_alternativas) != list(
    configuracion_comparacion["descripciones"]
):
    raise ValueError("Las categorías no coinciden con la configuración inicial.")

# Creamos una copia sin modificar la configuración respaldada
configuracion_alternativa = copy.deepcopy(
    configuracion_comparacion
)
configuracion_alternativa["descripciones"] = (
    descripciones_alternativas.copy()
)

# Mostramos los cambios para documentar el experimento
tabla_descripciones = pd.DataFrame({
    "Categoría": list(descripciones_alternativas),
    "Descripción inicial": list(
        configuracion_comparacion["descripciones"].values()
    ),
    "Descripción alternativa": list(
        descripciones_alternativas.values()
    )
})

with pd.option_context("display.max_colwidth", None):
    display(tabla_descripciones)

print("Plantilla conservada:", configuracion_alternativa["plantilla"])
print("Consultas de desarrollo:", len(consultas_desarrollo))

print("=" * 60)
print("✅ Configuración alternativa preparada.")
print("=" * 60)

,Categoría,Descripción inicial,Descripción alternativa
0,LIFE_INSURANCE,información general o consulta de información sobre un seguro de vida,"qué es un seguro de vida, para qué sirve o cómo consultar información general de mi póliza de vida"
1,COVERAGE,"consulta de coberturas, exclusiones o modificación de la cobertura del seguro","qué cubre o excluye mi póliza y cómo ampliar, reducir o cambiar su cobertura"
2,QUOTE,"consulta de tarifas, estimación de costo o solicitud de cotización del seguro","cuánto cuesta el seguro, sus tarifas o cómo obtener y calcular una cotización"
3,PAYMENT,"realización y consulta de pagos del seguro, programación, métodos y problemas de pago","cómo pagar el seguro, consultar pagos realizados, programar pagos o reportar un problema de pago"
4,ENROLLMENT,"comparación, contratación, renovación o cancelación de pólizas, incluidos requisitos y cargos de terminación","cómo comparar o contratar pólizas, renovarlas o cancelarlas y conocer los cargos de cancelación"
5,CONTACT,"contacto con atención al cliente, un representante, la aseguradora o una persona","cómo llamar, enviar un correo o comunicarme con atención al cliente, la aseguradora o un agente humano"


Plantilla conservada: Esta consulta trata sobre {}.
Consultas de desarrollo: 60
✅ Configuración alternativa preparada.


### **🧠 5.8 Evaluación de la configuración alternativa**

Se evaluarán los tres modelos con las descripciones alternativas,
manteniendo las consultas, etiquetas y condiciones de ejecución.

Cada modelo realizará un calentamiento y tres repeticiones.
Se conservarán las predicciones, métricas por categoría
y tiempos individuales.

Se compararán la exactitud y el F1 macro con los resultados
iniciales para comprobar el efecto del cambio de descripciones.

**Estado:** Experimento sobre desarrollo.
El conjunto de prueba permanece sin evaluar.

In [ ]:
# ============================================================
# 5.8 Evaluación de la configuración alternativa
# ============================================================

import gc
import copy
import numpy as np
import pandas as pd
import torch

from time import perf_counter
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# Conservamos una copia de la configuración de este experimento
configuracion_experimento_alternativo = copy.deepcopy(
    configuracion_alternativa
)

descripciones = configuracion_experimento_alternativo["descripciones"]
categorias = list(descripciones)

categoria_por_descripcion = {
    descripcion: categoria
    for categoria, descripcion in descripciones.items()
}

parametros = {
    "candidate_labels": list(descripciones.values()),
    "hypothesis_template": configuracion_experimento_alternativo["plantilla"],
    "multi_label": configuracion_experimento_alternativo["multi_label"],
    "batch_size": configuracion_experimento_alternativo["batch_size"]
}

# Conservamos cada modelo evaluado en un diccionario independiente
resultados_alternativa = {}

for nombre_modelo, identificador in modelos_comparacion.items():
    print("=" * 60)
    print("Modelo:", nombre_modelo)

    # Liberamos el modelo anterior antes de cargar el siguiente
    for variable in ["clasificador", "clasificador_comparacion"]:
        anterior = globals().pop(variable, None)
        if anterior is not None:
            del anterior

    gc.collect()

    if dispositivo_comparacion == 0:
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    # Medimos la carga por separado
    inicio_carga = perf_counter()

    clasificador_comparacion = pipeline(
        task="zero-shot-classification",
        model=identificador,
        device=dispositivo_comparacion,
        torch_dtype=torch.float32
    )

    if dispositivo_comparacion == 0:
        torch.cuda.synchronize()

    tiempo_carga = perf_counter() - inicio_carga

    # Realizamos el mismo calentamiento
    _ = clasificador_comparacion(
        consultas_desarrollo[:2],
        **parametros
    )

    if dispositivo_comparacion == 0:
        torch.cuda.synchronize()

    tiempos = []
    etiquetas_repeticiones = []

    # Medimos tres ejecuciones sobre las mismas consultas
    for repeticion in range(
        configuracion_experimento_alternativo["repeticiones"]
    ):
        if dispositivo_comparacion == 0:
            torch.cuda.synchronize()

        inicio = perf_counter()

        salidas = clasificador_comparacion(
            consultas_desarrollo,
            **parametros
        )

        if dispositivo_comparacion == 0:
            torch.cuda.synchronize()

        duracion = perf_counter() - inicio
        tiempos.append(duracion)

        etiquetas = [
            categoria_por_descripcion[salida["labels"][0]]
            for salida in salidas
        ]
        etiquetas_repeticiones.append(etiquetas)

        if repeticion == 0:
            predicciones = etiquetas.copy()
            puntuaciones = [
                salida["scores"][0] for salida in salidas
            ]

        print(
            f"Repetición {repeticion + 1}: "
            f"{duracion:.2f} segundos"
        )

    # Calculamos métricas generales
    precision, exhaustividad, f1, _ = (
        precision_recall_fscore_support(
            etiquetas_desarrollo,
            predicciones,
            labels=categorias,
            average="macro",
            zero_division=0
        )
    )

    metricas = {
        "Modelo": nombre_modelo,
        "Exactitud": accuracy_score(
            etiquetas_desarrollo, predicciones
        ),
        "Precisión macro": precision,
        "Exhaustividad macro": exhaustividad,
        "F1 macro": f1,
        "Tiempo medio total (s)": float(np.mean(tiempos)),
        "Desviación de tiempo (s)": float(np.std(tiempos, ddof=1)),
        "Tiempo medio por consulta (s)": float(
            np.mean(tiempos) / len(consultas_desarrollo)
        )
    }

    # Calculamos métricas por categoría
    p_clase, r_clase, f1_clase, soporte = (
        precision_recall_fscore_support(
            etiquetas_desarrollo,
            predicciones,
            labels=categorias,
            average=None,
            zero_division=0
        )
    )

    tabla_categorias = pd.DataFrame({
        "Categoría": categorias,
        "Precisión": p_clase,
        "Exhaustividad": r_clase,
        "F1": f1_clase,
        "Consultas": soporte
    })

    # Conservamos las predicciones vinculadas con las consultas
    tabla_predicciones = df_comparacion_desarrollo[
        ["id_consulta", "consulta_es", "categoria_referencia"]
    ].copy()

    tabla_predicciones["categoria_predicha"] = predicciones
    tabla_predicciones["puntuacion"] = puntuaciones
    tabla_predicciones["acierto"] = (
        tabla_predicciones["categoria_referencia"]
        .eq(tabla_predicciones["categoria_predicha"])
    )

    estables = all(
        etiquetas == predicciones
        for etiquetas in etiquetas_repeticiones
    )

    resultados_alternativa[nombre_modelo] = {
        "metricas": metricas,
        "metricas_categoria": tabla_categorias,
        "predicciones": tabla_predicciones,
        "tiempos": tiempos,
        "tiempo_carga": tiempo_carga,
        "etiquetas_estables": estables
    }

    display(pd.DataFrame([metricas]).round(4))
    display(tabla_categorias.round(4))

    print(
        "Aciertos:",
        int(tabla_predicciones["acierto"].sum()),
        "de",
        len(tabla_predicciones)
    )
    print("Etiquetas iguales entre repeticiones:", estables)

# Reunimos las métricas alternativas
df_metricas_alternativa = pd.DataFrame([
    resultado["metricas"]
    for resultado in resultados_alternativa.values()
])

# Comparamos por nombre de modelo con la configuración inicial
df_efecto_descripciones = df_comparacion_modelos[
    ["Modelo", "Exactitud", "F1 macro"]
].merge(
    df_metricas_alternativa[
        ["Modelo", "Exactitud", "F1 macro"]
    ],
    on="Modelo",
    suffixes=("_inicial", "_alternativa"),
    validate="one_to_one"
)

df_efecto_descripciones["Cambio exactitud (puntos porcentuales)"] = (
    df_efecto_descripciones["Exactitud_alternativa"]
    - df_efecto_descripciones["Exactitud_inicial"]
) * 100

df_efecto_descripciones["Cambio F1 macro"] = (
    df_efecto_descripciones["F1 macro_alternativa"]
    - df_efecto_descripciones["F1 macro_inicial"]
)

display(df_efecto_descripciones.round(4))

print("=" * 60)
print("✅ Configuración alternativa evaluada en los tres modelos.")
print("=" * 60)

Modelo: MiniLMv2 multilingüe


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Repetición 1: 5.27 segundos
Repetición 2: 5.48 segundos
Repetición 3: 6.72 segundos


,Modelo,Exactitud,Precisión macro,Exhaustividad macro,F1 macro,Tiempo medio total (s),Desviación de tiempo (s),Tiempo medio por consulta (s)
0,MiniLMv2 multilingüe,0.2333,0.2562,0.2333,0.1564,5.822,0.7851,0.097


,Categoría,Precisión,Exhaustividad,F1,Consultas
0,LIFE_INSURANCE,0.0000,0.0,0.0000,10
1,COVERAGE,0.1667,0.1,0.1250,10
2,QUOTE,0.5000,0.1,0.1667,10
3,PAYMENT,0.2041,1.0,0.3390,10
4,ENROLLMENT,0.6667,0.2,0.3077,10
5,CONTACT,0.0000,0.0,0.0000,10


...

,Modelo,Exactitud,Precisión macro,Exhaustividad macro,F1 macro,Tiempo medio total (s),Desviación de tiempo (s),Tiempo medio por consulta (s)
0,XLM-RoBERTa-large,0.5,0.5595,0.5,0.4664,6.9958,0.3114,0.1166


,Categoría,Precisión,Exhaustividad,F1,Consultas
0,LIFE_INSURANCE,0.3571,1.0,0.5263,10
1,COVERAGE,0.7143,0.5,0.5882,10
2,QUOTE,0.2857,0.4,0.3333,10
3,PAYMENT,1.0000,0.3,0.4615,10
4,ENROLLMENT,0.0000,0.0,0.0000,10
5,CONTACT,1.0000,0.8,0.8889,10


Aciertos: 30 de 60
Etiquetas iguales entre repeticiones: True


,Modelo,Exactitud_inicial,F1 macro_inicial,Exactitud_alternativa,F1 macro_alternativa,Cambio exactitud (puntos porcentuales),Cambio F1 macro
0,XLM-RoBERTa-large,0.6167,0.5988,0.5000,0.4664,-11.6667,-0.1324
1,MiniLMv2 multilingüe,0.3833,0.3437,0.2333,0.1564,-15.0000,-0.1873
2,mDeBERTa-v3-base,0.2333,0.1944,0.3833,0.3569,15.0000,0.1625


✅ Configuración alternativa evaluada en los tres modelos.


### 📝 Interpretación de la configuración alternativa

El cambio de descripciones produjo efectos diferentes en los tres modelos.

- MiniLMv2 pasó de 23 a 14 aciertos y su F1 macro disminuyó de 0.3437 a 0.1564.
- mDeBERTa pasó de 14 a 23 aciertos y su F1 macro aumentó de 0.1944 a 0.3569.
- XLM-RoBERTa pasó de 37 a 30 aciertos y su F1 macro disminuyó de 0.5988 a 0.4664.

Las predicciones coincidieron entre las tres repeticiones de cada modelo.

**Hallazgo principal:**

Las descripciones alternativas no mejoraron todos los modelos.
La clasificación zero-shot mostró sensibilidad a la redacción
de las categorías: mDeBERTa mejoró, mientras que MiniLMv2
y XLM-RoBERTa empeoraron.

**Decisión sobre desarrollo:**

La mejor combinación observada fue XLM-RoBERTa con las
descripciones iniciales, con 37 aciertos de 60 y un F1 macro
de 0.5988.

Para comparar los tres modelos bajo una configuración común,
se conservarán las descripciones iniciales en la evaluación
de prueba. La configuración alternativa quedará documentada
como experimento de sensibilidad.

**Limitación:**

Esta decisión se basa en 60 consultas de desarrollo.
La selección final requiere evaluar el conjunto de prueba
preparado y validado, sin ajustar las descripciones a partir
de sus resultados.

### **🧠 5.9 Registro de la configuración para prueba**

Se seleccionará la configuración inicial como configuración
común para evaluar los tres modelos sobre prueba.

Se utilizará F1 macro como criterio principal de comparación
y se mantendrán las etiquetas y los criterios de clasificación.

La configuración alternativa se conservará como parte
del análisis de sensibilidad.

**Estado:** Configuración común seleccionada sobre desarrollo.
Preparación del conjunto de prueba pendiente.

In [ ]:
# ============================================================
# 5.9 Registro de la configuración para prueba
# ============================================================

import copy

# Conservamos la configuración inicial sin modificar los experimentos
configuracion_prueba = copy.deepcopy(configuracion_comparacion)

decision_desarrollo = {
    "criterio_principal": "F1 macro",
    "configuracion_comun": "inicial",
    "candidato_preliminar": "XLM-RoBERTa-large",
    "motivo": (
        "Mayor F1 macro observado sobre desarrollo: 0.6275. "
        "La alternativa no mejoró de forma común los tres modelos."
    ),
    "estado": "selección final pendiente de evaluación sobre prueba"
}

# Mostramos la configuración seleccionada
with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            configuracion_prueba["descripciones"].items(),
            columns=["Categoría", "Descripción para prueba"]
        )
    )

print("Criterio principal:", decision_desarrollo["criterio_principal"])
print("Configuración común:", decision_desarrollo["configuracion_comun"])
print("Candidato preliminar:", decision_desarrollo["candidato_preliminar"])
print("Estado:", decision_desarrollo["estado"])

print("=" * 60)
print("✅ Configuración común para prueba registrada.")
print("=" * 60)

,Categoría,Descripción para prueba
0,LIFE_INSURANCE,información general o consulta de información sobre un seguro de vida
1,COVERAGE,"consulta de coberturas, exclusiones o modificación de la cobertura del seguro"
2,QUOTE,"consulta de tarifas, estimación de costo o solicitud de cotización del seguro"
3,PAYMENT,"realización y consulta de pagos del seguro, programación, métodos y problemas de pago"
4,ENROLLMENT,"comparación, contratación, renovación o cancelación de pólizas, incluidos requisitos y cargos de terminación"
5,CONTACT,"contacto con atención al cliente, un representante, la aseguradora o una persona"


Criterio principal: F1 macro
Configuración común: inicial
Candidato preliminar: XLM-RoBERTa-large
Estado: selección final pendiente de evaluación sobre prueba
✅ Configuración común para prueba registrada.


### **🧠 5.10 Respaldo de experimentos y cierre de desarrollo**

Se guardarán los resultados de las configuraciones inicial
y alternativa, sus descripciones y la decisión tomada
sobre desarrollo.

También se conservarán la muestra actualizada y la auditoría
de registros retirados.

**Estado:** Comparación sobre desarrollo completada.
Preparación y evaluación del conjunto de prueba pendientes.

In [ ]:
# ============================================================
# 5.10 Respaldo de experimentos y cierre de desarrollo
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import zipfile
from google.colab import files

print("=" * 60)
print("📦 Preparando respaldo de desarrollo y experimentos...")
print("=" * 60)

# ------------------------------------------------------------
# 1. Verificamos los insumos antes de crear el respaldo
# ------------------------------------------------------------

variables_requeridas = [
    "df_muestra_revision",
    "df_registros_retirados",
    "df_comparacion_desarrollo",
    "df_comparacion_modelos",
    "df_metricas_alternativa",
    "df_efecto_descripciones",
    "df_resultados_minilm",
    "tabla_metricas_minilm",
    "df_resultados_mdeberta",
    "tabla_metricas_mdeberta",
    "df_resultados_xlm_roberta",
    "tabla_metricas_xlm_roberta",
    "resultados_alternativa",
    "modelos_comparacion",
    "configuracion_comparacion",
    "configuracion_experimento_alternativo",
    "configuracion_prueba",
    "decision_desarrollo",
    "tiempos_minilm",
    "tiempos_mdeberta",
    "tiempos_xlm_roberta"
]

faltantes = [
    nombre
    for nombre in variables_requeridas
    if nombre not in globals()
]

if faltantes:
    raise ValueError(
        "Faltan insumos para respaldar los experimentos: "
        + ", ".join(faltantes)
        + ". Ejecuta los bloques anteriores."
    )

if len(df_comparacion_desarrollo) != 60:
    raise ValueError(
        "El conjunto utilizado en desarrollo debe contener 60 consultas."
    )

if df_comparacion_desarrollo["id_consulta"].duplicated().any():
    raise ValueError(
        "El conjunto de desarrollo contiene identificadores repetidos."
    )

if not df_comparacion_desarrollo["estado_revision"].eq(
    "validado_usuario"
).all():
    raise ValueError(
        "Hay consultas de desarrollo pendientes de validación."
    )

# ------------------------------------------------------------
# 2. Preparamos configuraciones y referencias de los resultados
# ------------------------------------------------------------

resultados_iniciales = {
    "minilm": (df_resultados_minilm, tabla_metricas_minilm),
    "mdeberta": (df_resultados_mdeberta, tabla_metricas_mdeberta),
    "xlm_roberta": (
        df_resultados_xlm_roberta,
        tabla_metricas_xlm_roberta
    )
}

# Usamos nombres descriptivos también para la alternativa
nombres_archivos = {
    "MiniLMv2 multilingüe": "minilm",
    "mDeBERTa-v3-base": "mdeberta",
    "XLM-RoBERTa-large": "xlm_roberta"
}

if set(resultados_alternativa) != set(nombres_archivos):
    raise ValueError(
        "Los resultados alternativos no contienen los tres modelos esperados."
    )

ids_desarrollo = set(df_comparacion_desarrollo["id_consulta"])

# Comprobamos que las predicciones correspondan al mismo conjunto
predicciones_a_verificar = {
    f"inicial_{nombre}": predicciones
    for nombre, (predicciones, _) in resultados_iniciales.items()
}

for nombre, resultado in resultados_alternativa.items():
    predicciones_a_verificar[
        f"alternativa_{nombres_archivos[nombre]}"
    ] = resultado["predicciones"]

for nombre, predicciones in predicciones_a_verificar.items():
    if (
        len(predicciones) != 60
        or predicciones["id_consulta"].duplicated().any()
        or set(predicciones["id_consulta"]) != ids_desarrollo
    ):
        raise ValueError(
            f"Las predicciones de {nombre} no corresponden "
            "a las 60 consultas de desarrollo."
        )

# Preparamos el registro de la alternativa
resumen_alternativa = {}

for nombre, resultado in resultados_alternativa.items():
    nombre_archivo = nombres_archivos[nombre]

    resumen_alternativa[nombre] = {
        "archivo_predicciones": (
            f"predicciones_alternativas_{nombre_archivo}.csv"
        ),
        "archivo_metricas_categoria": (
            f"categorias_alternativas_{nombre_archivo}.csv"
        ),
        "tiempos": [float(t) for t in resultado["tiempos"]],
        "tiempo_carga": float(resultado["tiempo_carga"]),
        "etiquetas_estables": bool(resultado["etiquetas_estables"])
    }

registro_cierre = {
    "modelos": modelos_comparacion,
    "configuracion_inicial": configuracion_comparacion,
    "configuracion_alternativa": configuracion_experimento_alternativo,
    "configuracion_prueba": configuracion_prueba,
    "decision_desarrollo": decision_desarrollo,
    "tiempos_iniciales": {
        "MiniLMv2 multilingüe": [float(t) for t in tiempos_minilm],
        "mDeBERTa-v3-base": [float(t) for t in tiempos_mdeberta],
        "XLM-RoBERTa-large": [
            float(t) for t in tiempos_xlm_roberta
        ]
    },
    "resultados_alternativa": resumen_alternativa,
    "estado_prueba": (
        "pendiente de completar traducción, validación y evaluación"
    )
}

# Verificamos que el registro pueda guardarse como JSON
texto_registro = json.dumps(
    registro_cierre,
    ensure_ascii=False,
    indent=2
)

texto_configuracion_prueba = json.dumps(
    {
        "modelos": modelos_comparacion,
        "configuracion_prueba": configuracion_prueba,
        "decision_desarrollo": decision_desarrollo
    },
    ensure_ascii=False,
    indent=2
)

# ------------------------------------------------------------
# 3. Creamos una carpeta nueva sin sobrescribir respaldos
# ------------------------------------------------------------

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_cierre = (
    Path("/content/revision_evaluacion")
    / f"cierre_desarrollo_{marca_version}"
)

carpeta_cierre.mkdir(parents=True, exist_ok=False)

# Registramos qué contiene cada archivo y para qué sirve
inventario_archivos = {}


def guardar_tabla(nombre, tabla, contenido, uso):
    """Guarda una tabla CSV y registra su finalidad."""
    tabla.to_csv(
        carpeta_cierre / nombre,
        index=False,
        encoding="utf-8-sig"
    )
    inventario_archivos[nombre] = {
        "contenido": contenido,
        "uso_posterior": uso,
        "registros": len(tabla)
    }


# ------------------------------------------------------------
# 4. Guardamos los datos y la auditoría
# ------------------------------------------------------------

guardar_tabla(
    "muestra_actualizada.csv",
    df_muestra_revision,
    "Muestra completa con el estado actual de revisión.",
    "Retomar la preparación del conjunto de prueba."
)

guardar_tabla(
    "desarrollo_utilizado.csv",
    df_comparacion_desarrollo,
    "Las 60 consultas validadas utilizadas en los experimentos.",
    "Cargar directamente el conjunto de desarrollo."
)

guardar_tabla(
    "registros_retirados.csv",
    df_registros_retirados,
    "Registros retirados y trazabilidad de los reemplazos.",
    "Consultar la auditoría; no utilizar como datos de evaluación."
)

# ------------------------------------------------------------
# 5. Guardamos las métricas de ambos experimentos
# ------------------------------------------------------------

guardar_tabla(
    "metricas_iniciales.csv",
    df_comparacion_modelos,
    "Métricas generales de los tres modelos con descripciones iniciales.",
    "Consultar y documentar la comparación inicial."
)

guardar_tabla(
    "metricas_alternativas.csv",
    df_metricas_alternativa,
    "Métricas generales con descripciones alternativas.",
    "Consultar el experimento de sensibilidad."
)

guardar_tabla(
    "efecto_descripciones.csv",
    df_efecto_descripciones,
    "Cambios de exactitud y F1 macro entre configuraciones.",
    "Justificar la elección de las descripciones para prueba."
)

# Guardamos predicciones y métricas por categoría de la inicial
for nombre, (
    predicciones,
    metricas_categoria
) in resultados_iniciales.items():

    guardar_tabla(
        f"predicciones_iniciales_{nombre}.csv",
        predicciones,
        f"Predicciones de {nombre} con la configuración inicial.",
        "Analizar aciertos y errores sin repetir la inferencia."
    )

    guardar_tabla(
        f"categorias_iniciales_{nombre}.csv",
        metricas_categoria,
        f"Métricas por categoría de {nombre}, configuración inicial.",
        "Consultar el rendimiento de cada categoría."
    )

# Guardamos los resultados de la alternativa
for nombre, resultado in resultados_alternativa.items():
    nombre_archivo = nombres_archivos[nombre]

    guardar_tabla(
        f"predicciones_alternativas_{nombre_archivo}.csv",
        resultado["predicciones"],
        f"Predicciones de {nombre} con la configuración alternativa.",
        "Analizar aciertos y errores del experimento alternativo."
    )

    guardar_tabla(
        f"categorias_alternativas_{nombre_archivo}.csv",
        resultado["metricas_categoria"],
        f"Métricas por categoría de {nombre}, configuración alternativa.",
        "Comparar el efecto de las descripciones por categoría."
    )

# ------------------------------------------------------------
# 6. Guardamos configuraciones y una guía de uso
# ------------------------------------------------------------

(carpeta_cierre / "registro_experimentos.json").write_text(
    texto_registro,
    encoding="utf-8"
)

inventario_archivos["registro_experimentos.json"] = {
    "contenido": "Modelos, configuraciones, tiempos y decisión de desarrollo.",
    "uso_posterior": "Consultar los parámetros y la trazabilidad experimental."
}

(carpeta_cierre / "configuracion_para_prueba.json").write_text(
    texto_configuracion_prueba,
    encoding="utf-8"
)

inventario_archivos["configuracion_para_prueba.json"] = {
    "contenido": "Modelos y configuración común fijada para prueba.",
    "uso_posterior": "Recuperar la configuración sin repetir los experimentos."
}

guia = [
    "RESPALDO DE DESARROLLO Y EXPERIMENTOS",
    "=" * 60,
    f"Versión UTC: {marca_version}",
    "",
    "INSUMOS PRINCIPALES PARA CONTINUAR",
    "",
    "1. muestra_actualizada.csv",
    "   Permite retomar la preparación de prueba.",
    "   No representa todavía un dataset completo validado.",
    "",
    "2. desarrollo_utilizado.csv",
    "   Contiene las 60 consultas validadas de desarrollo.",
    "   Puede cargarse directamente sin repetir su preparación.",
    "",
    "3. configuracion_para_prueba.json",
    "   Conserva los modelos y la configuración fijada para prueba.",
    "",
    "4. registros_retirados.csv",
    "   Conserva la auditoría de los reemplazos.",
    "",
    "ESTADO",
    "Desarrollo evaluado con las dos configuraciones.",
    "Prueba pendiente de completar traducción y validación.",
    "Selección final del modelo pendiente de evaluación sobre prueba.",
    "",
    "PARA RECUPERAR ESTE RESPALDO",
    "Descomprimir el ZIP y cargar los CSV y JSON necesarios.",
    "Los archivos no dependen de que siga activo este entorno de Colab.",
    "La evaluación con modelos requiere sus bibliotecas y pesos.",
    "",
    "INVENTARIO DE ARCHIVOS",
    "=" * 60
]

for nombre, detalles in inventario_archivos.items():
    guia.extend([
        "",
        nombre,
        f"  Contenido: {detalles['contenido']}",
        f"  Uso posterior: {detalles['uso_posterior']}"
    ])

(carpeta_cierre / "LEEME.txt").write_text(
    "\n".join(guia) + "\n",
    encoding="utf-8"
)

with (carpeta_cierre / "inventario_archivos.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(
        inventario_archivos,
        archivo,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 7. Comprimimos y verificamos el respaldo
# ------------------------------------------------------------

ruta_zip_cierre = Path(
    shutil.make_archive(
        str(carpeta_cierre),
        "zip",
        root_dir=str(carpeta_cierre)
    )
)

archivos_esperados = {
    ruta.name
    for ruta in carpeta_cierre.iterdir()
    if ruta.is_file()
}

with zipfile.ZipFile(ruta_zip_cierre, "r") as respaldo:
    archivo_danado = respaldo.testzip()
    archivos_incluidos = set(respaldo.namelist())

if archivo_danado is not None:
    raise ValueError(
        f"El ZIP contiene un archivo dañado: {archivo_danado}"
    )

if not archivos_esperados.issubset(archivos_incluidos):
    raise ValueError("El ZIP no contiene todos los archivos esperados.")

# ------------------------------------------------------------
# 8. Mostramos los insumos y descargamos a la computadora
# ------------------------------------------------------------

print("\n📁 Carpeta del respaldo en Colab:")
print(carpeta_cierre)

print("\n📦 ZIP que se descargará:")
print(ruta_zip_cierre.name)

print("\n📍 Ruta completa del ZIP en Colab:")
print(ruta_zip_cierre)

print("\n📋 Archivos incluidos:")
for nombre in sorted(archivos_esperados):
    print(f"  • {nombre}")

print("\n➡️ INSUMOS PARA LA SIGUIENTE ETAPA:")
print("  • muestra_actualizada.csv: continuar la preparación de prueba.")
print("  • configuracion_para_prueba.json: recuperar la configuración.")
print("  • desarrollo_utilizado.csv: recuperar desarrollo validado.")
print("  • registros_retirados.csv: conservar la auditoría.")
print("  • LEEME.txt: consultar el contenido y uso de los archivos.")

print("\nConsultas de desarrollo respaldadas:",
      len(df_comparacion_desarrollo))
print("Registros retirados conservados:",
      len(df_registros_retirados))
print("Número de archivos incluidos:", len(archivos_esperados))

print("\n⚠️ Este respaldo todavía no es el dataset final de prueba.")
print("Prueba pendiente de completar traducción y validación.")

print("\n" + "=" * 60)
print("✅ Respaldo creado y ZIP verificado.")
print("=" * 60)

print("\n📥 Iniciando descarga del ZIP a tu computadora...")
files.download(str(ruta_zip_cierre))

print("\nℹ️ La descarga fue solicitada al navegador.")
print("Comprueba que el ZIP aparezca en tu carpeta de descargas.")
print("Conserva este ZIP para recuperar los insumos en otra sesión.")

📦 Preparando respaldo de desarrollo y experimentos...

📁 Carpeta del respaldo en Colab:
/content/revision_evaluacion/cierre_desarrollo_20261010T191612134285Z

📦 ZIP que se descargará:
cierre_desarrollo_20261010T191612134285Z.zip

📍 Ruta completa del ZIP en Colab:
/content/revision_evaluacion/cierre_desarrollo_20261010T191612134285Z.zip

📋 Archivos incluidos:
  • LEEME.txt
  • categorias_alternativas_mdeberta.csv
  • categorias_alternativas_minilm.csv
  • categorias_alternativas_xlm_roberta.csv
  • categorias_iniciales_mdeberta.csv
  • categorias_iniciales_minilm.csv
  • categorias_iniciales_xlm_roberta.csv
  • configuracion_para_prueba.json
  • desarrollo_utilizado.csv
  • efecto_descripciones.csv
  • inventario_archivos.json
  • metricas_alternativas.csv
  • metricas_iniciales.csv
  • muestra_actualizada.csv
  • predicciones_alternativas_mdeberta.csv
  • predicciones_alternativas_minilm.csv
  • predicciones_alternativas_xlm_roberta.csv
  • predicciones_iniciales_mdeberta.csv
  • predi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


ℹ️ La descarga fue solicitada al navegador.
Comprueba que el ZIP aparezca en tu carpeta de descargas.
Conserva este ZIP para recuperar los insumos en otra sesión.


## **💻 6. Preparación del conjunto de prueba**

Se traducirán y revisarán las 240 consultas de prueba,
distribuidas inicialmente en 40 consultas por categoría.

Se aplicarán los criterios de etiquetado definidos durante
la preparación del conjunto de desarrollo.

Antes de evaluar los modelos, se comprobarán la pertinencia,
las traducciones, las etiquetas y los duplicados dentro
de prueba y entre desarrollo y prueba.

La configuración seleccionada permanecerá fija.
Las predicciones de prueba no se utilizarán para ajustar
las descripciones ni las etiquetas de referencia.

### **🧠 6.1 Revisión de LIFE_INSURANCE: primer bloque**

Se mostrarán las primeras diez consultas de prueba
de LIFE_INSURANCE para revisar su significado y preparar
las traducciones propuestas.

**Estado:** Primer bloque pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 6.1 Revisión de LIFE_INSURANCE: primer bloque
# ============================================================

# Recuperamos las consultas actuales del conjunto de prueba
df_prueba_revision = (
    df_muestra_revision[
        df_muestra_revision["conjunto"].eq("prueba")
    ]
    .sort_values("id_consulta")
    .copy()
)

# Comprobamos el tamaño y la distribución antes de comenzar
distribucion_prueba = (
    df_prueba_revision["categoria_original"]
    .value_counts()
    .reindex(categorias_candidatas, fill_value=0)
)

if (
    len(df_prueba_revision) != 240
    or not df_prueba_revision["id_consulta"].is_unique
    or not distribucion_prueba.eq(40).all()
):
    raise ValueError("El conjunto de prueba no tiene la distribución esperada.")

# Seleccionamos diez consultas sin ejecutar predicciones
df_bloque_prueba = df_prueba_revision[
    df_prueba_revision["categoria_original"].eq("LIFE_INSURANCE")
].iloc[:10].copy()

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_prueba[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Primer bloque de prueba presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
10,CONS_0011,need to see my fucking life insurance could uhelp me,LIFE_INSURANCE,information_life_insurance
11,CONS_0012,how can I get informationm about my fucking life insurance?,LIFE_INSURANCE,information_life_insurance
12,CONS_0013,"I would like to see my life insurance, where can I do it?",LIFE_INSURANCE,information_life_insurance
13,CONS_0014,"I have to hceck my life insurance, where do I do it?",LIFE_INSURANCE,information_life_insurance
14,CONS_0015,"I would like to know about my life insurance, find some information",LIFE_INSURANCE,information_life_insurance
15,CONS_0016,wanna check my fuycking life insurance where could i do it,LIFE_INSURANCE,information_life_insurance
16,CONS_0017,how could I see my fucking life insurance?,LIFE_INSURANCE,information_life_insurance
17,CONS_0018,"I'd like to check my fucking life insurance, where do I do it?",LIFE_INSURANCE,information_life_insurance
18,CONS_0019,need to check my fucking life insurance how could i do it,LIFE_INSURANCE,information_life_insurance
19,CONS_0020,i need information about my life insurance where could i find it,LIFE_INSURANCE,information_life_insurance


✅ Primer bloque de prueba presentado para revisión.


### **🧠 6.2 Traducción de LIFE_INSURANCE: primer bloque**

Se propondrán traducciones para CONS_0011 a CONS_0020,
conservando el significado y el tono original.

Se asignará LIFE_INSURANCE como etiqueta propuesta según
el criterio de consulta general del seguro de vida.

Se comprobarán traducciones repetidas entre los registros
ya traducidos, incluidas coincidencias entre desarrollo y prueba.
Los registros no se eliminarán automáticamente.

**Estado:** Traducciones y etiquetas propuestas.
Revisión de coincidencias y validación pendientes.

In [ ]:
# ============================================================
# 6.2 Traducción de LIFE_INSURANCE: primer bloque
# ============================================================

traducciones_prueba_vida_1 = {
    "CONS_0011": (
        "Necesito consultar mi seguro de vida, ¿puedes ayudarme?"
    ),
    "CONS_0012": (
        "¿Cómo puedo obtener información sobre mi seguro de vida?"
    ),
    "CONS_0013": (
        "Quisiera consultar mi seguro de vida, ¿dónde puedo hacerlo?"
    ),
    "CONS_0014": (
        "Tengo que revisar mi seguro de vida, ¿dónde puedo hacerlo?"
    ),
    "CONS_0015": (
        "Quisiera saber sobre mi seguro de vida, consigue información."
    ),
    "CONS_0016": (
        "Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?"
    ),
    "CONS_0017": (
        "¿Cómo puedo consultar mi seguro de vida?"
    ),
    "CONS_0018": (
        "Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?"
    ),
    "CONS_0019": (
        "Necesito revisar mi seguro de vida, ¿cómo puedo hacerlo?"
    ),
    "CONS_0020": (
        "Necesito información sobre mi seguro de vida, "
        "¿dónde puedo encontrarla?"
    )
}

# Verificamos la pertenencia de los registros al bloque
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_prueba_vida_1
)
bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != 10
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("prueba").all()
    or not bloque["categoria_original"].eq("LIFE_INSURANCE").all()
    or bloque["consulta_original"]
        .map(detectar_referencias_producto).map(bool).any()
):
    raise ValueError("El bloque no cumple las condiciones esperadas.")

# Registramos propuestas sin modificar la configuración de los modelos
muestra_actualizada = df_muestra_revision.copy()

for identificador, traduccion in traducciones_prueba_vida_1.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[
        mascara, "categoria_referencia"
    ] = "LIFE_INSURANCE"
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Consulta de información sobre el seguro de vida propio. "
        "Traducción y etiqueta propuestas; validación pendiente."
    )

df_muestra_revision = muestra_actualizada.copy()

# Buscamos duplicados entre todos los registros ya traducidos
revision_traducidos = df_muestra_revision.copy()
revision_traducidos["texto_es_normalizado"] = (
    revision_traducidos["consulta_es"].map(normalizar_contexto)
)

revision_traducidos = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].ne("")
].copy()

duplicados = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].duplicated(keep=False)
].copy()

# Mostramos únicamente los grupos que involucran este bloque
textos_del_bloque = set(
    revision_traducidos.loc[
        revision_traducidos["id_consulta"].isin(traducciones_prueba_vida_1),
        "texto_es_normalizado"
    ]
)

df_coincidencias_prueba_vida_1 = duplicados[
    duplicados["texto_es_normalizado"].isin(textos_del_bloque)
].sort_values(["texto_es_normalizado", "id_consulta"])

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

    print(
        "Grupos de traducciones repetidas en este bloque:",
        df_coincidencias_prueba_vida_1["texto_es_normalizado"].nunique()
    )

    if not df_coincidencias_prueba_vida_1.empty:
        display(
            df_coincidencias_prueba_vida_1[
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ]
        )

print("=" * 60)
print("✅ Primer bloque traducido y coincidencias comprobadas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
10,CONS_0011,"Necesito consultar mi seguro de vida, ¿puedes ayudarme?",LIFE_INSURANCE,propuesta_pendiente_revision
11,CONS_0012,¿Cómo puedo obtener información sobre mi seguro de vida?,LIFE_INSURANCE,propuesta_pendiente_revision
12,CONS_0013,"Quisiera consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
13,CONS_0014,"Tengo que revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
14,CONS_0015,"Quisiera saber sobre mi seguro de vida, consigue información.",LIFE_INSURANCE,propuesta_pendiente_revision
15,CONS_0016,"Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
16,CONS_0017,¿Cómo puedo consultar mi seguro de vida?,LIFE_INSURANCE,propuesta_pendiente_revision
17,CONS_0018,"Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
18,CONS_0019,"Necesito revisar mi seguro de vida, ¿cómo puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
19,CONS_0020,"Necesito información sobre mi seguro de vida, ¿dónde puedo encontrarla?",LIFE_INSURANCE,propuesta_pendiente_revision


Grupos de traducciones repetidas en este bloque: 2


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
8,CONS_0009,desarrollo,wana check my life insurance where can i do it,"Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
15,CONS_0016,prueba,wanna check my fuycking life insurance where could i do it,"Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
3,CONS_0004,desarrollo,"I would like to check my life insurance, where can I do it?","Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
17,CONS_0018,prueba,"I'd like to check my fucking life insurance, where do I do it?","Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE


✅ Primer bloque traducido y coincidencias comprobadas.


### **🧠 6.3 Revisión de LIFE_INSURANCE: segundo bloque**

Se examinarán las siguientes diez consultas de prueba
de LIFE_INSURANCE, correspondientes a CONS_0021 a CONS_0030.

Se revisarán los textos originales y sus intenciones
antes de preparar las traducciones propuestas.

**Estado:** Segundo bloque pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 6.3 Revisión de LIFE_INSURANCE: segundo bloque
# ============================================================

# Seleccionamos explícitamente los identificadores del segundo bloque
ids_bloque_2 = [
    f"CONS_{numero:04d}"
    for numero in range(21, 31)
]

df_bloque_prueba = (
    df_muestra_revision[
        df_muestra_revision["id_consulta"].isin(ids_bloque_2)
    ]
    .sort_values("id_consulta")
    .copy()
)

# Comprobamos que los registros pertenezcan al bloque esperado
if (
    len(df_bloque_prueba) != 10
    or not df_bloque_prueba["id_consulta"].is_unique
    or not df_bloque_prueba["conjunto"].eq("prueba").all()
    or not df_bloque_prueba["categoria_original"]
        .eq("LIFE_INSURANCE").all()
):
    raise ValueError("El segundo bloque no coincide con lo esperado.")

# Mostramos los textos completos para su revisión
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_prueba[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Segundo bloque de LIFE_INSURANCE presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
20,CONS_0021,how to see my life insuurance,LIFE_INSURANCE,information_life_insurance
21,CONS_0022,"I am looking for information about my fucking life insurance, could you help me?",LIFE_INSURANCE,information_life_insurance
22,CONS_0023,wanna know more about my life insurance can ya find some information,LIFE_INSURANCE,information_life_insurance
23,CONS_0024,"I need to check my fucking life insurance, could you help me?",LIFE_INSURANCE,information_life_insurance
24,CONS_0025,"I want to see my ilfe insurance, I need assistance",LIFE_INSURANCE,information_life_insurance
25,CONS_0026,I'm looking for information about my fucking life insurance i need assistance,LIFE_INSURANCE,information_life_insurance
26,CONS_0027,i want information about my fucking life insurance how could i get it,LIFE_INSURANCE,information_life_insurance
27,CONS_0028,is it possible to check my life insurance?,LIFE_INSURANCE,information_life_insurance
28,CONS_0029,"I want to see my fucking life insurance, where to do it?",LIFE_INSURANCE,information_life_insurance
29,CONS_0030,i need information about my fucking life insurance how could i get it,LIFE_INSURANCE,information_life_insurance


✅ Segundo bloque de LIFE_INSURANCE presentado para revisión.


### **🧠 6.4 Traducción de LIFE_INSURANCE: segundo bloque**

Se propondrán traducciones para CONS_0021 a CONS_0030,
conservando el significado y el tono original.

Se asignará LIFE_INSURANCE como etiqueta propuesta.
Se comprobarán traducciones repetidas entre todos los registros
traducidos hasta este momento.

Las coincidencias se mostrarán para revisión, sin modificar
las traducciones únicamente para hacerlas diferentes.

**Estado:** Traducciones y etiquetas propuestas.
Revisión de coincidencias y validación pendientes.

In [ ]:
# ============================================================
# 6.4 Traducción de LIFE_INSURANCE: segundo bloque
# ============================================================

# Definimos traducciones que conservan el significado y el tono
traducciones_prueba_vida_2 = {
    "CONS_0021": (
        "¿Cómo puedo consultar mi seguro de vida?"
    ),
    "CONS_0022": (
        "Busco información sobre mi seguro de vida, "
        "¿puedes ayudarme?"
    ),
    "CONS_0023": (
        "Quiero saber más sobre mi seguro de vida, "
        "¿puedes conseguir información?"
    ),
    "CONS_0024": (
        "Necesito revisar mi seguro de vida, "
        "¿puedes ayudarme?"
    ),
    "CONS_0025": (
        "Quiero consultar mi seguro de vida, necesito ayuda."
    ),
    "CONS_0026": (
        "Busco información sobre mi seguro de vida, "
        "necesito ayuda."
    ),
    "CONS_0027": (
        "Quiero información sobre mi seguro de vida, "
        "¿cómo puedo obtenerla?"
    ),
    "CONS_0028": (
        "¿Es posible revisar mi seguro de vida?"
    ),
    "CONS_0029": (
        "Quiero consultar mi seguro de vida, "
        "¿dónde puedo hacerlo?"
    ),
    "CONS_0030": (
        "Necesito información sobre mi seguro de vida, "
        "¿cómo puedo obtenerla?"
    )
}

# Verificamos los registros antes de modificar la muestra
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_prueba_vida_2
)

bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != 10
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("prueba").all()
    or not bloque["categoria_original"].eq("LIFE_INSURANCE").all()
    or bloque["consulta_original"]
        .map(detectar_referencias_producto).map(bool).any()
):
    raise ValueError("El segundo bloque requiere revisión.")

# Registramos las propuestas conservando la configuración de prueba
muestra_actualizada = df_muestra_revision.copy()

for identificador, traduccion in traducciones_prueba_vida_2.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[
        mascara, "categoria_referencia"
    ] = "LIFE_INSURANCE"
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Consulta de información sobre el seguro de vida propio. "
        "Traducción y etiqueta propuestas; validación pendiente."
    )

df_muestra_revision = muestra_actualizada.copy()

# Buscamos coincidencias entre todos los registros ya traducidos
revision_traducidos = df_muestra_revision.copy()

revision_traducidos["texto_es_normalizado"] = (
    revision_traducidos["consulta_es"].map(normalizar_contexto)
)

revision_traducidos = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].ne("")
].copy()

duplicados = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].duplicated(keep=False)
].copy()

# Mostramos todos los miembros de los grupos que involucran este bloque
textos_del_bloque = set(
    revision_traducidos.loc[
        revision_traducidos["id_consulta"].isin(
            traducciones_prueba_vida_2
        ),
        "texto_es_normalizado"
    ]
)

df_coincidencias_prueba_vida_2 = duplicados[
    duplicados["texto_es_normalizado"].isin(textos_del_bloque)
].sort_values(["texto_es_normalizado", "id_consulta"])

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

    print(
        "Grupos de traducciones repetidas en este bloque:",
        df_coincidencias_prueba_vida_2[
            "texto_es_normalizado"
        ].nunique()
    )

    if not df_coincidencias_prueba_vida_2.empty:
        display(
            df_coincidencias_prueba_vida_2[
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ]
        )

print("=" * 60)
print("✅ Segundo bloque traducido y coincidencias comprobadas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
20,CONS_0021,¿Cómo puedo consultar mi seguro de vida?,LIFE_INSURANCE,propuesta_pendiente_revision
21,CONS_0022,"Busco información sobre mi seguro de vida, ¿puedes ayudarme?",LIFE_INSURANCE,propuesta_pendiente_revision
22,CONS_0023,"Quiero saber más sobre mi seguro de vida, ¿puedes conseguir información?",LIFE_INSURANCE,propuesta_pendiente_revision
23,CONS_0024,"Necesito revisar mi seguro de vida, ¿puedes ayudarme?",LIFE_INSURANCE,propuesta_pendiente_revision
24,CONS_0025,"Quiero consultar mi seguro de vida, necesito ayuda.",LIFE_INSURANCE,propuesta_pendiente_revision
25,CONS_0026,"Busco información sobre mi seguro de vida, necesito ayuda.",LIFE_INSURANCE,propuesta_pendiente_revision
26,CONS_0027,"Quiero información sobre mi seguro de vida, ¿cómo puedo obtenerla?",LIFE_INSURANCE,propuesta_pendiente_revision
27,CONS_0028,¿Es posible revisar mi seguro de vida?,LIFE_INSURANCE,propuesta_pendiente_revision
28,CONS_0029,"Quiero consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
29,CONS_0030,"Necesito información sobre mi seguro de vida, ¿cómo puedo obtenerla?",LIFE_INSURANCE,propuesta_pendiente_revision


Grupos de traducciones repetidas en este bloque: 1


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
16,CONS_0017,prueba,how could I see my fucking life insurance?,¿Cómo puedo consultar mi seguro de vida?,LIFE_INSURANCE
20,CONS_0021,prueba,how to see my life insuurance,¿Cómo puedo consultar mi seguro de vida?,LIFE_INSURANCE


✅ Segundo bloque traducido y coincidencias comprobadas.


### **🧠 6.5 Revisión de LIFE_INSURANCE: tercer bloque**

Se examinarán las diez consultas de prueba correspondientes
a CONS_0031 a CONS_0040.

Se revisarán sus textos originales e intenciones antes
de preparar las traducciones y etiquetas propuestas.

**Estado:** Tercer bloque pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 6.5 Revisión de LIFE_INSURANCE: tercer bloque
# ============================================================

# Seleccionamos los identificadores del tercer bloque
ids_bloque_3 = [
    f"CONS_{numero:04d}"
    for numero in range(31, 41)
]

df_bloque_prueba = (
    df_muestra_revision[
        df_muestra_revision["id_consulta"].isin(ids_bloque_3)
    ]
    .sort_values("id_consulta")
    .copy()
)

# Verificamos la pertenencia al bloque esperado
if (
    len(df_bloque_prueba) != 10
    or not df_bloque_prueba["id_consulta"].is_unique
    or not df_bloque_prueba["conjunto"].eq("prueba").all()
    or not df_bloque_prueba["categoria_original"]
        .eq("LIFE_INSURANCE").all()
):
    raise ValueError("El tercer bloque no coincide con lo esperado.")

# Mostramos los textos completos para revisión
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_prueba[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Tercer bloque de LIFE_INSURANCE presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
30,CONS_0031,can you help me to check my fucking life insurance?,LIFE_INSURANCE,information_life_insurance
31,CONS_0032,could i check my life insurance,LIFE_INSURANCE,information_life_insurance
32,CONS_0033,"I wantto see my fucking life insurance, where to do it?",LIFE_INSURANCE,information_life_insurance
33,CONS_0034,"I have to see my fucking life insurance, how can I do it?",LIFE_INSURANCE,information_life_insurance
34,CONS_0035,"I would like to know about my life insurance, can yoi give me some information?",LIFE_INSURANCE,information_life_insurance
35,CONS_0036,"I would like to see my fucking life insurance, can you help me?",LIFE_INSURANCE,information_life_insurance
36,CONS_0037,where to check my fucking life insurance,LIFE_INSURANCE,information_life_insurance
37,CONS_0038,need to see my fucking life insurance,LIFE_INSURANCE,information_life_insurance
38,CONS_0039,"I'd like information about my life insurance, where could I find it?",LIFE_INSURANCE,information_life_insurance
39,CONS_0040,"I'd like to know more about my life insurance, will you find some information ?",LIFE_INSURANCE,information_life_insurance


✅ Tercer bloque de LIFE_INSURANCE presentado para revisión.


### **🧠 6.6 Traducción de LIFE_INSURANCE: tercer bloque**

Se propondrán traducciones para CONS_0031 a CONS_0040,
conservando el significado y el tono original.

CONS_0032 presenta una construcción incompleta.
Su traducción será una interpretación contextual que
requiere revisión específica.

Se comprobarán coincidencias textuales con todos los registros
traducidos, incluidos desarrollo y los bloques previos de prueba.

**Estado:** Traducciones y etiquetas propuestas.
Revisión de coincidencias y validación pendientes.

In [ ]:
# ============================================================
# 6.6 Traducción de LIFE_INSURANCE: tercer bloque
# ============================================================

# Definimos traducciones fieles sin introducir diferencias artificiales
traducciones_prueba_vida_3 = {
    "CONS_0031": "¿Puedes ayudarme a revisar mi seguro de vida?",
    "CONS_0032": "¿Podría revisar mi seguro de vida?",
    "CONS_0033": (
        "Quiero consultar mi seguro de vida, "
        "¿dónde puedo hacerlo?"
    ),
    "CONS_0034": (
        "Tengo que consultar mi seguro de vida, "
        "¿cómo puedo hacerlo?"
    ),
    "CONS_0035": (
        "Quisiera saber sobre mi seguro de vida, "
        "¿puedes darme información?"
    ),
    "CONS_0036": (
        "Quisiera consultar mi seguro de vida, "
        "¿puedes ayudarme?"
    ),
    "CONS_0037": "¿Dónde puedo revisar mi seguro de vida?",
    "CONS_0038": "Necesito consultar mi seguro de vida.",
    "CONS_0039": (
        "Quisiera información sobre mi seguro de vida, "
        "¿dónde puedo encontrarla?"
    ),
    "CONS_0040": (
        "Quisiera saber más sobre mi seguro de vida, "
        "¿puedes conseguir información?"
    )
}

# Verificamos los registros antes de modificar la muestra
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_prueba_vida_3
)
bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != 10
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("prueba").all()
    or not bloque["categoria_original"].eq("LIFE_INSURANCE").all()
    or bloque["consulta_original"]
        .map(detectar_referencias_producto).map(bool).any()
):
    raise ValueError("El tercer bloque requiere revisión.")

# Registramos propuestas sin cambiar la configuración de prueba
muestra_actualizada = df_muestra_revision.copy()

for identificador, traduccion in traducciones_prueba_vida_3.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[
        mascara, "categoria_referencia"
    ] = "LIFE_INSURANCE"
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Consulta de información sobre el seguro de vida propio. "
        "Traducción y etiqueta propuestas; validación pendiente."
    )

# Documentamos la incertidumbre del original incompleto
muestra_actualizada.loc[
    muestra_actualizada["id_consulta"].eq("CONS_0032"),
    "observaciones"
] = (
    "Original incompleto: could i check my life insurance. "
    "Traducción contextual propuesta; requiere revisión específica."
)

df_muestra_revision = muestra_actualizada.copy()

# Buscamos coincidencias entre todos los registros ya traducidos
revision_traducidos = df_muestra_revision.copy()
revision_traducidos["texto_es_normalizado"] = (
    revision_traducidos["consulta_es"].map(normalizar_contexto)
)
revision_traducidos = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].ne("")
].copy()

duplicados = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].duplicated(keep=False)
].copy()

textos_del_bloque = set(
    revision_traducidos.loc[
        revision_traducidos["id_consulta"].isin(
            traducciones_prueba_vida_3
        ),
        "texto_es_normalizado"
    ]
)

df_coincidencias_prueba_vida_3 = duplicados[
    duplicados["texto_es_normalizado"].isin(textos_del_bloque)
].sort_values(["texto_es_normalizado", "id_consulta"])

# Mostramos las propuestas y todos los miembros de grupos repetidos
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

    print(
        "Grupos de traducciones repetidas en este bloque:",
        df_coincidencias_prueba_vida_3["texto_es_normalizado"].nunique()
    )

    if not df_coincidencias_prueba_vida_3.empty:
        display(
            df_coincidencias_prueba_vida_3[
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ]
        )

print("=" * 60)
print("✅ Tercer bloque traducido y coincidencias comprobadas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
30,CONS_0031,¿Puedes ayudarme a revisar mi seguro de vida?,LIFE_INSURANCE,propuesta_pendiente_revision
31,CONS_0032,¿Podría revisar mi seguro de vida?,LIFE_INSURANCE,propuesta_pendiente_revision
32,CONS_0033,"Quiero consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
33,CONS_0034,"Tengo que consultar mi seguro de vida, ¿cómo puedo hacerlo?",LIFE_INSURANCE,propuesta_pendiente_revision
34,CONS_0035,"Quisiera saber sobre mi seguro de vida, ¿puedes darme información?",LIFE_INSURANCE,propuesta_pendiente_revision
35,CONS_0036,"Quisiera consultar mi seguro de vida, ¿puedes ayudarme?",LIFE_INSURANCE,propuesta_pendiente_revision
36,CONS_0037,¿Dónde puedo revisar mi seguro de vida?,LIFE_INSURANCE,propuesta_pendiente_revision
37,CONS_0038,Necesito consultar mi seguro de vida.,LIFE_INSURANCE,propuesta_pendiente_revision
38,CONS_0039,"Quisiera información sobre mi seguro de vida, ¿dónde puedo encontrarla?",LIFE_INSURANCE,propuesta_pendiente_revision
39,CONS_0040,"Quisiera saber más sobre mi seguro de vida, ¿puedes conseguir información?",LIFE_INSURANCE,propuesta_pendiente_revision


Grupos de traducciones repetidas en este bloque: 1


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
28,CONS_0029,prueba,"I want to see my fucking life insurance, where to do it?","Quiero consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
32,CONS_0033,prueba,"I wantto see my fucking life insurance, where to do it?","Quiero consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE


✅ Tercer bloque traducido y coincidencias comprobadas.


### **🧠 6.7 Revisión de LIFE_INSURANCE: cuarto bloque**

Se conservará CONS_0029 y se registrará CONS_0033 como
pendiente de reemplazo por traducción duplicada en prueba.

Se examinarán las diez consultas restantes de la categoría,
correspondientes a CONS_0041 a CONS_0050.

Los reemplazos se seleccionarán después de completar
la traducción de la categoría y revisar sus coincidencias.

**Estado:** Cuarto bloque pendiente de traducción.
Un duplicado pendiente de reemplazo.

In [ ]:
# ============================================================
# 6.7 Revisión de LIFE_INSURANCE: cuarto bloque
# ============================================================

# Comprobamos el duplicado antes de registrar su estado
par = df_muestra_revision[
    df_muestra_revision["id_consulta"].isin(
        ["CONS_0029", "CONS_0033"]
    )
].copy()

if (
    len(par) != 2
    or not par["id_consulta"].is_unique
    or not par["conjunto"].eq("prueba").all()
    or par["consulta_es"].map(normalizar_contexto).eq("").any()
    or par["consulta_es"].map(normalizar_contexto).nunique() != 1
):
    raise ValueError("El par no coincide con el duplicado esperado.")

# Preparamos el cuarto bloque
ids_bloque_4 = [
    f"CONS_{numero:04d}"
    for numero in range(41, 51)
]

df_bloque_prueba = (
    df_muestra_revision[
        df_muestra_revision["id_consulta"].isin(ids_bloque_4)
    ]
    .sort_values("id_consulta")
    .copy()
)

if (
    len(df_bloque_prueba) != 10
    or not df_bloque_prueba["id_consulta"].is_unique
    or not df_bloque_prueba["conjunto"].eq("prueba").all()
    or not df_bloque_prueba["categoria_original"]
        .eq("LIFE_INSURANCE").all()
):
    raise ValueError("El cuarto bloque no coincide con lo esperado.")

# Registramos el caso pendiente sin eliminarlo
mascara_duplicado = df_muestra_revision["id_consulta"].eq("CONS_0033")

df_muestra_revision.loc[
    mascara_duplicado, "estado_revision"
] = "reemplazo_pendiente_duplicado"

df_muestra_revision.loc[
    mascara_duplicado, "observaciones"
] = (
    "Traducción idéntica a CONS_0029, también de prueba. "
    "Se conserva CONS_0029; CONS_0033 requiere reemplazo."
)

# Mostramos los textos originales del cuarto bloque
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_prueba[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Duplicado registrado y cuarto bloque presentado.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
40,CONS_0041,"I need information about my life insurance, how coulf I find it?",LIFE_INSURANCE,information_life_insurance
41,CONS_0042,"I need information about my life insurance, how could I get it?",LIFE_INSURANCE,information_life_insurance
42,CONS_0043,I need assistance to see my life insurance,LIFE_INSURANCE,information_life_insurance
43,CONS_0044,i have to check my fuucking life insurance can i get some help,LIFE_INSURANCE,information_life_insurance
44,CONS_0045,"I'd like to know about my life insurance, how can I find more information?",LIFE_INSURANCE,information_life_insurance
45,CONS_0046,"I'd like information about my life insurance, where can I get it?",LIFE_INSURANCE,information_life_insurance
46,CONS_0047,where could i find information about my fucking life insurance,LIFE_INSURANCE,information_life_insurance
47,CONS_0048,i got to check mny fucking life insurance,LIFE_INSURANCE,information_life_insurance
48,CONS_0049,"I'd like to check my life insurance, can I get soem help?",LIFE_INSURANCE,information_life_insurance
49,CONS_0050,"I'm looking for information about my life insurance, how could I get it?",LIFE_INSURANCE,information_life_insurance


✅ Duplicado registrado y cuarto bloque presentado.


### **🧠 6.8 Traducción de LIFE_INSURANCE: cuarto bloque**

Se propondrán traducciones para CONS_0041 a CONS_0050,
conservando el significado y el tono original.

Se comprobarán traducciones repetidas entre todos los registros
traducidos hasta este momento, incluidos desarrollo y prueba.

Las coincidencias se conservarán para su revisión.
No se introducirán diferencias artificiales en las traducciones.

**Estado:** Cuarenta consultas de prueba de LIFE_INSURANCE
con traducción propuesta. Resolución de duplicados
y validación pendientes.

In [ ]:
# ============================================================
# 6.8 Traducción de LIFE_INSURANCE: cuarto bloque
# ============================================================

traducciones_prueba_vida_4 = {
    "CONS_0041": (
        "Necesito información sobre mi seguro de vida, "
        "¿cómo puedo encontrarla?"
    ),
    "CONS_0042": (
        "Necesito información sobre mi seguro de vida, "
        "¿cómo puedo obtenerla?"
    ),
    "CONS_0043": (
        "Necesito ayuda para consultar mi seguro de vida."
    ),
    "CONS_0044": (
        "Tengo que revisar mi seguro de vida, "
        "¿me pueden ayudar?"
    ),
    "CONS_0045": (
        "Quisiera saber sobre mi seguro de vida, "
        "¿cómo puedo encontrar más información?"
    ),
    "CONS_0046": (
        "Quisiera información sobre mi seguro de vida, "
        "¿dónde puedo obtenerla?"
    ),
    "CONS_0047": (
        "¿Dónde puedo encontrar información sobre mi seguro de vida?"
    ),
    "CONS_0048": (
        "Tengo que revisar mi seguro de vida."
    ),
    "CONS_0049": (
        "Quisiera revisar mi seguro de vida, ¿me pueden ayudar?"
    ),
    "CONS_0050": (
        "Busco información sobre mi seguro de vida, "
        "¿cómo puedo obtenerla?"
    )
}

# Verificamos el cuarto bloque antes de registrar las propuestas
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_prueba_vida_4
)

bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != 10
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("prueba").all()
    or not bloque["categoria_original"].eq("LIFE_INSURANCE").all()
    or bloque["consulta_original"]
        .map(detectar_referencias_producto).map(bool).any()
):
    raise ValueError("El cuarto bloque requiere revisión.")

# Registramos propuestas sin cambiar la configuración de prueba
muestra_actualizada = df_muestra_revision.copy()

for identificador, traduccion in traducciones_prueba_vida_4.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[
        mascara, "categoria_referencia"
    ] = "LIFE_INSURANCE"
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Consulta de información sobre el seguro de vida propio. "
        "Traducción y etiqueta propuestas; validación pendiente."
    )

df_muestra_revision = muestra_actualizada.copy()

# Recuperamos todos los registros traducidos
revision_traducidos = df_muestra_revision.copy()

revision_traducidos["texto_es_normalizado"] = (
    revision_traducidos["consulta_es"].map(normalizar_contexto)
)

revision_traducidos = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].ne("")
].copy()

duplicados = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].duplicated(keep=False)
].copy()

# Incluimos los grupos repetidos de toda la categoría de prueba
textos_prueba_vida = set(
    revision_traducidos.loc[
        revision_traducidos["conjunto"].eq("prueba")
        & revision_traducidos["categoria_original"].eq("LIFE_INSURANCE"),
        "texto_es_normalizado"
    ]
)

df_coincidencias_prueba_vida = duplicados[
    duplicados["texto_es_normalizado"].isin(textos_prueba_vida)
].sort_values(["texto_es_normalizado", "conjunto", "id_consulta"])

# Mostramos propuestas y coincidencias, sin eliminar registros
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

    print(
        "Grupos repetidos que involucran prueba de LIFE_INSURANCE:",
        df_coincidencias_prueba_vida[
            "texto_es_normalizado"
        ].nunique()
    )

    if not df_coincidencias_prueba_vida.empty:
        display(
            df_coincidencias_prueba_vida[
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ]
        )

print("=" * 60)
print("✅ Cuarto bloque traducido y coincidencias de la categoría comprobadas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
40,CONS_0041,"Necesito información sobre mi seguro de vida, ¿cómo puedo encontrarla?",LIFE_INSURANCE,propuesta_pendiente_revision
41,CONS_0042,"Necesito información sobre mi seguro de vida, ¿cómo puedo obtenerla?",LIFE_INSURANCE,propuesta_pendiente_revision
42,CONS_0043,Necesito ayuda para consultar mi seguro de vida.,LIFE_INSURANCE,propuesta_pendiente_revision
43,CONS_0044,"Tengo que revisar mi seguro de vida, ¿me pueden ayudar?",LIFE_INSURANCE,propuesta_pendiente_revision
44,CONS_0045,"Quisiera saber sobre mi seguro de vida, ¿cómo puedo encontrar más información?",LIFE_INSURANCE,propuesta_pendiente_revision
45,CONS_0046,"Quisiera información sobre mi seguro de vida, ¿dónde puedo obtenerla?",LIFE_INSURANCE,propuesta_pendiente_revision
46,CONS_0047,¿Dónde puedo encontrar información sobre mi seguro de vida?,LIFE_INSURANCE,propuesta_pendiente_revision
47,CONS_0048,Tengo que revisar mi seguro de vida.,LIFE_INSURANCE,propuesta_pendiente_revision
48,CONS_0049,"Quisiera revisar mi seguro de vida, ¿me pueden ayudar?",LIFE_INSURANCE,propuesta_pendiente_revision
49,CONS_0050,"Busco información sobre mi seguro de vida, ¿cómo puedo obtenerla?",LIFE_INSURANCE,propuesta_pendiente_revision


Grupos repetidos que involucran prueba de LIFE_INSURANCE: 6


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
29,CONS_0030,prueba,i need information about my fucking life insurance how could i get it,"Necesito información sobre mi seguro de vida, ¿cómo puedo obtenerla?",LIFE_INSURANCE
41,CONS_0042,prueba,"I need information about my life insurance, how could I get it?","Necesito información sobre mi seguro de vida, ¿cómo puedo obtenerla?",LIFE_INSURANCE
28,CONS_0029,prueba,"I want to see my fucking life insurance, where to do it?","Quiero consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
32,CONS_0033,prueba,"I wantto see my fucking life insurance, where to do it?","Quiero consultar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
8,CONS_0009,desarrollo,wana check my life insurance where can i do it,"Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
15,CONS_0016,prueba,wanna check my fuycking life insurance where could i do it,"Quiero revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
3,CONS_0004,desarrollo,"I would like to check my life insurance, where can I do it?","Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
17,CONS_0018,prueba,"I'd like to check my fucking life insurance, where do I do it?","Quisiera revisar mi seguro de vida, ¿dónde puedo hacerlo?",LIFE_INSURANCE
0,CONS_0001,desarrollo,i have to check my fucking life insurance can i get some help,"Tengo que revisar mi seguro de vida, ¿me pueden ayudar?",LIFE_INSURANCE
43,CONS_0044,prueba,i have to check my fuucking life insurance can i get some help,"Tengo que revisar mi seguro de vida, ¿me pueden ayudar?",LIFE_INSURANCE


✅ Cuarto bloque traducido y coincidencias de la categoría comprobadas.


### **🧠 6.9 Selección de reemplazos de LIFE_INSURANCE**

Se seleccionarán candidatos para CONS_0033 y CONS_0044.

Se conservará CONS_0029 frente al duplicado dentro de prueba.
Se conservará CONS_0001 de desarrollo frente a la coincidencia
con CONS_0044 de prueba.

Los candidatos conservarán la categoría LIFE_INSURANCE
y la intención information_life_insurance.

Sus traducciones deberán comprobarse frente a todos los textos
ya traducidos antes de incorporarlos.

**Estado:** Dos reemplazos candidatos pendientes de revisión.

In [ ]:
# ============================================================
# 6.9 Selección de reemplazos de LIFE_INSURANCE
# ============================================================

import pandas as pd

print("=" * 60)
print("🔎 Seleccionando seis reemplazos de LIFE_INSURANCE...")
print("=" * 60)

# ------------------------------------------------------------
# 1. Definimos las coincidencias detectadas en 6.8
# ------------------------------------------------------------

# Cada par indica: registro que conservamos, registro que reemplazamos
pares_duplicados_vida = [
    ("CONS_0009", "CONS_0016"),
    ("CONS_0004", "CONS_0018"),
    ("CONS_0017", "CONS_0021"),
    ("CONS_0029", "CONS_0033"),
    ("CONS_0030", "CONS_0042"),
    ("CONS_0001", "CONS_0044")
]

ids_reemplazar_vida = [
    reemplazar
    for conservar, reemplazar in pares_duplicados_vida
]

objetivos = df_muestra_revision.loc[
    df_muestra_revision["id_consulta"].isin(ids_reemplazar_vida)
].copy()

if (
    len(objetivos) != len(ids_reemplazar_vida)
    or not objetivos["id_consulta"].is_unique
    or not objetivos["conjunto"].eq("prueba").all()
    or not objetivos["categoria_original"].eq("LIFE_INSURANCE").all()
    or not objetivos["categoria_referencia"].eq("LIFE_INSURANCE").all()
):
    raise ValueError(
        "Los seis registros no coinciden con los casos esperados."
    )

# ------------------------------------------------------------
# 2. Comprobamos que las coincidencias siguen presentes
# ------------------------------------------------------------

for conservar, reemplazar in pares_duplicados_vida:
    par = df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(
            [conservar, reemplazar]
        )
    ]

    textos = (
        par["consulta_es"]
        .fillna("")
        .map(normalizar_contexto)
    )

    if (
        len(par) != 2
        or not par["id_consulta"].is_unique
        or textos.eq("").any()
        or textos.nunique() != 1
    ):
        raise ValueError(
            f"La coincidencia entre {conservar} y "
            f"{reemplazar} cambió. Revisa la salida de 6.8."
        )

# ------------------------------------------------------------
# 3. Excluimos originales utilizados y registros retirados
# ------------------------------------------------------------

textos_utilizados = set(
    df_muestra_revision["consulta_original"]
    .fillna("")
    .map(normalizar_contexto)
)

textos_utilizados.update(
    df_registros_retirados["consulta_original"]
    .fillna("")
    .map(normalizar_contexto)
)

reserva = df_elegibles.copy()

reserva["texto_revision"] = (
    reserva["instruction"]
    .fillna("")
    .map(normalizar_contexto)
)

# Excluimos referencias a otros productos de seguro
referencias_otros_productos = (
    reserva["instruction"]
    .fillna("")
    .map(detectar_referencias_producto)
    .map(bool)
)

reserva = (
    reserva.loc[
        reserva["texto_revision"].ne("")
        & ~reserva["texto_revision"].isin(textos_utilizados)
        & ~referencias_otros_productos
    ]
    .sort_values("texto_revision")
    .drop_duplicates("texto_revision")
    .copy()
)

# ------------------------------------------------------------
# 4. Seleccionamos candidatos de la misma categoría e intención
# ------------------------------------------------------------

propuestas_vida = []

for posicion, (_, original) in enumerate(
    objetivos.sort_values("id_consulta").iterrows()
):
    candidatos = reserva.loc[
        reserva["category"].eq(original["categoria_original"])
        & reserva["intent"].eq(original["intencion_original"])
        & ~reserva["texto_revision"].isin(textos_utilizados)
    ]

    if candidatos.empty:
        raise ValueError(
            "No hay candidatos disponibles para "
            f"{original['id_consulta']}."
        )

    # Conservamos una selección reproducible
    candidato = candidatos.sample(
        n=1,
        random_state=842 + posicion
    ).iloc[0]

    propuestas_vida.append({
        "id_a_reemplazar": original["id_consulta"],
        "conjunto": original["conjunto"],
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"]
    })

    # Evitamos seleccionar el mismo original para otro reemplazo
    textos_utilizados.add(candidato["texto_revision"])

# ------------------------------------------------------------
# 5. Conservamos las propuestas separadas de la muestra
# ------------------------------------------------------------

propuestas_temporales = pd.DataFrame(propuestas_vida)

if (
    len(propuestas_temporales) != 6
    or not propuestas_temporales["id_a_reemplazar"].is_unique
    or propuestas_temporales["consulta_original"]
        .map(normalizar_contexto).duplicated().any()
):
    raise ValueError(
        "Las propuestas no contienen seis candidatos distintos."
    )

df_reemplazos_prueba_vida = propuestas_temporales.copy()

with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazos_prueba_vida[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("\nCandidatos seleccionados:",
      len(df_reemplazos_prueba_vida))

print("Estado: propuestas pendientes de traducción y revisión.")
print("Las consultas de desarrollo permanecen conservadas.")
print(
    "Siguiente paso: revisar las traducciones y comprobar "
    "que no generen nuevas coincidencias."
)

print("=" * 60)
print("✅ Seis candidatos de LIFE_INSURANCE presentados para revisión.")
print("=" * 60)

🔎 Seleccionando seis reemplazos de LIFE_INSURANCE...


,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
0,CONS_0016,prueba,I'd like to chck my fucking life insurance could i get some help,LIFE_INSURANCE,information_life_insurance
1,CONS_0018,prueba,wanna check my fucking life insuranc can uhelp me,LIFE_INSURANCE,information_life_insurance
2,CONS_0021,prueba,"I'd like to know about my life insuarnce, where can I find more information?",LIFE_INSURANCE,information_life_insurance
3,CONS_0033,prueba,"I want to see my fucking life insurance, where do I do it?",LIFE_INSURANCE,information_life_insurance
4,CONS_0042,prueba,"I want to know about my fucking life insurance, could you give me some information?",LIFE_INSURANCE,information_life_insurance
5,CONS_0044,prueba,"I would like to check my life insurance, I need assistance",LIFE_INSURANCE,information_life_insurance



Candidatos seleccionados: 6
Estado: propuestas pendientes de traducción y revisión.
Las consultas de desarrollo permanecen conservadas.
Siguiente paso: revisar las traducciones y comprobar que no generen nuevas coincidencias.
✅ Seis candidatos de LIFE_INSURANCE presentados para revisión.


In [ ]:
# ============================================================
# 6.9.1 Nueva selección de candidatos que repiten el contenido
# ============================================================

import pandas as pd

print("=" * 60)
print("🔎 Sustituyendo tres candidatos con contenido repetido...")
print("=" * 60)

ids_reseleccionar = [
    "CONS_0016",
    "CONS_0021",
    "CONS_0033"
]

# Trabajamos sobre una copia de las propuestas
propuestas_actualizadas = df_reemplazos_prueba_vida.copy()

if (
    len(propuestas_actualizadas) != 6
    or not propuestas_actualizadas["id_a_reemplazar"].is_unique
    or not set(ids_reseleccionar).issubset(
        set(propuestas_actualizadas["id_a_reemplazar"])
    )
):
    raise ValueError(
        "Ejecuta primero 6.9 y comprueba sus seis propuestas."
    )

# Excluimos los originales actuales, los retirados y todos
# los candidatos anteriores, incluidos los tres descartados
textos_excluidos = set()

for tabla, columna in [
    (df_muestra_revision, "consulta_original"),
    (df_registros_retirados, "consulta_original"),
    (df_reemplazos_prueba_vida, "consulta_original")
]:
    textos_excluidos.update(
        tabla[columna].fillna("").map(normalizar_contexto)
    )

reserva_nueva = df_elegibles.copy()

reserva_nueva["texto_revision"] = (
    reserva_nueva["instruction"]
    .fillna("")
    .map(normalizar_contexto)
)

referencias_otros_productos = (
    reserva_nueva["instruction"]
    .fillna("")
    .map(detectar_referencias_producto)
    .map(bool)
)

reserva_nueva = (
    reserva_nueva.loc[
        reserva_nueva["texto_revision"].ne("")
        & ~reserva_nueva["texto_revision"].isin(textos_excluidos)
        & ~referencias_otros_productos
    ]
    .sort_values("texto_revision")
    .drop_duplicates("texto_revision")
    .copy()
)

# Seleccionamos nuevos originales de la misma categoría e intención
for posicion, identificador in enumerate(ids_reseleccionar):
    mascara = propuestas_actualizadas["id_a_reemplazar"].eq(
        identificador
    )
    propuesta_anterior = propuestas_actualizadas.loc[mascara].iloc[0]

    candidatos = reserva_nueva.loc[
        reserva_nueva["category"].eq(
            propuesta_anterior["categoria_original"]
        )
        & reserva_nueva["intent"].eq(
            propuesta_anterior["intencion_original"]
        )
        & ~reserva_nueva["texto_revision"].isin(textos_excluidos)
    ]

    if candidatos.empty:
        raise ValueError(
            f"No hay candidatos disponibles para {identificador}."
        )

    candidato = candidatos.sample(
        n=1,
        random_state=1842 + posicion
    ).iloc[0]

    campos = {
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"]
    }

    for columna, valor in campos.items():
        propuestas_actualizadas.loc[mascara, columna] = valor

    textos_excluidos.add(candidato["texto_revision"])

# Publicamos las propuestas solo después de completar la selección
df_reemplazos_prueba_vida = propuestas_actualizadas.copy()

with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazos_prueba_vida[
            [
                "id_a_reemplazar",
                "conjunto",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("\nLa muestra todavía no se ha modificado.")
print("Los nuevos candidatos requieren revisión de su traducción.")
print("=" * 60)
print("✅ Propuestas actualizadas y presentadas para revisión.")
print("=" * 60)

🔎 Sustituyendo tres candidatos con contenido repetido...


,id_a_reemplazar,conjunto,consulta_original,categoria_original,intencion_original
0,CONS_0016,prueba,"I would like to know more about my fucking life insuracne, can you give me some information?",LIFE_INSURANCE,information_life_insurance
1,CONS_0018,prueba,wanna check my fucking life insuranc can uhelp me,LIFE_INSURANCE,information_life_insurance
2,CONS_0021,prueba,"I would like to see my fucking life insurance, could I get some help?",LIFE_INSURANCE,information_life_insurance
3,CONS_0033,prueba,"I have ot see my fucking life insurance, will you help me?",LIFE_INSURANCE,information_life_insurance
4,CONS_0042,prueba,"I want to know about my fucking life insurance, could you give me some information?",LIFE_INSURANCE,information_life_insurance
5,CONS_0044,prueba,"I would like to check my life insurance, I need assistance",LIFE_INSURANCE,information_life_insurance



La muestra todavía no se ha modificado.
Los nuevos candidatos requieren revisión de su traducción.
✅ Propuestas actualizadas y presentadas para revisión.


In [ ]:
# ============================================================
# 6.9.2 Nueva selección del candidato para CONS_0021
# ============================================================

print("=" * 60)
print("🔎 Seleccionando otro candidato para CONS_0021...")
print("=" * 60)

identificador = "CONS_0021"

mascara = df_reemplazos_prueba_vida["id_a_reemplazar"].eq(
    identificador
)

if mascara.sum() != 1:
    raise ValueError("Debe existir una propuesta para CONS_0021.")

propuesta = df_reemplazos_prueba_vida.loc[mascara].iloc[0]

# Conservamos las exclusiones acumuladas en 6.9.1
textos_excluidos.update(
    df_reemplazos_prueba_vida["consulta_original"]
    .fillna("")
    .map(normalizar_contexto)
)

candidatos = reserva_nueva.loc[
    reserva_nueva["category"].eq(propuesta["categoria_original"])
    & reserva_nueva["intent"].eq(propuesta["intencion_original"])
    & ~reserva_nueva["texto_revision"].isin(textos_excluidos)
]

if candidatos.empty:
    raise ValueError("No hay candidatos disponibles para CONS_0021.")

candidato = candidatos.sample(
    n=1,
    random_state=2842
).iloc[0]

propuestas_actualizadas = df_reemplazos_prueba_vida.copy()

for columna, valor in {
    "indice_origen": candidato["indice_origen"],
    "consulta_original": candidato["instruction"],
    "categoria_original": candidato["category"],
    "intencion_original": candidato["intent"],
    "consulta_normalizada": candidato["consulta_normalizada"]
}.items():
    propuestas_actualizadas.loc[mascara, columna] = valor

df_reemplazos_prueba_vida = propuestas_actualizadas
textos_excluidos.add(candidato["texto_revision"])

with pd.option_context("display.max_colwidth", None):
    display(
        df_reemplazos_prueba_vida.loc[
            mascara,
            [
                "id_a_reemplazar",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("\nLa muestra todavía no se ha modificado.")
print("=" * 60)
print("✅ Nuevo candidato para CONS_0021 presentado para revisión.")
print("=" * 60)

🔎 Seleccionando otro candidato para CONS_0021...


,id_a_reemplazar,consulta_original,categoria_original,intencion_original
2,CONS_0021,"I would like to check my life insurance, I need help",LIFE_INSURANCE,information_life_insurance



La muestra todavía no se ha modificado.
✅ Nuevo candidato para CONS_0021 presentado para revisión.


In [ ]:
# ============================================================
# 6.9.3 Opciones de reemplazo para CONS_0021
# ============================================================

print("=" * 60)
print("🔎 Preparando opciones para revisar CONS_0021...")
print("=" * 60)

# Conservamos todas las exclusiones de las selecciones anteriores
textos_excluidos.update(
    df_reemplazos_prueba_vida["consulta_original"]
    .fillna("")
    .map(normalizar_contexto)
)

opciones = reserva_nueva.loc[
    reserva_nueva["category"].eq("LIFE_INSURANCE")
    & reserva_nueva["intent"].eq("information_life_insurance")
    & ~reserva_nueva["texto_revision"].isin(textos_excluidos)
].copy()

if opciones.empty:
    raise ValueError("No quedan opciones disponibles para CONS_0021.")

# Presentamos varias opciones sin incorporarlas a la muestra
df_opciones_cons_0021 = (
    opciones.sample(
        n=min(8, len(opciones)),
        random_state=3842
    )
    .reset_index(drop=True)
)

df_opciones_cons_0021.insert(
    0,
    "opcion",
    range(1, len(df_opciones_cons_0021) + 1)
)

with pd.option_context("display.max_colwidth", None):
    display(
        df_opciones_cons_0021[
            ["opcion", "instruction", "category", "intent"]
        ].rename(columns={
            "instruction": "consulta_original",
            "category": "categoria_original",
            "intent": "intencion_original"
        })
    )

print("\nLa muestra y las seis propuestas permanecen sin cambios.")
print("Revisaremos estas opciones antes de elegir el reemplazo.")
print("=" * 60)
print("✅ Opciones presentadas para revisión semántica.")
print("=" * 60)

🔎 Preparando opciones para revisar CONS_0021...


,opcion,consulta_original,categoria_original,intencion_original
0,1,i got to see my life insurance where to do it,LIFE_INSURANCE,information_life_insurance
1,2,im looking for information about my life insurance how can i find it,LIFE_INSURANCE,information_life_insurance
2,3,id like ot check my fucking life insurance how to do it,LIFE_INSURANCE,information_life_insurance
3,4,"I'd like to know about my fucking life insurance, how do I get more information?",LIFE_INSURANCE,information_life_insurance
4,5,i have to see my life insurance could ya help me,LIFE_INSURANCE,information_life_insurance
5,6,i got to check my life insurance where to do it,LIFE_INSURANCE,information_life_insurance
6,7,"I would like to see my fucking life insurance, I need assistance",LIFE_INSURANCE,information_life_insurance
7,8,i have to see my fucking life insurance where can i do it,LIFE_INSURANCE,information_life_insurance



La muestra y las seis propuestas permanecen sin cambios.
Revisaremos estas opciones antes de elegir el reemplazo.
✅ Opciones presentadas para revisión semántica.


### 🧠 6.10 Incorporación y respaldo de reemplazos de LIFE_INSURANCE

Se incorporarán los seis candidatos revisados, conservando sus
identificadores, categoría, intención y conjunto de prueba.

Se comprobará que sus traducciones no coincidan entre sí ni con
otros registros ya traducidos. También se verificará que las
consultas de desarrollo y la distribución de la muestra permanezcan
sin cambios.

Los registros reemplazados se conservarán en la auditoría.
Se generará y descargará un ZIP con la muestra actualizada,
la auditoría, las 40 consultas propuestas de LIFE_INSURANCE
y la configuración para prueba.

**Estado:** Cuarenta traducciones propuestas de LIFE_INSURANCE.
Confirmación del usuario pendiente. El respaldo es un avance;
todavía no constituye el dataset final validado.

In [ ]:
# ============================================================
# 6.10 Incorporación y respaldo de reemplazos de LIFE_INSURANCE
# ============================================================

import pandas as pd
import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("=" * 60)
print("🔎 Comprobando los seis reemplazos de LIFE_INSURANCE...")
print("=" * 60)

# ------------------------------------------------------------
# 1. Actualizamos la propuesta de CONS_0021 con la opción 2
# ------------------------------------------------------------

opcion_elegida = df_opciones_cons_0021.loc[
    df_opciones_cons_0021["opcion"].eq(2)
]

if len(opcion_elegida) != 1:
    raise ValueError("No se encuentra la opción 2 de 6.9.3.")

candidato_elegido = opcion_elegida.iloc[0]
propuestas = df_reemplazos_prueba_vida.copy()

mascara_propuesta = propuestas["id_a_reemplazar"].eq("CONS_0021")

if mascara_propuesta.sum() != 1:
    raise ValueError("Debe existir una propuesta para CONS_0021.")

for columna, valor in {
    "indice_origen": candidato_elegido["indice_origen"],
    "consulta_original": candidato_elegido["instruction"],
    "categoria_original": candidato_elegido["category"],
    "intencion_original": candidato_elegido["intent"],
    "consulta_normalizada": candidato_elegido["consulta_normalizada"]
}.items():
    propuestas.loc[mascara_propuesta, columna] = valor

# ------------------------------------------------------------
# 2. Definimos traducciones sin expresiones ofensivas
# ------------------------------------------------------------

traducciones_vida = {
    "CONS_0016": (
        "Quisiera saber más sobre mi seguro de vida, "
        "¿puedes darme información?"
    ),
    "CONS_0018": (
        "Quiero revisar mi seguro de vida, ¿puedes ayudarme?"
    ),
    "CONS_0021": (
        "Busco información sobre mi seguro de vida, "
        "¿cómo puedo encontrarla?"
    ),
    "CONS_0033": (
        "Tengo que consultar mi seguro de vida, ¿me ayudarás?"
    ),
    "CONS_0042": (
        "Quiero saber sobre mi seguro de vida, "
        "¿podrías darme información?"
    ),
    "CONS_0044": (
        "Quisiera revisar mi seguro de vida, necesito ayuda."
    )
}

ids_esperados = set(traducciones_vida)

if (
    len(propuestas) != 6
    or not propuestas["id_a_reemplazar"].is_unique
    or set(propuestas["id_a_reemplazar"]) != ids_esperados
):
    raise ValueError("Las propuestas no corresponden a los seis reemplazos.")

# ------------------------------------------------------------
# 3. Comprobamos originales, categoría e intención
# ------------------------------------------------------------

textos_ocupados = set(
    df_muestra_revision["consulta_original"]
    .fillna("")
    .map(normalizar_contexto)
)

textos_ocupados.update(
    df_registros_retirados["consulta_original"]
    .fillna("")
    .map(normalizar_contexto)
)

originales_propuestos = (
    propuestas["consulta_original"]
    .fillna("")
    .map(normalizar_contexto)
)

if (
    originales_propuestos.eq("").any()
    or originales_propuestos.duplicated().any()
    or originales_propuestos.isin(textos_ocupados).any()
):
    raise ValueError(
        "Hay originales vacíos, repetidos o utilizados anteriormente."
    )

for _, propuesta in propuestas.iterrows():
    identificador = propuesta["id_a_reemplazar"]

    registro = df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].eq(identificador)
    ]

    if (
        len(registro) != 1
        or not registro["conjunto"].eq("prueba").all()
        or not registro["categoria_original"].eq("LIFE_INSURANCE").all()
        or propuesta["conjunto"] != "prueba"
        or propuesta["categoria_original"] != "LIFE_INSURANCE"
        or propuesta["intencion_original"]
            != registro.iloc[0]["intencion_original"]
    ):
        raise ValueError(
            f"El reemplazo de {identificador} no coincide "
            "con la categoría, intención o conjunto esperado."
        )

    if detectar_referencias_producto(propuesta["consulta_original"]):
        raise ValueError(
            f"El candidato de {identificador} contiene "
            "referencias a otros productos y requiere revisión."
        )

# ------------------------------------------------------------
# 4. Aplicamos los cambios sobre una copia
# ------------------------------------------------------------

muestra_propuesta = df_muestra_revision.copy(deep=True)

desarrollo_antes = df_muestra_revision.loc[
    df_muestra_revision["conjunto"].eq("desarrollo")
].copy(deep=True)

distribucion_antes = (
    df_muestra_revision.groupby(
        ["categoria_original", "conjunto"]
    ).size()
)

fecha_reemplazo = datetime.now(timezone.utc).isoformat()

retirados_nuevos = df_muestra_revision.loc[
    df_muestra_revision["id_consulta"].isin(ids_esperados)
].copy()

retirados_nuevos["motivo_retiro"] = (
    "Traducción repetida tras retirar expresiones ofensivas."
)
retirados_nuevos["fecha_retiro_utc"] = fecha_reemplazo
retirados_nuevos["etapa_retiro"] = "6.10 prueba LIFE_INSURANCE"

for _, propuesta in propuestas.iterrows():
    identificador = propuesta["id_a_reemplazar"]
    mascara = muestra_propuesta["id_consulta"].eq(identificador)

    cambios = {
        "indice_origen": propuesta["indice_origen"],
        "consulta_original": propuesta["consulta_original"],
        "categoria_original": propuesta["categoria_original"],
        "intencion_original": propuesta["intencion_original"],
        "consulta_normalizada": propuesta["consulta_normalizada"],
        "consulta_es": traducciones_vida[identificador],
        "categoria_referencia": "LIFE_INSURANCE",
        "estado_revision": "propuesta_pendiente_revision",
        "observaciones": (
            "Original reemplazado por coincidencia de traducción. "
            "Traducción sin expresiones ofensivas. "
            "Confirmación del usuario pendiente."
        )
    }

    for columna, valor in cambios.items():
        muestra_propuesta.loc[mascara, columna] = valor

    # Evitamos conservar una validación anterior del registro retirado
    for columna in ["validado_por", "fecha_validacion_utc"]:
        if columna in muestra_propuesta.columns:
            muestra_propuesta.loc[mascara, columna] = ""

    if "revision_asistente" in muestra_propuesta.columns:
        muestra_propuesta.loc[mascara, "revision_asistente"] = "completada"

# ------------------------------------------------------------
# 5. Comprobamos coincidencias antes de modificar la muestra real
# ------------------------------------------------------------

textos_es = (
    muestra_propuesta["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

mascara_traducidos = textos_es.ne("")
mascara_repetidos = (
    mascara_traducidos
    & textos_es.duplicated(keep=False)
)

if mascara_repetidos.any():
    with pd.option_context("display.max_colwidth", None):
        display(
            muestra_propuesta.loc[
                mascara_repetidos,
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ].sort_values("consulta_es")
        )

    raise ValueError(
        "Persisten traducciones repetidas. "
        "Los reemplazos no se aplicaron a la muestra real."
    )

mascara_vida_prueba = (
    muestra_propuesta["conjunto"].eq("prueba")
    & muestra_propuesta["categoria_original"].eq("LIFE_INSURANCE")
)

if (
    mascara_vida_prueba.sum() != 40
    or textos_es.loc[mascara_vida_prueba].eq("").any()
    or not muestra_propuesta.loc[
        mascara_vida_prueba, "categoria_referencia"
    ].eq("LIFE_INSURANCE").all()
):
    raise ValueError(
        "Prueba de LIFE_INSURANCE debe conservar "
        "40 consultas traducidas con su categoría de referencia."
    )

desarrollo_despues = muestra_propuesta.loc[
    muestra_propuesta["conjunto"].eq("desarrollo"),
    desarrollo_antes.columns
]

if not desarrollo_antes.equals(desarrollo_despues):
    raise ValueError("Se detectó una modificación de desarrollo.")

distribucion_despues = (
    muestra_propuesta.groupby(
        ["categoria_original", "conjunto"]
    ).size()
)

if not distribucion_antes.equals(distribucion_despues):
    raise ValueError("Cambió la distribución de la muestra.")

auditoria_propuesta = pd.concat(
    [df_registros_retirados, retirados_nuevos],
    ignore_index=True,
    sort=False
)

# ------------------------------------------------------------
# 6. Guardamos un respaldo antes de publicar los cambios
# ------------------------------------------------------------

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_avance = (
    Path("/content/revision_evaluacion")
    / f"avance_prueba_vida_{marca_version}"
)
carpeta_avance.mkdir(parents=True, exist_ok=False)

muestra_propuesta.to_csv(
    carpeta_avance / "muestra_actualizada.csv",
    index=False,
    encoding="utf-8-sig"
)

auditoria_propuesta.to_csv(
    carpeta_avance / "registros_retirados.csv",
    index=False,
    encoding="utf-8-sig"
)

muestra_propuesta.loc[mascara_vida_prueba].to_csv(
    carpeta_avance / "prueba_life_insurance_propuesta.csv",
    index=False,
    encoding="utf-8-sig"
)

with (carpeta_avance / "configuracion_para_prueba.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(
        {
            "modelos": modelos_comparacion,
            "configuracion_prueba": configuracion_prueba,
            "decision_desarrollo": decision_desarrollo
        },
        archivo,
        ensure_ascii=False,
        indent=2
    )

(carpeta_avance / "LEEME.txt").write_text(
    "RESPALDO DE AVANCE: PRUEBA LIFE_INSURANCE\n\n"
    "muestra_actualizada.csv:\n"
    "  Muestra completa para continuar la preparación de prueba.\n\n"
    "registros_retirados.csv:\n"
    "  Auditoría acumulada, incluidos los seis reemplazos de 6.10.\n\n"
    "prueba_life_insurance_propuesta.csv:\n"
    "  Las 40 consultas traducidas de prueba de LIFE_INSURANCE.\n"
    "  Pendientes de confirmación del usuario.\n\n"
    "configuracion_para_prueba.json:\n"
    "  Modelos, configuración fijada y decisión de desarrollo.\n\n"
    "Este respaldo es un avance, no el dataset final validado.\n"
    "Conservar también el ZIP de cierre de desarrollo de 5.10.\n",
    encoding="utf-8"
)

ruta_zip_avance = Path(
    shutil.make_archive(
        str(carpeta_avance),
        "zip",
        root_dir=str(carpeta_avance)
    )
)

with zipfile.ZipFile(ruta_zip_avance, "r") as archivo_zip:
    if archivo_zip.testzip() is not None:
        raise ValueError("Falló la verificación del ZIP.")

# Publicamos los cambios después de comprobarlos y respaldarlos
df_muestra_revision = muestra_propuesta
df_registros_retirados = auditoria_propuesta
df_reemplazos_prueba_vida = propuestas

# ------------------------------------------------------------
# 7. Mostramos el resultado y solicitamos la descarga
# ------------------------------------------------------------

with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(ids_esperados),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

print("\nConsultas de prueba LIFE_INSURANCE traducidas: 40")
print("Grupos de traducciones repetidas: 0")
print("Reemplazos incorporados: 6")
print("Registros retirados conservados:", len(df_registros_retirados))
print("Estado: pendiente de confirmación del usuario.")

print("\n📁 Carpeta del respaldo:")
print(carpeta_avance)

print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_avance.name)

print("\n➡️ Para continuar: cargar muestra_actualizada.csv.")
print("➡️ Para auditoría: conservar registros_retirados.csv.")
print("⚠️ Este respaldo todavía no es el dataset final validado.")

print("=" * 60)
print("✅ Seis reemplazos incorporados y avance respaldado.")
print("=" * 60)

print("\n📥 Solicitando descarga del ZIP a tu computadora...")
files.download(str(ruta_zip_avance))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

🔎 Comprobando los seis reemplazos de LIFE_INSURANCE...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
15,CONS_0016,"I would like to know more about my fucking life insuracne, can you give me some information?","Quisiera saber más sobre mi seguro de vida, ¿puedes darme información?",LIFE_INSURANCE,propuesta_pendiente_revision
17,CONS_0018,wanna check my fucking life insuranc can uhelp me,"Quiero revisar mi seguro de vida, ¿puedes ayudarme?",LIFE_INSURANCE,propuesta_pendiente_revision
20,CONS_0021,im looking for information about my life insurance how can i find it,"Busco información sobre mi seguro de vida, ¿cómo puedo encontrarla?",LIFE_INSURANCE,propuesta_pendiente_revision
32,CONS_0033,"I have ot see my fucking life insurance, will you help me?","Tengo que consultar mi seguro de vida, ¿me ayudarás?",LIFE_INSURANCE,propuesta_pendiente_revision
41,CONS_0042,"I want to know about my fucking life insurance, could you give me some information?","Quiero saber sobre mi seguro de vida, ¿podrías darme información?",LIFE_INSURANCE,propuesta_pendiente_revision
43,CONS_0044,"I would like to check my life insurance, I need assistance","Quisiera revisar mi seguro de vida, necesito ayuda.",LIFE_INSURANCE,propuesta_pendiente_revision



Consultas de prueba LIFE_INSURANCE traducidas: 40
Grupos de traducciones repetidas: 0
Reemplazos incorporados: 6
Registros retirados conservados: 27
Estado: pendiente de confirmación del usuario.

📁 Carpeta del respaldo:
/content/revision_evaluacion/avance_prueba_vida_20261010T192640515033Z

📦 ZIP para conservar en tu computadora:
avance_prueba_vida_20261010T192640515033Z.zip

➡️ Para continuar: cargar muestra_actualizada.csv.
➡️ Para auditoría: conservar registros_retirados.csv.
⚠️ Este respaldo todavía no es el dataset final validado.
✅ Seis reemplazos incorporados y avance respaldado.

📥 Solicitando descarga del ZIP a tu computadora...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


### **🧠 6.11 Revisión de COVERAGE: primer bloque**

Se examinarán las primeras diez consultas de prueba de COVERAGE,
correspondientes a CONS_0061 a CONS_0070.

Se revisarán los textos actuales, incluidos los reemplazos,
para identificar consultas sobre coberturas, exclusiones
o modificaciones de cobertura.

**Estado:** Primer bloque pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 6.11 Revisión de COVERAGE: primer bloque
# ============================================================

# Seleccionamos los identificadores del primer bloque
ids_cobertura_1 = [
    f"CONS_{numero:04d}"
    for numero in range(61, 71)
]

df_bloque_prueba = (
    df_muestra_revision[
        df_muestra_revision["id_consulta"].isin(ids_cobertura_1)
    ]
    .sort_values("id_consulta")
    .copy()
)

# Verificamos la pertenencia al bloque esperado
if (
    len(df_bloque_prueba) != 10
    or not df_bloque_prueba["id_consulta"].is_unique
    or not df_bloque_prueba["conjunto"].eq("prueba").all()
    or not df_bloque_prueba["categoria_original"].eq("COVERAGE").all()
):
    raise ValueError("El bloque COVERAGE no coincide con lo esperado.")

# Mostramos los textos completos y las intenciones originales
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_bloque_prueba[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("=" * 60)
print("✅ Primer bloque de COVERAGE presentado para revisión.")
print("=" * 60)

,id_consulta,consulta_original,categoria_original,intencion_original
60,CONS_0061,im lookingf for information about the fucking coverage of my insurance policy how can i get it,COVERAGE,check_coverage
61,CONS_0062,could you help me to upgrade my fucking coverage?,COVERAGE,upgrade_coverage
62,CONS_0063,"I got to see my fucking coverage, could I get some help?",COVERAGE,check_coverage
63,CONS_0064,where can i upgrade the fucking coverage of my life insurance policy,COVERAGE,upgrade_coverage
64,CONS_0065,i need help ot upgrade my fucking coverage,COVERAGE,upgrade_coverage
65,CONS_0066,"I got to modify the coverage of my life insurance policy, where do I do it?",COVERAGE,change_coverage
66,CONS_0067,id like to know about the fucking coverage of my life insurance policy how can i get more information,COVERAGE,check_coverage
67,CONS_0068,id like to upgrade the fucking coverage of my insurance policy will ya help me,COVERAGE,upgrade_coverage
68,CONS_0069,I'd like to change the coverage of my insurance policy couldi get some help,COVERAGE,change_coverage
69,CONS_0070,"I'd like to see the coverage of my life insurance policy, whee to do it?",COVERAGE,check_coverage


✅ Primer bloque de COVERAGE presentado para revisión.


### **🧠 6.12 Traducción de COVERAGE: primer bloque**

Se propondrán traducciones para CONS_0061 a CONS_0070,
conservando el significado, el tono y el producto mencionado.

Se asignará COVERAGE como etiqueta propuesta para consultas
y solicitudes de ampliación o modificación de cobertura.

Se comprobarán coincidencias con todos los registros traducidos,
incluidos desarrollo y prueba.

**Estado:** Traducciones y etiquetas propuestas.
Revisión de coincidencias y validación pendientes.

In [ ]:
# ============================================================
# 6.12 Traducción de COVERAGE: primer bloque
# ============================================================

# Definimos traducciones fieles al significado y tono originales
traducciones_prueba_cobertura_1 = {
    "CONS_0061": (
        "Busco información sobre la cobertura "
        "de mi póliza de seguro, ¿cómo puedo obtenerla?"
    ),
    "CONS_0062": (
        "¿Podrías ayudarme a ampliar mi cobertura?"
    ),
    "CONS_0063": (
        "Tengo que consultar mi cobertura, "
        "¿me pueden ayudar?"
    ),
    "CONS_0064": (
        "¿Dónde puedo ampliar la cobertura "
        "de mi póliza de seguro de vida?"
    ),
    "CONS_0065": (
        "Necesito ayuda para ampliar mi cobertura."
    ),
    "CONS_0066": (
        "Tengo que modificar la cobertura de mi póliza "
        "de seguro de vida, ¿dónde puedo hacerlo?"
    ),
    "CONS_0067": (
        "Quisiera saber sobre la cobertura de mi póliza "
        "de seguro de vida, ¿cómo puedo obtener más información?"
    ),
    "CONS_0068": (
        "Quisiera ampliar la cobertura de mi póliza "
        "de seguro, ¿me ayudas?"
    ),
    "CONS_0069": (
        "Quisiera cambiar la cobertura de mi póliza de seguro, "
        "¿me pueden ayudar?"
    ),
    "CONS_0070": (
        "Quisiera consultar la cobertura de mi póliza "
        "de seguro de vida, ¿dónde puedo hacerlo?"
    )
}

# Verificamos el bloque antes de modificar la muestra
mascara_bloque = df_muestra_revision["id_consulta"].isin(
    traducciones_prueba_cobertura_1
)

bloque = df_muestra_revision.loc[mascara_bloque]

if (
    len(bloque) != 10
    or not bloque["id_consulta"].is_unique
    or not bloque["conjunto"].eq("prueba").all()
    or not bloque["categoria_original"].eq("COVERAGE").all()
    or bloque["consulta_original"]
        .map(detectar_referencias_producto).map(bool).any()
):
    raise ValueError("El primer bloque de COVERAGE requiere revisión.")

# Registramos propuestas sin modificar la configuración de prueba
muestra_actualizada = df_muestra_revision.copy()

for identificador, traduccion in traducciones_prueba_cobertura_1.items():
    mascara = muestra_actualizada["id_consulta"].eq(identificador)

    muestra_actualizada.loc[mascara, "consulta_es"] = traduccion
    muestra_actualizada.loc[
        mascara, "categoria_referencia"
    ] = "COVERAGE"
    muestra_actualizada.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"
    muestra_actualizada.loc[mascara, "observaciones"] = (
        "Consulta, ampliación o modificación de cobertura. "
        "Traducción y etiqueta propuestas; validación pendiente."
    )

df_muestra_revision = muestra_actualizada.copy()

# Recuperamos todos los registros traducidos
revision_traducidos = df_muestra_revision.copy()

revision_traducidos["texto_es_normalizado"] = (
    revision_traducidos["consulta_es"].map(normalizar_contexto)
)

revision_traducidos = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].ne("")
].copy()

# Identificamos todos los miembros de grupos repetidos
duplicados = revision_traducidos[
    revision_traducidos["texto_es_normalizado"].duplicated(keep=False)
].copy()

textos_del_bloque = set(
    revision_traducidos.loc[
        revision_traducidos["id_consulta"].isin(
            traducciones_prueba_cobertura_1
        ),
        "texto_es_normalizado"
    ]
)

df_coincidencias_prueba_cobertura_1 = duplicados[
    duplicados["texto_es_normalizado"].isin(textos_del_bloque)
].sort_values(["texto_es_normalizado", "conjunto", "id_consulta"])

# Mostramos las propuestas y las coincidencias del bloque
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            mascara_bloque,
            [
                "id_consulta",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ]
    )

    print(
        "Grupos de traducciones repetidas en este bloque:",
        df_coincidencias_prueba_cobertura_1[
            "texto_es_normalizado"
        ].nunique()
    )

    if not df_coincidencias_prueba_cobertura_1.empty:
        display(
            df_coincidencias_prueba_cobertura_1[
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ]
        )

print("=" * 60)
print("✅ Primer bloque de COVERAGE traducido y coincidencias comprobadas.")
print("=" * 60)

,id_consulta,consulta_es,categoria_referencia,estado_revision
60,CONS_0061,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",COVERAGE,propuesta_pendiente_revision
61,CONS_0062,¿Podrías ayudarme a ampliar mi cobertura?,COVERAGE,propuesta_pendiente_revision
62,CONS_0063,"Tengo que consultar mi cobertura, ¿me pueden ayudar?",COVERAGE,propuesta_pendiente_revision
63,CONS_0064,¿Dónde puedo ampliar la cobertura de mi póliza de seguro de vida?,COVERAGE,propuesta_pendiente_revision
64,CONS_0065,Necesito ayuda para ampliar mi cobertura.,COVERAGE,propuesta_pendiente_revision
65,CONS_0066,"Tengo que modificar la cobertura de mi póliza de seguro de vida, ¿dónde puedo hacerlo?",COVERAGE,propuesta_pendiente_revision
66,CONS_0067,"Quisiera saber sobre la cobertura de mi póliza de seguro de vida, ¿cómo puedo obtener más información?",COVERAGE,propuesta_pendiente_revision
67,CONS_0068,"Quisiera ampliar la cobertura de mi póliza de seguro, ¿me ayudas?",COVERAGE,propuesta_pendiente_revision
68,CONS_0069,"Quisiera cambiar la cobertura de mi póliza de seguro, ¿me pueden ayudar?",COVERAGE,propuesta_pendiente_revision
69,CONS_0070,"Quisiera consultar la cobertura de mi póliza de seguro de vida, ¿dónde puedo hacerlo?",COVERAGE,propuesta_pendiente_revision


Grupos de traducciones repetidas en este bloque: 1


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
56,CONS_0057,desarrollo,I'm looking for information about the coverage of my insurance policy how could i get ig,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",COVERAGE
60,CONS_0061,prueba,im lookingf for information about the fucking coverage of my insurance policy how can i get it,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",COVERAGE


✅ Primer bloque de COVERAGE traducido y coincidencias comprobadas.


### 🧠 **6.13 Revisión de COVERAGE: consultas restantes**

Se presentan las 30 consultas restantes de prueba de COVERAGE,
correspondientes a CONS_0071 a CONS_0100.

Se revisarán sus textos originales e intenciones antes de preparar
las traducciones al español sin expresiones ofensivas.

**Pendiente:** Reemplazar CONS_0061 por su coincidencia con
CONS_0057 de desarrollo.

**Estado:** Consultas restantes pendientes de traducción y revisión.

In [ ]:
# ============================================================
# 6.13 Revisión de COVERAGE: consultas restantes
# ============================================================

import pandas as pd

print("=" * 60)
print("🔎 Presentando las 30 consultas restantes de COVERAGE...")
print("=" * 60)

# Identificamos las consultas previstas para este bloque
ids_coverage_restantes = [
    f"CONS_{numero:04d}"
    for numero in range(71, 101)
]

df_revision_coverage_restantes = (
    df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(
            ids_coverage_restantes
        )
    ]
    .sort_values("id_consulta")
    .copy()
)

# Comprobamos cantidad, identificadores, categoría y conjunto
if (
    len(df_revision_coverage_restantes) != 30
    or not df_revision_coverage_restantes["id_consulta"].is_unique
    or set(df_revision_coverage_restantes["id_consulta"])
        != set(ids_coverage_restantes)
    or not df_revision_coverage_restantes["conjunto"].eq("prueba").all()
    or not df_revision_coverage_restantes[
        "categoria_original"
    ].eq("COVERAGE").all()
):
    raise ValueError(
        "Los registros no corresponden a las 30 consultas "
        "restantes de prueba de COVERAGE."
    )

# Mostramos todos los originales sin truncar su contenido
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", 40
):
    display(
        df_revision_coverage_restantes[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("\nConsultas presentadas:",
      len(df_revision_coverage_restantes))
print("La muestra permanece sin cambios.")
print("Pendiente de reemplazo: CONS_0061.")
print("Siguiente paso: preparar y revisar las traducciones.")

print("=" * 60)
print("✅ Consultas restantes de COVERAGE presentadas para revisión.")
print("=" * 60)

🔎 Presentando las 30 consultas restantes de COVERAGE...


,id_consulta,consulta_original,categoria_original,intencion_original
70,CONS_0071,I'd like to increase my coverag can ya help me,COVERAGE,upgrade_coverage
71,CONS_0072,wanna upgrade the coverage of my insurance policy how do i do it,COVERAGE,upgrade_coverage
72,CONS_0073,"I would like to know about my fucking coverage, could you find some information?",COVERAGE,check_coverage
73,CONS_0074,"I gotr to increase the coverage of my life insurance policy, where to do it?",COVERAGE,upgrade_coverage
74,CONS_0075,could ya help me check the covdrage of my insurance policy,COVERAGE,check_coverage
75,CONS_0076,"I would like to increase the coverage of my life insurance policy, how to do it?",COVERAGE,upgrade_coverage
76,CONS_0077,"I want to downgrade the fucking coverage of my life insurance policy, how do I do it?",COVERAGE,downgrade_coverage
77,CONS_0078,"I have to increase tbe coverage of my insurance policy, could you help me?",COVERAGE,upgrade_coverage
78,CONS_0079,"I got to reduce the fucking coverage of my life insurance policy, where to do it ?",COVERAGE,downgrade_coverage
79,CONS_0080,i have to upgrade my coverage i need help,COVERAGE,upgrade_coverage



Consultas presentadas: 30
La muestra permanece sin cambios.
Pendiente de reemplazo: CONS_0061.
Siguiente paso: preparar y revisar las traducciones.
✅ Consultas restantes de COVERAGE presentadas para revisión.


### 🧠 **6.14 Traducción de las consultas restantes de COVERAGE**

Se proponen traducciones sin expresiones ofensivas, conservando
la intención y las referencias al producto presentes en el original.

CONS_0095 queda pendiente de reemplazo porque menciona un seguro
de viaje. CONS_0061 continúa pendiente por su coincidencia con
una consulta de desarrollo.

Se comprobarán las traducciones repetidas entre todos los registros
traducidos. Las propuestas permanecerán pendientes de validación.

In [ ]:
# ============================================================
# 6.14 Traducción de las consultas restantes de COVERAGE
# ============================================================

import pandas as pd

print("=" * 60)
print("📝 Preparando traducciones restantes de COVERAGE...")
print("=" * 60)

traducciones_coverage_restantes = {
    "CONS_0071": "Quisiera ampliar mi cobertura, ¿puedes ayudarme?",
    "CONS_0072": (
        "Quiero ampliar la cobertura de mi póliza de seguro, "
        "¿cómo lo hago?"
    ),
    "CONS_0073": (
        "Quisiera saber sobre mi cobertura, "
        "¿podrías conseguir información?"
    ),
    "CONS_0074": (
        "Tengo que ampliar la cobertura de mi póliza de seguro de vida, "
        "¿dónde puedo hacerlo?"
    ),
    "CONS_0075": (
        "¿Puedes ayudarme a consultar la cobertura "
        "de mi póliza de seguro?"
    ),
    "CONS_0076": (
        "Quisiera ampliar la cobertura de mi póliza de seguro de vida, "
        "¿cómo puedo hacerlo?"
    ),
    "CONS_0077": (
        "Quiero reducir la cobertura de mi póliza de seguro de vida, "
        "¿cómo lo hago?"
    ),
    "CONS_0078": (
        "Tengo que ampliar la cobertura de mi póliza de seguro, "
        "¿puedes ayudarme?"
    ),
    "CONS_0079": (
        "Tengo que reducir la cobertura de mi póliza de seguro de vida, "
        "¿dónde puedo hacerlo?"
    ),
    "CONS_0080": "Tengo que ampliar mi cobertura, necesito ayuda.",
    "CONS_0081": "¿Dónde puedo ampliar mi cobertura?",
    "CONS_0082": "¿Es posible consultar mi cobertura?",
    "CONS_0083": (
        "Quiero ayuda para consultar la cobertura "
        "de mi póliza de seguro."
    ),
    "CONS_0084": "Tengo que reducir mi cobertura, necesito ayuda.",
    "CONS_0085": (
        "¿Es posible ampliar la cobertura de mi póliza de seguro?"
    ),
    "CONS_0086": (
        "Quiero consultar la cobertura de mi póliza de seguro, "
        "¿dónde puedo hacerlo?"
    ),
    "CONS_0087": "Tengo que consultar mi cobertura, necesito ayuda.",
    "CONS_0088": (
        "Quisiera información sobre la cobertura de mi póliza "
        "de seguro de vida, ¿cómo puedo encontrarla?"
    ),
    "CONS_0089": (
        "Necesito reducir la cobertura de mi póliza de seguro de vida, "
        "¿me pueden ayudar?"
    ),
    "CONS_0090": (
        "¿Sería posible consultar la cobertura de mi póliza de seguro?"
    ),
    "CONS_0091": (
        "Tengo que modificar la cobertura de mi póliza de seguro, "
        "¿me ayudarás?"
    ),
    "CONS_0092": (
        "Busco información sobre la cobertura de mi póliza de seguro, "
        "¿puedes ayudarme?"
    ),
    "CONS_0093": (
        "Quiero cambiar la cobertura de mi póliza de seguro, "
        "necesito ayuda."
    ),
    "CONS_0094": (
        "Tengo que consultar la cobertura de mi póliza de seguro, "
        "necesito ayuda."
    ),
    "CONS_0096": (
        "Quisiera ampliar la cobertura de mi póliza de seguro, "
        "¿cómo lo hago?"
    ),
    "CONS_0097": "¿Dónde puedo obtener información sobre mi cobertura?",
    "CONS_0098": "Quiero modificar mi cobertura, ¿me pueden ayudar?",
    "CONS_0099": "Necesito ayuda para reducir mi cobertura.",
    "CONS_0100": (
        "Quiero saber sobre la cobertura de mi póliza de seguro, "
        "¿dónde puedo encontrar más información?"
    )
}

# Trabajamos sobre una copia antes de publicar las propuestas
muestra_propuesta = df_muestra_revision.copy(deep=True)

for identificador, traduccion in traducciones_coverage_restantes.items():
    mascara = muestra_propuesta["id_consulta"].eq(identificador)
    registro = muestra_propuesta.loc[mascara]

    if (
        len(registro) != 1
        or not registro["conjunto"].eq("prueba").all()
        or not registro["categoria_original"].eq("COVERAGE").all()
    ):
        raise ValueError(
            f"{identificador} no coincide con el registro esperado."
        )

    muestra_propuesta.loc[mascara, "consulta_es"] = traduccion
    muestra_propuesta.loc[mascara, "categoria_referencia"] = "COVERAGE"
    muestra_propuesta.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"

# Señalamos el registro de seguro de viaje sin traducirlo
mascara_viaje = muestra_propuesta["id_consulta"].eq("CONS_0095")
registro_viaje = muestra_propuesta.loc[mascara_viaje]

if (
    len(registro_viaje) != 1
    or not registro_viaje["conjunto"].eq("prueba").all()
    or not registro_viaje["categoria_original"].eq("COVERAGE").all()
    or "travwl" not in normalizar_contexto(
        registro_viaje.iloc[0]["consulta_original"]
    )
):
    raise ValueError("CONS_0095 no coincide con el caso de seguro de viaje.")

muestra_propuesta.loc[mascara_viaje, "consulta_es"] = ""
muestra_propuesta.loc[
    mascara_viaje, "estado_revision"
] = "pendiente_reemplazo"

muestra_propuesta.loc[mascara_viaje, "observaciones"] = (
    "Referencia a seguro de viaje: travwl insurance. "
    "Pendiente de reemplazo por quedar fuera del alcance."
)

# Publicamos las traducciones propuestas
df_muestra_revision = muestra_propuesta

ids_bloque = [
    f"CONS_{numero:04d}"
    for numero in range(71, 101)
]

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(ids_bloque),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

# Buscamos coincidencias en todos los textos ya traducidos
textos_es = (
    df_muestra_revision["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

mascara_repetidos = (
    textos_es.ne("")
    & textos_es.duplicated(keep=False)
)

# Mostramos grupos repetidos que involucren prueba de COVERAGE
mascara_prueba_coverage = (
    df_muestra_revision["conjunto"].eq("prueba")
    & df_muestra_revision["categoria_original"].eq("COVERAGE")
)

textos_repetidos_coverage = set(
    textos_es.loc[
        mascara_repetidos & mascara_prueba_coverage
    ]
)

mascara_coincidencias = textos_es.isin(textos_repetidos_coverage)

print(
    "\nGrupos repetidos que involucran prueba de COVERAGE:",
    len(textos_repetidos_coverage)
)

if mascara_coincidencias.any():
    coincidencias = df_muestra_revision.loc[
        mascara_coincidencias,
        [
            "id_consulta",
            "conjunto",
            "consulta_original",
            "consulta_es",
            "categoria_referencia"
        ]
    ].copy()

    coincidencias["_orden"] = textos_es.loc[mascara_coincidencias]

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            coincidencias.sort_values(
                ["_orden", "id_consulta"]
            ).drop(columns="_orden")
        )

print("\nTraducciones propuestas en este bloque: 29")
print("Pendiente por coincidencia conocida: CONS_0061.")
print("Pendiente por seguro de viaje: CONS_0095.")
print("Revisaremos también las nuevas coincidencias mostradas.")
print("Estado: propuestas pendientes de validación.")

print("=" * 60)
print("✅ Traducciones propuestas y coincidencias comprobadas.")
print("=" * 60)

📝 Preparando traducciones restantes de COVERAGE...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
70,CONS_0071,I'd like to increase my coverag can ya help me,"Quisiera ampliar mi cobertura, ¿puedes ayudarme?",COVERAGE,propuesta_pendiente_revision
71,CONS_0072,wanna upgrade the coverage of my insurance policy how do i do it,"Quiero ampliar la cobertura de mi póliza de seguro, ¿cómo lo hago?",COVERAGE,propuesta_pendiente_revision
72,CONS_0073,"I would like to know about my fucking coverage, could you find some information?","Quisiera saber sobre mi cobertura, ¿podrías conseguir información?",COVERAGE,propuesta_pendiente_revision
73,CONS_0074,"I gotr to increase the coverage of my life insurance policy, where to do it?","Tengo que ampliar la cobertura de mi póliza de seguro de vida, ¿dónde puedo hacerlo?",COVERAGE,propuesta_pendiente_revision
74,CONS_0075,could ya help me check the covdrage of my insurance policy,¿Puedes ayudarme a consultar la cobertura de mi póliza de seguro?,COVERAGE,propuesta_pendiente_revision
75,CONS_0076,"I would like to increase the coverage of my life insurance policy, how to do it?","Quisiera ampliar la cobertura de mi póliza de seguro de vida, ¿cómo puedo hacerlo?",COVERAGE,propuesta_pendiente_revision
76,CONS_0077,"I want to downgrade the fucking coverage of my life insurance policy, how do I do it?","Quiero reducir la cobertura de mi póliza de seguro de vida, ¿cómo lo hago?",COVERAGE,propuesta_pendiente_revision
77,CONS_0078,"I have to increase tbe coverage of my insurance policy, could you help me?","Tengo que ampliar la cobertura de mi póliza de seguro, ¿puedes ayudarme?",COVERAGE,propuesta_pendiente_revision
78,CONS_0079,"I got to reduce the fucking coverage of my life insurance policy, where to do it ?","Tengo que reducir la cobertura de mi póliza de seguro de vida, ¿dónde puedo hacerlo?",COVERAGE,propuesta_pendiente_revision
79,CONS_0080,i have to upgrade my coverage i need help,"Tengo que ampliar mi cobertura, necesito ayuda.",COVERAGE,propuesta_pendiente_revision



Grupos repetidos que involucran prueba de COVERAGE: 1


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
56,CONS_0057,desarrollo,I'm looking for information about the coverage of my insurance policy how could i get ig,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",COVERAGE
60,CONS_0061,prueba,im lookingf for information about the fucking coverage of my insurance policy how can i get it,"Busco información sobre la cobertura de mi póliza de seguro, ¿cómo puedo obtenerla?",COVERAGE



Traducciones propuestas en este bloque: 29
Pendiente por coincidencia conocida: CONS_0061.
Pendiente por seguro de viaje: CONS_0095.
Revisaremos también las nuevas coincidencias mostradas.
Estado: propuestas pendientes de validación.
✅ Traducciones propuestas y coincidencias comprobadas.


### 🧠 **6.15 Selección de candidatos de reemplazo de COVERAGE**

Se presentan opciones para reemplazar CONS_0061 por traducción
repetida y CONS_0095 por referencia a un seguro de viaje.

Los candidatos conservarán la categoría y la intención del registro
reemplazado. Se excluirán originales utilizados, registros retirados
y referencias detectadas a otros productos.

**Estado:** Opciones pendientes de revisión. La muestra no se modifica.

In [ ]:
# ============================================================
# 6.15 Opciones de reemplazo de COVERAGE
# ============================================================

import pandas as pd
import re

print("=" * 60)
print("🔎 Preparando opciones para dos reemplazos de COVERAGE...")
print("=" * 60)

ids_reemplazar_coverage = ["CONS_0061", "CONS_0095"]

objetivos = df_muestra_revision.loc[
    df_muestra_revision["id_consulta"].isin(
        ids_reemplazar_coverage
    )
].sort_values("id_consulta").copy()

if (
    len(objetivos) != 2
    or not objetivos["id_consulta"].is_unique
    or not objetivos["conjunto"].eq("prueba").all()
    or not objetivos["categoria_original"].eq("COVERAGE").all()
):
    raise ValueError("Los dos registros no coinciden con los esperados.")

# Excluimos todos los originales utilizados o retirados
textos_excluidos_coverage = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_excluidos_coverage.update(
        tabla["consulta_original"]
        .fillna("")
        .map(normalizar_contexto)
    )

reserva_coverage = df_elegibles.copy()

reserva_coverage["texto_revision"] = (
    reserva_coverage["instruction"]
    .fillna("")
    .map(normalizar_contexto)
)

# Complementamos el detector con la variante encontrada en 6.13
def detectar_producto_candidatos_coverage(texto):
    normalizado = normalizar_contexto(texto)

    return bool(
        detectar_referencias_producto(texto)
        or re.search(r"\btravwl\b", normalizado)
    )

otros_productos = (
    reserva_coverage["instruction"]
    .fillna("")
    .map(detectar_producto_candidatos_coverage)
)

reserva_coverage = (
    reserva_coverage.loc[
        reserva_coverage["texto_revision"].ne("")
        & ~reserva_coverage["texto_revision"].isin(
            textos_excluidos_coverage
        )
        & ~otros_productos
    ]
    .sort_values("texto_revision")
    .drop_duplicates("texto_revision")
    .copy()
)

# Presentamos hasta cinco opciones por objetivo
partes_opciones = []

for posicion, (_, objetivo) in enumerate(objetivos.iterrows()):
    candidatos = reserva_coverage.loc[
        reserva_coverage["category"].eq(
            objetivo["categoria_original"]
        )
        & reserva_coverage["intent"].eq(
            objetivo["intencion_original"]
        )
    ]

    if candidatos.empty:
        raise ValueError(
            f"No hay candidatos para {objetivo['id_consulta']}."
        )

    seleccion = candidatos.sample(
        n=min(5, len(candidatos)),
        random_state=4842 + posicion
    ).copy()

    seleccion.insert(
        0, "id_a_reemplazar", objetivo["id_consulta"]
    )
    seleccion.insert(
        1, "opcion", range(1, len(seleccion) + 1)
    )

    partes_opciones.append(seleccion)

df_opciones_reemplazos_coverage = pd.concat(
    partes_opciones,
    ignore_index=True
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_opciones_reemplazos_coverage[
            [
                "id_a_reemplazar",
                "opcion",
                "instruction",
                "category",
                "intent"
            ]
        ].rename(columns={
            "instruction": "consulta_original",
            "category": "categoria_original",
            "intent": "intencion_original"
        })
    )

print("\nLa muestra permanece sin cambios.")
print("Revisaremos las opciones y sus posibles traducciones.")
print("La comprobación automática no sustituye la revisión semántica.")

print("=" * 60)
print("✅ Opciones de reemplazo de COVERAGE presentadas.")
print("=" * 60)

🔎 Preparando opciones para dos reemplazos de COVERAGE...


,id_a_reemplazar,opcion,consulta_original,categoria_original,intencion_original
0,CONS_0061,1,wanna know about the fucking coverage of my life insurance policy where can i get more information,COVERAGE,check_coverage
1,CONS_0061,2,I'd like to check the fucking coverage of my insuraqnce policy where to do it,COVERAGE,check_coverage
2,CONS_0061,3,I'd like to know about my coverage where can i find more informatgion,COVERAGE,check_coverage
3,CONS_0061,4,"I'd like to check my coverage, where to do it?",COVERAGE,check_coverage
4,CONS_0061,5,"I need to see the fucking coverage of my life insurance policy, could you help me?",COVERAGE,check_coverage
5,CONS_0095,1,"I want to reduce the coverage of my life insurance policy, help me",COVERAGE,downgrade_coverage
6,CONS_0095,2,how do I reduce the coverage of my insurance policy ?,COVERAGE,downgrade_coverage
7,CONS_0095,3,"I got to downgrade the coverage of my life insurance policy, how could I do it?",COVERAGE,downgrade_coverage
8,CONS_0095,4,wanna reduce my fuckingcoverage can uhelp me,COVERAGE,downgrade_coverage
9,CONS_0095,5,I'd like to reduce my fucking coverage where can i do it,COVERAGE,downgrade_coverage



La muestra permanece sin cambios.
Revisaremos las opciones y sus posibles traducciones.
La comprobación automática no sustituye la revisión semántica.
✅ Opciones de reemplazo de COVERAGE presentadas.


### 🧠 **6.16 Incorporación y respaldo de reemplazos de COVERAGE**

Se incorporarán los candidatos elegidos para CONS_0061 y CONS_0095,
conservando sus identificadores, categoría, intención y conjunto.

Se comprobarán las traducciones repetidas y la distribución de la
muestra. Los originales retirados se conservarán en la auditoría.

Se registrará la aprobación del usuario de las 40 traducciones de
LIFE_INSURANCE. Las 40 traducciones de COVERAGE permanecerán
pendientes de confirmación.

Se descargará un ZIP con la muestra actualizada, la auditoría,
los conjuntos de ambas categorías y la configuración para prueba.

**Estado:** Respaldo de avance; dataset de prueba aún incompleto.

In [ ]:
# ============================================================
# 6.16 Incorporación y respaldo de reemplazos de COVERAGE
# ============================================================

import pandas as pd
import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("=" * 60)
print("🔎 Comprobando los reemplazos de COVERAGE...")
print("=" * 60)

# ------------------------------------------------------------
# 1. Definimos las opciones y traducciones revisadas
# ------------------------------------------------------------

elecciones_coverage = {
    "CONS_0061": {
        "opcion": 3,
        "traduccion": (
            "Quisiera saber sobre mi cobertura, "
            "¿dónde puedo encontrar más información?"
        ),
        "motivo": "Traducción repetida con CONS_0057 de desarrollo."
    },
    "CONS_0095": {
        "opcion": 2,
        "traduccion": (
            "¿Cómo puedo reducir la cobertura de mi póliza de seguro?"
        ),
        "motivo": "Referencia a seguro de viaje: travwl insurance."
    }
}

muestra_propuesta = df_muestra_revision.copy(deep=True)
desarrollo_antes = df_muestra_revision.loc[
    df_muestra_revision["conjunto"].eq("desarrollo")
].copy(deep=True)

distribucion_antes = df_muestra_revision.groupby(
    ["categoria_original", "conjunto"]
).size()

fecha_actual = datetime.now(timezone.utc).isoformat()
retirados = []
textos_ocupados = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_ocupados.update(
        tabla["consulta_original"]
        .fillna("")
        .map(normalizar_contexto)
    )

# ------------------------------------------------------------
# 2. Verificamos e incorporamos los candidatos sobre una copia
# ------------------------------------------------------------

for identificador, eleccion in elecciones_coverage.items():
    opciones = df_opciones_reemplazos_coverage.loc[
        df_opciones_reemplazos_coverage["id_a_reemplazar"].eq(
            identificador
        )
        & df_opciones_reemplazos_coverage["opcion"].eq(
            eleccion["opcion"]
        )
    ]

    mascara = muestra_propuesta["id_consulta"].eq(identificador)
    original = muestra_propuesta.loc[mascara]

    if len(opciones) != 1 or len(original) != 1:
        raise ValueError(
            f"No se encuentra un registro único para {identificador}."
        )

    candidato = opciones.iloc[0]

    if (
        not original["conjunto"].eq("prueba").all()
        or not original["categoria_original"].eq("COVERAGE").all()
        or candidato["category"] != "COVERAGE"
        or candidato["intent"] != original.iloc[0]["intencion_original"]
    ):
        raise ValueError(
            f"Categoría, intención o conjunto incorrecto: {identificador}."
        )

    texto_candidato = normalizar_contexto(candidato["instruction"])

    if (
        not texto_candidato
        or texto_candidato in textos_ocupados
        or detectar_producto_candidatos_coverage(
            candidato["instruction"]
        )
    ):
        raise ValueError(
            f"El candidato de {identificador} requiere nueva revisión."
        )

    textos_ocupados.add(texto_candidato)

    registro_retirado = original.copy()
    registro_retirado["motivo_retiro"] = eleccion["motivo"]
    registro_retirado["fecha_retiro_utc"] = fecha_actual
    registro_retirado["etapa_retiro"] = "6.16 prueba COVERAGE"
    retirados.append(registro_retirado)

    cambios = {
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"],
        "consulta_es": eleccion["traduccion"],
        "categoria_referencia": "COVERAGE",
        "estado_revision": "propuesta_pendiente_revision",
        "observaciones": (
            "Original reemplazado: " + eleccion["motivo"]
            + " Traducción revisada por el asistente; "
            "confirmación del usuario pendiente."
        )
    }

    for columna, valor in cambios.items():
        muestra_propuesta.loc[mascara, columna] = valor

    for columna in ["validado_por", "fecha_validacion_utc"]:
        if columna in muestra_propuesta.columns:
            muestra_propuesta.loc[mascara, columna] = ""

    if "revision_asistente" in muestra_propuesta.columns:
        muestra_propuesta.loc[mascara, "revision_asistente"] = "completada"

# ------------------------------------------------------------
# 3. Comprobamos las traducciones y la distribución
# ------------------------------------------------------------

textos_es = (
    muestra_propuesta["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = textos_es.ne("") & textos_es.duplicated(keep=False)

if repetidos.any():
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            muestra_propuesta.loc[
                repetidos,
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ].sort_values("consulta_es")
        )

    raise ValueError(
        "Persisten traducciones repetidas. "
        "La muestra original no se modificó."
    )

mascara_vida = (
    muestra_propuesta["conjunto"].eq("prueba")
    & muestra_propuesta["categoria_original"].eq("LIFE_INSURANCE")
)

mascara_coverage = (
    muestra_propuesta["conjunto"].eq("prueba")
    & muestra_propuesta["categoria_original"].eq("COVERAGE")
)

for categoria, mascara in [
    ("LIFE_INSURANCE", mascara_vida),
    ("COVERAGE", mascara_coverage)
]:
    if (
        mascara.sum() != 40
        or textos_es.loc[mascara].eq("").any()
        or not muestra_propuesta.loc[
            mascara, "categoria_referencia"
        ].eq(categoria).all()
    ):
        raise ValueError(
            f"{categoria} debe contener 40 consultas "
            "traducidas con su referencia correspondiente."
        )

distribucion_despues = muestra_propuesta.groupby(
    ["categoria_original", "conjunto"]
).size()

if not distribucion_antes.equals(distribucion_despues):
    raise ValueError("Cambió la distribución de la muestra.")

# ------------------------------------------------------------
# 4. Registramos la aprobación previa de LIFE_INSURANCE
# ------------------------------------------------------------

# El usuario confirmó estas 40 traducciones en la conversación
for columna in ["validado_por", "fecha_validacion_utc"]:
    if columna not in muestra_propuesta.columns:
        muestra_propuesta[columna] = ""

muestra_propuesta.loc[
    mascara_vida, "estado_revision"
] = "validado_usuario"

muestra_propuesta.loc[
    mascara_vida, "validado_por"
] = "usuario"

# Conservamos fechas anteriores si la celda vuelve a ejecutarse
sin_fecha = (
    muestra_propuesta["fecha_validacion_utc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

muestra_propuesta.loc[
    mascara_vida & sin_fecha, "fecha_validacion_utc"
] = fecha_actual

# Verificamos que desarrollo conserva todos sus campos anteriores
desarrollo_despues = muestra_propuesta.loc[
    muestra_propuesta["conjunto"].eq("desarrollo"),
    desarrollo_antes.columns
]

if not desarrollo_antes.equals(desarrollo_despues):
    raise ValueError("Se detectó una modificación de desarrollo.")

auditoria_propuesta = pd.concat(
    [df_registros_retirados] + retirados,
    ignore_index=True,
    sort=False
)

# ------------------------------------------------------------
# 5. Guardamos y verificamos el respaldo del avance
# ------------------------------------------------------------

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_avance_coverage = (
    Path("/content/revision_evaluacion")
    / f"avance_prueba_coverage_{marca_version}"
)
carpeta_avance_coverage.mkdir(parents=True, exist_ok=False)

tablas_guardar = {
    "muestra_actualizada.csv": muestra_propuesta,
    "registros_retirados.csv": auditoria_propuesta,
    "prueba_life_insurance_validada.csv": (
        muestra_propuesta.loc[mascara_vida]
    ),
    "prueba_coverage_propuesta.csv": (
        muestra_propuesta.loc[mascara_coverage]
    )
}

for nombre, tabla in tablas_guardar.items():
    tabla.to_csv(
        carpeta_avance_coverage / nombre,
        index=False,
        encoding="utf-8-sig"
    )

with (
    carpeta_avance_coverage / "configuracion_para_prueba.json"
).open("w", encoding="utf-8") as archivo:
    json.dump(
        {
            "modelos": modelos_comparacion,
            "configuracion_prueba": configuracion_prueba,
            "decision_desarrollo": decision_desarrollo
        },
        archivo,
        ensure_ascii=False,
        indent=2
    )

(carpeta_avance_coverage / "LEEME.txt").write_text(
    "RESPALDO DE AVANCE DE PRUEBA\n\n"
    "muestra_actualizada.csv:\n"
    "  Insumo principal para continuar las categorías restantes.\n\n"
    "registros_retirados.csv:\n"
    "  Auditoría acumulada de los reemplazos.\n\n"
    "prueba_life_insurance_validada.csv:\n"
    "  40 consultas aprobadas por el usuario.\n\n"
    "prueba_coverage_propuesta.csv:\n"
    "  40 traducciones pendientes de confirmación del usuario.\n\n"
    "configuracion_para_prueba.json:\n"
    "  Modelos y configuración fijada para evaluación de prueba.\n\n"
    "Prueba todavía incompleta: faltan QUOTE, PAYMENT, "
    "ENROLLMENT y CONTACT.\n"
    "Conservar también el respaldo de cierre de desarrollo.\n",
    encoding="utf-8"
)

ruta_zip_coverage = Path(
    shutil.make_archive(
        str(carpeta_avance_coverage),
        "zip",
        root_dir=str(carpeta_avance_coverage)
    )
)

with zipfile.ZipFile(ruta_zip_coverage, "r") as respaldo:
    if respaldo.testzip() is not None:
        raise ValueError("Falló la verificación del ZIP.")

# Publicamos los cambios después de verificar y guardar
df_muestra_revision = muestra_propuesta
df_registros_retirados = auditoria_propuesta

# ------------------------------------------------------------
# 6. Mostramos los resultados y descargamos
# ------------------------------------------------------------

with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(
                elecciones_coverage
            ),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

print("\nPrueba LIFE_INSURANCE validada: 40 consultas.")
print("Prueba COVERAGE traducida: 40 consultas.")
print("COVERAGE: confirmación del usuario pendiente.")
print("Grupos de traducciones repetidas: 0")
print("Registros retirados conservados:", len(df_registros_retirados))

print("\n📁 Carpeta del respaldo:")
print(carpeta_avance_coverage)

print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_coverage.name)

print("\n➡️ Para continuar: muestra_actualizada.csv.")
print("➡️ Para auditoría: registros_retirados.csv.")
print("⚠️ Este respaldo todavía no es el dataset final de prueba.")

print("=" * 60)
print("✅ Reemplazos de COVERAGE incorporados y avance respaldado.")
print("=" * 60)

print("\n📥 Solicitando descarga del ZIP...")
files.download(str(ruta_zip_coverage))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

🔎 Comprobando los reemplazos de COVERAGE...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
60,CONS_0061,I'd like to know about my coverage where can i find more informatgion,"Quisiera saber sobre mi cobertura, ¿dónde puedo encontrar más información?",COVERAGE,propuesta_pendiente_revision
94,CONS_0095,how do I reduce the coverage of my insurance policy ?,¿Cómo puedo reducir la cobertura de mi póliza de seguro?,COVERAGE,propuesta_pendiente_revision



Prueba LIFE_INSURANCE validada: 40 consultas.
Prueba COVERAGE traducida: 40 consultas.
COVERAGE: confirmación del usuario pendiente.
Grupos de traducciones repetidas: 0
Registros retirados conservados: 29

📁 Carpeta del respaldo:
/content/revision_evaluacion/avance_prueba_coverage_20261010T193246882495Z

📦 ZIP para conservar en tu computadora:
avance_prueba_coverage_20261010T193246882495Z.zip

➡️ Para continuar: muestra_actualizada.csv.
➡️ Para auditoría: registros_retirados.csv.
⚠️ Este respaldo todavía no es el dataset final de prueba.
✅ Reemplazos de COVERAGE incorporados y avance respaldado.

📥 Solicitando descarga del ZIP...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


### 🧠 **6.17 Validación de COVERAGE y revisión de QUOTE**

Se registra la aprobación del usuario de las 40 traducciones
de prueba de COVERAGE.

Se presentan las consultas CONS_0111 a CONS_0150 para revisar
sus referencias al producto y sus intenciones relacionadas
con tarifas, costos y cotizaciones.

**Estado:** COVERAGE validado. QUOTE pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 6.17 Validación de COVERAGE y revisión de QUOTE
# ============================================================

import pandas as pd
from datetime import datetime, timezone

print("=" * 60)
print("✅ Registrando aprobación de COVERAGE...")
print("=" * 60)

mascara_coverage = (
    df_muestra_revision["conjunto"].eq("prueba")
    & df_muestra_revision["categoria_original"].eq("COVERAGE")
)

registros_coverage = df_muestra_revision.loc[mascara_coverage]

textos_coverage = (
    registros_coverage["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

if (
    len(registros_coverage) != 40
    or textos_coverage.eq("").any()
    or textos_coverage.duplicated().any()
    or not registros_coverage[
        "categoria_referencia"
    ].eq("COVERAGE").all()
):
    raise ValueError(
        "COVERAGE debe contener 40 traducciones completas "
        "y distintas con su categoría de referencia."
    )

# Comprobamos también coincidencias con otras consultas traducidas
textos_todos = (
    df_muestra_revision["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = (
    textos_todos.ne("")
    & textos_todos.duplicated(keep=False)
)

if (repetidos & mascara_coverage).any():
    raise ValueError(
        "Hay coincidencias de COVERAGE con otros registros traducidos."
    )

# Preparamos la siguiente categoría antes de publicar cambios
ids_quote_prueba = [
    f"CONS_{numero:04d}"
    for numero in range(111, 151)
]

df_revision_quote_prueba = (
    df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(ids_quote_prueba)
    ]
    .sort_values("id_consulta")
    .copy()
)

if (
    len(df_revision_quote_prueba) != 40
    or not df_revision_quote_prueba["id_consulta"].is_unique
    or set(df_revision_quote_prueba["id_consulta"])
        != set(ids_quote_prueba)
    or not df_revision_quote_prueba["conjunto"].eq("prueba").all()
    or not df_revision_quote_prueba[
        "categoria_original"
    ].eq("QUOTE").all()
):
    raise ValueError(
        "Los registros no corresponden a las 40 consultas "
        "de prueba de QUOTE."
    )

# Registramos la confirmación explícita del usuario
fecha_actual = datetime.now(timezone.utc).isoformat()

for columna in ["validado_por", "fecha_validacion_utc"]:
    if columna not in df_muestra_revision.columns:
        df_muestra_revision[columna] = ""

df_muestra_revision.loc[
    mascara_coverage, "estado_revision"
] = "validado_usuario"

df_muestra_revision.loc[
    mascara_coverage, "validado_por"
] = "usuario"

sin_fecha = (
    df_muestra_revision["fecha_validacion_utc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

df_muestra_revision.loc[
    mascara_coverage & sin_fecha,
    "fecha_validacion_utc"
] = fecha_actual

print("\nConsultas de prueba COVERAGE validadas: 40")
print("Validación registrada en memoria.")
print("Se incluirá en el siguiente respaldo descargable.")

print("\n" + "=" * 60)
print("🔎 Presentando las 40 consultas de prueba de QUOTE...")
print("=" * 60)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_revision_quote_prueba[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("\nConsultas de QUOTE presentadas: 40")
print("Estado: pendientes de traducción y revisión.")

print("=" * 60)
print("✅ COVERAGE validado y QUOTE presentado para revisión.")
print("=" * 60)

✅ Registrando aprobación de COVERAGE...

Consultas de prueba COVERAGE validadas: 40
Validación registrada en memoria.
Se incluirá en el siguiente respaldo descargable.

🔎 Presentando las 40 consultas de prueba de QUOTE...


,id_consulta,consulta_original,categoria_original,intencion_original
110,CONS_0111,I'd like to estimate a pe insurance quote how could i do it,QUOTE,calculate_insurance_quote
111,CONS_0112,wanna check the fucking life insurance rates i need assistance,QUOTE,check_rates
112,CONS_0113,"I want to know more about the rates, can you find some information?",QUOTE,check_rates
113,CONS_0114,could ushow me the fucking rates,QUOTE,check_rates
114,CONS_0115,wanna estimate a fucking insurance quote can uhelp me,QUOTE,calculate_insurance_quote
115,CONS_0116,"I want to know more about the life insurance rates , will you find some information?",QUOTE,check_rates
116,CONS_0117,help me estimating an insurance quote,QUOTE,calculate_insurance_quote
117,CONS_0118,"I need an insurance quote, will you help me estimating it?",QUOTE,calculate_insurance_quote
118,CONS_0119,i want a fuckin life insurance quote i wanna estimate it,QUOTE,calculate_insurance_quote
119,CONS_0120,I'd like to check the fucking rates could i get some hel,QUOTE,check_rates



Consultas de QUOTE presentadas: 40
Estado: pendientes de traducción y revisión.
✅ COVERAGE validado y QUOTE presentado para revisión.


### 🧠 **6.18 Traducción y revisión de QUOTE**

Se proponen traducciones de 38 consultas sobre tarifas y cotizaciones,
sin expresiones ofensivas y conservando las referencias al producto
presentes en los originales.

CONS_0111 queda pendiente de reemplazo por la referencia ambigua
«pe insurance». CONS_0136 queda pendiente por mencionar seguro
de salud.

Se mostrarán las traducciones repetidas dentro de prueba y las
coincidencias con desarrollo.

**Estado:** Traducciones propuestas; reemplazos y validación pendientes.

In [ ]:
# ============================================================
# 6.18 Traducción y revisión de QUOTE
# ============================================================

import pandas as pd

print("=" * 60)
print("📝 Preparando traducciones de prueba de QUOTE...")
print("=" * 60)

traducciones_quote = {
    "CONS_0112": "Quiero consultar las tarifas del seguro de vida, necesito ayuda.",
    "CONS_0113": "Quiero saber más sobre las tarifas, ¿puedes conseguir información?",
    "CONS_0114": "¿Puedes mostrarme las tarifas?",
    "CONS_0115": "Quiero estimar una cotización de seguro, ¿puedes ayudarme?",
    "CONS_0116": "Quiero saber más sobre las tarifas del seguro de vida, ¿puedes conseguir información?",
    "CONS_0117": "Ayúdame a estimar una cotización de seguro.",
    "CONS_0118": "Necesito una cotización de seguro, ¿me ayudarás a estimarla?",
    "CONS_0119": "Quiero una cotización de seguro de vida, quiero estimarla.",
    "CONS_0120": "Quisiera consultar las tarifas, ¿me pueden ayudar?",
    "CONS_0121": "Necesito las tarifas, ¿cómo puedo consultarlas?",
    "CONS_0122": "Quisiera información sobre las tarifas, ¿dónde puedo obtenerla?",
    "CONS_0123": "Quiero saber más sobre las tarifas del seguro de vida, dame información.",
    "CONS_0124": "Quiero una cotización de seguro de vida, ¿puedo calcularla?",
    "CONS_0125": "Busco una cotización de seguro, ¿puedes ayudarme a encontrarla?",
    "CONS_0126": "Necesito las tarifas, quisiera consultarlas.",
    "CONS_0127": "Busco las tarifas, ¿cómo puedo encontrarlas?",
    "CONS_0128": "Necesito una cotización de seguro, ¿podrías estimarla?",
    "CONS_0129": "Quiero ayuda para consultar las tarifas del seguro.",
    "CONS_0130": "Busco las tarifas, ¿puedes ayudarme a encontrarlas?",
    "CONS_0131": "Quiero una cotización de seguro de vida, ¿puedes ayudarme a calcularla?",
    "CONS_0132": "Busco una cotización de seguro de vida, ¿puedes ayudarme a encontrarla?",
    "CONS_0133": "Quisiera estimar una cotización de seguro de vida, ¿dónde puedo hacerlo?",
    "CONS_0134": "Estoy intentando encontrar las tarifas del seguro, ¿me pueden ayudar?",
    "CONS_0135": "Quiero las tarifas del seguro, ¿puedes ayudarme a consultarlas?",
    "CONS_0137": "Busco información sobre las tarifas del seguro de vida, necesito ayuda.",
    "CONS_0138": "Necesito una cotización de seguro, ¿cómo puedo estimarla?",
    "CONS_0139": "Quisiera consultar las tarifas, ¿dónde puedo hacerlo?",
    "CONS_0140": "Tengo que calcular una cotización de seguro de vida, necesito ayuda.",
    "CONS_0141": "Quiero consultar las tarifas del seguro, ¿cómo lo hago?",
    "CONS_0142": "Tengo que estimar una cotización de seguro de vida, ¿me pueden ayudar?",
    "CONS_0143": "Necesito una cotización de seguro, ¿puedes ayudarme a estimarla?",
    "CONS_0144": "Quisiera saber sobre las tarifas, ¿dónde puedo encontrar más información?",
    "CONS_0145": "Quiero estimar una cotización de seguro, ayúdame.",
    "CONS_0146": "Quisiera información sobre las tarifas del seguro, ¿cómo puedo obtenerla?",
    "CONS_0147": "Necesito una cotización de seguro, ¿me ayudarás a estimarla?",
    "CONS_0148": "Quiero calcular una cotización de seguro, ¿dónde puedo hacerlo?",
    "CONS_0149": "Busco información sobre las tarifas del seguro de vida, necesito ayuda.",
    "CONS_0150": "Quisiera saber sobre las tarifas."
}

motivos_quote = {
    "CONS_0111": "Referencia ambigua a producto: pe insurance.",
    "CONS_0136": "Referencia a seguro de salud: ehalth insurance."
}

ids_quote = {
    f"CONS_{numero:04d}"
    for numero in range(111, 151)
}

if set(traducciones_quote) | set(motivos_quote) != ids_quote:
    raise ValueError("La configuración no incluye los 40 registros previstos.")

muestra_propuesta = df_muestra_revision.copy(deep=True)

# Comprobamos todos los registros antes de publicar cambios
registros_quote = muestra_propuesta.loc[
    muestra_propuesta["id_consulta"].isin(ids_quote)
]

if (
    len(registros_quote) != 40
    or not registros_quote["id_consulta"].is_unique
    or not registros_quote["conjunto"].eq("prueba").all()
    or not registros_quote["categoria_original"].eq("QUOTE").all()
):
    raise ValueError("Los registros de QUOTE no coinciden con los esperados.")

for identificador, traduccion in traducciones_quote.items():
    mascara = muestra_propuesta["id_consulta"].eq(identificador)

    muestra_propuesta.loc[mascara, "consulta_es"] = traduccion
    muestra_propuesta.loc[mascara, "categoria_referencia"] = "QUOTE"
    muestra_propuesta.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"

for identificador, motivo in motivos_quote.items():
    mascara = muestra_propuesta["id_consulta"].eq(identificador)

    muestra_propuesta.loc[mascara, "consulta_es"] = ""
    muestra_propuesta.loc[mascara, "categoria_referencia"] = ""
    muestra_propuesta.loc[
        mascara, "estado_revision"
    ] = "pendiente_reemplazo"
    muestra_propuesta.loc[mascara, "observaciones"] = motivo

df_muestra_revision = muestra_propuesta

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(ids_quote),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

# Detectamos coincidencias entre todos los registros traducidos
textos_es = (
    df_muestra_revision["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = textos_es.ne("") & textos_es.duplicated(keep=False)

mascara_quote_prueba = (
    df_muestra_revision["conjunto"].eq("prueba")
    & df_muestra_revision["categoria_original"].eq("QUOTE")
)

textos_repetidos_quote = set(
    textos_es.loc[repetidos & mascara_quote_prueba]
)

print(
    "\nGrupos repetidos que involucran prueba de QUOTE:",
    len(textos_repetidos_quote)
)

if textos_repetidos_quote:
    mascara_coincidencias = textos_es.isin(textos_repetidos_quote)

    coincidencias_quote = df_muestra_revision.loc[
        mascara_coincidencias,
        [
            "id_consulta",
            "conjunto",
            "consulta_original",
            "consulta_es",
            "categoria_referencia"
        ]
    ].copy()

    coincidencias_quote["_orden"] = textos_es.loc[mascara_coincidencias]

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            coincidencias_quote.sort_values(
                ["_orden", "id_consulta"]
            ).drop(columns="_orden")
        )

print("\nTraducciones propuestas: 38")
print("Pendientes por referencia al producto: CONS_0111 y CONS_0136.")
print("Las coincidencias mostradas requieren revisión y reemplazo.")
print("Estado: confirmación del usuario pendiente.")

print("=" * 60)
print("✅ Traducciones de QUOTE propuestas y coincidencias comprobadas.")
print("=" * 60)

📝 Preparando traducciones de prueba de QUOTE...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
110,CONS_0111,I'd like to estimate a pe insurance quote how could i do it,,,pendiente_reemplazo
111,CONS_0112,wanna check the fucking life insurance rates i need assistance,"Quiero consultar las tarifas del seguro de vida, necesito ayuda.",QUOTE,propuesta_pendiente_revision
112,CONS_0113,"I want to know more about the rates, can you find some information?","Quiero saber más sobre las tarifas, ¿puedes conseguir información?",QUOTE,propuesta_pendiente_revision
113,CONS_0114,could ushow me the fucking rates,¿Puedes mostrarme las tarifas?,QUOTE,propuesta_pendiente_revision
114,CONS_0115,wanna estimate a fucking insurance quote can uhelp me,"Quiero estimar una cotización de seguro, ¿puedes ayudarme?",QUOTE,propuesta_pendiente_revision
115,CONS_0116,"I want to know more about the life insurance rates , will you find some information?","Quiero saber más sobre las tarifas del seguro de vida, ¿puedes conseguir información?",QUOTE,propuesta_pendiente_revision
116,CONS_0117,help me estimating an insurance quote,Ayúdame a estimar una cotización de seguro.,QUOTE,propuesta_pendiente_revision
117,CONS_0118,"I need an insurance quote, will you help me estimating it?","Necesito una cotización de seguro, ¿me ayudarás a estimarla?",QUOTE,propuesta_pendiente_revision
118,CONS_0119,i want a fuckin life insurance quote i wanna estimate it,"Quiero una cotización de seguro de vida, quiero estimarla.",QUOTE,propuesta_pendiente_revision
119,CONS_0120,I'd like to check the fucking rates could i get some hel,"Quisiera consultar las tarifas, ¿me pueden ayudar?",QUOTE,propuesta_pendiente_revision



Grupos repetidos que involucran prueba de QUOTE: 2


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
136,CONS_0137,prueba,"I am looking for information about the fucking life insurance rates, I neec assistance","Busco información sobre las tarifas del seguro de vida, necesito ayuda.",QUOTE
148,CONS_0149,prueba,"I am looking for information about the fucking life insurance rates, I need help","Busco información sobre las tarifas del seguro de vida, necesito ayuda.",QUOTE
117,CONS_0118,prueba,"I need an insurance quote, will you help me estimating it?","Necesito una cotización de seguro, ¿me ayudarás a estimarla?",QUOTE
146,CONS_0147,prueba,"I need an insurance quote, will you help me to estimate it?","Necesito una cotización de seguro, ¿me ayudarás a estimarla?",QUOTE



Traducciones propuestas: 38
Pendientes por referencia al producto: CONS_0111 y CONS_0136.
Las coincidencias mostradas requieren revisión y reemplazo.
Estado: confirmación del usuario pendiente.
✅ Traducciones de QUOTE propuestas y coincidencias comprobadas.


### 🧠 **6.19 Selección de candidatos de reemplazo de QUOTE**

Se presentan opciones para cuatro reemplazos: dos por referencias
al producto y dos por traducciones repetidas dentro de prueba.

Se conservarán la categoría y la intención de cada registro.
Los candidatos se revisarán antes de traducirlos e incorporarlos.

**Estado:** Opciones pendientes de revisión. La muestra no se modifica.

In [ ]:
# ============================================================
# 6.19 Opciones de reemplazo de QUOTE
# ============================================================

import pandas as pd
import re

print("=" * 60)
print("🔎 Preparando opciones para cuatro reemplazos de QUOTE...")
print("=" * 60)

ids_reemplazar_quote = [
    "CONS_0111",
    "CONS_0136",
    "CONS_0147",
    "CONS_0149"
]

objetivos_quote = df_muestra_revision.loc[
    df_muestra_revision["id_consulta"].isin(ids_reemplazar_quote)
].sort_values("id_consulta").copy()

if (
    len(objetivos_quote) != 4
    or not objetivos_quote["id_consulta"].is_unique
    or not objetivos_quote["conjunto"].eq("prueba").all()
    or not objetivos_quote["categoria_original"].eq("QUOTE").all()
):
    raise ValueError("Los cuatro objetivos no coinciden con los esperados.")

# Verificamos las dos coincidencias detectadas en 6.18
for conservar, reemplazar in [
    ("CONS_0118", "CONS_0147"),
    ("CONS_0137", "CONS_0149")
]:
    par = df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(
            [conservar, reemplazar]
        )
    ]

    textos = par["consulta_es"].fillna("").map(normalizar_contexto)

    if (
        len(par) != 2
        or not par["id_consulta"].is_unique
        or textos.eq("").any()
        or textos.nunique() != 1
    ):
        raise ValueError(
            f"La coincidencia de {reemplazar} cambió. Revisa 6.18."
        )

# Excluimos originales utilizados y registros retirados
textos_excluidos_quote = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_excluidos_quote.update(
        tabla["consulta_original"]
        .fillna("")
        .map(normalizar_contexto)
    )

reserva_quote = df_elegibles.copy()

reserva_quote["texto_revision"] = (
    reserva_quote["instruction"]
    .fillna("")
    .map(normalizar_contexto)
)

# Complementamos el detector con las variantes encontradas
def detectar_producto_candidatos_quote(texto):
    normalizado = normalizar_contexto(texto)

    return bool(
        detectar_referencias_producto(texto)
        or re.search(r"\b(?:travwl|ehalth)\b", normalizado)
        or re.search(r"\bpe\s+insur[a-z]*\b", normalizado)
    )

otros_productos = (
    reserva_quote["instruction"]
    .fillna("")
    .map(detectar_producto_candidatos_quote)
)

reserva_quote = (
    reserva_quote.loc[
        reserva_quote["texto_revision"].ne("")
        & ~reserva_quote["texto_revision"].isin(
            textos_excluidos_quote
        )
        & ~otros_productos
    ]
    .sort_values("texto_revision")
    .drop_duplicates("texto_revision")
    .copy()
)

# Presentamos hasta cinco opciones por objetivo.
# Evitamos repetir candidatos entre las cuatro listas.
partes_opciones_quote = []
textos_presentados_quote = set()

for posicion, (_, objetivo) in enumerate(objetivos_quote.iterrows()):
    candidatos = reserva_quote.loc[
        reserva_quote["category"].eq(
            objetivo["categoria_original"]
        )
        & reserva_quote["intent"].eq(
            objetivo["intencion_original"]
        )
        & ~reserva_quote["texto_revision"].isin(
            textos_presentados_quote
        )
    ]

    if candidatos.empty:
        raise ValueError(
            f"No hay candidatos disponibles para {objetivo['id_consulta']}."
        )

    seleccion = candidatos.sample(
        n=min(5, len(candidatos)),
        random_state=5842 + posicion
    ).copy()

    textos_presentados_quote.update(seleccion["texto_revision"])

    seleccion.insert(
        0, "id_a_reemplazar", objetivo["id_consulta"]
    )
    seleccion.insert(
        1, "opcion", range(1, len(seleccion) + 1)
    )

    partes_opciones_quote.append(seleccion)

df_opciones_reemplazos_quote = pd.concat(
    partes_opciones_quote,
    ignore_index=True
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_opciones_reemplazos_quote[
            [
                "id_a_reemplazar",
                "opcion",
                "instruction",
                "category",
                "intent"
            ]
        ].rename(columns={
            "instruction": "consulta_original",
            "category": "categoria_original",
            "intent": "intencion_original"
        })
    )

print("\nObjetivos de reemplazo: 4")
print("La muestra permanece sin cambios.")
print("Revisaremos los originales y sus posibles traducciones.")
print("Todavía no se han incorporado candidatos ni generado un respaldo.")

print("=" * 60)
print("✅ Opciones de reemplazo de QUOTE presentadas para revisión.")
print("=" * 60)

🔎 Preparando opciones para cuatro reemplazos de QUOTE...


,id_a_reemplazar,opcion,consulta_original,categoria_original,intencion_original
0,CONS_0111,1,I'm looking for an insuraance quote could uhelp me to find it,QUOTE,calculate_insurance_quote
1,CONS_0111,2,I got to estimate an life insurance quote,QUOTE,calculate_insurance_quote
2,CONS_0111,3,need to calculate a fucking ttavel insurance quote could ya help me,QUOTE,calculate_insurance_quote
3,CONS_0111,4,im looking for a fucking insurance quote help me,QUOTE,calculate_insurance_quote
4,CONS_0111,5,I'm lookng for an insurance quote help me,QUOTE,calculate_insurance_quote
5,CONS_0136,1,"I am triyng to find the insurance rates, I need help",QUOTE,check_rates
6,CONS_0136,2,"I need the fucking life insurance rates, how can I see them?",QUOTE,check_rates
7,CONS_0136,3,I'd like to check the fucking life insurance ratyes could ya help me,QUOTE,check_rates
8,CONS_0136,4,how can I find information about the artes?,QUOTE,check_rates
9,CONS_0136,5,"I want to see the fuckinghealth insurance rates, I need assistance",QUOTE,check_rates



Objetivos de reemplazo: 4
La muestra permanece sin cambios.
Revisaremos los originales y sus posibles traducciones.
Todavía no se han incorporado candidatos ni generado un respaldo.
✅ Opciones de reemplazo de QUOTE presentadas para revisión.


### 🧠 **6.20 Incorporación y respaldo de reemplazos de QUOTE**

Se incorporarán cuatro candidatos de QUOTE, conservando sus
identificadores, categoría, intención y conjunto de prueba.

Se comprobarán las traducciones repetidas y se conservarán
los originales retirados en la auditoría. Los registros previamente
validados permanecerán sin cambios.

Se descargará un ZIP con la muestra actualizada, la auditoría,
las consultas propuestas de QUOTE y la configuración para prueba.

**Estado:** QUOTE pendiente de confirmación del usuario.
Dataset de prueba todavía incompleto.

In [ ]:
# ============================================================
# 6.20 Incorporación y respaldo de reemplazos de QUOTE
# ============================================================

import pandas as pd
import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone

print("=" * 60)
print("🔎 Comprobando cuatro reemplazos de QUOTE...")
print("=" * 60)

elecciones_quote = {
    "CONS_0111": {
        "opcion": 2,
        "traduccion": "Tengo que estimar una cotización de seguro de vida.",
        "motivo": "Referencia ambigua al producto: pe insurance."
    },
    "CONS_0136": {
        "opcion": 4,
        "traduccion": "¿Cómo puedo encontrar información sobre las tarifas?",
        "motivo": "Referencia a seguro de salud: ehalth insurance."
    },
    "CONS_0147": {
        "opcion": 5,
        "traduccion": "¿Cómo puedo estimar una cotización de seguro de vida?",
        "motivo": "Traducción repetida con CONS_0118."
    },
    "CONS_0149": {
        "opcion": 1,
        "traduccion": (
            "Necesito información sobre las tarifas del seguro, "
            "¿dónde puedo obtenerla?"
        ),
        "motivo": "Traducción repetida con CONS_0137."
    }
}

# Trabajamos sobre copias; publicamos los cambios al final
muestra_quote_propuesta = df_muestra_revision.copy(deep=True)
fecha_actual = datetime.now(timezone.utc).isoformat()
retirados_quote = []

textos_ocupados = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_ocupados.update(
        tabla["consulta_original"].fillna("").map(normalizar_contexto)
    )

for identificador, eleccion in elecciones_quote.items():
    opciones = df_opciones_reemplazos_quote.loc[
        df_opciones_reemplazos_quote["id_a_reemplazar"].eq(identificador)
        & df_opciones_reemplazos_quote["opcion"].eq(eleccion["opcion"])
    ]

    mascara = muestra_quote_propuesta["id_consulta"].eq(identificador)
    original = muestra_quote_propuesta.loc[mascara]

    if len(opciones) != 1 or len(original) != 1:
        raise ValueError(f"Registro o candidato no único: {identificador}.")

    candidato = opciones.iloc[0]

    if (
        not original["conjunto"].eq("prueba").all()
        or not original["categoria_original"].eq("QUOTE").all()
        or candidato["category"] != "QUOTE"
        or candidato["intent"] != original.iloc[0]["intencion_original"]
    ):
        raise ValueError(
            f"Categoría, intención o conjunto incorrecto: {identificador}."
        )

    texto_candidato = normalizar_contexto(candidato["instruction"])

    if (
        not texto_candidato
        or texto_candidato in textos_ocupados
        or detectar_producto_candidatos_quote(candidato["instruction"])
    ):
        raise ValueError(
            f"El candidato de {identificador} requiere nueva revisión."
        )

    textos_ocupados.add(texto_candidato)

    retirado = original.copy()
    retirado["motivo_retiro"] = eleccion["motivo"]
    retirado["fecha_retiro_utc"] = fecha_actual
    retirado["etapa_retiro"] = "6.20 prueba QUOTE"
    retirados_quote.append(retirado)

    cambios = {
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"],
        "consulta_es": eleccion["traduccion"],
        "categoria_referencia": "QUOTE",
        "estado_revision": "propuesta_pendiente_revision",
        "observaciones": (
            "Original reemplazado: " + eleccion["motivo"]
            + " Traducción revisada por el asistente; "
            "confirmación del usuario pendiente."
        )
    }

    for columna, valor in cambios.items():
        muestra_quote_propuesta.loc[mascara, columna] = valor

    for columna in ["validado_por", "fecha_validacion_utc"]:
        if columna in muestra_quote_propuesta.columns:
            muestra_quote_propuesta.loc[mascara, columna] = ""

    if "revision_asistente" in muestra_quote_propuesta.columns:
        muestra_quote_propuesta.loc[
            mascara, "revision_asistente"
        ] = "completada"

# Comprobamos todos los textos ya traducidos
textos_es = (
    muestra_quote_propuesta["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = textos_es.ne("") & textos_es.duplicated(keep=False)

if repetidos.any():
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            muestra_quote_propuesta.loc[
                repetidos,
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ].sort_values("consulta_es")
        )

    raise ValueError(
        "Persisten traducciones repetidas. "
        "La muestra original no se modificó."
    )

mascara_quote = (
    muestra_quote_propuesta["conjunto"].eq("prueba")
    & muestra_quote_propuesta["categoria_original"].eq("QUOTE")
)

if (
    mascara_quote.sum() != 40
    or textos_es.loc[mascara_quote].eq("").any()
    or not muestra_quote_propuesta.loc[
        mascara_quote, "categoria_referencia"
    ].eq("QUOTE").all()
):
    raise ValueError("QUOTE debe conservar 40 consultas traducidas.")

# Comprobamos que todos los registros ajenos a los reemplazos
# permanecen exactamente iguales
mascara_otros = ~df_muestra_revision["id_consulta"].isin(
    elecciones_quote
)

if not df_muestra_revision.loc[mascara_otros].equals(
    muestra_quote_propuesta.loc[mascara_otros]
):
    raise ValueError("Se modificaron registros ajenos a los reemplazos.")

distribucion_antes = df_muestra_revision.groupby(
    ["categoria_original", "conjunto"]
).size()

distribucion_despues = muestra_quote_propuesta.groupby(
    ["categoria_original", "conjunto"]
).size()

if not distribucion_antes.equals(distribucion_despues):
    raise ValueError("Cambió la distribución de la muestra.")

auditoria_quote_propuesta = pd.concat(
    [df_registros_retirados] + retirados_quote,
    ignore_index=True,
    sort=False
)

# Guardamos un respaldo nuevo del avance
marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_avance_quote = (
    Path("/content/revision_evaluacion")
    / f"avance_prueba_quote_{marca_version}"
)
carpeta_avance_quote.mkdir(parents=True, exist_ok=False)

tablas_guardar = {
    "muestra_actualizada.csv": muestra_quote_propuesta,
    "registros_retirados.csv": auditoria_quote_propuesta,
    "prueba_quote_propuesta.csv": (
        muestra_quote_propuesta.loc[mascara_quote]
    ),
    "prueba_validada_hasta_coverage.csv": (
        muestra_quote_propuesta.loc[
            muestra_quote_propuesta["conjunto"].eq("prueba")
            & muestra_quote_propuesta["categoria_original"].isin(
                ["LIFE_INSURANCE", "COVERAGE"]
            )
            & muestra_quote_propuesta["estado_revision"].eq(
                "validado_usuario"
            )
        ]
    )
}

for nombre, tabla in tablas_guardar.items():
    tabla.to_csv(
        carpeta_avance_quote / nombre,
        index=False,
        encoding="utf-8-sig"
    )

with (
    carpeta_avance_quote / "configuracion_para_prueba.json"
).open("w", encoding="utf-8") as archivo:
    json.dump(
        {
            "modelos": modelos_comparacion,
            "configuracion_prueba": configuracion_prueba,
            "decision_desarrollo": decision_desarrollo
        },
        archivo,
        ensure_ascii=False,
        indent=2
    )

(carpeta_avance_quote / "LEEME.txt").write_text(
    "RESPALDO DE AVANCE: QUOTE\n\n"
    "muestra_actualizada.csv:\n"
    "  Insumo principal para continuar la preparación de prueba.\n\n"
    "registros_retirados.csv:\n"
    "  Auditoría acumulada de los reemplazos.\n\n"
    "prueba_quote_propuesta.csv:\n"
    "  40 consultas traducidas de QUOTE; validación pendiente.\n\n"
    "prueba_validada_hasta_coverage.csv:\n"
    "  Consultas de prueba aprobadas de LIFE_INSURANCE y COVERAGE.\n\n"
    "configuracion_para_prueba.json:\n"
    "  Modelos y configuración fijada para prueba.\n\n"
    "Faltan PAYMENT, ENROLLMENT y CONTACT.\n"
    "Este avance todavía no es el dataset final validado.\n"
    "Conservar también el respaldo de cierre de desarrollo.\n",
    encoding="utf-8"
)

ruta_zip_quote = Path(
    shutil.make_archive(
        str(carpeta_avance_quote),
        "zip",
        root_dir=str(carpeta_avance_quote)
    )
)

archivos_esperados = {
    ruta.name
    for ruta in carpeta_avance_quote.iterdir()
    if ruta.is_file()
}

with zipfile.ZipFile(ruta_zip_quote, "r") as respaldo:
    if (
        respaldo.testzip() is not None
        or not archivos_esperados.issubset(set(respaldo.namelist()))
    ):
        raise ValueError("Falló la verificación del ZIP.")

# Publicamos después de verificar y respaldar
df_muestra_revision = muestra_quote_propuesta
df_registros_retirados = auditoria_quote_propuesta

with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(elecciones_quote),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

print("\nConsultas de prueba QUOTE traducidas: 40")
print("Reemplazos incorporados: 4")
print("Grupos de traducciones repetidas: 0")
print("Registros retirados conservados:", len(df_registros_retirados))
print("QUOTE: confirmación del usuario pendiente.")

print("\n📁 Carpeta del respaldo:")
print(carpeta_avance_quote)
print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_quote.name)

print("\n📋 Archivos incluidos y uso posterior:")
print("  • muestra_actualizada.csv: continuar la preparación.")
print("  • registros_retirados.csv: consultar la auditoría.")
print("  • prueba_quote_propuesta.csv: revisar y validar QUOTE.")
print("  • prueba_validada_hasta_coverage.csv: recuperar prueba aprobada.")
print("  • configuracion_para_prueba.json: recuperar la configuración.")
print("  • LEEME.txt: consultar la guía del respaldo.")

print("\n⚠️ Dataset de prueba todavía incompleto.")
print("=" * 60)
print("✅ Reemplazos de QUOTE incorporados y avance respaldado.")
print("=" * 60)

from google.colab import files

print("\n📥 Solicitando descarga del ZIP...")
files.download(str(ruta_zip_quote))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

🔎 Comprobando cuatro reemplazos de QUOTE...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
110,CONS_0111,I got to estimate an life insurance quote,Tengo que estimar una cotización de seguro de vida.,QUOTE,propuesta_pendiente_revision
135,CONS_0136,how can I find information about the artes?,¿Cómo puedo encontrar información sobre las tarifas?,QUOTE,propuesta_pendiente_revision
146,CONS_0147,how do i estimate a life insuranbce quote,¿Cómo puedo estimar una cotización de seguro de vida?,QUOTE,propuesta_pendiente_revision
148,CONS_0149,i need information about the insurance rates where can i egt it,"Necesito información sobre las tarifas del seguro, ¿dónde puedo obtenerla?",QUOTE,propuesta_pendiente_revision



Consultas de prueba QUOTE traducidas: 40
Reemplazos incorporados: 4
Grupos de traducciones repetidas: 0
Registros retirados conservados: 33
QUOTE: confirmación del usuario pendiente.

📁 Carpeta del respaldo:
/content/revision_evaluacion/avance_prueba_quote_20261010T193801096349Z

📦 ZIP para conservar en tu computadora:
avance_prueba_quote_20261010T193801096349Z.zip

📋 Archivos incluidos y uso posterior:
  • muestra_actualizada.csv: continuar la preparación.
  • registros_retirados.csv: consultar la auditoría.
  • prueba_quote_propuesta.csv: revisar y validar QUOTE.
  • prueba_validada_hasta_coverage.csv: recuperar prueba aprobada.
  • configuracion_para_prueba.json: recuperar la configuración.
  • LEEME.txt: consultar la guía del respaldo.

⚠️ Dataset de prueba todavía incompleto.
✅ Reemplazos de QUOTE incorporados y avance respaldado.

📥 Solicitando descarga del ZIP...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


### 🧠 **6.21 Validación de QUOTE y revisión de PAYMENT**

Se registra la aprobación del usuario de las 40 traducciones
de prueba de QUOTE.

Se presentan las consultas CONS_0161 a CONS_0200 para revisar
sus referencias al producto e intenciones relacionadas con pagos,
métodos de pago, programación y problemas de pago.

**Estado:** QUOTE validado. PAYMENT pendiente de traducción y revisión.

In [ ]:
# ============================================================
# 6.21 Validación de QUOTE y revisión de PAYMENT
# ============================================================

import pandas as pd
from datetime import datetime, timezone

print("=" * 60)
print("✅ Registrando aprobación de QUOTE...")
print("=" * 60)

mascara_quote = (
    df_muestra_revision["conjunto"].eq("prueba")
    & df_muestra_revision["categoria_original"].eq("QUOTE")
)

registros_quote = df_muestra_revision.loc[mascara_quote]

textos_quote = (
    registros_quote["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

if (
    len(registros_quote) != 40
    or not registros_quote["id_consulta"].is_unique
    or textos_quote.eq("").any()
    or textos_quote.duplicated().any()
    or not registros_quote["categoria_referencia"].eq("QUOTE").all()
):
    raise ValueError(
        "QUOTE debe contener 40 traducciones completas "
        "y distintas con su categoría de referencia."
    )

# Comprobamos coincidencias con todos los registros traducidos
textos_todos = (
    df_muestra_revision["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = (
    textos_todos.ne("")
    & textos_todos.duplicated(keep=False)
)

if (repetidos & mascara_quote).any():
    raise ValueError(
        "Hay coincidencias de QUOTE con otros registros traducidos."
    )

# Preparamos la siguiente categoría antes de registrar cambios
ids_payment_prueba = [
    f"CONS_{numero:04d}"
    for numero in range(161, 201)
]

df_revision_payment_prueba = (
    df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(ids_payment_prueba)
    ]
    .sort_values("id_consulta")
    .copy()
)

if (
    len(df_revision_payment_prueba) != 40
    or not df_revision_payment_prueba["id_consulta"].is_unique
    or set(df_revision_payment_prueba["id_consulta"])
        != set(ids_payment_prueba)
    or not df_revision_payment_prueba["conjunto"].eq("prueba").all()
    or not df_revision_payment_prueba[
        "categoria_original"
    ].eq("PAYMENT").all()
):
    raise ValueError(
        "Los registros no corresponden a las 40 consultas "
        "de prueba de PAYMENT."
    )

# Registramos la confirmación explícita del usuario
fecha_actual = datetime.now(timezone.utc).isoformat()

for columna in ["validado_por", "fecha_validacion_utc"]:
    if columna not in df_muestra_revision.columns:
        df_muestra_revision[columna] = ""

df_muestra_revision.loc[
    mascara_quote, "estado_revision"
] = "validado_usuario"

df_muestra_revision.loc[
    mascara_quote, "validado_por"
] = "usuario"

sin_fecha = (
    df_muestra_revision["fecha_validacion_utc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

df_muestra_revision.loc[
    mascara_quote & sin_fecha,
    "fecha_validacion_utc"
] = fecha_actual

print("\nConsultas de prueba QUOTE validadas: 40")
print("Validación registrada en memoria.")
print("Se incluirá en el siguiente respaldo descargable.")

print("\n" + "=" * 60)
print("🔎 Presentando las 40 consultas de prueba de PAYMENT...")
print("=" * 60)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_revision_payment_prueba[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("\nConsultas de PAYMENT presentadas: 40")
print("Estado: pendientes de traducción y revisión.")

print("=" * 60)
print("✅ QUOTE validado y PAYMENT presentado para revisión.")
print("=" * 60)

✅ Registrando aprobación de QUOTE...

Consultas de prueba QUOTE validadas: 40
Validación registrada en memoria.
Se incluirá en el siguiente respaldo descargable.

🔎 Presentando las 40 consultas de prueba de PAYMENT...


,id_consulta,consulta_original,categoria_original,intencion_original
160,CONS_0161,"I want to check your fucking payment methods, could I get some help?",PAYMENT,payment_methods
161,CONS_0162,iwant assistance to pay,PAYMENT,pay
162,CONS_0163,need to see ur fucking payment options how could i do it,PAYMENT,payment_methods
163,CONS_0164,I'd like to schedule the fucking payments will uhelp me,PAYMENT,schedule_payments
164,CONS_0165,i got to make transfers were could i do it,PAYMENT,pay
165,CONS_0166,I'd like to know more about ur fucking payment options give me some informtion,PAYMENT,payment_methods
166,CONS_0167,can you help me to schedule the fucking payents?,PAYMENT,schedule_payments
167,CONS_0168,"I get a fucking error when I try to make a payment, can you notify of it?",PAYMENT,report_payment_issue
168,CONS_0169,"I'd like to check my fucking payments, could I get some help?",PAYMENT,check_payments
169,CONS_0170,"I cannot pay, help me report a fucking payment issue",PAYMENT,report_payment_issue



Consultas de PAYMENT presentadas: 40
Estado: pendientes de traducción y revisión.
✅ QUOTE validado y PAYMENT presentado para revisión.


### 🧠 **6.22 Traducción y revisión de PAYMENT**

Se proponen traducciones de las 40 consultas de prueba de PAYMENT,
sin expresiones ofensivas y conservando su intención.

Se comprobarán las traducciones repetidas dentro de prueba
y las coincidencias con desarrollo.

**Estado:** Traducciones propuestas; revisión de coincidencias
y validación del usuario pendientes.

In [ ]:
# ============================================================
# 6.22 Traducción y revisión de PAYMENT
# ============================================================

import pandas as pd

print("=" * 60)
print("📝 Preparando las 40 traducciones de PAYMENT...")
print("=" * 60)

traducciones_payment = {
    "CONS_0161": "Quiero consultar sus métodos de pago, ¿me pueden ayudar?",
    "CONS_0162": "Quiero ayuda para pagar.",
    "CONS_0163": "Necesito consultar sus opciones de pago, ¿cómo puedo hacerlo?",
    "CONS_0164": "Quisiera programar los pagos, ¿me ayudarás?",
    "CONS_0165": "Tengo que hacer transferencias, ¿dónde puedo hacerlo?",
    "CONS_0166": "Quisiera saber más sobre sus opciones de pago, dame información.",
    "CONS_0167": "¿Puedes ayudarme a programar los pagos?",
    "CONS_0168": "Me aparece un error cuando intento hacer un pago, ¿puedes reportarlo?",
    "CONS_0169": "Quisiera consultar mis pagos, ¿me pueden ayudar?",
    "CONS_0170": "No puedo pagar, ayúdame a reportar un problema de pago.",
    "CONS_0171": "Quisiera consultar sus métodos de pago, ¿cómo lo hago?",
    "CONS_0172": "El pago no se procesó, necesito ayuda para reportar un problema de pago.",
    "CONS_0173": "Tengo que hacer un pago, ¿dónde puedo hacerlo?",
    "CONS_0174": "¿Puedo pagar con tarjeta?",
    "CONS_0175": "No puedo hacer una transferencia, quiero ayuda para reportar un problema de pago.",
    "CONS_0176": "Tengo que programar los pagos, ¿me pueden ayudar?",
    "CONS_0177": "Tengo que pagar con tarjeta de crédito, ayúdame.",
    "CONS_0178": "¿Dónde puedo consultar sus opciones de pago?",
    "CONS_0179": "Quisiera programar los pagos, ¿puedes ayudarme?",
    "CONS_0180": "Tengo que programar los pagos, ¿dónde puedo hacerlo?",
    "CONS_0181": "¿Dónde puedo consultar sus opciones de pago?",
    "CONS_0182": "Necesito ayuda para reportar un problema de pago.",
    "CONS_0183": "No puedo hacer una transferencia, ayúdame a reportar problemas de pago.",
    "CONS_0184": "Quisiera pagar con tarjeta de crédito, necesito ayuda.",
    "CONS_0185": "Necesito programar los pagos, ¿dónde puedo hacerlo?",
    "CONS_0186": "Quiero programar los pagos, ¿dónde puedo hacerlo?",
    "CONS_0187": "Necesito consultar mis pagos a la compañía de seguros de vida, ¿dónde puedo hacerlo?",
    "CONS_0188": "¿Dónde puedo consultar sus métodos de pago?",
    "CONS_0189": "Necesito ayuda para pagar.",
    "CONS_0190": "No puedo hacer transferencias, necesito ayuda para reportar problemas de pago.",
    "CONS_0191": "Tengo que hacer una transferencia, ¿dónde puedo hacerlo?",
    "CONS_0192": "Tengo que consultar sus métodos de pago, ¿cómo puedo hacerlo?",
    "CONS_0193": "Tengo que programar los pagos, ayúdame.",
    "CONS_0194": "Tengo que consultar mis pagos, ¿cómo puedo hacerlo?",
    "CONS_0195": "¿Sería posible consultar sus opciones de pago?",
    "CONS_0196": "Necesito consultar mis pagos a la compañía de seguros de vida, ¿dónde puedo hacerlo?",
    "CONS_0197": "Tengo que programar los pagos, ¿me pueden ayudar?",
    "CONS_0198": "Quisiera información sobre sus opciones de pago.",
    "CONS_0199": "Quisiera saber sobre sus opciones de pago, ¿dónde puedo obtener más información?",
    "CONS_0200": "¿Puedes ayudarme a pagar con tarjeta?"
}

ids_payment = {
    f"CONS_{numero:04d}"
    for numero in range(161, 201)
}

if set(traducciones_payment) != ids_payment:
    raise ValueError("Faltan traducciones de los 40 registros previstos.")

muestra_propuesta = df_muestra_revision.copy(deep=True)

registros_payment = muestra_propuesta.loc[
    muestra_propuesta["id_consulta"].isin(ids_payment)
]

if (
    len(registros_payment) != 40
    or not registros_payment["id_consulta"].is_unique
    or not registros_payment["conjunto"].eq("prueba").all()
    or not registros_payment["categoria_original"].eq("PAYMENT").all()
):
    raise ValueError(
        "Los registros no corresponden a las 40 consultas de PAYMENT."
    )

# Aplicamos las traducciones únicamente a esta categoría de prueba
for identificador, traduccion in traducciones_payment.items():
    mascara = muestra_propuesta["id_consulta"].eq(identificador)

    muestra_propuesta.loc[mascara, "consulta_es"] = traduccion
    muestra_propuesta.loc[mascara, "categoria_referencia"] = "PAYMENT"
    muestra_propuesta.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"

df_muestra_revision = muestra_propuesta

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(ids_payment),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

# Comprobamos coincidencias con todos los registros traducidos
textos_es = (
    df_muestra_revision["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = textos_es.ne("") & textos_es.duplicated(keep=False)

mascara_payment_prueba = (
    df_muestra_revision["conjunto"].eq("prueba")
    & df_muestra_revision["categoria_original"].eq("PAYMENT")
)

textos_repetidos_payment = set(
    textos_es.loc[repetidos & mascara_payment_prueba]
)

print(
    "\nGrupos repetidos que involucran prueba de PAYMENT:",
    len(textos_repetidos_payment)
)

if textos_repetidos_payment:
    mascara_coincidencias = textos_es.isin(textos_repetidos_payment)

    coincidencias_payment = df_muestra_revision.loc[
        mascara_coincidencias,
        [
            "id_consulta",
            "conjunto",
            "consulta_original",
            "consulta_es",
            "categoria_referencia"
        ]
    ].copy()

    coincidencias_payment["_orden"] = textos_es.loc[
        mascara_coincidencias
    ]

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            coincidencias_payment.sort_values(
                ["_orden", "id_consulta"]
            ).drop(columns="_orden")
        )

print("\nTraducciones propuestas: 40")
print("Las coincidencias mostradas requieren revisión.")
print("Estado: confirmación del usuario pendiente.")
print("Este bloque registra propuestas en memoria.")
print("El siguiente respaldo incluirá la validación de QUOTE.")

print("=" * 60)
print("✅ Traducciones de PAYMENT propuestas y coincidencias comprobadas.")
print("=" * 60)

📝 Preparando las 40 traducciones de PAYMENT...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
160,CONS_0161,"I want to check your fucking payment methods, could I get some help?","Quiero consultar sus métodos de pago, ¿me pueden ayudar?",PAYMENT,propuesta_pendiente_revision
161,CONS_0162,iwant assistance to pay,Quiero ayuda para pagar.,PAYMENT,propuesta_pendiente_revision
162,CONS_0163,need to see ur fucking payment options how could i do it,"Necesito consultar sus opciones de pago, ¿cómo puedo hacerlo?",PAYMENT,propuesta_pendiente_revision
163,CONS_0164,I'd like to schedule the fucking payments will uhelp me,"Quisiera programar los pagos, ¿me ayudarás?",PAYMENT,propuesta_pendiente_revision
164,CONS_0165,i got to make transfers were could i do it,"Tengo que hacer transferencias, ¿dónde puedo hacerlo?",PAYMENT,propuesta_pendiente_revision
165,CONS_0166,I'd like to know more about ur fucking payment options give me some informtion,"Quisiera saber más sobre sus opciones de pago, dame información.",PAYMENT,propuesta_pendiente_revision
166,CONS_0167,can you help me to schedule the fucking payents?,¿Puedes ayudarme a programar los pagos?,PAYMENT,propuesta_pendiente_revision
167,CONS_0168,"I get a fucking error when I try to make a payment, can you notify of it?","Me aparece un error cuando intento hacer un pago, ¿puedes reportarlo?",PAYMENT,propuesta_pendiente_revision
168,CONS_0169,"I'd like to check my fucking payments, could I get some help?","Quisiera consultar mis pagos, ¿me pueden ayudar?",PAYMENT,propuesta_pendiente_revision
169,CONS_0170,"I cannot pay, help me report a fucking payment issue","No puedo pagar, ayúdame a reportar un problema de pago.",PAYMENT,propuesta_pendiente_revision



Grupos repetidos que involucran prueba de PAYMENT: 3


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
186,CONS_0187,prueba,"I need to check my payments to the life insurance company , where can I do it?","Necesito consultar mis pagos a la compañía de seguros de vida, ¿dónde puedo hacerlo?",PAYMENT
195,CONS_0196,prueba,need to see my fucking payments to the life insurance company where can i do it,"Necesito consultar mis pagos a la compañía de seguros de vida, ¿dónde puedo hacerlo?",PAYMENT
175,CONS_0176,prueba,"I got to schedule the payments, couod I get some help?","Tengo que programar los pagos, ¿me pueden ayudar?",PAYMENT
196,CONS_0197,prueba,i have to schedule the fucking payments could i egt some help,"Tengo que programar los pagos, ¿me pueden ayudar?",PAYMENT
177,CONS_0178,prueba,where do i check ur payment options,¿Dónde puedo consultar sus opciones de pago?,PAYMENT
180,CONS_0181,prueba,where could i check ur payment options,¿Dónde puedo consultar sus opciones de pago?,PAYMENT



Traducciones propuestas: 40
Las coincidencias mostradas requieren revisión.
Estado: confirmación del usuario pendiente.
Este bloque registra propuestas en memoria.
El siguiente respaldo incluirá la validación de QUOTE.
✅ Traducciones de PAYMENT propuestas y coincidencias comprobadas.


### 🧠 **6.23 Selección de candidatos de reemplazo de PAYMENT**

Se presentan opciones para reemplazar CONS_0181, CONS_0196
y CONS_0197 por traducciones repetidas dentro de prueba.

Se conservarán la categoría y la intención de cada registro.
Los candidatos se revisarán antes de traducirlos e incorporarlos.

**Estado:** Opciones pendientes de revisión. La muestra no se modifica.

In [ ]:
# ============================================================
# 6.23 Opciones de reemplazo de PAYMENT
# ============================================================

import pandas as pd
import re

print("=" * 60)
print("🔎 Preparando opciones para tres reemplazos de PAYMENT...")
print("=" * 60)

pares_duplicados_payment = [
    ("CONS_0178", "CONS_0181"),
    ("CONS_0187", "CONS_0196"),
    ("CONS_0176", "CONS_0197")
]

ids_reemplazar_payment = [
    reemplazar
    for conservar, reemplazar in pares_duplicados_payment
]

objetivos_payment = df_muestra_revision.loc[
    df_muestra_revision["id_consulta"].isin(
        ids_reemplazar_payment
    )
].sort_values("id_consulta").copy()

if (
    len(objetivos_payment) != 3
    or not objetivos_payment["id_consulta"].is_unique
    or not objetivos_payment["conjunto"].eq("prueba").all()
    or not objetivos_payment["categoria_original"].eq("PAYMENT").all()
):
    raise ValueError("Los tres objetivos no coinciden con los esperados.")

# Comprobamos las coincidencias detectadas en 6.22
for conservar, reemplazar in pares_duplicados_payment:
    par = df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(
            [conservar, reemplazar]
        )
    ]

    textos = par["consulta_es"].fillna("").map(normalizar_contexto)

    if (
        len(par) != 2
        or not par["id_consulta"].is_unique
        or textos.eq("").any()
        or textos.nunique() != 1
    ):
        raise ValueError(
            f"La coincidencia de {reemplazar} cambió. Revisa 6.22."
        )

# Excluimos originales utilizados y registros retirados
textos_excluidos_payment = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_excluidos_payment.update(
        tabla["consulta_original"]
        .fillna("")
        .map(normalizar_contexto)
    )

reserva_payment = df_elegibles.copy()

reserva_payment["texto_revision"] = (
    reserva_payment["instruction"]
    .fillna("")
    .map(normalizar_contexto)
)

# Complementamos el detector con variantes observadas.
# La revisión posterior también comprobará errores no previstos.
def detectar_producto_candidatos_payment(texto):
    normalizado = normalizar_contexto(texto)

    return bool(
        detectar_referencias_producto(texto)
        or re.search(
            r"\b(?:travwl|ttavel|ehalth)\b",
            normalizado
        )
        or re.search(r"\bpe\s+insur[a-z]*\b", normalizado)
        or re.search(
            r"fucking(?:health|travel|pet|auto|car|home)",
            normalizado
        )
    )

otros_productos = (
    reserva_payment["instruction"]
    .fillna("")
    .map(detectar_producto_candidatos_payment)
)

reserva_payment = (
    reserva_payment.loc[
        reserva_payment["texto_revision"].ne("")
        & ~reserva_payment["texto_revision"].isin(
            textos_excluidos_payment
        )
        & ~otros_productos
    ]
    .sort_values("texto_revision")
    .drop_duplicates("texto_revision")
    .copy()
)

# Presentamos hasta cinco opciones por objetivo
partes_opciones_payment = []
textos_presentados_payment = set()

for posicion, (_, objetivo) in enumerate(
    objetivos_payment.iterrows()
):
    candidatos = reserva_payment.loc[
        reserva_payment["category"].eq(
            objetivo["categoria_original"]
        )
        & reserva_payment["intent"].eq(
            objetivo["intencion_original"]
        )
        & ~reserva_payment["texto_revision"].isin(
            textos_presentados_payment
        )
    ]

    if candidatos.empty:
        raise ValueError(
            f"No hay candidatos para {objetivo['id_consulta']}."
        )

    seleccion = candidatos.sample(
        n=min(5, len(candidatos)),
        random_state=6842 + posicion
    ).copy()

    textos_presentados_payment.update(
        seleccion["texto_revision"]
    )

    seleccion.insert(
        0, "id_a_reemplazar", objetivo["id_consulta"]
    )
    seleccion.insert(
        1, "opcion", range(1, len(seleccion) + 1)
    )

    partes_opciones_payment.append(seleccion)

df_opciones_reemplazos_payment = pd.concat(
    partes_opciones_payment,
    ignore_index=True
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_opciones_reemplazos_payment[
            [
                "id_a_reemplazar",
                "opcion",
                "instruction",
                "category",
                "intent"
            ]
        ].rename(columns={
            "instruction": "consulta_original",
            "category": "categoria_original",
            "intent": "intencion_original"
        })
    )

print("\nObjetivos de reemplazo: 3")
print("La muestra permanece sin cambios.")
print("Revisaremos los originales y sus posibles traducciones.")
print("Este bloque presenta opciones; todavía no genera un respaldo.")

print("=" * 60)
print("✅ Opciones de reemplazo de PAYMENT presentadas para revisión.")
print("=" * 60)

🔎 Preparando opciones para tres reemplazos de PAYMENT...


,id_a_reemplazar,opcion,consulta_original,categoria_original,intencion_original
0,CONS_0181,1,im looking for information about ur fucking payment methods how can i find it,PAYMENT,payment_methods
1,CONS_0181,2,i need help seeing ur fucking payment options,PAYMENT,payment_methods
2,CONS_0181,3,id like to pay by debit card where can i do it,PAYMENT,payment_methods
3,CONS_0181,4,could you give me information about your fucking payment options?,PAYMENT,payment_methods
4,CONS_0181,5,I want information about your fucking payment options,PAYMENT,payment_methods
5,CONS_0196,1,"I'd like to check my fucking payments to the life insurance company, how to do it?",PAYMENT,check_payments
6,CONS_0196,2,"I want to see my fucking payments to the life insurance company , could you help me?",PAYMENT,check_payments
7,CONS_0196,3,help me checking my fucking payments to the life insurance company,PAYMENT,check_payments
8,CONS_0196,4,is it possible to see my fucking payments,PAYMENT,check_payments
9,CONS_0196,5,"I would like to check my payments, where can I do it ?",PAYMENT,check_payments



Objetivos de reemplazo: 3
La muestra permanece sin cambios.
Revisaremos los originales y sus posibles traducciones.
Este bloque presenta opciones; todavía no genera un respaldo.
✅ Opciones de reemplazo de PAYMENT presentadas para revisión.


### 🧠 **6.24 Incorporación y respaldo de reemplazos de PAYMENT**

Se incorporarán tres candidatos de PAYMENT, conservando sus
identificadores, categoría, intención y conjunto de prueba.

Se comprobarán las traducciones repetidas y se conservarán
los originales retirados en la auditoría. Los demás registros
permanecerán sin cambios.

Se descargará un ZIP con la muestra actualizada, la auditoría,
las consultas propuestas de PAYMENT y la configuración para prueba.

**Estado:** PAYMENT pendiente de confirmación del usuario.
Faltan ENROLLMENT y CONTACT para completar la preparación de prueba.

In [ ]:
# ============================================================
# 6.24 Incorporación y respaldo de reemplazos de PAYMENT
# ============================================================

import pandas as pd
import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("=" * 60)
print("🔎 Comprobando tres reemplazos de PAYMENT...")
print("=" * 60)

elecciones_payment = {
    "CONS_0181": {
        "opcion": 1,
        "traduccion": (
            "Busco información sobre sus métodos de pago, "
            "¿cómo puedo encontrarla?"
        ),
        "motivo": "Traducción repetida con CONS_0178."
    },
    "CONS_0196": {
        "opcion": 4,
        "traduccion": "¿Es posible consultar mis pagos?",
        "motivo": "Traducción repetida con CONS_0187."
    },
    "CONS_0197": {
        "opcion": 4,
        "traduccion": "Quisiera programar los pagos.",
        "motivo": "Traducción repetida con CONS_0176."
    }
}

# ------------------------------------------------------------
# 1. Preparamos copias y excluimos originales utilizados
# ------------------------------------------------------------

muestra_propuesta = df_muestra_revision.copy(deep=True)
fecha_actual = datetime.now(timezone.utc).isoformat()
retirados_payment = []
textos_ocupados = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_ocupados.update(
        tabla["consulta_original"]
        .fillna("")
        .map(normalizar_contexto)
    )

# ------------------------------------------------------------
# 2. Comprobamos e incorporamos las opciones sobre una copia
# ------------------------------------------------------------

for identificador, eleccion in elecciones_payment.items():
    opciones = df_opciones_reemplazos_payment.loc[
        df_opciones_reemplazos_payment["id_a_reemplazar"].eq(
            identificador
        )
        & df_opciones_reemplazos_payment["opcion"].eq(
            eleccion["opcion"]
        )
    ]

    mascara = muestra_propuesta["id_consulta"].eq(identificador)
    original = muestra_propuesta.loc[mascara]

    if len(opciones) != 1 or len(original) != 1:
        raise ValueError(
            f"Registro o candidato no único: {identificador}."
        )

    candidato = opciones.iloc[0]

    if (
        not original["conjunto"].eq("prueba").all()
        or not original["categoria_original"].eq("PAYMENT").all()
        or candidato["category"] != "PAYMENT"
        or candidato["intent"] != original.iloc[0]["intencion_original"]
    ):
        raise ValueError(
            f"Categoría, intención o conjunto incorrecto: {identificador}."
        )

    texto_candidato = normalizar_contexto(candidato["instruction"])

    if (
        not texto_candidato
        or texto_candidato in textos_ocupados
        or detectar_producto_candidatos_payment(
            candidato["instruction"]
        )
    ):
        raise ValueError(
            f"El candidato de {identificador} requiere nueva revisión."
        )

    textos_ocupados.add(texto_candidato)

    retirado = original.copy()
    retirado["motivo_retiro"] = eleccion["motivo"]
    retirado["fecha_retiro_utc"] = fecha_actual
    retirado["etapa_retiro"] = "6.24 prueba PAYMENT"
    retirados_payment.append(retirado)

    cambios = {
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"],
        "consulta_es": eleccion["traduccion"],
        "categoria_referencia": "PAYMENT",
        "estado_revision": "propuesta_pendiente_revision",
        "observaciones": (
            "Original reemplazado: " + eleccion["motivo"]
            + " Traducción revisada por el asistente; "
            "confirmación del usuario pendiente."
        )
    }

    for columna, valor in cambios.items():
        muestra_propuesta.loc[mascara, columna] = valor

    # Retiramos cualquier validación anterior del registro reemplazado
    for columna in ["validado_por", "fecha_validacion_utc"]:
        if columna in muestra_propuesta.columns:
            muestra_propuesta.loc[mascara, columna] = ""

    if "revision_asistente" in muestra_propuesta.columns:
        muestra_propuesta.loc[
            mascara, "revision_asistente"
        ] = "completada"

# ------------------------------------------------------------
# 3. Verificamos coincidencias y conservación de la muestra
# ------------------------------------------------------------

textos_es = (
    muestra_propuesta["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = textos_es.ne("") & textos_es.duplicated(keep=False)

if repetidos.any():
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            muestra_propuesta.loc[
                repetidos,
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ].sort_values("consulta_es")
        )

    raise ValueError(
        "Persisten traducciones repetidas. "
        "La muestra original no se modificó."
    )

mascara_payment = (
    muestra_propuesta["conjunto"].eq("prueba")
    & muestra_propuesta["categoria_original"].eq("PAYMENT")
)

if (
    mascara_payment.sum() != 40
    or textos_es.loc[mascara_payment].eq("").any()
    or not muestra_propuesta.loc[
        mascara_payment, "categoria_referencia"
    ].eq("PAYMENT").all()
):
    raise ValueError(
        "PAYMENT debe conservar 40 consultas traducidas."
    )

mascara_otros = ~df_muestra_revision["id_consulta"].isin(
    elecciones_payment
)

if not df_muestra_revision.loc[mascara_otros].equals(
    muestra_propuesta.loc[mascara_otros]
):
    raise ValueError(
        "Se modificaron registros ajenos a los reemplazos."
    )

distribucion_antes = df_muestra_revision.groupby(
    ["categoria_original", "conjunto"]
).size()

distribucion_despues = muestra_propuesta.groupby(
    ["categoria_original", "conjunto"]
).size()

if not distribucion_antes.equals(distribucion_despues):
    raise ValueError("Cambió la distribución de la muestra.")

auditoria_propuesta = pd.concat(
    [df_registros_retirados] + retirados_payment,
    ignore_index=True,
    sort=False
)

# Recuperamos las tres categorías de prueba ya aprobadas
mascara_prueba_validada = (
    muestra_propuesta["conjunto"].eq("prueba")
    & muestra_propuesta["categoria_original"].isin(
        ["LIFE_INSURANCE", "COVERAGE", "QUOTE"]
    )
    & muestra_propuesta["estado_revision"].eq("validado_usuario")
)

if mascara_prueba_validada.sum() != 120:
    raise ValueError(
        "Deben existir 120 consultas de prueba validadas "
        "entre LIFE_INSURANCE, COVERAGE y QUOTE."
    )

# ------------------------------------------------------------
# 4. Guardamos los insumos y una guía de uso
# ------------------------------------------------------------

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_avance_payment = (
    Path("/content/revision_evaluacion")
    / f"avance_prueba_payment_{marca_version}"
)
carpeta_avance_payment.mkdir(parents=True, exist_ok=False)

tablas_guardar = {
    "muestra_actualizada.csv": muestra_propuesta,
    "registros_retirados.csv": auditoria_propuesta,
    "prueba_payment_propuesta.csv": (
        muestra_propuesta.loc[mascara_payment]
    ),
    "prueba_validada_hasta_quote.csv": (
        muestra_propuesta.loc[mascara_prueba_validada]
    )
}

for nombre, tabla in tablas_guardar.items():
    tabla.to_csv(
        carpeta_avance_payment / nombre,
        index=False,
        encoding="utf-8-sig"
    )

with (
    carpeta_avance_payment / "configuracion_para_prueba.json"
).open("w", encoding="utf-8") as archivo:
    json.dump(
        {
            "modelos": modelos_comparacion,
            "configuracion_prueba": configuracion_prueba,
            "decision_desarrollo": decision_desarrollo
        },
        archivo,
        ensure_ascii=False,
        indent=2
    )

(carpeta_avance_payment / "LEEME.txt").write_text(
    "RESPALDO DE AVANCE: PAYMENT\n\n"
    "muestra_actualizada.csv:\n"
    "  Insumo principal para continuar la preparación de prueba.\n\n"
    "registros_retirados.csv:\n"
    "  Auditoría acumulada de los reemplazos.\n\n"
    "prueba_payment_propuesta.csv:\n"
    "  40 consultas traducidas de PAYMENT; validación pendiente.\n\n"
    "prueba_validada_hasta_quote.csv:\n"
    "  120 consultas aprobadas de LIFE_INSURANCE, COVERAGE y QUOTE.\n\n"
    "configuracion_para_prueba.json:\n"
    "  Modelos y configuración fijada para prueba.\n\n"
    "Faltan ENROLLMENT y CONTACT.\n"
    "Este avance todavía no es el dataset final validado.\n"
    "Conservar también el respaldo de cierre de desarrollo.\n",
    encoding="utf-8"
)

# ------------------------------------------------------------
# 5. Comprimimos, verificamos y publicamos los cambios
# ------------------------------------------------------------

ruta_zip_payment = Path(
    shutil.make_archive(
        str(carpeta_avance_payment),
        "zip",
        root_dir=str(carpeta_avance_payment)
    )
)

archivos_esperados = {
    ruta.name
    for ruta in carpeta_avance_payment.iterdir()
    if ruta.is_file()
}

with zipfile.ZipFile(ruta_zip_payment, "r") as respaldo:
    if (
        respaldo.testzip() is not None
        or not archivos_esperados.issubset(
            set(respaldo.namelist())
        )
    ):
        raise ValueError("Falló la verificación del ZIP.")

df_muestra_revision = muestra_propuesta
df_registros_retirados = auditoria_propuesta

with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(
                elecciones_payment
            ),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

print("\nConsultas de prueba PAYMENT traducidas: 40")
print("Reemplazos incorporados: 3")
print("Grupos de traducciones repetidas: 0")
print("Consultas de prueba previamente validadas: 120")
print("Registros retirados conservados:",
      len(df_registros_retirados))
print("PAYMENT: confirmación del usuario pendiente.")

print("\n📁 Carpeta del respaldo:")
print(carpeta_avance_payment)

print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_payment.name)

print("\n📋 Archivos incluidos y uso posterior:")
print("  • muestra_actualizada.csv: continuar la preparación.")
print("  • registros_retirados.csv: consultar la auditoría.")
print("  • prueba_payment_propuesta.csv: revisar y validar PAYMENT.")
print("  • prueba_validada_hasta_quote.csv: recuperar prueba aprobada.")
print("  • configuracion_para_prueba.json: recuperar la configuración.")
print("  • LEEME.txt: consultar la guía del respaldo.")

print("\n⚠️ Dataset de prueba todavía incompleto.")
print("=" * 60)
print("✅ Reemplazos de PAYMENT incorporados y avance respaldado.")
print("=" * 60)

print("\n📥 Solicitando descarga del ZIP...")
files.download(str(ruta_zip_payment))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

🔎 Comprobando tres reemplazos de PAYMENT...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
180,CONS_0181,im looking for information about ur fucking payment methods how can i find it,"Busco información sobre sus métodos de pago, ¿cómo puedo encontrarla?",PAYMENT,propuesta_pendiente_revision
195,CONS_0196,is it possible to see my fucking payments,¿Es posible consultar mis pagos?,PAYMENT,propuesta_pendiente_revision
196,CONS_0197,I would like to scheduel the payments,Quisiera programar los pagos.,PAYMENT,propuesta_pendiente_revision



Consultas de prueba PAYMENT traducidas: 40
Reemplazos incorporados: 3
Grupos de traducciones repetidas: 0
Consultas de prueba previamente validadas: 120
Registros retirados conservados: 36
PAYMENT: confirmación del usuario pendiente.

📁 Carpeta del respaldo:
/content/revision_evaluacion/avance_prueba_payment_20261010T194203887664Z

📦 ZIP para conservar en tu computadora:
avance_prueba_payment_20261010T194203887664Z.zip

📋 Archivos incluidos y uso posterior:
  • muestra_actualizada.csv: continuar la preparación.
  • registros_retirados.csv: consultar la auditoría.
  • prueba_payment_propuesta.csv: revisar y validar PAYMENT.
  • prueba_validada_hasta_quote.csv: recuperar prueba aprobada.
  • configuracion_para_prueba.json: recuperar la configuración.
  • LEEME.txt: consultar la guía del respaldo.

⚠️ Dataset de prueba todavía incompleto.
✅ Reemplazos de PAYMENT incorporados y avance respaldado.

📥 Solicitando descarga del ZIP...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


### 🧠 **6.25 Validación de PAYMENT y revisión de ENROLLMENT**

Se registra la aprobación del usuario de las 40 traducciones
de prueba de PAYMENT.

Se presentan las consultas CONS_0211 a CONS_0250 para revisar
las referencias al producto y las intenciones de comparación,
contratación, renovación y cancelación de pólizas.

Los cargos vinculados a cancelación o terminación se mantienen
en ENROLLMENT conforme al criterio acordado.

**Estado:** PAYMENT validado. ENROLLMENT pendiente de traducción
y revisión.

In [ ]:
# ============================================================
# 6.25 Validación de PAYMENT y revisión de ENROLLMENT
# ============================================================

import pandas as pd
from datetime import datetime, timezone

print("=" * 60)
print("✅ Registrando aprobación de PAYMENT...")
print("=" * 60)

mascara_payment = (
    df_muestra_revision["conjunto"].eq("prueba")
    & df_muestra_revision["categoria_original"].eq("PAYMENT")
)

registros_payment = df_muestra_revision.loc[mascara_payment]

textos_payment = (
    registros_payment["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

if (
    len(registros_payment) != 40
    or not registros_payment["id_consulta"].is_unique
    or textos_payment.eq("").any()
    or textos_payment.duplicated().any()
    or not registros_payment["categoria_referencia"].eq("PAYMENT").all()
):
    raise ValueError(
        "PAYMENT debe contener 40 traducciones completas "
        "y distintas con su categoría de referencia."
    )

# Comprobamos coincidencias con todos los registros traducidos
textos_todos = (
    df_muestra_revision["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = (
    textos_todos.ne("")
    & textos_todos.duplicated(keep=False)
)

if (repetidos & mascara_payment).any():
    raise ValueError(
        "Hay coincidencias de PAYMENT con otros registros traducidos."
    )

# Preparamos la siguiente categoría antes de registrar cambios
ids_enrollment_prueba = [
    f"CONS_{numero:04d}"
    for numero in range(211, 251)
]

df_revision_enrollment_prueba = (
    df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(
            ids_enrollment_prueba
        )
    ]
    .sort_values("id_consulta")
    .copy()
)

if (
    len(df_revision_enrollment_prueba) != 40
    or not df_revision_enrollment_prueba["id_consulta"].is_unique
    or set(df_revision_enrollment_prueba["id_consulta"])
        != set(ids_enrollment_prueba)
    or not df_revision_enrollment_prueba["conjunto"].eq("prueba").all()
    or not df_revision_enrollment_prueba[
        "categoria_original"
    ].eq("ENROLLMENT").all()
):
    raise ValueError(
        "Los registros no corresponden a las 40 consultas "
        "de prueba de ENROLLMENT."
    )

# Registramos la confirmación explícita del usuario
fecha_actual = datetime.now(timezone.utc).isoformat()

for columna in ["validado_por", "fecha_validacion_utc"]:
    if columna not in df_muestra_revision.columns:
        df_muestra_revision[columna] = ""

df_muestra_revision.loc[
    mascara_payment, "estado_revision"
] = "validado_usuario"

df_muestra_revision.loc[
    mascara_payment, "validado_por"
] = "usuario"

sin_fecha = (
    df_muestra_revision["fecha_validacion_utc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

df_muestra_revision.loc[
    mascara_payment & sin_fecha,
    "fecha_validacion_utc"
] = fecha_actual

print("\nConsultas de prueba PAYMENT validadas: 40")
print("Validación registrada en memoria.")
print("Se incluirá en el siguiente respaldo descargable.")

print("\n" + "=" * 60)
print("🔎 Presentando las 40 consultas de prueba de ENROLLMENT...")
print("=" * 60)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_revision_enrollment_prueba[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("\nConsultas de ENROLLMENT presentadas: 40")
print("Estado: pendientes de traducción y revisión.")

print("=" * 60)
print("✅ PAYMENT validado y ENROLLMENT presentado para revisión.")
print("=" * 60)

✅ Registrando aprobación de PAYMENT...

Consultas de prueba PAYMENT validadas: 40
Validación registrada en memoria.
Se incluirá en el siguiente respaldo descargable.

🔎 Presentando las 40 consultas de prueba de ENROLLMENT...


,id_consulta,consulta_original,categoria_original,intencion_original
210,CONS_0211,I'd like to see the early termination charges could i get som help,ENROLLMENT,cancellation_fees
211,CONS_0212,i cant pay for myhealth insurance policy could ya help me to cancel it,ENROLLMENT,cancel_insurance_policy
212,CONS_0213,where could I find information about the fucking cancellation fees?,ENROLLMENT,cancellation_fees
213,CONS_0214,"I don't want to renew my life insurance policy, I need help cancelling it",ENROLLMENT,cancel_insurance_policy
214,CONS_0215,wjere can I get information about the fucking early termination fees?,ENROLLMENT,cancellation_fees
215,CONS_0216,"I got to see the fucking early termination fees , how can I do it?",ENROLLMENT,cancellation_fees
216,CONS_0217,I'd like to check the cancellation chargeshow do i do it,ENROLLMENT,cancellation_fees
217,CONS_0218,I want to se the fucking early termination charges,ENROLLMENT,cancellation_fees
218,CONS_0219,"I would like to know more about the cancellation fees , will you give me some information?",ENROLLMENT,cancellation_fees
219,CONS_0220,"I want to see the fucking early termination charges, how can I do it?",ENROLLMENT,cancellation_fees



Consultas de ENROLLMENT presentadas: 40
Estado: pendientes de traducción y revisión.
✅ PAYMENT validado y ENROLLMENT presentado para revisión.


### 🧠 **6.26 Traducción y revisión de ENROLLMENT**

Se proponen traducciones de 36 consultas sobre comparación,
contratación, cancelación y cargos por terminación de pólizas.

CONS_0212, CONS_0224, CONS_0225 y CONS_0234 quedan pendientes
de reemplazo por referencias a seguros de salud, mascotas
o automóvil.

Se comprobarán las traducciones repetidas dentro de prueba
y las coincidencias con desarrollo.

**Estado:** Traducciones propuestas; reemplazos y validación pendientes.

In [ ]:
# ============================================================
# 6.26 Traducción y revisión de ENROLLMENT
# ============================================================

import pandas as pd

print("=" * 60)
print("📝 Preparando traducciones de ENROLLMENT...")
print("=" * 60)

traducciones_enrollment = {
    "CONS_0211": "Quisiera consultar los cargos por terminación anticipada, ¿me pueden ayudar?",
    "CONS_0213": "¿Dónde puedo encontrar información sobre los cargos por cancelación?",
    "CONS_0214": "No quiero renovar mi póliza de seguro de vida, necesito ayuda para cancelarla.",
    "CONS_0215": "¿Dónde puedo obtener información sobre los cargos por terminación anticipada?",
    "CONS_0216": "Tengo que consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",
    "CONS_0217": "Quisiera consultar los cargos por cancelación, ¿cómo lo hago?",
    "CONS_0218": "Quiero consultar los cargos por terminación anticipada.",
    "CONS_0219": "Quisiera saber más sobre los cargos por cancelación, ¿me darás información?",
    "CONS_0220": "Quiero consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",
    "CONS_0221": "Tengo que consultar los cargos por terminación anticipada, ¿dónde puedo hacerlo?",
    "CONS_0222": "Necesito consultar los cargos por terminación anticipada, ayúdame.",
    "CONS_0223": "¿Puedes ayudarme a consultar los cargos por terminación anticipada?",
    "CONS_0226": "No quiero mi póliza de seguro, ¿puedes cancelarla?",
    "CONS_0227": "Quisiera información sobre los cargos por terminación anticipada, ¿cómo puedo obtenerla?",
    "CONS_0228": "Quiero consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",
    "CONS_0229": "Tengo que consultar los cargos por cancelación, ¿me pueden ayudar?",
    "CONS_0230": "No quiero mi póliza de seguro de vida, necesito ayuda para cancelarla.",
    "CONS_0231": "Quisiera información sobre los cargos por cancelación, ¿cómo puedo obtenerla?",
    "CONS_0232": "Quisiera información sobre las distintas pólizas de seguro de vida, ¿dónde puedo obtenerla?",
    "CONS_0233": "¿Cómo puedo contratar una póliza de seguro?",
    "CONS_0235": "Quiero saber sobre las distintas pólizas de seguro.",
    "CONS_0236": "Quisiera saber más sobre los cargos por terminación anticipada, ¿puedes darme información?",
    "CONS_0237": "Quisiera consultar los cargos por terminación anticipada, necesito ayuda.",
    "CONS_0238": "No quiero que se renueve mi póliza de seguro.",
    "CONS_0239": "Quisiera consultar los cargos por terminación anticipada, ¿cómo lo hago?",
    "CONS_0240": "No quiero mi póliza de seguro, necesito ayuda para cancelarla.",
    "CONS_0241": "¿Podría consultar los cargos por cancelación?",
    "CONS_0242": "Quisiera consultar los cargos por terminación anticipada, ¿me pueden ayudar?",
    "CONS_0243": "Quiero saber más sobre los cargos por cancelación, ¿puedes conseguir información?",
    "CONS_0244": "Tengo que consultar los cargos por terminación anticipada, ayúdame.",
    "CONS_0245": "Necesito consultar los cargos por terminación anticipada, ¿puedes ayudarme?",
    "CONS_0246": "Busco información sobre los cargos por terminación anticipada, ¿me ayudarás?",
    "CONS_0247": "Necesito cancelar mi póliza de seguro de vida, ¿puedes ayudarme?",
    "CONS_0248": "Tengo que consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",
    "CONS_0249": "Necesito información sobre los cargos por terminación anticipada, ¿cómo puedo obtenerla?",
    "CONS_0250": "Tengo que consultar los cargos por cancelación, ¿dónde puedo hacerlo?"
}

motivos_enrollment = {
    "CONS_0212": "Referencia a seguro de salud: myhealth insurance.",
    "CONS_0224": "Referencia a seguro de salud: heslth insurance.",
    "CONS_0225": "Referencia a seguro de mascotas: mypet insurance.",
    "CONS_0234": "Referencia a seguro de automóvil: auot insurance."
}

ids_enrollment = {
    f"CONS_{numero:04d}"
    for numero in range(211, 251)
}

if (
    set(traducciones_enrollment) | set(motivos_enrollment)
    != ids_enrollment
    or set(traducciones_enrollment) & set(motivos_enrollment)
):
    raise ValueError(
        "La configuración debe incluir los 40 registros sin solapamientos."
    )

# Trabajamos sobre una copia antes de publicar cambios
muestra_propuesta = df_muestra_revision.copy(deep=True)

registros = muestra_propuesta.loc[
    muestra_propuesta["id_consulta"].isin(ids_enrollment)
]

if (
    len(registros) != 40
    or not registros["id_consulta"].is_unique
    or not registros["conjunto"].eq("prueba").all()
    or not registros["categoria_original"].eq("ENROLLMENT").all()
):
    raise ValueError(
        "Los registros no corresponden a las 40 consultas de ENROLLMENT."
    )

for identificador, traduccion in traducciones_enrollment.items():
    mascara = muestra_propuesta["id_consulta"].eq(identificador)

    muestra_propuesta.loc[mascara, "consulta_es"] = traduccion
    muestra_propuesta.loc[
        mascara, "categoria_referencia"
    ] = "ENROLLMENT"
    muestra_propuesta.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"

for identificador, motivo in motivos_enrollment.items():
    mascara = muestra_propuesta["id_consulta"].eq(identificador)

    muestra_propuesta.loc[mascara, "consulta_es"] = ""
    muestra_propuesta.loc[mascara, "categoria_referencia"] = ""
    muestra_propuesta.loc[
        mascara, "estado_revision"
    ] = "pendiente_reemplazo"
    muestra_propuesta.loc[mascara, "observaciones"] = motivo

df_muestra_revision = muestra_propuesta

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(ids_enrollment),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

# Detectamos coincidencias entre todos los registros traducidos
textos_es = (
    df_muestra_revision["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = textos_es.ne("") & textos_es.duplicated(keep=False)

mascara_enrollment_prueba = (
    df_muestra_revision["conjunto"].eq("prueba")
    & df_muestra_revision["categoria_original"].eq("ENROLLMENT")
)

textos_repetidos_enrollment = set(
    textos_es.loc[repetidos & mascara_enrollment_prueba]
)

print(
    "\nGrupos repetidos que involucran prueba de ENROLLMENT:",
    len(textos_repetidos_enrollment)
)

if textos_repetidos_enrollment:
    mascara_coincidencias = textos_es.isin(
        textos_repetidos_enrollment
    )

    coincidencias_enrollment = df_muestra_revision.loc[
        mascara_coincidencias,
        [
            "id_consulta",
            "conjunto",
            "consulta_original",
            "consulta_es",
            "categoria_referencia"
        ]
    ].copy()

    coincidencias_enrollment["_orden"] = textos_es.loc[
        mascara_coincidencias
    ]

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            coincidencias_enrollment.sort_values(
                ["_orden", "id_consulta"]
            ).drop(columns="_orden")
        )

print("\nTraducciones propuestas: 36")
print("Pendientes por otros productos:")
print("  • CONS_0212: salud.")
print("  • CONS_0224: salud.")
print("  • CONS_0225: mascotas.")
print("  • CONS_0234: automóvil.")
print("Las coincidencias mostradas requieren revisión.")
print("Estado: confirmación del usuario pendiente.")
print("El siguiente respaldo incluirá la validación de PAYMENT.")

print("=" * 60)
print("✅ Traducciones de ENROLLMENT propuestas y coincidencias comprobadas.")
print("=" * 60)

📝 Preparando traducciones de ENROLLMENT...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
210,CONS_0211,I'd like to see the early termination charges could i get som help,"Quisiera consultar los cargos por terminación anticipada, ¿me pueden ayudar?",ENROLLMENT,propuesta_pendiente_revision
211,CONS_0212,i cant pay for myhealth insurance policy could ya help me to cancel it,,,pendiente_reemplazo
212,CONS_0213,where could I find information about the fucking cancellation fees?,¿Dónde puedo encontrar información sobre los cargos por cancelación?,ENROLLMENT,propuesta_pendiente_revision
213,CONS_0214,"I don't want to renew my life insurance policy, I need help cancelling it","No quiero renovar mi póliza de seguro de vida, necesito ayuda para cancelarla.",ENROLLMENT,propuesta_pendiente_revision
214,CONS_0215,wjere can I get information about the fucking early termination fees?,¿Dónde puedo obtener información sobre los cargos por terminación anticipada?,ENROLLMENT,propuesta_pendiente_revision
215,CONS_0216,"I got to see the fucking early termination fees , how can I do it?","Tengo que consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",ENROLLMENT,propuesta_pendiente_revision
216,CONS_0217,I'd like to check the cancellation chargeshow do i do it,"Quisiera consultar los cargos por cancelación, ¿cómo lo hago?",ENROLLMENT,propuesta_pendiente_revision
217,CONS_0218,I want to se the fucking early termination charges,Quiero consultar los cargos por terminación anticipada.,ENROLLMENT,propuesta_pendiente_revision
218,CONS_0219,"I would like to know more about the cancellation fees , will you give me some information?","Quisiera saber más sobre los cargos por cancelación, ¿me darás información?",ENROLLMENT,propuesta_pendiente_revision
219,CONS_0220,"I want to see the fucking early termination charges, how can I do it?","Quiero consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",ENROLLMENT,propuesta_pendiente_revision



Grupos repetidos que involucran prueba de ENROLLMENT: 4


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
207,CONS_0208,desarrollo,"I don't want to reenw my life insurance policy, I need assistance to cancel it","No quiero renovar mi póliza de seguro de vida, necesito ayuda para cancelarla.",ENROLLMENT
213,CONS_0214,prueba,"I don't want to renew my life insurance policy, I need help cancelling it","No quiero renovar mi póliza de seguro de vida, necesito ayuda para cancelarla.",ENROLLMENT
219,CONS_0220,prueba,"I want to see the fucking early termination charges, how can I do it?","Quiero consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",ENROLLMENT
227,CONS_0228,prueba,wanna see the fucking eaely termination charges how could i do it,"Quiero consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",ENROLLMENT
210,CONS_0211,prueba,I'd like to see the early termination charges could i get som help,"Quisiera consultar los cargos por terminación anticipada, ¿me pueden ayudar?",ENROLLMENT
241,CONS_0242,prueba,I'd like to check the fucking early termination charges could i get some help,"Quisiera consultar los cargos por terminación anticipada, ¿me pueden ayudar?",ENROLLMENT
215,CONS_0216,prueba,"I got to see the fucking early termination fees , how can I do it?","Tengo que consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",ENROLLMENT
247,CONS_0248,prueba,i got to see the early termination charges how could i do it,"Tengo que consultar los cargos por terminación anticipada, ¿cómo puedo hacerlo?",ENROLLMENT



Traducciones propuestas: 36
Pendientes por otros productos:
  • CONS_0212: salud.
  • CONS_0224: salud.
  • CONS_0225: mascotas.
  • CONS_0234: automóvil.
Las coincidencias mostradas requieren revisión.
Estado: confirmación del usuario pendiente.
El siguiente respaldo incluirá la validación de PAYMENT.
✅ Traducciones de ENROLLMENT propuestas y coincidencias comprobadas.


### 🧠 **6.27 Selección de candidatos de reemplazo de ENROLLMENT**

Se presentan opciones para ocho reemplazos: cuatro por referencias
a otros productos y cuatro por traducciones repetidas.

Los candidatos conservarán la categoría y la intención del registro
reemplazado. Se revisarán sus originales y traducciones antes
de incorporarlos.

**Estado:** Opciones pendientes de revisión. La muestra no se modifica.

In [ ]:
# ============================================================
# 6.27 Opciones de reemplazo de ENROLLMENT
# ============================================================

import pandas as pd
import re

print("=" * 60)
print("🔎 Preparando opciones para ocho reemplazos de ENROLLMENT...")
print("=" * 60)

ids_reemplazar_enrollment = [
    "CONS_0212",
    "CONS_0214",
    "CONS_0224",
    "CONS_0225",
    "CONS_0228",
    "CONS_0234",
    "CONS_0242",
    "CONS_0248"
]

objetivos_enrollment = df_muestra_revision.loc[
    df_muestra_revision["id_consulta"].isin(
        ids_reemplazar_enrollment
    )
].sort_values("id_consulta").copy()

if (
    len(objetivos_enrollment) != 8
    or not objetivos_enrollment["id_consulta"].is_unique
    or not objetivos_enrollment["conjunto"].eq("prueba").all()
    or not objetivos_enrollment[
        "categoria_original"
    ].eq("ENROLLMENT").all()
):
    raise ValueError("Los ocho objetivos no coinciden con los esperados.")

# Comprobamos las coincidencias detectadas en 6.26
for conservar, reemplazar in [
    ("CONS_0208", "CONS_0214"),
    ("CONS_0220", "CONS_0228"),
    ("CONS_0211", "CONS_0242"),
    ("CONS_0216", "CONS_0248")
]:
    par = df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(
            [conservar, reemplazar]
        )
    ]

    textos = par["consulta_es"].fillna("").map(normalizar_contexto)

    if (
        len(par) != 2
        or not par["id_consulta"].is_unique
        or textos.eq("").any()
        or textos.nunique() != 1
    ):
        raise ValueError(
            f"La coincidencia de {reemplazar} cambió. Revisa 6.26."
        )

# Excluimos originales utilizados y retirados
textos_excluidos_enrollment = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_excluidos_enrollment.update(
        tabla["consulta_original"]
        .fillna("")
        .map(normalizar_contexto)
    )

reserva_enrollment = df_elegibles.copy()

reserva_enrollment["texto_revision"] = (
    reserva_enrollment["instruction"]
    .fillna("")
    .map(normalizar_contexto)
)

# Complementamos el detector con variantes ya observadas.
# La revisión semántica comprobará también errores no previstos.
def detectar_producto_candidatos_enrollment(texto):
    normalizado = normalizar_contexto(texto)

    return bool(
        detectar_referencias_producto(texto)
        or re.search(
            r"\b(?:travwl|ttavel|ehalth|heslth|auot)\b",
            normalizado
        )
        or re.search(r"\bpe\s+insur[a-z]*\b", normalizado)
        or re.search(
            r"(?:my|fucking)(?:health|pet|travel|auto|car|home)",
            normalizado
        )
    )

otros_productos = (
    reserva_enrollment["instruction"]
    .fillna("")
    .map(detectar_producto_candidatos_enrollment)
)

reserva_enrollment = (
    reserva_enrollment.loc[
        reserva_enrollment["texto_revision"].ne("")
        & ~reserva_enrollment["texto_revision"].isin(
            textos_excluidos_enrollment
        )
        & ~otros_productos
    ]
    .sort_values("texto_revision")
    .drop_duplicates("texto_revision")
    .copy()
)

# Seleccionamos opciones distintas entre las ocho listas
partes_opciones_enrollment = []
textos_presentados_enrollment = set()

for posicion, (_, objetivo) in enumerate(
    objetivos_enrollment.iterrows()
):
    candidatos = reserva_enrollment.loc[
        reserva_enrollment["category"].eq(
            objetivo["categoria_original"]
        )
        & reserva_enrollment["intent"].eq(
            objetivo["intencion_original"]
        )
        & ~reserva_enrollment["texto_revision"].isin(
            textos_presentados_enrollment
        )
    ]

    if candidatos.empty:
        raise ValueError(
            f"No hay candidatos para {objetivo['id_consulta']}."
        )

    seleccion = candidatos.sample(
        n=min(5, len(candidatos)),
        random_state=7842 + posicion
    ).copy()

    textos_presentados_enrollment.update(
        seleccion["texto_revision"]
    )

    seleccion.insert(
        0, "id_a_reemplazar", objetivo["id_consulta"]
    )
    seleccion.insert(
        1, "opcion", range(1, len(seleccion) + 1)
    )

    partes_opciones_enrollment.append(seleccion)

df_opciones_reemplazos_enrollment = pd.concat(
    partes_opciones_enrollment,
    ignore_index=True
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_opciones_reemplazos_enrollment[
            [
                "id_a_reemplazar",
                "opcion",
                "instruction",
                "category",
                "intent"
            ]
        ].rename(columns={
            "instruction": "consulta_original",
            "category": "categoria_original",
            "intent": "intencion_original"
        })
    )

print("\nObjetivos de reemplazo: 8")
print("La muestra permanece sin cambios.")
print("Revisaremos los candidatos antes de incorporarlos.")
print("Este bloque presenta opciones; todavía no genera un respaldo.")

print("=" * 60)
print("✅ Opciones de reemplazo de ENROLLMENT presentadas para revisión.")
print("=" * 60)

🔎 Preparando opciones para ocho reemplazos de ENROLLMENT...


,id_a_reemplazar,opcion,consulta_original,categoria_original,intencion_original
0,CONS_0212,1,I want to cancel my fucking insurance policy,ENROLLMENT,cancel_insurance_policy
1,CONS_0212,2,"I got to cancel my life insurance policy, could I get some help ?",ENROLLMENT,cancel_insurance_policy
2,CONS_0212,3,i dont wanna renew my life insurance policy would it be possible to cancel it,ENROLLMENT,cancel_insurance_policy
3,CONS_0212,4,"I do not want to renew my fucking life insurance policy, will you help me to cancel it?",ENROLLMENT,cancel_insurance_policy
4,CONS_0212,5,wanna cancel my fucking insurance policy i need assistance,ENROLLMENT,cancel_insurance_policy
5,CONS_0214,1,"I don't want my fucking insurance poliyc, I need help cancelling it",ENROLLMENT,cancel_insurance_policy
6,CONS_0214,2,i do not want my fucking insurance policy help me cancelling it,ENROLLMENT,cancel_insurance_policy
7,CONS_0214,3,"I don't want my fucking life insurance policy, can you help me canceling it?",ENROLLMENT,cancel_insurance_policy
8,CONS_0214,4,help me cancel myu fucking insurance policy,ENROLLMENT,cancel_insurance_policy
9,CONS_0214,5,i do not want my insurance policy can i cancel it,ENROLLMENT,cancel_insurance_policy



Objetivos de reemplazo: 8
La muestra permanece sin cambios.
Revisaremos los candidatos antes de incorporarlos.
Este bloque presenta opciones; todavía no genera un respaldo.
✅ Opciones de reemplazo de ENROLLMENT presentadas para revisión.


### 🧠 **6.28 Incorporación y respaldo de reemplazos de ENROLLMENT**

Se incorporarán ocho candidatos, conservando sus identificadores,
categoría, intención y conjunto de prueba.

Se comprobarán las traducciones repetidas y la conservación de los
demás registros. Los originales retirados quedarán en la auditoría.

Se descargará un ZIP con la muestra actualizada, la auditoría,
las consultas propuestas de ENROLLMENT, las consultas de prueba
previamente validadas y la configuración para prueba.

**Estado:** ENROLLMENT pendiente de confirmación del usuario.
CONTACT pendiente de preparación.

In [ ]:
# ============================================================
# 6.28 Incorporación y respaldo de reemplazos de ENROLLMENT
# ============================================================

import pandas as pd
import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("=" * 60)
print("🔎 Comprobando ocho reemplazos de ENROLLMENT...")
print("=" * 60)

# ------------------------------------------------------------
# 1. Definimos las opciones y traducciones revisadas
# ------------------------------------------------------------

elecciones_enrollment = {
    "CONS_0212": {
        "opcion": 1,
        "traduccion": "Quiero cancelar mi póliza de seguro.",
        "motivo": "Referencia a seguro de salud: myhealth insurance."
    },
    "CONS_0214": {
        "opcion": 4,
        "traduccion": "Ayúdame a cancelar mi póliza de seguro.",
        "motivo": "Traducción repetida con CONS_0208 de desarrollo."
    },
    "CONS_0224": {
        "opcion": 4,
        "traduccion": (
            "Necesito una póliza de seguro, ¿dónde puedo contratarla?"
        ),
        "motivo": "Referencia a seguro de salud: heslth insurance."
    },
    "CONS_0225": {
        "opcion": 2,
        "traduccion": (
            "No quiero que se renueve mi póliza de seguro, "
            "¿cómo puedo cancelarla?"
        ),
        "motivo": "Referencia a seguro de mascotas: mypet insurance."
    },
    "CONS_0228": {
        "opcion": 2,
        "traduccion": (
            "Busco información sobre los cargos por cancelación, "
            "¿cómo puedo obtenerla?"
        ),
        "motivo": "Traducción repetida con CONS_0220."
    },
    "CONS_0234": {
        "opcion": 5,
        "traduccion": "¿Es posible contratar una póliza de seguro?",
        "motivo": "Referencia a seguro de automóvil: auot insurance."
    },
    "CONS_0242": {
        "opcion": 4,
        "traduccion": (
            "Busco información sobre los cargos por cancelación, ayúdame."
        ),
        "motivo": "Traducción repetida con CONS_0211."
    },
    "CONS_0248": {
        "opcion": 1,
        "traduccion": (
            "Necesito información sobre los cargos por terminación anticipada."
        ),
        "motivo": "Traducción repetida con CONS_0216."
    }
}

# Complementamos el detector con la variante observada en 6.27
def detectar_producto_reemplazo_enrollment(texto):
    return bool(
        detectar_producto_candidatos_enrollment(texto)
        or "teavel" in normalizar_contexto(texto).split()
    )

# ------------------------------------------------------------
# 2. Aplicamos los cambios sobre una copia
# ------------------------------------------------------------

muestra_propuesta = df_muestra_revision.copy(deep=True)
fecha_actual = datetime.now(timezone.utc).isoformat()
retirados_enrollment = []
textos_ocupados = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_ocupados.update(
        tabla["consulta_original"]
        .fillna("")
        .map(normalizar_contexto)
    )

for identificador, eleccion in elecciones_enrollment.items():
    opciones = df_opciones_reemplazos_enrollment.loc[
        df_opciones_reemplazos_enrollment["id_a_reemplazar"].eq(
            identificador
        )
        & df_opciones_reemplazos_enrollment["opcion"].eq(
            eleccion["opcion"]
        )
    ]

    mascara = muestra_propuesta["id_consulta"].eq(identificador)
    original = muestra_propuesta.loc[mascara]

    if len(opciones) != 1 or len(original) != 1:
        raise ValueError(
            f"Registro o candidato no único: {identificador}."
        )

    candidato = opciones.iloc[0]

    if (
        not original["conjunto"].eq("prueba").all()
        or not original["categoria_original"].eq("ENROLLMENT").all()
        or candidato["category"] != "ENROLLMENT"
        or candidato["intent"] != original.iloc[0]["intencion_original"]
    ):
        raise ValueError(
            f"Categoría, intención o conjunto incorrecto: {identificador}."
        )

    texto_candidato = normalizar_contexto(candidato["instruction"])

    if (
        not texto_candidato
        or texto_candidato in textos_ocupados
        or detectar_producto_reemplazo_enrollment(
            candidato["instruction"]
        )
    ):
        raise ValueError(
            f"El candidato de {identificador} requiere nueva revisión."
        )

    textos_ocupados.add(texto_candidato)

    retirado = original.copy()
    retirado["motivo_retiro"] = eleccion["motivo"]
    retirado["fecha_retiro_utc"] = fecha_actual
    retirado["etapa_retiro"] = "6.28 prueba ENROLLMENT"
    retirados_enrollment.append(retirado)

    cambios = {
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"],
        "consulta_es": eleccion["traduccion"],
        "categoria_referencia": "ENROLLMENT",
        "estado_revision": "propuesta_pendiente_revision",
        "observaciones": (
            "Original reemplazado: " + eleccion["motivo"]
            + " Traducción revisada por el asistente; "
            "confirmación del usuario pendiente."
        )
    }

    for columna, valor in cambios.items():
        muestra_propuesta.loc[mascara, columna] = valor

    for columna in ["validado_por", "fecha_validacion_utc"]:
        if columna in muestra_propuesta.columns:
            muestra_propuesta.loc[mascara, columna] = ""

    if "revision_asistente" in muestra_propuesta.columns:
        muestra_propuesta.loc[
            mascara, "revision_asistente"
        ] = "completada"

# ------------------------------------------------------------
# 3. Comprobamos traducciones y conservación de los datos
# ------------------------------------------------------------

textos_es = (
    muestra_propuesta["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = textos_es.ne("") & textos_es.duplicated(keep=False)

if repetidos.any():
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            muestra_propuesta.loc[
                repetidos,
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ].sort_values("consulta_es")
        )

    raise ValueError(
        "Persisten traducciones repetidas. "
        "La muestra original no se modificó."
    )

mascara_enrollment = (
    muestra_propuesta["conjunto"].eq("prueba")
    & muestra_propuesta["categoria_original"].eq("ENROLLMENT")
)

if (
    mascara_enrollment.sum() != 40
    or textos_es.loc[mascara_enrollment].eq("").any()
    or not muestra_propuesta.loc[
        mascara_enrollment, "categoria_referencia"
    ].eq("ENROLLMENT").all()
):
    raise ValueError(
        "ENROLLMENT debe conservar 40 consultas traducidas."
    )

mascara_otros = ~df_muestra_revision["id_consulta"].isin(
    elecciones_enrollment
)

if not df_muestra_revision.loc[mascara_otros].equals(
    muestra_propuesta.loc[mascara_otros]
):
    raise ValueError(
        "Se modificaron registros ajenos a los reemplazos."
    )

distribucion_antes = df_muestra_revision.groupby(
    ["categoria_original", "conjunto"]
).size()

distribucion_despues = muestra_propuesta.groupby(
    ["categoria_original", "conjunto"]
).size()

if not distribucion_antes.equals(distribucion_despues):
    raise ValueError("Cambió la distribución de la muestra.")

auditoria_propuesta = pd.concat(
    [df_registros_retirados] + retirados_enrollment,
    ignore_index=True,
    sort=False
)

mascara_prueba_validada = (
    muestra_propuesta["conjunto"].eq("prueba")
    & muestra_propuesta["categoria_original"].isin(
        ["LIFE_INSURANCE", "COVERAGE", "QUOTE", "PAYMENT"]
    )
    & muestra_propuesta["estado_revision"].eq("validado_usuario")
)

if mascara_prueba_validada.sum() != 160:
    raise ValueError(
        "Deben existir 160 consultas de prueba previamente validadas."
    )

# ------------------------------------------------------------
# 4. Guardamos un respaldo nuevo con una guía de los insumos
# ------------------------------------------------------------

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_avance_enrollment = (
    Path("/content/revision_evaluacion")
    / f"avance_prueba_enrollment_{marca_version}"
)
carpeta_avance_enrollment.mkdir(parents=True, exist_ok=False)

tablas_guardar = {
    "muestra_actualizada.csv": muestra_propuesta,
    "registros_retirados.csv": auditoria_propuesta,
    "prueba_enrollment_propuesta.csv": (
        muestra_propuesta.loc[mascara_enrollment]
    ),
    "prueba_validada_hasta_payment.csv": (
        muestra_propuesta.loc[mascara_prueba_validada]
    )
}

for nombre, tabla in tablas_guardar.items():
    tabla.to_csv(
        carpeta_avance_enrollment / nombre,
        index=False,
        encoding="utf-8-sig"
    )

with (
    carpeta_avance_enrollment / "configuracion_para_prueba.json"
).open("w", encoding="utf-8") as archivo:
    json.dump(
        {
            "modelos": modelos_comparacion,
            "configuracion_prueba": configuracion_prueba,
            "decision_desarrollo": decision_desarrollo
        },
        archivo,
        ensure_ascii=False,
        indent=2
    )

(carpeta_avance_enrollment / "LEEME.txt").write_text(
    "RESPALDO DE AVANCE: ENROLLMENT\n\n"
    "muestra_actualizada.csv:\n"
    "  Insumo principal para continuar con CONTACT.\n\n"
    "registros_retirados.csv:\n"
    "  Auditoría acumulada de los reemplazos.\n\n"
    "prueba_enrollment_propuesta.csv:\n"
    "  40 consultas traducidas de ENROLLMENT; validación pendiente.\n\n"
    "prueba_validada_hasta_payment.csv:\n"
    "  160 consultas aprobadas de las cuatro categorías anteriores.\n\n"
    "configuracion_para_prueba.json:\n"
    "  Modelos y configuración fijada para prueba.\n\n"
    "CONTACT está pendiente de preparación.\n"
    "Este avance todavía no es el dataset final validado.\n"
    "Conservar también el respaldo de cierre de desarrollo.\n",
    encoding="utf-8"
)

# ------------------------------------------------------------
# 5. Comprimimos, verificamos y publicamos los cambios
# ------------------------------------------------------------

ruta_zip_enrollment = Path(
    shutil.make_archive(
        str(carpeta_avance_enrollment),
        "zip",
        root_dir=str(carpeta_avance_enrollment)
    )
)

archivos_esperados = {
    ruta.name
    for ruta in carpeta_avance_enrollment.iterdir()
    if ruta.is_file()
}

with zipfile.ZipFile(ruta_zip_enrollment, "r") as respaldo:
    if (
        respaldo.testzip() is not None
        or not archivos_esperados.issubset(set(respaldo.namelist()))
    ):
        raise ValueError("Falló la verificación del ZIP.")

df_muestra_revision = muestra_propuesta
df_registros_retirados = auditoria_propuesta

with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(
                elecciones_enrollment
            ),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

print("\nConsultas de prueba ENROLLMENT traducidas: 40")
print("Reemplazos incorporados: 8")
print("Grupos de traducciones repetidas: 0")
print("Consultas de prueba previamente validadas: 160")
print("Registros retirados conservados:",
      len(df_registros_retirados))
print("ENROLLMENT: confirmación del usuario pendiente.")

print("\n📁 Carpeta del respaldo:")
print(carpeta_avance_enrollment)

print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_enrollment.name)

print("\n📋 Archivos incluidos y uso posterior:")
print("  • muestra_actualizada.csv: continuar con CONTACT.")
print("  • registros_retirados.csv: consultar la auditoría.")
print("  • prueba_enrollment_propuesta.csv: revisar y validar ENROLLMENT.")
print("  • prueba_validada_hasta_payment.csv: recuperar prueba aprobada.")
print("  • configuracion_para_prueba.json: recuperar la configuración.")
print("  • LEEME.txt: consultar la guía del respaldo.")

print("\n⚠️ Dataset de prueba todavía incompleto.")
print("=" * 60)
print("✅ Reemplazos de ENROLLMENT incorporados y avance respaldado.")
print("=" * 60)

print("\n📥 Solicitando descarga del ZIP...")
files.download(str(ruta_zip_enrollment))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

🔎 Comprobando ocho reemplazos de ENROLLMENT...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
211,CONS_0212,I want to cancel my fucking insurance policy,Quiero cancelar mi póliza de seguro.,ENROLLMENT,propuesta_pendiente_revision
213,CONS_0214,help me cancel myu fucking insurance policy,Ayúdame a cancelar mi póliza de seguro.,ENROLLMENT,propuesta_pendiente_revision
223,CONS_0224,"I nede an insurance policy, where can I buy it?","Necesito una póliza de seguro, ¿dónde puedo contratarla?",ENROLLMENT,propuesta_pendiente_revision
224,CONS_0225,i dont want my insurance polify to be renewed how can i cancel it,"No quiero que se renueve mi póliza de seguro, ¿cómo puedo cancelarla?",ENROLLMENT,propuesta_pendiente_revision
227,CONS_0228,"I'm looking for information about the cancellation fees, how can I get it?","Busco información sobre los cargos por cancelación, ¿cómo puedo obtenerla?",ENROLLMENT,propuesta_pendiente_revision
233,CONS_0234,is ity possible to buy an insurance policy?,¿Es posible contratar una póliza de seguro?,ENROLLMENT,propuesta_pendiente_revision
241,CONS_0242,im looking for information about the fucking cancellation fees help me,"Busco información sobre los cargos por cancelación, ayúdame.",ENROLLMENT,propuesta_pendiente_revision
247,CONS_0248,i need information about the fucking early termination fees,Necesito información sobre los cargos por terminación anticipada.,ENROLLMENT,propuesta_pendiente_revision



Consultas de prueba ENROLLMENT traducidas: 40
Reemplazos incorporados: 8
Grupos de traducciones repetidas: 0
Consultas de prueba previamente validadas: 160
Registros retirados conservados: 44
ENROLLMENT: confirmación del usuario pendiente.

📁 Carpeta del respaldo:
/content/revision_evaluacion/avance_prueba_enrollment_20261010T194616599198Z

📦 ZIP para conservar en tu computadora:
avance_prueba_enrollment_20261010T194616599198Z.zip

📋 Archivos incluidos y uso posterior:
  • muestra_actualizada.csv: continuar con CONTACT.
  • registros_retirados.csv: consultar la auditoría.
  • prueba_enrollment_propuesta.csv: revisar y validar ENROLLMENT.
  • prueba_validada_hasta_payment.csv: recuperar prueba aprobada.
  • configuracion_para_prueba.json: recuperar la configuración.
  • LEEME.txt: consultar la guía del respaldo.

⚠️ Dataset de prueba todavía incompleto.
✅ Reemplazos de ENROLLMENT incorporados y avance respaldado.

📥 Solicitando descarga del ZIP...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


### 🧠 **6.29 Validación de ENROLLMENT y revisión de CONTACT**

Se registra la aprobación del usuario de las 40 traducciones
de prueba de ENROLLMENT.

Se presentan las consultas CONS_0261 a CONS_0300 para revisar
las solicitudes de contacto con atención al cliente, la aseguradora,
representantes o agentes humanos.

**Estado:** ENROLLMENT validado. CONTACT pendiente de traducción
y revisión.

In [ ]:
# ============================================================
# 6.29 Validación de ENROLLMENT y revisión de CONTACT
# ============================================================

import pandas as pd
from datetime import datetime, timezone

print("=" * 60)
print("✅ Registrando aprobación de ENROLLMENT...")
print("=" * 60)

mascara_enrollment = (
    df_muestra_revision["conjunto"].eq("prueba")
    & df_muestra_revision["categoria_original"].eq("ENROLLMENT")
)

registros = df_muestra_revision.loc[mascara_enrollment]

textos = (
    registros["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

if (
    len(registros) != 40
    or not registros["id_consulta"].is_unique
    or textos.eq("").any()
    or textos.duplicated().any()
    or not registros["categoria_referencia"].eq("ENROLLMENT").all()
):
    raise ValueError(
        "ENROLLMENT debe contener 40 traducciones completas "
        "y distintas con su categoría de referencia."
    )

# Comprobamos coincidencias con todos los registros traducidos
textos_todos = (
    df_muestra_revision["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = (
    textos_todos.ne("")
    & textos_todos.duplicated(keep=False)
)

if (repetidos & mascara_enrollment).any():
    raise ValueError(
        "Hay coincidencias de ENROLLMENT con otros registros traducidos."
    )

# Preparamos CONTACT antes de registrar cambios
ids_contact_prueba = [
    f"CONS_{numero:04d}"
    for numero in range(261, 301)
]

df_revision_contact_prueba = (
    df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(ids_contact_prueba)
    ]
    .sort_values("id_consulta")
    .copy()
)

if (
    len(df_revision_contact_prueba) != 40
    or not df_revision_contact_prueba["id_consulta"].is_unique
    or set(df_revision_contact_prueba["id_consulta"])
        != set(ids_contact_prueba)
    or not df_revision_contact_prueba["conjunto"].eq("prueba").all()
    or not df_revision_contact_prueba[
        "categoria_original"
    ].eq("CONTACT").all()
):
    raise ValueError(
        "Los registros no corresponden a las 40 consultas "
        "de prueba de CONTACT."
    )

# Registramos la confirmación explícita del usuario
fecha_actual = datetime.now(timezone.utc).isoformat()

for columna in ["validado_por", "fecha_validacion_utc"]:
    if columna not in df_muestra_revision.columns:
        df_muestra_revision[columna] = ""

df_muestra_revision.loc[
    mascara_enrollment, "estado_revision"
] = "validado_usuario"

df_muestra_revision.loc[
    mascara_enrollment, "validado_por"
] = "usuario"

sin_fecha = (
    df_muestra_revision["fecha_validacion_utc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

df_muestra_revision.loc[
    mascara_enrollment & sin_fecha,
    "fecha_validacion_utc"
] = fecha_actual

print("\nConsultas de prueba ENROLLMENT validadas: 40")
print("Validación registrada en memoria.")
print("Se incluirá en el siguiente respaldo descargable.")

print("\n" + "=" * 60)
print("🔎 Presentando las 40 consultas de prueba de CONTACT...")
print("=" * 60)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_revision_contact_prueba[
            [
                "id_consulta",
                "consulta_original",
                "categoria_original",
                "intencion_original"
            ]
        ]
    )

print("\nConsultas de CONTACT presentadas: 40")
print("Estado: pendientes de traducción y revisión.")
print("CONTACT es la última categoría por preparar.")

print("=" * 60)
print("✅ ENROLLMENT validado y CONTACT presentado para revisión.")
print("=" * 60)

✅ Registrando aprobación de ENROLLMENT...

Consultas de prueba ENROLLMENT validadas: 40
Validación registrada en memoria.
Se incluirá en el siguiente respaldo descargable.

🔎 Presentando las 40 consultas de prueba de CONTACT...


,id_consulta,consulta_original,categoria_original,intencion_original
260,CONS_0261,"I need to speak with the insurance company, how could I do it?",CONTACT,agent
261,CONS_0262,"I have to speak with the insurance company, help me",CONTACT,agent
262,CONS_0263,i have to talk to the insurance company how can i do it,CONTACT,agent
263,CONS_0264,"I don't understand you, could you help me to speak withn a fucking human agent?",CONTACT,human_agent
264,CONS_0265,could ya help me to speak with the insurance company,CONTACT,agent
265,CONS_0266,can I speak with fucking customer service?,CONTACT,customer_service
266,CONS_0267,"I got to talk to fucking customer support, how to do it?",CONTACT,customer_service
267,CONS_0268,i got to talk to my fucking insurance representative,CONTACT,insurance_representative
268,CONS_0269,"I'd like to speak with my fucking inusrance representative, could you help me?",CONTACT,insurance_representative
269,CONS_0270,i have to speak with my insurance representaitve will ya help me,CONTACT,insurance_representative



Consultas de CONTACT presentadas: 40
Estado: pendientes de traducción y revisión.
CONTACT es la última categoría por preparar.
✅ ENROLLMENT validado y CONTACT presentado para revisión.


### 🧠 **6.30 Traducción y revisión de CONTACT**

Se proponen traducciones de las 40 consultas de prueba de CONTACT,
sin expresiones ofensivas y conservando los interlocutores
y medios de contacto mencionados en los originales.

Se comprobarán las traducciones repetidas dentro de prueba
y las coincidencias con desarrollo.

**Estado:** Última categoría traducida; revisión de coincidencias
y validación del usuario pendientes.

In [ ]:
# ============================================================
# 6.30 Traducción y revisión de CONTACT
# ============================================================

import pandas as pd

print("=" * 60)
print("📝 Preparando las 40 traducciones de CONTACT...")
print("=" * 60)

traducciones_contact = {
    "CONS_0261": "Necesito hablar con la compañía de seguros, ¿cómo puedo hacerlo?",
    "CONS_0262": "Tengo que hablar con la compañía de seguros, ayúdame.",
    "CONS_0263": "Tengo que hablar con la compañía de seguros, ¿cómo puedo hacerlo?",
    "CONS_0264": "No te entiendo, ¿podrías ayudarme a hablar con un agente humano?",
    "CONS_0265": "¿Puedes ayudarme a hablar con la compañía de seguros?",
    "CONS_0266": "¿Puedo hablar con atención al cliente?",
    "CONS_0267": "Tengo que hablar con atención al cliente, ¿cómo puedo hacerlo?",
    "CONS_0268": "Tengo que hablar con mi representante de seguros.",
    "CONS_0269": "Quisiera hablar con mi representante de seguros, ¿puedes ayudarme?",
    "CONS_0270": "Tengo que hablar con mi representante de seguros, ¿me ayudarás?",
    "CONS_0271": "Tengo que hablar con la compañía de seguros, ¿cómo puedo hacerlo?",
    "CONS_0272": "No te entiendo, ¿podrías ayudarme a hablar con un agente?",
    "CONS_0273": "Quisiera hablar con mi representante de seguros, ¿cómo puedo hacerlo?",
    "CONS_0274": "Tengo que hablar con alguien, ¿me ayudarás?",
    "CONS_0275": "Quiero hablar con un agente humano, ¿cómo lo hago?",
    "CONS_0276": "¿Puedo hablar con atención al cliente?",
    "CONS_0277": "Tengo que llamar a atención al cliente, ¿cómo lo hago?",
    "CONS_0278": "Necesito enviar un correo a atención al cliente, ¿dónde puedo hacerlo?",
    "CONS_0279": "No te entiendo, tengo que hablar con un agente humano.",
    "CONS_0280": "No te entiendo, ¿puedes ayudarme a hablar con alguien?",
    "CONS_0281": "Quiero ayuda para hablar con atención al cliente.",
    "CONS_0282": "Necesito ayuda para hablar con mi representante de seguros.",
    "CONS_0283": "Tengo que hablar con la compañía de seguros, ¿dónde puedo hacerlo?",
    "CONS_0284": "Quiero hablar con mi representante de seguros, ¿puedes ayudarme?",
    "CONS_0285": "Necesito hablar con mi representante de seguros, ¿puedes ayudarme?",
    "CONS_0286": "Quisiera hablar con la compañía de seguros, ¿dónde puedo hacerlo?",
    "CONS_0287": "Necesito hablar con la aseguradora, ¿cómo puedo hacerlo?",
    "CONS_0288": "Necesito enviar un correo a atención al cliente, ¿cómo puedo hacerlo?",
    "CONS_0289": "Quisiera hablar con mi representante de seguros, necesito ayuda.",
    "CONS_0290": "Necesito hablar con atención al cliente, ¿dónde puedo hacerlo?",
    "CONS_0291": "Tengo que hablar con alguien, necesito ayuda.",
    "CONS_0292": "Quiero hablar con atención al cliente, ¿me pueden ayudar?",
    "CONS_0293": "No te entiendo, ¿puedo hablar con una persona?",
    "CONS_0294": "Quiero enviar un correo a atención al cliente, ¿dónde puedo hacerlo?",
    "CONS_0295": "Tengo que hablar con la aseguradora, ¿dónde puedo hacerlo?",
    "CONS_0296": "Quiero ayuda para llamar a atención al cliente.",
    "CONS_0297": "Tengo que hablar con atención al cliente, ¿me ayudarás?",
    "CONS_0298": "Quisiera hablar con mi representante de seguros, ¿cómo puedo hacerlo?",
    "CONS_0299": "Tengo que hablar con una persona.",
    "CONS_0300": "Quisiera hablar con mi representante de seguros, ¿puedes ayudarme?"
}

ids_contact = {
    f"CONS_{numero:04d}"
    for numero in range(261, 301)
}

if set(traducciones_contact) != ids_contact:
    raise ValueError("Faltan traducciones de los 40 registros previstos.")

muestra_propuesta = df_muestra_revision.copy(deep=True)

registros_contact = muestra_propuesta.loc[
    muestra_propuesta["id_consulta"].isin(ids_contact)
]

if (
    len(registros_contact) != 40
    or not registros_contact["id_consulta"].is_unique
    or not registros_contact["conjunto"].eq("prueba").all()
    or not registros_contact["categoria_original"].eq("CONTACT").all()
):
    raise ValueError(
        "Los registros no corresponden a las 40 consultas de CONTACT."
    )

# Aplicamos las traducciones únicamente a CONTACT de prueba
for identificador, traduccion in traducciones_contact.items():
    mascara = muestra_propuesta["id_consulta"].eq(identificador)

    muestra_propuesta.loc[mascara, "consulta_es"] = traduccion
    muestra_propuesta.loc[mascara, "categoria_referencia"] = "CONTACT"
    muestra_propuesta.loc[
        mascara, "estado_revision"
    ] = "propuesta_pendiente_revision"

df_muestra_revision = muestra_propuesta

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(ids_contact),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

# Detectamos coincidencias con todos los registros traducidos
textos_es = (
    df_muestra_revision["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

repetidos = textos_es.ne("") & textos_es.duplicated(keep=False)

mascara_contact_prueba = (
    df_muestra_revision["conjunto"].eq("prueba")
    & df_muestra_revision["categoria_original"].eq("CONTACT")
)

textos_repetidos_contact = set(
    textos_es.loc[repetidos & mascara_contact_prueba]
)

print(
    "\nGrupos repetidos que involucran prueba de CONTACT:",
    len(textos_repetidos_contact)
)

if textos_repetidos_contact:
    mascara_coincidencias = textos_es.isin(textos_repetidos_contact)

    coincidencias_contact = df_muestra_revision.loc[
        mascara_coincidencias,
        [
            "id_consulta",
            "conjunto",
            "consulta_original",
            "consulta_es",
            "categoria_referencia"
        ]
    ].copy()

    coincidencias_contact["_orden"] = textos_es.loc[
        mascara_coincidencias
    ]

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            coincidencias_contact.sort_values(
                ["_orden", "id_consulta"]
            ).drop(columns="_orden")
        )

print("\nTraducciones propuestas de CONTACT: 40")
print("Las coincidencias mostradas requieren revisión.")
print("Estado: confirmación del usuario pendiente.")
print("El siguiente respaldo incluirá la validación de ENROLLMENT.")
print("Antes del dataset final, resolveremos todas las coincidencias.")

print("=" * 60)
print("✅ Traducciones de CONTACT propuestas y coincidencias comprobadas.")
print("=" * 60)

📝 Preparando las 40 traducciones de CONTACT...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
260,CONS_0261,"I need to speak with the insurance company, how could I do it?","Necesito hablar con la compañía de seguros, ¿cómo puedo hacerlo?",CONTACT,propuesta_pendiente_revision
261,CONS_0262,"I have to speak with the insurance company, help me","Tengo que hablar con la compañía de seguros, ayúdame.",CONTACT,propuesta_pendiente_revision
262,CONS_0263,i have to talk to the insurance company how can i do it,"Tengo que hablar con la compañía de seguros, ¿cómo puedo hacerlo?",CONTACT,propuesta_pendiente_revision
263,CONS_0264,"I don't understand you, could you help me to speak withn a fucking human agent?","No te entiendo, ¿podrías ayudarme a hablar con un agente humano?",CONTACT,propuesta_pendiente_revision
264,CONS_0265,could ya help me to speak with the insurance company,¿Puedes ayudarme a hablar con la compañía de seguros?,CONTACT,propuesta_pendiente_revision
265,CONS_0266,can I speak with fucking customer service?,¿Puedo hablar con atención al cliente?,CONTACT,propuesta_pendiente_revision
266,CONS_0267,"I got to talk to fucking customer support, how to do it?","Tengo que hablar con atención al cliente, ¿cómo puedo hacerlo?",CONTACT,propuesta_pendiente_revision
267,CONS_0268,i got to talk to my fucking insurance representative,Tengo que hablar con mi representante de seguros.,CONTACT,propuesta_pendiente_revision
268,CONS_0269,"I'd like to speak with my fucking inusrance representative, could you help me?","Quisiera hablar con mi representante de seguros, ¿puedes ayudarme?",CONTACT,propuesta_pendiente_revision
269,CONS_0270,i have to speak with my insurance representaitve will ya help me,"Tengo que hablar con mi representante de seguros, ¿me ayudarás?",CONTACT,propuesta_pendiente_revision



Grupos repetidos que involucran prueba de CONTACT: 4


,id_consulta,conjunto,consulta_original,consulta_es,categoria_referencia
272,CONS_0273,prueba,I'd like to speak with my fucking insurance representative how could i do it,"Quisiera hablar con mi representante de seguros, ¿cómo puedo hacerlo?",CONTACT
297,CONS_0298,prueba,id like to speak with my insurance representative how could i do it,"Quisiera hablar con mi representante de seguros, ¿cómo puedo hacerlo?",CONTACT
268,CONS_0269,prueba,"I'd like to speak with my fucking inusrance representative, could you help me?","Quisiera hablar con mi representante de seguros, ¿puedes ayudarme?",CONTACT
299,CONS_0300,prueba,"I woulpd like to talk to my insurance representative, can you help me?","Quisiera hablar con mi representante de seguros, ¿puedes ayudarme?",CONTACT
262,CONS_0263,prueba,i have to talk to the insurance company how can i do it,"Tengo que hablar con la compañía de seguros, ¿cómo puedo hacerlo?",CONTACT
270,CONS_0271,prueba,"I got to speak with the insuracne company, how to do it?","Tengo que hablar con la compañía de seguros, ¿cómo puedo hacerlo?",CONTACT
265,CONS_0266,prueba,can I speak with fucking customer service?,¿Puedo hablar con atención al cliente?,CONTACT
275,CONS_0276,prueba,can i taok to customer support,¿Puedo hablar con atención al cliente?,CONTACT



Traducciones propuestas de CONTACT: 40
Las coincidencias mostradas requieren revisión.
Estado: confirmación del usuario pendiente.
El siguiente respaldo incluirá la validación de ENROLLMENT.
Antes del dataset final, resolveremos todas las coincidencias.
✅ Traducciones de CONTACT propuestas y coincidencias comprobadas.


### 🧠 **6.31 Selección de candidatos de reemplazo de CONTACT**

Se presentan opciones para reemplazar cuatro consultas por
traducciones repetidas dentro de prueba.

Los candidatos conservarán la categoría y la intención del registro
reemplazado. Se revisarán antes de traducirlos e incorporarlos.

**Estado:** Opciones pendientes de revisión. La muestra no se modifica.

In [ ]:
# ============================================================
# 6.31 Opciones de reemplazo de CONTACT
# ============================================================

import pandas as pd
import re

print("=" * 60)
print("🔎 Preparando opciones para cuatro reemplazos de CONTACT...")
print("=" * 60)

pares_duplicados_contact = [
    ("CONS_0263", "CONS_0271"),
    ("CONS_0266", "CONS_0276"),
    ("CONS_0273", "CONS_0298"),
    ("CONS_0269", "CONS_0300")
]

ids_reemplazar_contact = [
    reemplazar
    for conservar, reemplazar in pares_duplicados_contact
]

objetivos_contact = df_muestra_revision.loc[
    df_muestra_revision["id_consulta"].isin(
        ids_reemplazar_contact
    )
].sort_values("id_consulta").copy()

if (
    len(objetivos_contact) != 4
    or not objetivos_contact["id_consulta"].is_unique
    or not objetivos_contact["conjunto"].eq("prueba").all()
    or not objetivos_contact["categoria_original"].eq("CONTACT").all()
):
    raise ValueError("Los cuatro objetivos no coinciden con los esperados.")

# Comprobamos las coincidencias detectadas en 6.30
for conservar, reemplazar in pares_duplicados_contact:
    par = df_muestra_revision.loc[
        df_muestra_revision["id_consulta"].isin(
            [conservar, reemplazar]
        )
    ]

    textos = par["consulta_es"].fillna("").map(normalizar_contexto)

    if (
        len(par) != 2
        or not par["id_consulta"].is_unique
        or textos.eq("").any()
        or textos.nunique() != 1
    ):
        raise ValueError(
            f"La coincidencia de {reemplazar} cambió. Revisa 6.30."
        )

# Excluimos los originales utilizados y retirados
textos_excluidos_contact = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_excluidos_contact.update(
        tabla["consulta_original"]
        .fillna("")
        .map(normalizar_contexto)
    )

reserva_contact = df_elegibles.copy()

reserva_contact["texto_revision"] = (
    reserva_contact["instruction"]
    .fillna("")
    .map(normalizar_contexto)
)

def detectar_producto_candidatos_contact(texto):
    normalizado = normalizar_contexto(texto)

    return bool(
        detectar_referencias_producto(texto)
        or re.search(
            r"\b(?:travwl|ttavel|teavel|ehalth|heslth|auot)\b",
            normalizado
        )
        or re.search(r"\bpe\s+insur[a-z]*\b", normalizado)
        or re.search(
            r"(?:my|fucking)(?:health|pet|travel|auto|car|home)",
            normalizado
        )
    )

otros_productos = (
    reserva_contact["instruction"]
    .fillna("")
    .map(detectar_producto_candidatos_contact)
)

reserva_contact = (
    reserva_contact.loc[
        reserva_contact["texto_revision"].ne("")
        & ~reserva_contact["texto_revision"].isin(
            textos_excluidos_contact
        )
        & ~otros_productos
    ]
    .sort_values("texto_revision")
    .drop_duplicates("texto_revision")
    .copy()
)

# Presentamos hasta cinco opciones por objetivo
partes_opciones_contact = []
textos_presentados_contact = set()

for posicion, (_, objetivo) in enumerate(objetivos_contact.iterrows()):
    candidatos = reserva_contact.loc[
        reserva_contact["category"].eq(
            objetivo["categoria_original"]
        )
        & reserva_contact["intent"].eq(
            objetivo["intencion_original"]
        )
        & ~reserva_contact["texto_revision"].isin(
            textos_presentados_contact
        )
    ]

    if candidatos.empty:
        raise ValueError(
            f"No hay candidatos para {objetivo['id_consulta']}."
        )

    seleccion = candidatos.sample(
        n=min(5, len(candidatos)),
        random_state=8842 + posicion
    ).copy()

    textos_presentados_contact.update(seleccion["texto_revision"])

    seleccion.insert(
        0, "id_a_reemplazar", objetivo["id_consulta"]
    )
    seleccion.insert(
        1, "opcion", range(1, len(seleccion) + 1)
    )

    partes_opciones_contact.append(seleccion)

df_opciones_reemplazos_contact = pd.concat(
    partes_opciones_contact,
    ignore_index=True
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        df_opciones_reemplazos_contact[
            [
                "id_a_reemplazar",
                "opcion",
                "instruction",
                "category",
                "intent"
            ]
        ].rename(columns={
            "instruction": "consulta_original",
            "category": "categoria_original",
            "intent": "intencion_original"
        })
    )

print("\nObjetivos de reemplazo: 4")
print("La muestra permanece sin cambios.")
print("Revisaremos los candidatos antes de incorporarlos.")
print("Este bloque presenta opciones; todavía no genera un respaldo.")

print("=" * 60)
print("✅ Opciones de reemplazo de CONTACT presentadas para revisión.")
print("=" * 60)

🔎 Preparando opciones para cuatro reemplazos de CONTACT...


,id_a_reemplazar,opcion,consulta_original,categoria_original,intencion_original
0,CONS_0271,1,id like to talk to the insurer how to do it,CONTACT,agent
1,CONS_0271,2,i need assistance talkingto the insurance company,CONTACT,agent
2,CONS_0271,3,would it be possible to speak with the insurer,CONTACT,agent
3,CONS_0271,4,i got to speak with the insurace company,CONTACT,agent
4,CONS_0271,5,"I need to speak with the inusrance company, can you help me?",CONTACT,agent
5,CONS_0276,1,I'd like to talk to fucking client service help me,CONTACT,customer_service
6,CONS_0276,2,"I want to contact customer service, where do I do it?",CONTACT,customer_service
7,CONS_0276,3,need to contact customer support how do i do it,CONTACT,customer_service
8,CONS_0276,4,"I have to send an email to fucking customer support, how could I do it?",CONTACT,customer_service
9,CONS_0276,5,id like to speak with customer support could uhelp me,CONTACT,customer_service



Objetivos de reemplazo: 4
La muestra permanece sin cambios.
Revisaremos los candidatos antes de incorporarlos.
Este bloque presenta opciones; todavía no genera un respaldo.
✅ Opciones de reemplazo de CONTACT presentadas para revisión.


### 🧠 **6.32 Incorporación y respaldo de reemplazos de CONTACT**

Se incorporarán cuatro candidatos de CONTACT, conservando
sus identificadores, categoría, intención y conjunto de prueba.

Se comprobará que las 300 consultas estén traducidas y que no
existan traducciones repetidas. Los originales retirados quedarán
conservados en la auditoría.

Se descargará un respaldo de la muestra completa y de la
configuración para prueba.

**Estado:** CONTACT pendiente de confirmación del usuario.
Después se realizará la validación final y la exportación del dataset.

In [ ]:
# ============================================================
# 6.32 Incorporación y respaldo de reemplazos de CONTACT
# ============================================================

import pandas as pd
import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("=" * 60)
print("🔎 Comprobando cuatro reemplazos de CONTACT...")
print("=" * 60)

elecciones_contact = {
    "CONS_0271": {
        "opcion": 3,
        "traduccion": "¿Sería posible hablar con la aseguradora?",
        "motivo": "Traducción repetida con CONS_0263."
    },
    "CONS_0276": {
        "opcion": 4,
        "traduccion": (
            "Tengo que enviar un correo a atención al cliente, "
            "¿cómo puedo hacerlo?"
        ),
        "motivo": "Traducción repetida con CONS_0266."
    },
    "CONS_0298": {
        "opcion": 1,
        "traduccion": (
            "Tengo que hablar con mi representante de seguros, "
            "¿dónde puedo hacerlo?"
        ),
        "motivo": "Traducción repetida con CONS_0273."
    },
    "CONS_0300": {
        "opcion": 4,
        "traduccion": (
            "Quiero ayuda para hablar con mi representante de seguros."
        ),
        "motivo": "Traducción repetida con CONS_0269."
    }
}

# ------------------------------------------------------------
# 1. Aplicamos los cambios sobre una copia
# ------------------------------------------------------------

muestra_propuesta = df_muestra_revision.copy(deep=True)
fecha_actual = datetime.now(timezone.utc).isoformat()
retirados_contact = []
textos_ocupados = set()

for tabla in [df_muestra_revision, df_registros_retirados]:
    textos_ocupados.update(
        tabla["consulta_original"]
        .fillna("")
        .map(normalizar_contexto)
    )

for identificador, eleccion in elecciones_contact.items():
    opciones = df_opciones_reemplazos_contact.loc[
        df_opciones_reemplazos_contact["id_a_reemplazar"].eq(
            identificador
        )
        & df_opciones_reemplazos_contact["opcion"].eq(
            eleccion["opcion"]
        )
    ]

    mascara = muestra_propuesta["id_consulta"].eq(identificador)
    original = muestra_propuesta.loc[mascara]

    if len(opciones) != 1 or len(original) != 1:
        raise ValueError(
            f"Registro o candidato no único: {identificador}."
        )

    candidato = opciones.iloc[0]

    if (
        not original["conjunto"].eq("prueba").all()
        or not original["categoria_original"].eq("CONTACT").all()
        or candidato["category"] != "CONTACT"
        or candidato["intent"] != original.iloc[0]["intencion_original"]
    ):
        raise ValueError(
            f"Categoría, intención o conjunto incorrecto: {identificador}."
        )

    texto_candidato = normalizar_contexto(candidato["instruction"])

    if (
        not texto_candidato
        or texto_candidato in textos_ocupados
        or detectar_producto_candidatos_contact(
            candidato["instruction"]
        )
    ):
        raise ValueError(
            f"El candidato de {identificador} requiere nueva revisión."
        )

    textos_ocupados.add(texto_candidato)

    retirado = original.copy()
    retirado["motivo_retiro"] = eleccion["motivo"]
    retirado["fecha_retiro_utc"] = fecha_actual
    retirado["etapa_retiro"] = "6.32 prueba CONTACT"
    retirados_contact.append(retirado)

    cambios = {
        "indice_origen": candidato["indice_origen"],
        "consulta_original": candidato["instruction"],
        "categoria_original": candidato["category"],
        "intencion_original": candidato["intent"],
        "consulta_normalizada": candidato["consulta_normalizada"],
        "consulta_es": eleccion["traduccion"],
        "categoria_referencia": "CONTACT",
        "estado_revision": "propuesta_pendiente_revision",
        "observaciones": (
            "Original reemplazado: " + eleccion["motivo"]
            + " Traducción revisada por el asistente; "
            "confirmación del usuario pendiente."
        )
    }

    for columna, valor in cambios.items():
        muestra_propuesta.loc[mascara, columna] = valor

    for columna in ["validado_por", "fecha_validacion_utc"]:
        if columna in muestra_propuesta.columns:
            muestra_propuesta.loc[mascara, columna] = ""

    if "revision_asistente" in muestra_propuesta.columns:
        muestra_propuesta.loc[
            mascara, "revision_asistente"
        ] = "completada"

# ------------------------------------------------------------
# 2. Comprobamos los 300 registros traducidos
# ------------------------------------------------------------

if (
    len(muestra_propuesta) != 300
    or not muestra_propuesta["id_consulta"].is_unique
):
    raise ValueError("La muestra debe conservar 300 identificadores únicos.")

textos_es = (
    muestra_propuesta["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

if textos_es.eq("").any():
    raise ValueError("Todavía hay consultas sin traducción.")

repetidos = textos_es.duplicated(keep=False)

if repetidos.any():
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None
    ):
        display(
            muestra_propuesta.loc[
                repetidos,
                [
                    "id_consulta",
                    "conjunto",
                    "consulta_original",
                    "consulta_es",
                    "categoria_referencia"
                ]
            ].sort_values("consulta_es")
        )

    raise ValueError(
        "Persisten traducciones repetidas. "
        "La muestra original no se modificó."
    )

if not muestra_propuesta["categoria_referencia"].eq(
    muestra_propuesta["categoria_original"]
).all():
    raise ValueError("Hay categorías de referencia inconsistentes.")

mascara_contact = (
    muestra_propuesta["conjunto"].eq("prueba")
    & muestra_propuesta["categoria_original"].eq("CONTACT")
)

if mascara_contact.sum() != 40:
    raise ValueError("CONTACT debe conservar 40 consultas de prueba.")

# Comprobamos que los demás registros permanecen iguales
mascara_otros = ~df_muestra_revision["id_consulta"].isin(
    elecciones_contact
)

if not df_muestra_revision.loc[mascara_otros].equals(
    muestra_propuesta.loc[mascara_otros]
):
    raise ValueError("Se modificaron registros ajenos a los reemplazos.")

distribucion_antes = df_muestra_revision.groupby(
    ["categoria_original", "conjunto"]
).size()

distribucion_despues = muestra_propuesta.groupby(
    ["categoria_original", "conjunto"]
).size()

if not distribucion_antes.equals(distribucion_despues):
    raise ValueError("Cambió la distribución de la muestra.")

mascara_prueba_validada = (
    muestra_propuesta["conjunto"].eq("prueba")
    & muestra_propuesta["categoria_original"].isin(
        ["LIFE_INSURANCE", "COVERAGE", "QUOTE", "PAYMENT", "ENROLLMENT"]
    )
    & muestra_propuesta["estado_revision"].eq("validado_usuario")
)

if mascara_prueba_validada.sum() != 200:
    raise ValueError(
        "Deben existir 200 consultas de prueba previamente validadas."
    )

auditoria_propuesta = pd.concat(
    [df_registros_retirados] + retirados_contact,
    ignore_index=True,
    sort=False
)

# ------------------------------------------------------------
# 3. Guardamos el respaldo completo del avance
# ------------------------------------------------------------

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_avance_contact = (
    Path("/content/revision_evaluacion")
    / f"avance_prueba_contact_{marca_version}"
)
carpeta_avance_contact.mkdir(parents=True, exist_ok=False)

tablas_guardar = {
    "muestra_actualizada.csv": muestra_propuesta,
    "registros_retirados.csv": auditoria_propuesta,
    "prueba_contact_propuesta.csv": (
        muestra_propuesta.loc[mascara_contact]
    ),
    "prueba_validada_hasta_enrollment.csv": (
        muestra_propuesta.loc[mascara_prueba_validada]
    )
}

for nombre, tabla in tablas_guardar.items():
    tabla.to_csv(
        carpeta_avance_contact / nombre,
        index=False,
        encoding="utf-8-sig"
    )

with (
    carpeta_avance_contact / "configuracion_para_prueba.json"
).open("w", encoding="utf-8") as archivo:
    json.dump(
        {
            "modelos": modelos_comparacion,
            "configuracion_prueba": configuracion_prueba,
            "decision_desarrollo": decision_desarrollo
        },
        archivo,
        ensure_ascii=False,
        indent=2
    )

(carpeta_avance_contact / "LEEME.txt").write_text(
    "RESPALDO DE AVANCE: CONTACT\n\n"
    "muestra_actualizada.csv:\n"
    "  Las 300 consultas traducidas y su estado de revisión.\n"
    "  Insumo principal para la validación final.\n\n"
    "registros_retirados.csv:\n"
    "  Auditoría acumulada de los reemplazos.\n\n"
    "prueba_contact_propuesta.csv:\n"
    "  40 consultas de CONTACT; confirmación del usuario pendiente.\n\n"
    "prueba_validada_hasta_enrollment.csv:\n"
    "  200 consultas de prueba previamente aprobadas.\n\n"
    "configuracion_para_prueba.json:\n"
    "  Modelos y configuración fijada para prueba.\n\n"
    "Pendiente: aprobar CONTACT y exportar el dataset final.\n"
    "Conservar también el respaldo de cierre de desarrollo.\n",
    encoding="utf-8"
)

ruta_zip_contact = Path(
    shutil.make_archive(
        str(carpeta_avance_contact),
        "zip",
        root_dir=str(carpeta_avance_contact)
    )
)

archivos_esperados = {
    ruta.name
    for ruta in carpeta_avance_contact.iterdir()
    if ruta.is_file()
}

with zipfile.ZipFile(ruta_zip_contact, "r") as respaldo:
    if (
        respaldo.testzip() is not None
        or not archivos_esperados.issubset(set(respaldo.namelist()))
    ):
        raise ValueError("Falló la verificación del ZIP.")

# Publicamos los cambios después de verificar y respaldar
df_muestra_revision = muestra_propuesta
df_registros_retirados = auditoria_propuesta

# ------------------------------------------------------------
# 4. Mostramos el resultado y descargamos
# ------------------------------------------------------------

with pd.option_context("display.max_colwidth", None):
    display(
        df_muestra_revision.loc[
            df_muestra_revision["id_consulta"].isin(elecciones_contact),
            [
                "id_consulta",
                "consulta_original",
                "consulta_es",
                "categoria_referencia",
                "estado_revision"
            ]
        ].sort_values("id_consulta")
    )

print("\nConsultas totales traducidas: 300")
print("Consultas de CONTACT traducidas: 40")
print("Reemplazos incorporados: 4")
print("Grupos de traducciones repetidas: 0")
print("Consultas de prueba previamente validadas: 200")
print("Registros retirados conservados:", len(df_registros_retirados))
print("CONTACT: confirmación del usuario pendiente.")

print("\n📁 Carpeta del respaldo:")
print(carpeta_avance_contact)

print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_contact.name)

print("\n📋 Archivos incluidos y uso posterior:")
print("  • muestra_actualizada.csv: realizar la validación final.")
print("  • registros_retirados.csv: consultar la auditoría.")
print("  • prueba_contact_propuesta.csv: revisar y validar CONTACT.")
print("  • prueba_validada_hasta_enrollment.csv: recuperar prueba aprobada.")
print("  • configuracion_para_prueba.json: recuperar la configuración.")
print("  • LEEME.txt: consultar la guía del respaldo.")

print("\nPendiente: confirmación de CONTACT y exportación final.")
print("=" * 60)
print("✅ Reemplazos de CONTACT incorporados y avance respaldado.")
print("=" * 60)

print("\n📥 Solicitando descarga del ZIP...")
files.download(str(ruta_zip_contact))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

🔎 Comprobando cuatro reemplazos de CONTACT...


,id_consulta,consulta_original,consulta_es,categoria_referencia,estado_revision
270,CONS_0271,would it be possible to speak with the insurer,¿Sería posible hablar con la aseguradora?,CONTACT,propuesta_pendiente_revision
275,CONS_0276,"I have to send an email to fucking customer support, how could I do it?","Tengo que enviar un correo a atención al cliente, ¿cómo puedo hacerlo?",CONTACT,propuesta_pendiente_revision
297,CONS_0298,"I have to speak with my insurance representative, where to do it?","Tengo que hablar con mi representante de seguros, ¿dónde puedo hacerlo?",CONTACT,propuesta_pendiente_revision
299,CONS_0300,i want hnelp talking to my insurance representative,Quiero ayuda para hablar con mi representante de seguros.,CONTACT,propuesta_pendiente_revision



Consultas totales traducidas: 300
Consultas de CONTACT traducidas: 40
Reemplazos incorporados: 4
Grupos de traducciones repetidas: 0
Consultas de prueba previamente validadas: 200
Registros retirados conservados: 48
CONTACT: confirmación del usuario pendiente.

📁 Carpeta del respaldo:
/content/revision_evaluacion/avance_prueba_contact_20261010T195039112192Z

📦 ZIP para conservar en tu computadora:
avance_prueba_contact_20261010T195039112192Z.zip

📋 Archivos incluidos y uso posterior:
  • muestra_actualizada.csv: realizar la validación final.
  • registros_retirados.csv: consultar la auditoría.
  • prueba_contact_propuesta.csv: revisar y validar CONTACT.
  • prueba_validada_hasta_enrollment.csv: recuperar prueba aprobada.
  • configuracion_para_prueba.json: recuperar la configuración.
  • LEEME.txt: consultar la guía del respaldo.

Pendiente: confirmación de CONTACT y exportación final.
✅ Reemplazos de CONTACT incorporados y avance respaldado.

📥 Solicitando descarga del ZIP...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


### 🧠 **6.33 Validación y exportación del dataset final**

Se registra la aprobación del usuario de las 40 traducciones
de CONTACT y se comprueban las 300 consultas de la muestra.

Se verifican identificadores, traducciones completas, categorías,
validación del usuario y distribución: 10 consultas de desarrollo
y 40 de prueba por categoría.

Se exportan los datasets, la trazabilidad, la configuración para
evaluación y un código de carga independiente de la preparación.

**Alcance:** Se han revisado las traducciones y eliminado las
coincidencias textuales detectadas. Pueden persistir consultas
semánticamente similares.

**Estado:** Dataset preparado para evaluación. La evaluación sobre
prueba y la selección final del modelo siguen pendientes.

In [ ]:
# ============================================================
# 6.33 Validación y exportación del dataset final
# ============================================================

import pandas as pd
import json
import hashlib
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("=" * 60)
print("🔎 Validando el dataset completo...")
print("=" * 60)

# ------------------------------------------------------------
# 1. Preparamos una copia y registramos aprobación de CONTACT
# ------------------------------------------------------------

dataset_validado = df_muestra_revision.copy(deep=True)

categorias_finales = [
    "LIFE_INSURANCE",
    "COVERAGE",
    "QUOTE",
    "PAYMENT",
    "ENROLLMENT",
    "CONTACT"
]

mascara_contact = (
    dataset_validado["conjunto"].eq("prueba")
    & dataset_validado["categoria_original"].eq("CONTACT")
)

if mascara_contact.sum() != 40:
    raise ValueError("CONTACT debe contener 40 consultas de prueba.")

fecha_actual = datetime.now(timezone.utc).isoformat()

for columna in ["validado_por", "fecha_validacion_utc"]:
    if columna not in dataset_validado.columns:
        dataset_validado[columna] = ""

dataset_validado.loc[
    mascara_contact, "estado_revision"
] = "validado_usuario"

dataset_validado.loc[
    mascara_contact, "validado_por"
] = "usuario"

sin_fecha = (
    dataset_validado["fecha_validacion_utc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

dataset_validado.loc[
    mascara_contact & sin_fecha, "fecha_validacion_utc"
] = fecha_actual

# ------------------------------------------------------------
# 2. Comprobamos integridad y distribución
# ------------------------------------------------------------

if (
    len(dataset_validado) != 300
    or dataset_validado["id_consulta"].isna().any()
    or dataset_validado["id_consulta"].astype(str).str.strip().eq("").any()
    or not dataset_validado["id_consulta"].is_unique
):
    raise ValueError("Se requieren 300 identificadores completos y únicos.")

if set(dataset_validado["conjunto"]) != {"desarrollo", "prueba"}:
    raise ValueError("Hay conjuntos distintos de desarrollo y prueba.")

if set(dataset_validado["categoria_original"]) != set(categorias_finales):
    raise ValueError("Las categorías no coinciden con las seis previstas.")

if not dataset_validado["categoria_referencia"].eq(
    dataset_validado["categoria_original"]
).all():
    raise ValueError("Hay categorías de referencia inconsistentes.")

if not dataset_validado["estado_revision"].eq("validado_usuario").all():
    raise ValueError("Todavía hay consultas pendientes de validación.")

textos_es = (
    dataset_validado["consulta_es"]
    .fillna("")
    .map(normalizar_contexto)
)

originales = (
    dataset_validado["consulta_original"]
    .fillna("")
    .map(normalizar_contexto)
)

if textos_es.eq("").any() or originales.eq("").any():
    raise ValueError("Hay traducciones u originales vacíos.")

if textos_es.duplicated().any():
    raise ValueError("Persisten traducciones repetidas.")

if originales.duplicated().any():
    raise ValueError("Persisten originales repetidos.")

if dataset_validado["consulta_es"].str.contains(
    r"\bpinche\b", case=False, regex=True, na=False
).any():
    raise ValueError("Todavía aparece 'pinche' en las traducciones.")

resumen_dataset = pd.crosstab(
    dataset_validado["categoria_referencia"],
    dataset_validado["conjunto"]
).reindex(
    index=categorias_finales,
    columns=["desarrollo", "prueba"],
    fill_value=0
)

if not (
    resumen_dataset["desarrollo"].eq(10).all()
    and resumen_dataset["prueba"].eq(40).all()
):
    raise ValueError("Se requieren 10 de desarrollo y 40 de prueba por categoría.")

# Comprobamos que desarrollo conserva los textos y referencias
# utilizados en los experimentos de la sección 5
campos_comparacion = [
    "id_consulta", "consulta_es", "categoria_referencia"
]

desarrollo_actual = (
    dataset_validado.loc[
        dataset_validado["conjunto"].eq("desarrollo"),
        campos_comparacion
    ]
    .sort_values("id_consulta")
    .reset_index(drop=True)
)

desarrollo_evaluado = (
    df_comparacion_desarrollo[campos_comparacion]
    .sort_values("id_consulta")
    .reset_index(drop=True)
)

if not desarrollo_actual.equals(desarrollo_evaluado):
    raise ValueError(
        "Desarrollo cambió respecto a los datos utilizados en la sección 5."
    )

dataset_validado = dataset_validado.sort_values(
    "id_consulta"
).reset_index(drop=True)

# ------------------------------------------------------------
# 3. Preparamos datasets de uso directo
# ------------------------------------------------------------

# Estos archivos contienen los campos necesarios para evaluar.
# La trazabilidad completa se guarda en un archivo separado.
columnas_uso = [
    "id_consulta",
    "consulta_es",
    "categoria_referencia"
]

df_dataset_desarrollo = dataset_validado.loc[
    dataset_validado["conjunto"].eq("desarrollo"),
    columnas_uso
].copy()

df_dataset_prueba = dataset_validado.loc[
    dataset_validado["conjunto"].eq("prueba"),
    columnas_uso
].copy()

resumen_dataset["Total"] = resumen_dataset.sum(axis=1)
resumen_dataset.index.name = "Categoría"

# ------------------------------------------------------------
# 4. Creamos una versión nueva y guardamos los insumos
# ------------------------------------------------------------

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_dataset_final = (
    Path("/content/revision_evaluacion")
    / f"dataset_final_{marca_version}"
)
carpeta_dataset_final.mkdir(parents=True, exist_ok=False)

tablas_exportar = {
    "dataset_desarrollo.csv": df_dataset_desarrollo,
    "dataset_prueba.csv": df_dataset_prueba,
    "dataset_completo_trazabilidad.csv": dataset_validado,
    "registros_retirados.csv": df_registros_retirados,
    "distribucion_dataset.csv": resumen_dataset.reset_index()
}

for nombre, tabla in tablas_exportar.items():
    tabla.to_csv(
        carpeta_dataset_final / nombre,
        index=False,
        encoding="utf-8-sig"
    )

with (
    carpeta_dataset_final / "configuracion_para_prueba.json"
).open("w", encoding="utf-8") as archivo:
    json.dump(
        {
            "modelos": modelos_comparacion,
            "configuracion_prueba": configuracion_prueba,
            "decision_desarrollo": decision_desarrollo
        },
        archivo,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 5. Incluimos un código de carga para futuras sesiones
# ------------------------------------------------------------

codigo_carga = '''
# Cargar los insumos finales sin repetir la preparación del dataset
import pandas as pd
import json
import hashlib
from pathlib import Path

# Ajustar esta ruta a la carpeta donde se descomprimió el ZIP
carpeta_insumos = Path("/content/dataset_final")

# Verificar que los archivos coincidan con el respaldo exportado
with (carpeta_insumos / "manifest.json").open(
    encoding="utf-8"
) as archivo:
    manifest = json.load(archivo)

for nombre, datos in manifest["archivos"].items():
    ruta = carpeta_insumos / nombre
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta el insumo: {ruta}")
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    if huella != datos["sha256"]:
        raise ValueError(f"El archivo cambió: {nombre}")

df_dataset_desarrollo = pd.read_csv(
    carpeta_insumos / "dataset_desarrollo.csv",
    encoding="utf-8-sig",
    keep_default_na=False
)
df_dataset_prueba = pd.read_csv(
    carpeta_insumos / "dataset_prueba.csv",
    encoding="utf-8-sig",
    keep_default_na=False
)

with (carpeta_insumos / "configuracion_para_prueba.json").open(
    encoding="utf-8"
) as archivo:
    registro_configuracion = json.load(archivo)

modelos_comparacion = registro_configuracion["modelos"]
configuracion_prueba = registro_configuracion["configuracion_prueba"]
decision_desarrollo = registro_configuracion["decision_desarrollo"]

consultas_prueba = df_dataset_prueba["consulta_es"].tolist()
etiquetas_prueba = df_dataset_prueba["categoria_referencia"].tolist()

print("=" * 60)
print("✅ Insumos verificados y cargados.")
print("Desarrollo:", len(df_dataset_desarrollo))
print("Prueba:", len(df_dataset_prueba))
print("Configuración para prueba recuperada.")
print("No es necesario repetir el muestreo ni las traducciones.")
print("La inferencia requiere cargar las bibliotecas y los modelos.")
print("=" * 60)
'''.lstrip()

(carpeta_dataset_final / "cargar_insumos.py").write_text(
    codigo_carga,
    encoding="utf-8"
)

guia_final = """DATASET FINAL PARA EVALUACIÓN
========================================

INSUMOS PRINCIPALES
dataset_desarrollo.csv
  60 consultas: 10 por categoría.
  Uso: desarrollo y consulta de los datos evaluados en la sección 5.

dataset_prueba.csv
  240 consultas: 40 por categoría.
  Uso: evaluación sobre prueba con la configuración fijada.

configuracion_para_prueba.json
  Modelos, configuración común y decisión preliminar de desarrollo.
  Uso: recuperar los parámetros para evaluar.

TRAZABILIDAD
dataset_completo_trazabilidad.csv
  Los 300 registros con originales, traducciones y revisión.

registros_retirados.csv
  Auditoría de los registros reemplazados.
  No utilizar como datos de evaluación.

distribucion_dataset.csv
  Cantidades por categoría y conjunto.

RECUPERACIÓN
cargar_insumos.py
  Código de carga y verificación independiente de la preparación.
  Ajustar carpeta_insumos a la carpeta donde se descomprimió el ZIP.
  En Colab se puede subir el ZIP, descomprimirlo en
  /content/dataset_final y ejecutar este código.

manifest.json
  Versión, cantidades y huellas SHA-256 de los archivos.

ALCANCE
Traducciones revisadas y aprobadas por el usuario.
Sin coincidencias textuales según la normalización utilizada.
Pueden persistir consultas semánticamente similares.
Los originales conservan las expresiones ofensivas para trazabilidad;
las traducciones se prepararon sin ellas.

SIGUIENTE PASO
Evaluar los modelos sobre dataset_prueba.csv usando la configuración
fijada. No ajustar las descripciones a partir de resultados de prueba.
La evaluación y la selección final del modelo están pendientes.

Conservar también el ZIP de cierre de desarrollo de la sección 5.10.
"""

(carpeta_dataset_final / "LEEME.txt").write_text(
    guia_final,
    encoding="utf-8"
)

# Guardamos cantidades y huellas para verificar futuras cargas
manifest = {
    "version_utc": marca_version,
    "estado": "dataset validado; evaluación de prueba pendiente",
    "consultas_desarrollo": len(df_dataset_desarrollo),
    "consultas_prueba": len(df_dataset_prueba),
    "registros_retirados": len(df_registros_retirados),
    "categorias": categorias_finales,
    "archivos": {}
}

for ruta in sorted(carpeta_dataset_final.iterdir()):
    if ruta.is_file():
        manifest["archivos"][ruta.name] = {
            "bytes": ruta.stat().st_size,
            "sha256": hashlib.sha256(ruta.read_bytes()).hexdigest()
        }

with (carpeta_dataset_final / "manifest.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(manifest, archivo, ensure_ascii=False, indent=2)

# ------------------------------------------------------------
# 6. Comprimimos y verificamos el respaldo
# ------------------------------------------------------------

ruta_zip_dataset_final = Path(
    shutil.make_archive(
        str(carpeta_dataset_final),
        "zip",
        root_dir=str(carpeta_dataset_final)
    )
)

archivos_esperados = {
    ruta.name
    for ruta in carpeta_dataset_final.iterdir()
    if ruta.is_file()
}

with zipfile.ZipFile(ruta_zip_dataset_final, "r") as respaldo:
    if (
        respaldo.testzip() is not None
        or not archivos_esperados.issubset(set(respaldo.namelist()))
    ):
        raise ValueError("Falló la verificación del ZIP final.")

# Publicamos la validación después de guardar y verificar
df_muestra_revision = dataset_validado

# ------------------------------------------------------------
# 7. Mostramos los insumos y descargamos
# ------------------------------------------------------------

display(resumen_dataset.reset_index())

print("\nConsultas de desarrollo validadas:", len(df_dataset_desarrollo))
print("Consultas de prueba validadas:", len(df_dataset_prueba))
print("Consultas totales validadas:", len(dataset_validado))
print("Traducciones vacías: 0")
print("Traducciones repetidas: 0")
print("Registros retirados conservados:", len(df_registros_retirados))

print("\n📁 Carpeta de los insumos finales en Colab:")
print(carpeta_dataset_final)

print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_dataset_final.name)

print("\n📋 Archivos incluidos:")
for nombre in sorted(archivos_esperados):
    print(f"  • {nombre}")

print("\n➡️ PARA EVALUAR: dataset_prueba.csv")
print("➡️ CONFIGURACIÓN: configuracion_para_prueba.json")
print("➡️ PARA CARGAR EN OTRA SESIÓN: cargar_insumos.py")
print("➡️ GUÍA DE USO: LEEME.txt")
print("➡️ AUDITORÍA: registros_retirados.csv")

print("\nEstos archivos permiten reutilizar el dataset")
print("sin repetir el muestreo, las traducciones ni los reemplazos.")
print("Pendiente: evaluación sobre prueba y selección final del modelo.")

print("=" * 60)
print("✅ Dataset final validado, exportado y ZIP verificado.")
print("=" * 60)

print("\n📥 Solicitando descarga del dataset final...")
files.download(str(ruta_zip_dataset_final))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

🔎 Validando el dataset completo...


conjunto,Categoría,desarrollo,prueba,Total
0,LIFE_INSURANCE,10,40,50
1,COVERAGE,10,40,50
2,QUOTE,10,40,50
3,PAYMENT,10,40,50
4,ENROLLMENT,10,40,50
5,CONTACT,10,40,50



Consultas de desarrollo validadas: 60
Consultas de prueba validadas: 240
Consultas totales validadas: 300
Traducciones vacías: 0
Traducciones repetidas: 0
Registros retirados conservados: 48

📁 Carpeta de los insumos finales en Colab:
/content/revision_evaluacion/dataset_final_20261010T195212647311Z

📦 ZIP para conservar en tu computadora:
dataset_final_20261010T195212647311Z.zip

📋 Archivos incluidos:
  • LEEME.txt
  • cargar_insumos.py
  • configuracion_para_prueba.json
  • dataset_completo_trazabilidad.csv
  • dataset_desarrollo.csv
  • dataset_prueba.csv
  • distribucion_dataset.csv
  • manifest.json
  • registros_retirados.csv

➡️ PARA EVALUAR: dataset_prueba.csv
➡️ CONFIGURACIÓN: configuracion_para_prueba.json
➡️ PARA CARGAR EN OTRA SESIÓN: cargar_insumos.py
➡️ GUÍA DE USO: LEEME.txt
➡️ AUDITORÍA: registros_retirados.csv

Estos archivos permiten reutilizar el dataset
sin repetir el muestreo, las traducciones ni los reemplazos.
Pendiente: evaluación sobre prueba y selección final

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


## 💻 **7. Evaluación final de los modelos sobre prueba**

Se evaluarán MiniLMv2 multilingüe, mDeBERTa-v3-base y
XLM-RoBERTa-large sobre las 240 consultas de prueba validadas,
con 40 consultas por categoría.

Se utilizará la configuración inicial fijada durante desarrollo,
sin modificar las descripciones a partir de los resultados de prueba.

La comparación incluirá exactitud, precisión macro, exhaustividad
macro, F1 macro, resultados por categoría y tiempos de ejecución.
El criterio principal será F1 macro.

Los resultados y la selección final se respaldarán en archivos
descargables, conservando su relación con el dataset utilizado.

**Insumos:** dataset_prueba.csv y configuracion_para_prueba.json,
incluidos en el ZIP del dataset final.

### 🧠 **7.1 Carga del dataset final y preparación de prueba**

Se cargan las 240 consultas de prueba y la configuración fijada
durante desarrollo. Se verifica la integridad de los archivos y
la distribución de 40 consultas por categoría.

Los tres modelos se evaluarán sobre las mismas consultas y con
las descripciones iniciales. No se ajustará la configuración
a partir de los resultados de prueba.

**Estado:** Insumos cargados; evaluación pendiente.

In [ ]:
# ============================================================
# 7.1 Carga del dataset final y preparación de prueba
# ============================================================

import pandas as pd
import json
import hashlib
import torch
from pathlib import Path

print("=" * 60)
print("📂 Cargando los insumos finales para evaluar sobre prueba...")
print("=" * 60)

# En esta sesión utilizamos la carpeta generada en 6.33.
# En una sesión nueva, sube y descomprime el ZIP final,
# y cambia esta ruta por la carpeta donde quedaron sus archivos.
carpeta_insumos = Path(
    "/content/revision_evaluacion/dataset_final_20261010T195212647311Z"
)

if not carpeta_insumos.is_dir():
    raise FileNotFoundError(
        f"No se encuentra la carpeta: {carpeta_insumos}\n"
        "Sube y descomprime el ZIP del dataset final y ajusta "
        "carpeta_insumos. No necesitas repetir su preparación."
    )

# ------------------------------------------------------------
# 1. Verificamos la integridad del respaldo
# ------------------------------------------------------------

with (carpeta_insumos / "manifest.json").open(
    encoding="utf-8"
) as archivo:
    manifest_prueba = json.load(archivo)

for nombre, datos in manifest_prueba["archivos"].items():
    ruta = carpeta_insumos / nombre

    if not ruta.is_file():
        raise FileNotFoundError(f"Falta el archivo: {ruta}")

    huella_actual = hashlib.sha256(ruta.read_bytes()).hexdigest()

    if huella_actual != datos["sha256"]:
        raise ValueError(
            f"El archivo {nombre} cambió respecto al respaldo final."
        )

# ------------------------------------------------------------
# 2. Cargamos los datos y la configuración guardada
# ------------------------------------------------------------

df_dataset_desarrollo = pd.read_csv(
    carpeta_insumos / "dataset_desarrollo.csv",
    encoding="utf-8-sig",
    keep_default_na=False
)

df_dataset_prueba = pd.read_csv(
    carpeta_insumos / "dataset_prueba.csv",
    encoding="utf-8-sig",
    keep_default_na=False
)

with (carpeta_insumos / "configuracion_para_prueba.json").open(
    encoding="utf-8"
) as archivo:
    registro_configuracion = json.load(archivo)

modelos_comparacion = registro_configuracion["modelos"]
configuracion_prueba = registro_configuracion["configuracion_prueba"]
decision_desarrollo = registro_configuracion["decision_desarrollo"]

categorias_prueba = manifest_prueba["categorias"]

# ------------------------------------------------------------
# 3. Verificamos las consultas de prueba
# ------------------------------------------------------------

columnas_requeridas = {
    "id_consulta",
    "consulta_es",
    "categoria_referencia"
}

if not columnas_requeridas.issubset(df_dataset_prueba.columns):
    raise ValueError("El dataset de prueba no tiene los campos requeridos.")

if (
    len(df_dataset_prueba) != 240
    or not df_dataset_prueba["id_consulta"].is_unique
    or df_dataset_prueba["consulta_es"].str.strip().eq("").any()
):
    raise ValueError(
        "Se requieren 240 consultas de prueba completas "
        "con identificadores únicos."
    )

if set(df_dataset_prueba["id_consulta"]) & set(
    df_dataset_desarrollo["id_consulta"]
):
    raise ValueError("Hay identificadores compartidos entre los conjuntos.")

distribucion_prueba = (
    df_dataset_prueba["categoria_referencia"]
    .value_counts()
    .reindex(categorias_prueba, fill_value=0)
)

if (
    set(df_dataset_prueba["categoria_referencia"])
        != set(categorias_prueba)
    or not distribucion_prueba.eq(40).all()
):
    raise ValueError("Se requieren 40 consultas por cada categoría.")

# Conservamos un orden estable para todos los modelos
df_dataset_prueba = (
    df_dataset_prueba.sort_values("id_consulta")
    .reset_index(drop=True)
)

consultas_prueba = df_dataset_prueba["consulta_es"].tolist()
etiquetas_prueba = df_dataset_prueba["categoria_referencia"].tolist()

# ------------------------------------------------------------
# 4. Identificamos el dispositivo disponible
# ------------------------------------------------------------

dispositivo_prueba = 0 if torch.cuda.is_available() else -1

nombre_dispositivo = (
    torch.cuda.get_device_name(0)
    if dispositivo_prueba == 0
    else "CPU"
)

display(
    distribucion_prueba.rename_axis("Categoría")
    .reset_index(name="Consultas")
)

print("\n📁 Insumos cargados desde:")
print(carpeta_insumos)

print("\nConsultas de prueba:", len(consultas_prueba))
print("Dispositivo:", nombre_dispositivo)
print("Modelos:", len(modelos_comparacion))
print("Configuración recuperada del respaldo final:")

print(json.dumps(
    configuracion_prueba,
    ensure_ascii=False,
    indent=2
))

print("\nLos archivos se verificaron contra el manifiesto.")
print("No se ha ejecutado inferencia sobre prueba.")
print("Siguiente paso: evaluar los tres modelos con esta configuración.")

print("=" * 60)
print("✅ Dataset final y configuración de prueba cargados.")
print("=" * 60)

📂 Cargando los insumos finales para evaluar sobre prueba...


,Categoría,Consultas
0,LIFE_INSURANCE,40
1,COVERAGE,40
2,QUOTE,40
3,PAYMENT,40
4,ENROLLMENT,40
5,CONTACT,40



📁 Insumos cargados desde:
/content/revision_evaluacion/dataset_final_20261010T195212647311Z

Consultas de prueba: 240
Dispositivo: Tesla T4
Modelos: 3
Configuración recuperada del respaldo final:
{
  "descripciones": {
    "LIFE_INSURANCE": "información general o consulta de información sobre un seguro de vida",
    "COVERAGE": "consulta de coberturas, exclusiones o modificación de la cobertura del seguro",
    "QUOTE": "consulta de tarifas, estimación de costo o solicitud de cotización del seguro",
    "PAYMENT": "realización y consulta de pagos del seguro, programación, métodos y problemas de pago",
    "ENROLLMENT": "comparación, contratación, renovación o cancelación de pólizas, incluidos requisitos y cargos de terminación",
    "CONTACT": "contacto con atención al cliente, un representante, la aseguradora o una persona"
  },
  "plantilla": "Esta consulta trata sobre {}.",
  "multi_label": false,
  "batch_size": 1,
  "repeticiones": 3,
  "precision": "float32"
}

Los archivos se v

### 🧠 **7.2 Evaluación de los tres modelos sobre prueba**

Se evaluarán los tres modelos sobre las mismas 240 consultas,
utilizando la configuración fijada durante desarrollo.

Se realizarán tres repeticiones por modelo, con calentamiento previo.
El tiempo de carga se registrará por separado. Las métricas se
calcularán con las predicciones de la primera repetición y se
comprobará la estabilidad de las etiquetas entre repeticiones.

Se guardarán resultados generales, métricas por categoría,
predicciones, errores, matrices de confusión y tiempos.

**Criterio principal:** F1 macro.
**Estado:** Evaluación de prueba y respaldo de resultados.

In [ ]:
# ============================================================
# 7.2 Evaluación de los tres modelos sobre prueba
# ============================================================

import gc
import time
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datetime import datetime, timezone
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
from google.colab import files

print("=" * 60)
print("🔎 Iniciando evaluación de los tres modelos sobre prueba...")
print("=" * 60)

# ------------------------------------------------------------
# 1. Recuperamos los parámetros fijados
# ------------------------------------------------------------

descripciones = configuracion_prueba["descripciones"]
categorias = list(categorias_prueba)

if set(descripciones) != set(categorias):
    raise ValueError("Las descripciones no coinciden con las categorías.")

if configuracion_prueba["precision"] != "float32":
    raise ValueError("La evaluación prevista requiere precisión float32.")

if len(df_dataset_prueba) != 240:
    raise ValueError("Se requieren las 240 consultas cargadas en 7.1.")

# Los identificadores se recuperan del respaldo, no se redefinen
if not isinstance(modelos_comparacion, dict):
    raise ValueError(
        "modelos_comparacion debe ser un diccionario "
        "con nombre del modelo e identificador de Hugging Face."
    )

if (
    len(modelos_comparacion) != 3
    or not all(isinstance(v, str) for v in modelos_comparacion.values())
):
    raise ValueError("Comprueba los tres identificadores de modelos guardados.")

etiquetas_candidatas = [descripciones[c] for c in categorias]

if len(set(etiquetas_candidatas)) != len(categorias):
    raise ValueError("Las descripciones de las categorías deben ser distintas.")

descripcion_a_categoria = {
    descripciones[c]: c for c in categorias
}

repeticiones = configuracion_prueba["repeticiones"]
tamanio_lote = configuracion_prueba["batch_size"]

if repeticiones != 3 or tamanio_lote != 1:
    raise ValueError("La configuración fijada requiere tres repeticiones y lote 1.")

# Liberamos los clasificadores conocidos de los experimentos anteriores
for nombre_variable in [
    "clasificador_minilm",
    "clasificador_mdeberta",
    "clasificador_xlm_roberta",
    "clasificador",
    "clasificador_prueba"
]:
    globals().pop(nombre_variable, None)

gc.collect()

if dispositivo_prueba == 0:
    torch.cuda.empty_cache()

resultados_prueba = {}
filas_metricas_prueba = []


def sincronizar_prueba():
    """Espera a que finalicen las operaciones de GPU antes de medir."""
    if dispositivo_prueba == 0:
        torch.cuda.synchronize()


# ------------------------------------------------------------
# 2. Evaluamos cada modelo
# ------------------------------------------------------------

for numero, (nombre, identificador_modelo) in enumerate(
    modelos_comparacion.items(), start=1
):
    print("\n" + "=" * 60)
    print(f"Modelo {numero}/3: {nombre}")
    print("Identificador:", identificador_modelo)
    print("=" * 60)

    sincronizar_prueba()
    inicio_carga = time.perf_counter()

    clasificador_prueba = pipeline(
        "zero-shot-classification",
        model=identificador_modelo,
        device=dispositivo_prueba,
        dtype=torch.float32
    )

    sincronizar_prueba()
    tiempo_carga = time.perf_counter() - inicio_carga

    try:
        # Calentamiento separado de las mediciones
        with torch.inference_mode():
            clasificador_prueba(
                consultas_prueba[:2],
                candidate_labels=etiquetas_candidatas,
                hypothesis_template=configuracion_prueba["plantilla"],
                multi_label=configuracion_prueba["multi_label"],
                batch_size=tamanio_lote
            )

        sincronizar_prueba()

        tiempos = []
        etiquetas_repeticiones = []
        salida_primera = None

        for repeticion in range(repeticiones):
            sincronizar_prueba()
            inicio = time.perf_counter()

            with torch.inference_mode():
                salida = clasificador_prueba(
                    consultas_prueba,
                    candidate_labels=etiquetas_candidatas,
                    hypothesis_template=configuracion_prueba["plantilla"],
                    multi_label=configuracion_prueba["multi_label"],
                    batch_size=tamanio_lote
                )

            sincronizar_prueba()
            duracion = time.perf_counter() - inicio
            tiempos.append(duracion)

            predichas = [
                descripcion_a_categoria[r["labels"][0]]
                for r in salida
            ]

            etiquetas_repeticiones.append(predichas)

            if repeticion == 0:
                salida_primera = salida

            print(
                f"Repetición {repeticion + 1}: "
                f"{duracion:.2f} segundos"
            )

        predichas = etiquetas_repeticiones[0]

        etiquetas_estables = all(
            etiquetas == predichas
            for etiquetas in etiquetas_repeticiones[1:]
        )

        precision_macro, recall_macro, f1_macro, _ = (
            precision_recall_fscore_support(
                etiquetas_prueba,
                predichas,
                labels=categorias,
                average="macro",
                zero_division=0
            )
        )

        metricas = {
            "Modelo": nombre,
            "Exactitud": float(accuracy_score(
                etiquetas_prueba, predichas
            )),
            "Precisión macro": float(precision_macro),
            "Exhaustividad macro": float(recall_macro),
            "F1 macro": float(f1_macro),
            "Tiempo medio total (s)": float(np.mean(tiempos)),
            "Desviación de tiempo (s)": float(np.std(tiempos, ddof=1)),
            "Tiempo medio por consulta (s)": float(
                np.mean(tiempos) / len(consultas_prueba)
            )
        }

        precision, recall, f1, soporte = (
            precision_recall_fscore_support(
                etiquetas_prueba,
                predichas,
                labels=categorias,
                average=None,
                zero_division=0
            )
        )

        metricas_categoria = pd.DataFrame({
            "Categoría": categorias,
            "Precisión": precision,
            "Exhaustividad": recall,
            "F1": f1,
            "Consultas": soporte
        })

        predicciones = df_dataset_prueba.copy()
        predicciones["categoria_predicha"] = predichas
        predicciones["puntuacion"] = [
            float(r["scores"][0]) for r in salida_primera
        ]
        predicciones["acierto"] = predicciones[
            "categoria_referencia"
        ].eq(predicciones["categoria_predicha"])

        errores = predicciones.loc[
            ~predicciones["acierto"]
        ].copy()

        matriz = pd.DataFrame(
            confusion_matrix(
                etiquetas_prueba,
                predichas,
                labels=categorias
            ),
            index=categorias,
            columns=categorias
        )
        matriz.index.name = "Categoría de referencia"
        matriz.columns.name = "Categoría predicha"

        resultados_prueba[nombre] = {
            "metricas": metricas,
            "metricas_categoria": metricas_categoria,
            "predicciones": predicciones,
            "errores": errores,
            "matriz_confusion": matriz,
            "tiempos": [float(t) for t in tiempos],
            "tiempo_carga": float(tiempo_carga),
            "etiquetas_estables": bool(etiquetas_estables)
        }

        filas_metricas_prueba.append(metricas)

        display(pd.DataFrame([metricas]).round(4))
        display(metricas_categoria.round(4))

        print("Aciertos:", int(predicciones["acierto"].sum()), "de 240")
        print("Tiempo de carga:", round(tiempo_carga, 2), "segundos")
        print("Etiquetas iguales entre repeticiones:", etiquetas_estables)

    finally:
        # Liberamos el modelo también si falla su evaluación
        del clasificador_prueba
        gc.collect()

        if dispositivo_prueba == 0:
            torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Consolidamos y respaldamos los resultados
# ------------------------------------------------------------

df_comparacion_prueba = (
    pd.DataFrame(filas_metricas_prueba)
    .sort_values("F1 macro", ascending=False)
    .reset_index(drop=True)
)

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_resultados_prueba = (
    Path("/content/revision_evaluacion")
    / f"evaluacion_prueba_{marca_version}"
)
carpeta_resultados_prueba.mkdir(parents=True, exist_ok=False)

df_comparacion_prueba.to_csv(
    carpeta_resultados_prueba / "comparacion_prueba.csv",
    index=False,
    encoding="utf-8-sig"
)

df_dataset_prueba.to_csv(
    carpeta_resultados_prueba / "dataset_prueba_utilizado.csv",
    index=False,
    encoding="utf-8-sig"
)

registro_evaluacion = {
    "version_utc": marca_version,
    "version_dataset": manifest_prueba["version_utc"],
    "dispositivo": nombre_dispositivo,
    "consultas": len(df_dataset_prueba),
    "configuracion_prueba": configuracion_prueba,
    "modelos": {},
    "versiones": {
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__
    }
}

for numero, (nombre, resultado) in enumerate(
    resultados_prueba.items(), start=1
):
    prefijo = f"modelo_{numero}"

    for tipo in ["predicciones", "errores", "metricas_categoria"]:
        resultado[tipo].to_csv(
            carpeta_resultados_prueba / f"{prefijo}_{tipo}.csv",
            index=False,
            encoding="utf-8-sig"
        )

    resultado["matriz_confusion"].to_csv(
        carpeta_resultados_prueba / f"{prefijo}_matriz_confusion.csv",
        encoding="utf-8-sig"
    )

    registro_evaluacion["modelos"][nombre] = {
        "identificador": modelos_comparacion[nombre],
        "prefijo_archivos": prefijo,
        "metricas": resultado["metricas"],
        "tiempos": resultado["tiempos"],
        "tiempo_carga": resultado["tiempo_carga"],
        "etiquetas_estables": resultado["etiquetas_estables"]
    }

with (
    carpeta_resultados_prueba / "registro_evaluacion.json"
).open("w", encoding="utf-8") as archivo:
    json.dump(
        registro_evaluacion,
        archivo,
        ensure_ascii=False,
        indent=2
    )

(carpeta_resultados_prueba / "LEEME.txt").write_text(
    "RESULTADOS DE EVALUACIÓN SOBRE PRUEBA\n\n"
    "comparacion_prueba.csv: métricas generales de los tres modelos.\n"
    "dataset_prueba_utilizado.csv: las 240 consultas evaluadas.\n"
    "modelo_N_predicciones.csv: etiquetas, puntuaciones y aciertos.\n"
    "modelo_N_errores.csv: consultas clasificadas incorrectamente.\n"
    "modelo_N_metricas_categoria.csv: resultados por categoría.\n"
    "modelo_N_matriz_confusion.csv: filas de referencia; columnas predichas.\n"
    "registro_evaluacion.json: relación entre nombres y archivos, "
    "configuración, tiempos y versión del dataset.\n\n"
    "Los tiempos medidos incluyen el procesamiento de la pipeline.\n"
    "El promedio por consulta no es una medición aislada de latencia.\n"
    "La puntuación del modelo no representa una probabilidad calibrada.\n"
    "Conservar también el ZIP del dataset final y el cierre de desarrollo.\n"
    "Estos resultados permiten analizar la evaluación sin repetir inferencia.\n",
    encoding="utf-8"
)

ruta_zip_resultados_prueba = Path(
    shutil.make_archive(
        str(carpeta_resultados_prueba),
        "zip",
        root_dir=str(carpeta_resultados_prueba)
    )
)

archivos_esperados = {
    ruta.name
    for ruta in carpeta_resultados_prueba.iterdir()
    if ruta.is_file()
}

with zipfile.ZipFile(ruta_zip_resultados_prueba, "r") as respaldo:
    if (
        respaldo.testzip() is not None
        or not archivos_esperados.issubset(set(respaldo.namelist()))
    ):
        raise ValueError("Falló la verificación del ZIP de resultados.")

print("\n📊 Comparación sobre prueba:")
display(df_comparacion_prueba.round(4))

print("\n📁 Resultados guardados en:")
print(carpeta_resultados_prueba)

print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_resultados_prueba.name)

print("\n➡️ PARA COMPARAR: comparacion_prueba.csv")
print("➡️ PARA ANALIZAR ERRORES: modelo_N_errores.csv")
print("➡️ PARA IDENTIFICAR CADA MODELO: registro_evaluacion.json")

print("=" * 60)
print("✅ Tres modelos evaluados y resultados respaldados.")
print("=" * 60)

print("\n📥 Solicitando descarga de los resultados...")
files.download(str(ruta_zip_resultados_prueba))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

🔎 Iniciando evaluación de los tres modelos sobre prueba...

Modelo 1/3: MiniLMv2 multilingüe
Identificador: MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Repetición 1: 24.66 segundos
Repetición 2: 18.10 segundos
Repetición 3: 12.82 segundos


,Modelo,Exactitud,Precisión macro,Exhaustividad macro,F1 macro,Tiempo medio total (s),Desviación de tiempo (s),Tiempo medio por consulta (s)
0,MiniLMv2 multilingüe,0.4417,0.4116,0.4417,0.4002,18.5258,5.9288,0.0772


,Categoría,Precisión,Exhaustividad,F1,Consultas
0,LIFE_INSURANCE,0.1111,0.025,0.0408,40
1,COVERAGE,0.4074,0.275,0.3284,40
2,QUOTE,0.6538,0.425,0.5152,40
3,PAYMENT,0.1190,0.125,0.1220,40
4,ENROLLMENT,0.3864,0.850,0.5312,40
5,CONTACT,0.7917,0.950,0.8636,40


...

,Modelo,Exactitud,Precisión macro,Exhaustividad macro,F1 macro,Tiempo medio total (s),Desviación de tiempo (s),Tiempo medio por consulta (s)
0,XLM-RoBERTa-large,0.6583,0.7460,0.6583,0.6492,26.6031,0.0942,0.1108
1,MiniLMv2 multilingüe,0.4417,0.4116,0.4417,0.4002,18.5258,5.9288,0.0772
2,mDeBERTa-v3-base,0.3125,0.5596,0.3125,0.2893,34.0984,0.2110,0.1421



📁 Resultados guardados en:
/content/revision_evaluacion/evaluacion_prueba_20261010T200128828138Z

📦 ZIP para conservar en tu computadora:
evaluacion_prueba_20261010T200128828138Z.zip

➡️ PARA COMPARAR: comparacion_prueba.csv
➡️ PARA ANALIZAR ERRORES: modelo_N_errores.csv
➡️ PARA IDENTIFICAR CADA MODELO: registro_evaluacion.json
✅ Tres modelos evaluados y resultados respaldados.

📥 Solicitando descarga de los resultados...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


### 🧠 **7.3 Interpretación de resultados y selección final**

XLM-RoBERTa-large obtuvo el mayor F1 macro sobre las 240 consultas
de prueba: 0.6492, con 158 aciertos y una exactitud de 65.83 %.
Este resultado respalda su selección para esta actividad con las
descripciones iniciales fijadas durante desarrollo.

MiniLMv2 obtuvo un F1 macro de 0.4002 y mDeBERTa de 0.2893.
Los tres modelos conservaron las mismas etiquetas entre repeticiones.

**Resultados por categoría:**

XLM-RoBERTa identificó las 40 consultas de LIFE_INSURANCE,
pero también asignó esta categoría a 45 consultas incorrectas.
Su precisión fue de 0.4706, lo que muestra una tendencia a
clasificar solicitudes específicas como información general.

En PAYMENT y ENROLLMENT acertó 17 de 40 consultas por categoría.
Estas categorías siguen siendo áreas de mejora, aunque en
ENROLLMENT todas sus predicciones fueron correctas.

MiniLMv2 obtuvo el mayor F1 de CONTACT: 0.8636.
mDeBERTa obtuvo el mayor F1 de QUOTE: 0.6102.
Estas ventajas puntuales no superaron el resultado global
de XLM-RoBERTa.

**Tiempos:**

MiniLMv2 registró el menor tiempo medio, pero presentó una
variación considerable entre repeticiones: de 24.66 a 12.82 segundos.
XLM-RoBERTa mostró tiempos más consistentes, con un promedio
de 26.6031 segundos para las 240 consultas.

Los tiempos corresponden a esta ejecución en Tesla T4 y al
procesamiento completo de la pipeline. El promedio por consulta
no representa una medición aislada de latencia.

**Selección final:**

Se selecciona XLM-RoBERTa-large con la configuración inicial
por obtener el mayor F1 macro entre los tres modelos evaluados.

**Limitaciones:**

La conclusión corresponde a esta muestra traducida y revisada.
Aunque se eliminaron coincidencias textuales, pueden persistir
consultas semánticamente similares entre desarrollo y prueba.

Las tres repeticiones verifican estabilidad de predicciones y
variación temporal; no equivalen a tres muestras independientes.

El modelo seleccionado cometió 82 errores de 240 consultas.
Su elección representa el mejor resultado de esta comparación;
no demuestra por sí sola que sea suficiente para uso operativo.

### 🧠 **7.4 Respaldo de la selección final y configuración reutilizable**

Se registra el modelo con mayor F1 macro sobre prueba:
XLM-RoBERTa-large, con la configuración inicial.

Se guardan la decisión, la configuración, los datasets y los
resultados de evaluación en un paquete descargable. Se incluye
un código para cargar el modelo y clasificar nuevas consultas.

**Estado:** Comparación completada y selección final respaldada.
El paquete conserva la configuración; los pesos del modelo
se descargarán desde Hugging Face cuando se cargue.

In [ ]:
# ============================================================
# 7.4 Respaldo de selección final y configuración reutilizable
# ============================================================

import pandas as pd
import json
import shutil
import zipfile
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("=" * 60)
print("📦 Preparando el paquete final de la comparación...")
print("=" * 60)

# ------------------------------------------------------------
# 1. Seleccionamos por el criterio fijado: F1 macro
# ------------------------------------------------------------

comparacion_final = (
    df_comparacion_prueba
    .sort_values("F1 macro", ascending=False)
    .reset_index(drop=True)
)

if (
    len(comparacion_final) != 3
    or not comparacion_final["Modelo"].is_unique
    or comparacion_final["F1 macro"].isna().any()
):
    raise ValueError("La comparación debe contener tres modelos válidos.")

if comparacion_final.loc[0, "F1 macro"] == comparacion_final.loc[1, "F1 macro"]:
    raise ValueError(
        "Hay empate en F1 macro. Debe documentarse antes de seleccionar."
    )

modelo_seleccionado = comparacion_final.loc[0, "Modelo"]

if modelo_seleccionado not in resultados_prueba:
    raise ValueError("No se encuentran los resultados del modelo seleccionado.")

resultado_seleccionado = resultados_prueba[modelo_seleccionado]
predicciones_seleccionadas = resultado_seleccionado["predicciones"]

if (
    len(predicciones_seleccionadas) != 240
    or not predicciones_seleccionadas["id_consulta"].is_unique
    or set(predicciones_seleccionadas["id_consulta"])
        != set(df_dataset_prueba["id_consulta"])
):
    raise ValueError("Las predicciones no corresponden al dataset de prueba.")

if not resultado_seleccionado["etiquetas_estables"]:
    raise ValueError(
        "Las etiquetas del modelo seleccionado cambiaron entre repeticiones."
    )

aciertos = int(predicciones_seleccionadas["acierto"].sum())

configuracion_modelo_final = {
    "nombre_modelo": modelo_seleccionado,
    "identificador_hugging_face": modelos_comparacion[modelo_seleccionado],
    "configuracion": configuracion_prueba,
    "version_dataset": manifest_prueba["version_utc"],
    "criterio_seleccion": "F1 macro sobre prueba",
    "categorias": list(categorias_prueba)
}

decision_final = {
    "fecha_registro_utc": datetime.now(timezone.utc).isoformat(),
    "modelo_seleccionado": modelo_seleccionado,
    "identificador_hugging_face": modelos_comparacion[modelo_seleccionado],
    "criterio": "mayor F1 macro sobre prueba",
    "consultas_prueba": 240,
    "aciertos": aciertos,
    "errores": 240 - aciertos,
    "metricas": resultado_seleccionado["metricas"],
    "etiquetas_estables": resultado_seleccionado["etiquetas_estables"],
    "alcance": (
        "Selección entre los tres modelos evaluados sobre la muestra "
        "traducida y revisada, con la configuración fijada en desarrollo."
    )
}

# ------------------------------------------------------------
# 2. Creamos una versión nueva del paquete final
# ------------------------------------------------------------

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_paquete_final = (
    Path("/content/revision_evaluacion")
    / f"paquete_final_comparacion_{marca_version}"
)
carpeta_paquete_final.mkdir(parents=True, exist_ok=False)

# Incluimos el dataset y los resultados ya verificados
if not carpeta_insumos.is_dir():
    raise FileNotFoundError("No se encuentra la carpeta del dataset final.")

if not carpeta_resultados_prueba.is_dir():
    raise FileNotFoundError("No se encuentra la carpeta de resultados de 7.2.")

shutil.copytree(
    carpeta_insumos,
    carpeta_paquete_final / "dataset"
)

shutil.copytree(
    carpeta_resultados_prueba,
    carpeta_paquete_final / "resultados_prueba"
)

for nombre, contenido in {
    "decision_final.json": decision_final,
    "configuracion_modelo_final.json": configuracion_modelo_final
}.items():
    with (carpeta_paquete_final / nombre).open(
        "w", encoding="utf-8"
    ) as archivo:
        json.dump(
            contenido,
            archivo,
            ensure_ascii=False,
            indent=2
        )

# ------------------------------------------------------------
# 3. Incluimos código para clasificar nuevas consultas
# ------------------------------------------------------------

codigo_uso = '''
# Clasificar nuevas consultas con el modelo seleccionado
# Requiere: torch y transformers instalados.
import json
import torch
from pathlib import Path
from transformers import pipeline

# Ajustar a la carpeta donde se descomprimió el paquete final
carpeta_paquete = Path("/content/paquete_final")

with (carpeta_paquete / "configuracion_modelo_final.json").open(
    encoding="utf-8"
) as archivo:
    registro_modelo = json.load(archivo)

configuracion = registro_modelo["configuracion"]
categorias = registro_modelo["categorias"]
descripciones = configuracion["descripciones"]

descripcion_a_categoria = {
    descripciones[categoria]: categoria
    for categoria in categorias
}

clasificador_final = pipeline(
    "zero-shot-classification",
    model=registro_modelo["identificador_hugging_face"],
    device=0 if torch.cuda.is_available() else -1,
    dtype=torch.float32
)

def clasificar_consultas(consultas):
    """Clasifica una consulta o una lista de consultas en español."""
    if isinstance(consultas, str):
        consultas = [consultas]

    if not isinstance(consultas, list) or not consultas:
        raise ValueError("Introduce una consulta o una lista no vacía.")

    if any(
        not isinstance(texto, str) or not texto.strip()
        for texto in consultas
    ):
        raise ValueError("Todas las consultas deben ser textos no vacíos.")

    with torch.inference_mode():
        salidas = clasificador_final(
            consultas,
            candidate_labels=[
                descripciones[categoria] for categoria in categorias
            ],
            hypothesis_template=configuracion["plantilla"],
            multi_label=configuracion["multi_label"],
            batch_size=configuracion["batch_size"]
        )

    return [
        {
            "consulta": texto,
            "categoria_predicha": descripcion_a_categoria[
                salida["labels"][0]
            ],
            "puntuacion": float(salida["scores"][0])
        }
        for texto, salida in zip(consultas, salidas)
    ]

print("✅ Modelo seleccionado cargado:", registro_modelo["nombre_modelo"])
print("Usa clasificar_consultas('Tu consulta en español').")
print("La puntuación no es una probabilidad calibrada de acierto.")
'''.lstrip()

(carpeta_paquete_final / "usar_modelo_seleccionado.py").write_text(
    codigo_uso,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 4. Documentamos los insumos y su uso
# ------------------------------------------------------------

guia = f"""PAQUETE FINAL DE COMPARACIÓN
========================================

MODELO SELECCIONADO
{modelo_seleccionado}
{modelos_comparacion[modelo_seleccionado]}

CRITERIO
Mayor F1 macro sobre las 240 consultas de prueba.
Aciertos: {aciertos}
Errores: {240 - aciertos}

INSUMOS PARA REUTILIZAR EL MODELO
configuracion_modelo_final.json
  Identificador del modelo, categorías, descripciones y plantilla.

usar_modelo_seleccionado.py
  Código para cargar el modelo y clasificar nuevas consultas.
  Ajustar carpeta_paquete a la ubicación donde se descomprimió el ZIP.
  Requiere torch, transformers y acceso a los pesos de Hugging Face.
  Los pesos no están incluidos en este paquete.

INSUMOS PARA RECUPERAR EL DATASET
dataset/dataset_desarrollo.csv
  60 consultas de desarrollo.

dataset/dataset_prueba.csv
  240 consultas de prueba.

dataset/dataset_completo_trazabilidad.csv
  Originales, traducciones y validaciones de los 300 registros.

dataset/registros_retirados.csv
  Auditoría de los reemplazos.

dataset/cargar_insumos.py
  Código para cargar el dataset sin repetir su preparación.
  Ajustar su ruta a la subcarpeta dataset de este paquete.

RESULTADOS Y DECISIÓN
decision_final.json
  Selección final y métricas del modelo elegido.

resultados_prueba/comparacion_prueba.csv
  Comparación de los tres modelos.

resultados_prueba/registro_evaluacion.json
  Configuración, tiempos y correspondencia de modelos y archivos.

resultados_prueba/modelo_N_*.csv
  Predicciones, errores, métricas por categoría y matrices de confusión.

ALCANCE
El paquete permite recuperar los datos y analizar los resultados
sin repetir el muestreo, las traducciones ni la evaluación.
La carga del modelo sí requiere sus bibliotecas y pesos.

La selección corresponde a esta muestra y configuración.
El rendimiento observado no demuestra suficiencia para uso operativo.
Pueden persistir consultas semánticamente similares entre conjuntos.

Conservar también el respaldo de experimentos de desarrollo de 5.10.
"""

(carpeta_paquete_final / "LEEME_FINAL.txt").write_text(
    guia,
    encoding="utf-8"
)

# Guardamos huellas de todos los archivos del paquete
manifest_paquete = {
    "version_utc": marca_version,
    "modelo_seleccionado": modelo_seleccionado,
    "archivos": {}
}

for ruta in sorted(carpeta_paquete_final.rglob("*")):
    if ruta.is_file():
        nombre_relativo = ruta.relative_to(
            carpeta_paquete_final
        ).as_posix()

        manifest_paquete["archivos"][nombre_relativo] = {
            "bytes": ruta.stat().st_size,
            "sha256": hashlib.sha256(ruta.read_bytes()).hexdigest()
        }

with (carpeta_paquete_final / "manifest_paquete.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(
        manifest_paquete,
        archivo,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 5. Comprimimos, verificamos y descargamos
# ------------------------------------------------------------

ruta_zip_paquete_final = Path(
    shutil.make_archive(
        str(carpeta_paquete_final),
        "zip",
        root_dir=str(carpeta_paquete_final)
    )
)

archivos_esperados = {
    ruta.relative_to(carpeta_paquete_final).as_posix()
    for ruta in carpeta_paquete_final.rglob("*")
    if ruta.is_file()
}

with zipfile.ZipFile(ruta_zip_paquete_final, "r") as respaldo:
    if (
        respaldo.testzip() is not None
        or not archivos_esperados.issubset(set(respaldo.namelist()))
    ):
        raise ValueError("Falló la verificación del paquete final.")

print("\nModelo seleccionado:", modelo_seleccionado)
print("Criterio: mayor F1 macro sobre prueba.")
print("F1 macro:", round(decision_final["metricas"]["F1 macro"], 4))
print("Aciertos:", aciertos, "de 240")

print("\n📁 Carpeta del paquete final:")
print(carpeta_paquete_final)

print("\n📦 ZIP principal para conservar en tu computadora:")
print(ruta_zip_paquete_final.name)

print("\n➡️ DECISIÓN: decision_final.json")
print("➡️ CONFIGURACIÓN: configuracion_modelo_final.json")
print("➡️ CLASIFICAR NUEVAS CONSULTAS: usar_modelo_seleccionado.py")
print("➡️ DATASETS Y AUDITORÍA: carpeta dataset")
print("➡️ MÉTRICAS Y ERRORES: carpeta resultados_prueba")
print("➡️ GUÍA: LEEME_FINAL.txt")

print("\nNo se repitió la evaluación ni se modificaron las descripciones.")

print("=" * 60)
print("✅ Selección final registrada y paquete verificado.")
print("=" * 60)

print("\n📥 Solicitando descarga del paquete final...")
files.download(str(ruta_zip_paquete_final))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

📦 Preparando el paquete final de la comparación...

Modelo seleccionado: XLM-RoBERTa-large
Criterio: mayor F1 macro sobre prueba.
F1 macro: 0.6492
Aciertos: 158 de 240

📁 Carpeta del paquete final:
/content/revision_evaluacion/paquete_final_comparacion_20261010T200350292712Z

📦 ZIP principal para conservar en tu computadora:
paquete_final_comparacion_20261010T200350292712Z.zip

➡️ DECISIÓN: decision_final.json
➡️ CONFIGURACIÓN: configuracion_modelo_final.json
➡️ CLASIFICAR NUEVAS CONSULTAS: usar_modelo_seleccionado.py
➡️ DATASETS Y AUDITORÍA: carpeta dataset
➡️ MÉTRICAS Y ERRORES: carpeta resultados_prueba
➡️ GUÍA: LEEME_FINAL.txt

No se repitió la evaluación ni se modificaron las descripciones.
✅ Selección final registrada y paquete verificado.

📥 Solicitando descarga del paquete final...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


### 🧠 **7.5 Inventario de evidencias e insumos para el reporte**

Se genera un inventario de los archivos finales, indicando su
contenido, uso posterior y relación con el reporte.

Se incluye una guía con los resultados observados, las limitaciones
y los insumos para continuar sin repetir la preparación ni la evaluación.

La correspondencia con las instrucciones y la rúbrica se completará
cuando se incorporen sus requisitos exactos.

**Estado:** Evidencias documentadas; revisión de cumplimiento
de la rúbrica pendiente.

In [ ]:
# ============================================================
# 7.5 Inventario de evidencias e insumos para el reporte
# ============================================================

import pandas as pd
import json
import hashlib
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("=" * 60)
print("📋 Reuniendo evidencias e insumos para continuar...")
print("=" * 60)

# ------------------------------------------------------------
# 1. Ubicamos el paquete final y el cierre de desarrollo
# ------------------------------------------------------------

# En otra sesión, ajustar estas rutas a los respaldos descomprimidos.
paquete = Path(
    "/content/revision_evaluacion/"
    "paquete_final_comparacion_20261010T200350292712Z"
)

cierre_desarrollo = Path(
    "/content/revision_evaluacion/"
    "cierre_desarrollo_20261010T191612134285Z"
)

if not paquete.is_dir():
    raise FileNotFoundError(
        "No se encuentra el paquete final. "
        "Descomprime su ZIP y ajusta la ruta 'paquete'."
    )

if not cierre_desarrollo.is_dir():
    raise FileNotFoundError(
        "No se encuentra el cierre de desarrollo de 5.10. "
        "Descomprime ese ZIP y ajusta 'cierre_desarrollo'."
    )

# Verificamos las huellas del paquete final
with (paquete / "manifest_paquete.json").open(
    encoding="utf-8"
) as archivo:
    manifest_paquete = json.load(archivo)

for nombre, datos in manifest_paquete["archivos"].items():
    ruta = paquete / nombre

    if not ruta.is_file():
        raise FileNotFoundError(f"Falta la evidencia: {ruta}")

    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()

    if huella != datos["sha256"]:
        raise ValueError(f"La evidencia cambió: {nombre}")

with (paquete / "decision_final.json").open(
    encoding="utf-8"
) as archivo:
    decision = json.load(archivo)

with (paquete / "configuracion_modelo_final.json").open(
    encoding="utf-8"
) as archivo:
    configuracion_final = json.load(archivo)

with (
    paquete / "resultados_prueba" / "registro_evaluacion.json"
).open(encoding="utf-8") as archivo:
    registro_prueba = json.load(archivo)

# ------------------------------------------------------------
# 2. Describimos cada evidencia y su uso
# ------------------------------------------------------------

def describir_archivo(nombre):
    """Devuelve contenido y uso para las evidencias conocidas."""
    if nombre == "dataset_desarrollo.csv":
        return (
            "60 consultas validadas de desarrollo.",
            "Explicar la separación de datos y recuperar desarrollo."
        )
    if nombre in ["dataset_prueba.csv", "dataset_prueba_utilizado.csv"]:
        return (
            "240 consultas de prueba.",
            "Recuperar los datos y documentar la muestra evaluada."
        )
    if nombre in [
        "dataset_completo_trazabilidad.csv", "muestra_actualizada.csv"
    ]:
        return (
            "Originales, traducciones, conjuntos y estados de revisión.",
            "Documentar la preparación y trazabilidad del dataset."
        )
    if nombre == "desarrollo_utilizado.csv":
        return (
            "Registros utilizados en los experimentos de desarrollo.",
            "Identificar los datos de los experimentos inicial y alternativo."
        )
    if nombre == "registros_retirados.csv":
        return (
            "Auditoría de registros reemplazados.",
            "Justificar exclusiones y reemplazos; no usar para evaluar."
        )
    if nombre == "distribucion_dataset.csv":
        return (
            "Cantidades por categoría y conjunto.",
            "Presentar la composición del dataset."
        )
    if nombre == "decision_final.json":
        return (
            "Modelo seleccionado, criterio y métricas.",
            "Documentar la selección final."
        )
    if nombre == "configuracion_modelo_final.json":
        return (
            "Identificador del modelo y parámetros de clasificación.",
            "Recuperar el modelo elegido y explicar la configuración."
        )
    if nombre == "configuracion_para_prueba.json":
        return (
            "Configuración fijada antes de evaluar prueba.",
            "Documentar el protocolo y recuperar sus parámetros."
        )
    if nombre == "comparacion_prueba.csv":
        return (
            "Métricas generales de los tres modelos sobre prueba.",
            "Construir la tabla principal de resultados."
        )
    if nombre in ["metricas_iniciales.csv", "metricas_alternativas.csv"]:
        return (
            "Métricas de desarrollo con las descripciones indicadas.",
            "Documentar los experimentos previos a prueba."
        )
    if nombre == "efecto_descripciones.csv":
        return (
            "Cambios de exactitud y F1 macro entre descripciones.",
            "Explicar la sensibilidad a la redacción de las categorías."
        )
    if "matriz_confusion" in nombre:
        return (
            "Conteos por categoría real y predicha.",
            "Analizar las confusiones del modelo."
        )
    if "predicciones" in nombre:
        return (
            "Predicciones individuales, puntuaciones y aciertos.",
            "Revisar ejemplos y calcular análisis adicionales sin inferencia."
        )
    if "errores" in nombre:
        return (
            "Consultas clasificadas incorrectamente.",
            "Sustentar el análisis cualitativo de errores."
        )
    if "categorias_" in nombre or "metricas_categoria" in nombre:
        return (
            "Precisión, exhaustividad y F1 por categoría.",
            "Comparar fortalezas y dificultades por categoría."
        )
    if nombre in ["registro_evaluacion.json", "registro_experimentos.json"]:
        return (
            "Configuraciones, modelos, tiempos y registros experimentales.",
            "Documentar reproducibilidad y condiciones de ejecución."
        )
    if nombre == "usar_modelo_seleccionado.py":
        return (
            "Código para clasificar nuevas consultas.",
            "Continuar con el modelo sin repetir su comparación."
        )
    if nombre == "cargar_insumos.py":
        return (
            "Código de carga y verificación del dataset.",
            "Recuperar los datos sin repetir su preparación."
        )
    if "manifest" in nombre:
        return (
            "Metadatos y huellas de integridad.",
            "Comprobar la identidad de los archivos respaldados."
        )
    if "LEEME" in nombre:
        return (
            "Guía de los archivos y su utilización.",
            "Consultar instrucciones de recuperación."
        )
    if nombre == "inventario_archivos.json":
        return (
            "Descripción de los archivos del cierre de desarrollo.",
            "Consultar la finalidad de los insumos."
        )
    return (
        "Archivo auxiliar conservado en el respaldo.",
        "Consultar junto con la guía de su carpeta."
    )

# ------------------------------------------------------------
# 3. Creamos un paquete que reúne ambas etapas
# ------------------------------------------------------------

marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_evidencias = (
    Path("/content/revision_evaluacion")
    / f"evidencias_reporte_{marca_version}"
)
carpeta_evidencias.mkdir(parents=True, exist_ok=False)

shutil.copytree(
    paquete,
    carpeta_evidencias / "paquete_final"
)
shutil.copytree(
    cierre_desarrollo,
    carpeta_evidencias / "experimentos_desarrollo"
)

filas_inventario = []

for etapa, carpeta in [
    ("Dataset y evaluación final", carpeta_evidencias / "paquete_final"),
    ("Experimentos de desarrollo",
     carpeta_evidencias / "experimentos_desarrollo")
]:
    for ruta in sorted(carpeta.rglob("*")):
        if ruta.is_file():
            contenido, uso = describir_archivo(ruta.name)
            filas_inventario.append({
                "etapa": etapa,
                "archivo": ruta.relative_to(
                    carpeta_evidencias
                ).as_posix(),
                "contenido": contenido,
                "uso_para_continuar_o_reportar": uso,
                "bytes": ruta.stat().st_size,
                "sha256": hashlib.sha256(
                    ruta.read_bytes()
                ).hexdigest()
            })

df_inventario_evidencias = pd.DataFrame(filas_inventario)

df_inventario_evidencias.to_csv(
    carpeta_evidencias / "inventario_evidencias.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 4. Relacionamos temas del reporte con sus evidencias
# ------------------------------------------------------------

# Son temas sugeridos; no sustituyen los requisitos de la rúbrica.
temas_reporte = [
    (
        "Preparación y distribución del dataset",
        "paquete_final/dataset/distribucion_dataset.csv; "
        "paquete_final/dataset/dataset_completo_trazabilidad.csv"
    ),
    (
        "Revisión de traducciones y reemplazos",
        "paquete_final/dataset/dataset_completo_trazabilidad.csv; "
        "paquete_final/dataset/registros_retirados.csv"
    ),
    (
        "Modelos y configuración experimental",
        "paquete_final/resultados_prueba/registro_evaluacion.json; "
        "experimentos_desarrollo/registro_experimentos.json"
    ),
    (
        "Experimento de descripciones alternativas",
        "experimentos_desarrollo/metricas_iniciales.csv; "
        "experimentos_desarrollo/metricas_alternativas.csv; "
        "experimentos_desarrollo/efecto_descripciones.csv"
    ),
    (
        "Comparación de resultados sobre prueba",
        "paquete_final/resultados_prueba/comparacion_prueba.csv"
    ),
    (
        "Resultados por categoría y análisis de errores",
        "paquete_final/resultados_prueba/modelo_N_metricas_categoria.csv; "
        "modelo_N_matriz_confusion.csv; modelo_N_errores.csv"
    ),
    (
        "Selección final y limitaciones",
        "paquete_final/decision_final.json; "
        "paquete_final/resultados_prueba/comparacion_prueba.csv"
    ),
    (
        "Reutilización de insumos",
        "paquete_final/usar_modelo_seleccionado.py; "
        "paquete_final/dataset/cargar_insumos.py"
    )
]

df_guia_reporte = pd.DataFrame(
    temas_reporte,
    columns=["tema_sugerido", "evidencias"]
)

df_guia_reporte["requisito_exacto_de_rubrica"] = (
    "Pendiente de incorporar instrucciones y rúbrica."
)
df_guia_reporte["estado_cumplimiento"] = (
    "No verificado contra la rúbrica."
)

df_guia_reporte.to_csv(
    carpeta_evidencias / "guia_evidencias_reporte.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 5. Generamos una guía de redacción con resultados reales
# ------------------------------------------------------------

comparacion = pd.read_csv(
    paquete / "resultados_prueba" / "comparacion_prueba.csv",
    encoding="utf-8-sig"
)

lineas_resultados = []

for _, fila in comparacion.iterrows():
    lineas_resultados.append(
        f"- {fila['Modelo']}: exactitud "
        f"{fila['Exactitud']:.4f}; F1 macro "
        f"{fila['F1 macro']:.4f}; tiempo medio total "
        f"{fila['Tiempo medio total (s)']:.4f} segundos."
    )

metricas_elegido = decision["metricas"]

guia_reporte = f"""# Guía de evidencias para redactar el reporte

## Estado de los requisitos

Las instrucciones y la rúbrica exactas todavía deben incorporarse.
Esta guía reúne evidencias; no certifica cumplimiento de la rúbrica.

## Datos y preparación

Se prepararon 300 consultas de seis categorías:
60 de desarrollo y 240 de prueba.
Cada categoría tiene 10 consultas de desarrollo y 40 de prueba.

Las traducciones fueron revisadas y aprobadas por el usuario.
Se conservaron los originales y la auditoría de 48 registros retirados.
Se eliminaron las coincidencias textuales detectadas con la normalización
utilizada. Pueden persistir consultas semánticamente similares.

Consultar los archivos de la subcarpeta paquete_final/dataset.

## Protocolo

Se evaluaron tres modelos con una configuración común fijada en desarrollo.
La plantilla fue: {configuracion_final['configuracion']['plantilla']}
El tamaño de lote fue 1 y se realizaron tres repeticiones.
La evaluación de prueba se ejecutó en {registro_prueba['dispositivo']}.
El criterio principal fue F1 macro.

Las métricas corresponden a la primera repetición.
Las etiquetas fueron estables entre repeticiones.
Estas repeticiones no representan muestras independientes.

## Experimento de desarrollo

Se compararon descripciones iniciales y alternativas sobre desarrollo.
La configuración inicial se conservó para la comparación común en prueba.
Consultar experimentos_desarrollo/efecto_descripciones.csv y sus registros.
No atribuir los cambios entre reconstrucciones del dataset exclusivamente
a la eliminación de expresiones ofensivas.

## Resultados sobre prueba

{chr(10).join(lineas_resultados)}

Consultar comparacion_prueba.csv para los valores completos.

## Selección final

Modelo: {decision['modelo_seleccionado']}
Identificador: {decision['identificador_hugging_face']}
F1 macro: {metricas_elegido['F1 macro']:.4f}
Aciertos: {decision['aciertos']} de {decision['consultas_prueba']}
Errores: {decision['errores']}

Se seleccionó por obtener el mayor F1 macro entre los modelos comparados.

## Análisis por categoría y errores

Usar las métricas por categoría, matrices de confusión y archivos de errores.
La correspondencia entre modelo_N y el nombre del modelo se encuentra en
resultados_prueba/registro_evaluacion.json.

Elegir ejemplos directamente de los archivos de errores.
No inventar consultas ni causas de las confusiones.

## Limitaciones

- Muestra traducida, revisada y limitada a seis categorías.
- Posible similitud semántica entre consultas de ambos conjuntos.
- Los tiempos dependen de esta ejecución y del entorno.
- MiniLMv2 mostró variación considerable de tiempos entre repeticiones.
- El tiempo por consulta es un promedio del procesamiento completo.
- Las puntuaciones no son probabilidades calibradas.
- La selección no demuestra suficiencia para uso operativo.

## Continuación sin repetir el trabajo

Para recuperar datos:
paquete_final/dataset/cargar_insumos.py

Para clasificar nuevas consultas:
paquete_final/usar_modelo_seleccionado.py

Ajustar las rutas dentro de esos archivos a la ubicación del paquete
descomprimido. La carga del modelo requiere sus bibliotecas y pesos.

## Pendiente para terminar el reporte

1. Incorporar las instrucciones y la rúbrica exactas.
2. Relacionar cada requisito con una evidencia del inventario.
3. Preparar las tablas y figuras exigidas.
4. Redactar el reporte en el formato y extensión solicitados.
5. Verificar las fuentes y referencias requeridas.
6. Revisar el cumplimiento antes de entregar.
"""

(carpeta_evidencias / "GUIA_REPORTE.md").write_text(
    guia_reporte,
    encoding="utf-8"
)

(carpeta_evidencias / "LEEME_EVIDENCIAS.txt").write_text(
    "PAQUETE DE EVIDENCIAS PARA CONTINUAR Y REDACTAR EL REPORTE\n\n"
    "inventario_evidencias.csv:\n"
    "  Archivos, contenido, uso y huellas de integridad.\n\n"
    "guia_evidencias_reporte.csv:\n"
    "  Temas del reporte y evidencias disponibles.\n"
    "  Correspondencia con rúbrica pendiente.\n\n"
    "GUIA_REPORTE.md:\n"
    "  Resultados, limitaciones y guía de redacción.\n\n"
    "paquete_final/:\n"
    "  Dataset, auditoría, evaluación de prueba y selección final.\n\n"
    "experimentos_desarrollo/:\n"
    "  Comparaciones inicial y alternativa de desarrollo.\n\n"
    "Este paquete reúne los archivos necesarios para recuperar los datos,\n"
    "analizar resultados y continuar sin repetir la preparación.\n"
    "No incluye los pesos de los modelos ni el notebook original.\n"
    "Conservar el notebook como evidencia del procedimiento ejecutado.\n"
    "Las instrucciones y la rúbrica deben incorporarse para revisar\n"
    "el cumplimiento exacto del reporte.\n",
    encoding="utf-8"
)

# ------------------------------------------------------------
# 6. Comprimimos, verificamos y descargamos
# ------------------------------------------------------------

ruta_zip_evidencias = Path(
    shutil.make_archive(
        str(carpeta_evidencias),
        "zip",
        root_dir=str(carpeta_evidencias)
    )
)

archivos_esperados = {
    ruta.relative_to(carpeta_evidencias).as_posix()
    for ruta in carpeta_evidencias.rglob("*")
    if ruta.is_file()
}

with zipfile.ZipFile(ruta_zip_evidencias, "r") as respaldo:
    if (
        respaldo.testzip() is not None
        or not archivos_esperados.issubset(set(respaldo.namelist()))
    ):
        raise ValueError("Falló la verificación del ZIP de evidencias.")

display(
    df_guia_reporte[["tema_sugerido", "evidencias"]]
)

print("\nArchivos de evidencia inventariados:",
      len(df_inventario_evidencias))
print("Modelo seleccionado:", decision["modelo_seleccionado"])

print("\n📁 Carpeta de evidencias:")
print(carpeta_evidencias)

print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_evidencias.name)

print("\n➡️ PARA UBICAR ARCHIVOS: inventario_evidencias.csv")
print("➡️ PARA REDACTAR: GUIA_REPORTE.md")
print("➡️ PARA RELACIONAR CON RÚBRICA: guia_evidencias_reporte.csv")
print("➡️ PARA CONTINUAR: paquete_final/")
print("➡️ EXPERIMENTOS PREVIOS: experimentos_desarrollo/")

print("\nPendiente: incorporar instrucciones y rúbrica exactas.")
print("Conserva también el notebook original.")

print("=" * 60)
print("✅ Evidencias reunidas, documentadas y ZIP verificado.")
print("=" * 60)

print("\n📥 Solicitando descarga del paquete de evidencias...")
files.download(str(ruta_zip_evidencias))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

📋 Reuniendo evidencias e insumos para continuar...


,tema_sugerido,evidencias
0,Preparación y distribución del dataset,paquete_final/dataset/distribucion_dataset.csv...
1,Revisión de traducciones y reemplazos,paquete_final/dataset/dataset_completo_trazabi...
2,Modelos y configuración experimental,paquete_final/resultados_prueba/registro_evalu...
3,Experimento de descripciones alternativas,experimentos_desarrollo/metricas_iniciales.csv...
4,Comparación de resultados sobre prueba,paquete_final/resultados_prueba/comparacion_pr...
5,Resultados por categoría y análisis de errores,paquete_final/resultados_prueba/modelo_N_metri...
6,Selección final y limitaciones,paquete_final/decision_final.json; paquete_fin...
7,Reutilización de insumos,paquete_final/usar_modelo_seleccionado.py; paq...



Archivos de evidencia inventariados: 52
Modelo seleccionado: XLM-RoBERTa-large

📁 Carpeta de evidencias:
/content/revision_evaluacion/evidencias_reporte_20261010T200713130418Z

📦 ZIP para conservar en tu computadora:
evidencias_reporte_20261010T200713130418Z.zip

➡️ PARA UBICAR ARCHIVOS: inventario_evidencias.csv
➡️ PARA REDACTAR: GUIA_REPORTE.md
➡️ PARA RELACIONAR CON RÚBRICA: guia_evidencias_reporte.csv
➡️ PARA CONTINUAR: paquete_final/
➡️ EXPERIMENTOS PREVIOS: experimentos_desarrollo/

Pendiente: incorporar instrucciones y rúbrica exactas.
Conserva también el notebook original.
✅ Evidencias reunidas, documentadas y ZIP verificado.

📥 Solicitando descarga del paquete de evidencias...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.


### 📦 **7.6 Exportación de insumos y cierre del primer Colab**

Se genera el archivo INSUMOS_COLAB_2_COMPARACION_MODELOS.zip,
que será el insumo de entrada del segundo Colab.

Incluye los datasets validados de desarrollo y prueba, la
configuración fijada, la trazabilidad y la auditoría de reemplazos.

El segundo Colab utilizará estos archivos para comparar los tres
modelos sin repetir el muestreo, las traducciones ni las revisiones.

**Cierre:** Este notebook se conserva como evidencia de la preparación
del dataset y de los experimentos realizados.

In [ ]:
# ============================================================
# 7.6 Exportación de insumos y cierre del primer Colab
# ============================================================

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("=" * 60)
print("📦 Preparando el insumo de entrada del segundo Colab...")
print("=" * 60)

# Utilizamos los archivos finales exportados en 6.33
carpeta_origen = Path(
    "/content/revision_evaluacion/"
    "dataset_final_20261010T195212647311Z"
)

if not carpeta_origen.is_dir():
    raise FileNotFoundError(
        "No se encuentra el dataset final. "
        "Descomprime su respaldo y ajusta carpeta_origen."
    )

# Verificamos los archivos contra el manifiesto original
with (carpeta_origen / "manifest.json").open(
    encoding="utf-8"
) as archivo:
    manifest_original = json.load(archivo)

archivos_incluir = {
    "dataset_desarrollo.csv": "60 consultas validadas de desarrollo.",
    "dataset_prueba.csv": "240 consultas validadas de prueba.",
    "configuracion_para_prueba.json": (
        "Identificadores de los tres modelos y configuración fijada."
    ),
    "dataset_completo_trazabilidad.csv": (
        "300 registros con originales, traducciones y validaciones."
    ),
    "registros_retirados.csv": "Auditoría de los 48 registros reemplazados.",
    "distribucion_dataset.csv": "Cantidades por categoría y conjunto."
}

for nombre in archivos_incluir:
    ruta = carpeta_origen / nombre

    if not ruta.is_file():
        raise FileNotFoundError(f"Falta el insumo: {ruta}")

    if nombre not in manifest_original["archivos"]:
        raise ValueError(f"{nombre} no está registrado en el manifiesto.")

    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()

    if huella != manifest_original["archivos"][nombre]["sha256"]:
        raise ValueError(f"El archivo {nombre} cambió desde su exportación.")

# Nombre claro y estable para identificar el insumo principal
nombre_zip = "INSUMOS_COLAB_2_COMPARACION_MODELOS.zip"

# Guardamos cada ejecución en una carpeta nueva
marca_version = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_entrega = (
    Path("/content/revision_evaluacion")
    / f"entrega_colab_2_{marca_version}"
)
carpeta_entrega.mkdir(parents=True, exist_ok=False)

ruta_zip_insumos_colab_2 = carpeta_entrega / nombre_zip

guia = """INSUMOS PARA EL SEGUNDO COLAB
========================================

ARCHIVO DE ENTRADA
INSUMOS_COLAB_2_COMPARACION_MODELOS.zip

PASOS EN EL SEGUNDO COLAB
1. Configurar GPU e instalar las dependencias necesarias.
2. Subir este ZIP y descomprimirlo en /content/insumos_colab_2.
3. Verificar los archivos con manifest_insumos.json.
4. Cargar dataset_desarrollo.csv y dataset_prueba.csv.
5. Recuperar los modelos y parámetros desde
   configuracion_para_prueba.json.
6. Ejecutar la comparación y guardar sus resultados.

ARCHIVOS PRINCIPALES
dataset_desarrollo.csv:
  60 consultas; 10 por categoría.

dataset_prueba.csv:
  240 consultas; 40 por categoría.

Ambos datasets contienen:
  id_consulta: identificador del registro.
  consulta_es: consulta traducida al español.
  categoria_referencia: etiqueta validada para evaluar.

configuracion_para_prueba.json:
  Identificadores de los tres modelos, descripciones y parámetros.
  La decisión de desarrollo es preliminar; no es una nueva evaluación.

ARCHIVOS DE TRAZABILIDAD
dataset_completo_trazabilidad.csv:
  300 registros con originales, traducciones, conjuntos y validaciones.

registros_retirados.csv:
  Auditoría de 48 registros reemplazados; no utilizar para evaluar.

distribucion_dataset.csv:
  Cantidades por categoría y conjunto.

manifest_insumos.json:
  Versión del dataset y huellas SHA-256 de los archivos.

ALCANCE
No es necesario ejecutar el primer Colab para usar estos insumos.
El primer notebook se conserva como evidencia del procedimiento.
Los pesos de los modelos no están incluidos.
Los resultados de los experimentos anteriores están en los respaldos
de evidencias, separados de este paquete de entrada.
"""

manifest_insumos = {
    "finalidad": "Entrada del segundo Colab de comparación de modelos",
    "version_dataset_utc": manifest_original["version_utc"],
    "fecha_empaquetado_utc": marca_version,
    "consultas_desarrollo": manifest_original["consultas_desarrollo"],
    "consultas_prueba": manifest_original["consultas_prueba"],
    "categorias": manifest_original["categorias"],
    "archivos": {
        nombre: {
            "contenido": descripcion,
            "sha256": manifest_original["archivos"][nombre]["sha256"]
        }
        for nombre, descripcion in archivos_incluir.items()
    }
}

# Los archivos quedan directamente en la raíz del ZIP
with zipfile.ZipFile(
    ruta_zip_insumos_colab_2,
    mode="x",
    compression=zipfile.ZIP_DEFLATED
) as respaldo:
    for nombre in archivos_incluir:
        respaldo.write(
            carpeta_origen / nombre,
            arcname=nombre
        )

    respaldo.writestr("LEEME_COLAB_2.txt", guia)
    respaldo.writestr(
        "manifest_insumos.json",
        json.dumps(
            manifest_insumos,
            ensure_ascii=False,
            indent=2
        )
    )

# Verificamos integridad y contenido del paquete
esperados = set(archivos_incluir) | {
    "LEEME_COLAB_2.txt",
    "manifest_insumos.json"
}

with zipfile.ZipFile(ruta_zip_insumos_colab_2, "r") as respaldo:
    if (
        respaldo.testzip() is not None
        or set(respaldo.namelist()) != esperados
    ):
        raise ValueError("Falló la verificación del ZIP de insumos.")

print("\n📦 ARCHIVO QUE DEBES SUBIR AL SEGUNDO COLAB:")
print(nombre_zip)

print("\n📍 Ubicación del respaldo en este entorno:")
print(ruta_zip_insumos_colab_2)

print("\n📋 Contenido:")
for nombre, descripcion in archivos_incluir.items():
    print(f"  • {nombre}: {descripcion}")

print("  • LEEME_COLAB_2.txt: instrucciones para continuar.")
print("  • manifest_insumos.json: versión e integridad de archivos.")

print("\n✅ Desarrollo: 60 consultas validadas.")
print("✅ Prueba: 240 consultas validadas.")
print("✅ Configuración de los tres modelos incluida.")
print("✅ No necesitas repetir la preparación del dataset.")

print("\nConserva este ZIP y el notebook del primer Colab.")
print("Los respaldos anteriores siguen disponibles.")

print("=" * 60)
print("✅ PRIMER COLAB CERRADO: insumos del segundo Colab preparados.")
print("=" * 60)

print("\n📥 Solicitando descarga del archivo principal...")
files.download(str(ruta_zip_insumos_colab_2))
print("ℹ️ Comprueba que el ZIP aparezca en tus descargas.")

📦 Preparando el insumo de entrada del segundo Colab...

📦 ARCHIVO QUE DEBES SUBIR AL SEGUNDO COLAB:
INSUMOS_COLAB_2_COMPARACION_MODELOS.zip

📍 Ubicación del respaldo en este entorno:
/content/revision_evaluacion/entrega_colab_2_20261010T201851393107Z/INSUMOS_COLAB_2_COMPARACION_MODELOS.zip

📋 Contenido:
  • dataset_desarrollo.csv: 60 consultas validadas de desarrollo.
  • dataset_prueba.csv: 240 consultas validadas de prueba.
  • configuracion_para_prueba.json: Identificadores de los tres modelos y configuración fijada.
  • dataset_completo_trazabilidad.csv: 300 registros con originales, traducciones y validaciones.
  • registros_retirados.csv: Auditoría de los 48 registros reemplazados.
  • distribucion_dataset.csv: Cantidades por categoría y conjunto.
  • LEEME_COLAB_2.txt: instrucciones para continuar.
  • manifest_insumos.json: versión e integridad de archivos.

✅ Desarrollo: 60 consultas validadas.
✅ Prueba: 240 consultas validadas.
✅ Configuración de los tres modelos incluida.
✅ 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tus descargas.
